In [0]:
# Journey Model — gold layer
#
# One quality-controlled gold product per live silver product: 94 materialized views
# that read a silver table, keep the rows that belong on the research surface,
# and repair or remove what is wrong with the rest.
#
# Silver is the layer for a researcher who wants to make their own decisions
# about what to keep: it publishes everything and records what is doubtful about
# it. Gold is for the researcher who does not want to make those decisions, so
# gold makes them here.
#
# Every doubt silver records is put on the same ladder:
#
#   fix       the value is wrong and the true value is known — write it
#   null      the value is wrong and the true value is not known — remove it
#   drop      nulling it leaves nothing the row is about — remove the row
#   advisory  the data does not say which of two values is wrong — count it
#
# What lands on the last rung is not what was too much trouble to fix. It is the
# set of checks whose own assumption is the doubtful part: an ordering rule that
# fires on four rows in five is describing two clocks, not two events in the
# wrong order, and an event that disagrees with a birth date cannot be corrected
# when 2,051 people in the spine died before they were born. Each of those says
# so, in the rule's own comment, with the measurement behind it.
#
# Each product gets a cell of its own below, holding the column list, the
# expectation rules and the column comments only it uses, then the flow itself.
# Products are grouped by plane: spine first, then clinical, events, text and
# reference.
#
# House rules every flow in this notebook follows:
#   * fixes and nulls live in the SELECT, so the value a consumer reads is the
#     value the rule produced and there is nothing to remember to apply;
#   * row drops are expectations, so Lakeflow reports how many rows each one
#     removed on every update;
#   * every rule carries the reason it exists and the measurement behind it,
#     taken from the profile of silver captured on 24 August 2026;
#   * every read goes through _src and uses the production Silver products;
#     the pipeline target controls where Gold is published.

In [0]:
# ==== Imports and the declarative pipelines API ====

# The declarative pipelines API lives in pyspark.pipelines on the serverless
# CURRENT channel. The fallback keeps the notebook runnable on a runtime that
# still only has the dlt module, at the cost of the refresh_policy hint.
from pyspark.sql import functions as F

try:
    from pyspark import pipelines as dp
except ImportError:
    import dlt

    class _DpShim:
        @staticmethod
        def materialized_view(**kwargs):
            kwargs.pop("refresh_policy", None)
            return dlt.table(**kwargs)

        @staticmethod
        def temporary_view(**kwargs):
            return dlt.view(**kwargs)

    dp = _DpShim()


def _expect_all(rules):
    """Record a rule per row without changing what is published."""
    if hasattr(dp, "expect_all"):
        return dp.expect_all(rules)
    try:
        import dlt as _dlt

        return _dlt.expect_all(rules)
    except (ImportError, AttributeError):
        return lambda function: function


def _expect_all_or_drop(rules):
    """Keep only the rows a rule holds for, and count the ones it removed."""
    if hasattr(dp, "expect_all_or_drop"):
        return dp.expect_all_or_drop(rules)
    import dlt as _dlt

    return _dlt.expect_all_or_drop(rules)

In [0]:
# ==== Where this pipeline reads and writes ====

# Source is the production Silver Journey model. Publication remains relative:
# the Lakeflow pipeline target supplies the Gold catalog and schema.


def _n(name):
    """Flatten a logical Gold plane into a target-relative table name."""
    logical_schema, table = name.split(".", 1)
    if not logical_schema.startswith("gold_"):
        raise ValueError(f"unexpected flow schema in {name!r}")
    plane = logical_schema[len("gold_") :]
    return "_" + plane + table if table.startswith("_") else plane + "_" + table


def _src(flat):
    """The production Silver table a Gold product reads."""
    return "4_prod.silver." + flat


def _with_comments(df, comments):
    """Attach each column's comment to the schema the flow returns."""
    for column_name in df.columns:
        if column_name in comments:
            df = df.withMetadata(column_name, {"comment": comments[column_name]})
    return df

In [0]:
# ==== The spine keys every cross-table check is measured against ====

# One deduplicated key column per spine table, plus the person's own birth and
# death, held as temporary views so the 46 products that check against them read
# the spine once rather than 46 times.


@dp.temporary_view(name="_gold_person_keys")
def _gold_person_keys():
    return (
        spark.read.table(_src("spine_person"))
        .select(F.col("person_id").alias("_qc_person_id"))
        .where(F.col("_qc_person_id").isNotNull())
        .dropDuplicates(["_qc_person_id"])
    )


@dp.temporary_view(name="_gold_encounter_keys")
def _gold_encounter_keys():
    return (
        spark.read.table(_src("spine_encounter"))
        .select(F.col("encounter_id").alias("_qc_encounter_id"))
        .where(F.col("_qc_encounter_id").isNotNull())
        .dropDuplicates(["_qc_encounter_id"])
    )


@dp.temporary_view(name="_gold_episode_keys")
def _gold_episode_keys():
    return (
        spark.read.table(_src("spine_episode"))
        .select(F.col("episode_id").alias("_qc_episode_id"))
        .where(F.col("_qc_episode_id").isNotNull())
        .dropDuplicates(["_qc_episode_id"])
    )


@dp.temporary_view(name="_gold_person_dates")
def _gold_person_dates():
    # The two dates are read through the same expressions gold_spine.person
    # publishes them with, rather than raw from silver. A flag measured against a
    # value gold has removed would contradict the person row a researcher joins
    # it to: a death recorded before its own birth is not in the person table, so
    # nothing may be flagged as happening after it, and a birth held at the
    # source's "date of birth unknown" marker is not a birth to compare against.
    return (
        spark.read.table(_src("spine_person"))
        .select(
            F.col("person_id").alias("_qc_date_person_id"),
            F.expr("CASE WHEN CAST(`birth_datetime` AS DATE) = DATE'1899-12-30' OR CAST(`birth_datetime` AS DATE) = DATE'1900-01-01' OR CAST(`birth_datetime` AS DATE) = DATE'2100-12-31' OR CAST(`birth_datetime` AS DATE) < DATE'1901-01-01' AND CAST(`birth_datetime` AS DATE) NOT IN (DATE'1800-01-01', DATE'1899-12-30', DATE'1900-01-01') THEN NULL ELSE `birth_datetime` END").alias("_qc_birth_datetime"),
            F.expr("CASE WHEN `deceased_datetime` < `birth_datetime` THEN NULL ELSE `deceased_datetime` END").alias("_qc_deceased_datetime"),
        )
        .where(F.col("_qc_date_person_id").isNotNull())
        .dropDuplicates(["_qc_date_person_id"])
    )

In [0]:
# ==== Reading a silver product and repairing its cross-table problems ====


def _qc(table_name, select_exprs, fk_columns=(), date_flags=()):
    """Select a silver product's gold columns and repair what the spine contradicts.

    fk_columns names the keys whose parent is looked up in the spine. A key that
    names a parent the spine does not have is set to NULL, because a pointer that
    cannot be followed is worse than no pointer: it invites a join that silently
    drops the row. Silver keeps the orphan and flags it, for anyone who wants to
    study the gap itself.

    Nothing is dropped here. Where losing the pointer leaves the row with nothing
    to describe — a link with one end missing, an address belonging to no one —
    the product's own mandatory rules drop it, so the loss is reported rather
    than hidden inside a join.

    date_flags names the checks against the person's own dates, and each is
    published as a boolean:

        event_before_birth      the event predates the person's birth
        event_after_death_30d   the event is more than 30 days after their death

    Thirty days of grace absorbs the routine lag between a death being recorded
    and the last of a person's activity being filed against them. Both are
    measured against the dates gold_spine.person publishes, so a flag never
    disagrees with the person row it belongs to.

    These two stay published rather than acted on, because the spine's own birth
    and death dates are not sound enough to correct an event against: 2,051
    people carry a death recorded before their birth. Which side of the
    disagreement is wrong is not something the data says, so gold hands the
    reader the flag instead of a guess.

    A product that asks for neither is simply read and selected.
    """
    df = spark.read.table(_src(table_name)).selectExpr(*select_exprs)

    if "person_id" in fk_columns:
        parent = spark.read.table("_gold_person_keys")
        df = (
            df.join(parent, df.person_id == parent._qc_person_id, "left")
            .withColumn(
                "person_id",
                F.when(F.col("_qc_person_id").isNotNull(), F.col("person_id")),
            )
            .drop("_qc_person_id")
        )
    if "encounter_id" in fk_columns:
        parent = spark.read.table("_gold_encounter_keys")
        df = (
            df.join(parent, df.encounter_id == parent._qc_encounter_id, "left")
            .withColumn(
                "encounter_id",
                F.when(F.col("_qc_encounter_id").isNotNull(), F.col("encounter_id")),
            )
            .drop("_qc_encounter_id")
        )
    if "episode_id" in fk_columns:
        parent = spark.read.table("_gold_episode_keys")
        df = (
            df.join(parent, df.episode_id == parent._qc_episode_id, "left")
            .withColumn(
                "episode_id",
                F.when(F.col("_qc_episode_id").isNotNull(), F.col("episode_id")),
            )
            .drop("_qc_episode_id")
        )

    if date_flags:
        dates = spark.read.table("_gold_person_dates")
        df = df.join(dates, df.person_id == dates._qc_date_person_id, "left")
        if "event_before_birth" in date_flags:
            df = df.withColumn(
                "event_before_birth",
                F.coalesce(
                    F.col("event_datetime") < F.col("_qc_birth_datetime"),
                    F.lit(False),
                ),
            )
        if "event_after_death_30d" in date_flags:
            df = df.withColumn(
                "event_after_death_30d",
                F.coalesce(
                    F.col("event_datetime")
                    > F.col("_qc_deceased_datetime") + F.expr("INTERVAL 30 DAYS"),
                    F.lit(False),
                ),
            )
        df = df.drop("_qc_date_person_id", "_qc_birth_datetime", "_qc_deceased_datetime")

    return df

In [0]:
# ======== The spine ========
#
# Person, encounter and episode: the identities every other product hangs off.
# 13 products follow.

In [0]:
# ==== journey_spine.care_participation ====

SPINE_CARE_PARTICIPATION_SELECT = [
    "`care_participation_id` AS `care_participation_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`encounter_id` AS `encounter_id`",
    "`journey_id` AS `journey_id`",
    "`service_id` AS `service_id`",
    "`practitioner_id` AS `practitioner_id`",
    "`role` AS `role`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 8 of 144,668,217 rows (5.53e-06%) when profiled on 2026-08-24.
    #
    # 1900-01-01 is a placeholder low date rather than a date in 1900; on a non-birth field
    # it carries no more meaning than it does on a birth one. Hit 1 of 144,668,217 rows
    # (6.91e-07%) when profiled on 2026-08-24.
    #
    # A date before 1901 that is not one of the known placeholders. Nothing in this estate
    # predates the twentieth century, so these are mistyped or mis-scaled rather than early.
    # Hit 1 of 144,668,217 rows (6.91e-07%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`valid_from` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`valid_from` AS DATE)) < 9999 OR CAST(`valid_from` AS DATE) = DATE'1900-01-01' OR CAST(`valid_from` AS DATE) < DATE'1901-01-01' AND CAST(`valid_from` AS DATE) NOT IN (DATE'1800-01-01', DATE'1899-12-30', DATE'1900-01-01') THEN NULL ELSE `valid_from` END AS `valid_from`",
    "`valid_to` AS `valid_to`",
    "`construction_rule` AS `construction_rule`",
    "`construction_version` AS `construction_version`",
    "`source_table` AS `source_table`",
    "`source_row_id` AS `source_row_id`",
    "`load_batch_id` AS `load_batch_id`",
    "`loaded_at` AS `loaded_at`",
]

SPINE_CARE_PARTICIPATION_ADVISORY_RULES = {
    # This bounds a period of validity, and a future end is exactly how the source says a
    # record is still current -- nulling it would assert the record is valid forever, which
    # is a stronger and worse claim than the one being corrected. Seen on 3 of 144,668,217
    # rows (2.07e-06%) when profiled on 2026-08-24.
    "gold.spine.care_participation.valid_from.future_owner":
        "NOT COALESCE((CAST(`valid_from` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",
}

SPINE_CARE_PARTICIPATION_COLUMN_COMMENTS = {
    "care_participation_id": "Deterministic participation primary key.",
    "subject_key": "Always-populated subject join key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier.",
    "encounter_id": "Encounter context.",
    "journey_id": "Journey context when available.",
    "service_id": "Service-registration context when available.",
    "practitioner_id": "Practitioner reference.",
    "role": "Source-field-derived participation role.",
    "valid_from":
        "Participation validity start. Gold QC transform rules: gold.spine.care_participation.valid_from.beyond_2100_below_9999_owner, gold.spine.care_participation.valid_from.d1900_01_01_owner, gold.spine.care_participation.valid_from.pre1901_other_owner.",
    "valid_to": "Participation validity end.",
    "construction_rule": "Governed derivation rule.",
    "construction_version": "Governed derivation-rule version.",
    "source_table": "Fully qualified bronze source table.",
    "source_row_id": "Source encounter row identity.",
    "load_batch_id": "Deterministic bronze batch token.",
    "loaded_at": "Bronze load timestamp.",
}

@dp.materialized_view(
    name=_n("gold_spine.care_participation"),
    comment=(
        "One practitioner role attribution to a subject in encounter context. Gold QC twin of "
        "the silver product: 2 columns are repaired or nulled, 1 check(s) are advisory. Each "
        "rule states its reason in the pipeline notebook, and Lakeflow expectation metrics "
        "report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all(SPINE_CARE_PARTICIPATION_ADVISORY_RULES)
def gold_spine_care_participation():
    """Quality-controlled twin of journey_spine.care_participation."""
    # 54 rows point at a person_id the spine does not have. The pointer is nulled so it
    # cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    df = _qc(
        "spine_care_participation",
        SPINE_CARE_PARTICIPATION_SELECT,
        fk_columns=["person_id"],
    )
    return _with_comments(df, SPINE_CARE_PARTICIPATION_COLUMN_COMMENTS)

In [0]:
# ==== journey_spine.encounter ====

SPINE_ENCOUNTER_SELECT = [
    "`encounter_id` AS `encounter_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`parent_encounter_id` AS `parent_encounter_id`",
    "`parentage_status` AS `parentage_status`",
    "`encounter_level` AS `encounter_level`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 40,784,129 of 48,222,739 rows
    # (84.6%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`class_code` AS STRING))) = '0' THEN NULL ELSE `class_code` END AS `class_code`",
    "`class_display` AS `class_display`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 80,610 of 48,222,739 rows
    # (0.167%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`type_code` AS STRING))) = '0' THEN NULL ELSE `type_code` END AS `type_code`",
    "`type_display` AS `type_display`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 80,561 of 48,222,739 rows
    # (0.167%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`type_class_code` AS STRING))) = '0' THEN NULL ELSE `type_class_code` END AS `type_class_code`",
    "`type_class_display` AS `type_class_display`",
    "`status_code` AS `status_code`",
    "`status_display` AS `status_display`",
    "`period_start` AS `period_start`",
    "`period_end` AS `period_end`",
    "`arrival_method` AS `arrival_method`",
    "`arrival_confidence` AS `arrival_confidence`",
    "`departure_method` AS `departure_method`",
    "`departure_confidence` AS `departure_confidence`",
    "`length_of_stay_minutes` AS `length_of_stay_minutes`",
    "`scheduled_start` AS `scheduled_start`",

    # scheduled_end cannot precede scheduled_start. The start is the better-attested of the
    # two, so the end is what goes and the row keeps its scheduled_start. Hit 4,927 of
    # 48,222,739 rows (0.0102%) when profiled on 2026-08-24.
    "CASE WHEN `scheduled_start` IS NOT NULL AND `scheduled_end` IS NOT NULL AND `scheduled_start` > `scheduled_end` THEN NULL ELSE `scheduled_end` END AS `scheduled_end`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 4 of 48,222,739 rows (8.29e-06%) when profiled on 2026-08-24.
    #
    # 1900-01-01 is a placeholder low date rather than a date in 1900; on a non-birth field
    # it carries no more meaning than it does on a birth one. Hit 1 of 48,222,739 rows
    # (2.07e-06%) when profiled on 2026-08-24.
    #
    # A date before 1901 that is not one of the known placeholders. Nothing in this estate
    # predates the twentieth century, so these are mistyped or mis-scaled rather than early.
    # Hit 1 of 48,222,739 rows (2.07e-06%) when profiled on 2026-08-24.
    #
    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 2 of 48,222,739 rows (4.15e-06%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`registration_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE CASE WHEN CAST(`registration_datetime` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`registration_datetime` AS DATE)) < 9999 OR CAST(`registration_datetime` AS DATE) = DATE'1900-01-01' OR CAST(`registration_datetime` AS DATE) < DATE'1901-01-01' AND CAST(`registration_datetime` AS DATE) NOT IN (DATE'1800-01-01', DATE'1899-12-30', DATE'1900-01-01') THEN NULL ELSE `registration_datetime` END END AS `registration_datetime`",
    "`inpatient_admit_datetime` AS `inpatient_admit_datetime`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 4 of 48,222,739 rows (8.29e-06%) when profiled on 2026-08-24.
    #
    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 1 of 48,222,739 rows (2.07e-06%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`discharge_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE CASE WHEN CAST(`discharge_datetime` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`discharge_datetime` AS DATE)) < 9999 THEN NULL ELSE `discharge_datetime` END END AS `discharge_datetime`",

    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 1 of 48,222,739 rows (2.07e-06%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`workflow_complete_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `workflow_complete_datetime` END AS `workflow_complete_datetime`",
    "`raw_arrival_datetime` AS `raw_arrival_datetime`",
    "`raw_departure_datetime` AS `raw_departure_datetime`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 43,583,370 of 48,222,739 rows
    # (90.4%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`admission_source_code` AS STRING))) = '0' THEN NULL ELSE `admission_source_code` END AS `admission_source_code`",
    "`admission_source_display` AS `admission_source_display`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 37,436,947 of 48,222,739 rows
    # (77.6%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`discharge_destination_code` AS STRING))) = '0' THEN NULL ELSE `discharge_destination_code` END AS `discharge_destination_code`",
    "`discharge_destination_display` AS `discharge_destination_display`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 4,036,807 of 48,222,739 rows
    # (8.37%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`responsible_service_code` AS STRING))) = '0' THEN NULL ELSE `responsible_service_code` END AS `responsible_service_code`",
    "`responsible_service_display` AS `responsible_service_display`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 48,222,739 of 48,222,739 rows
    # (100%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`specialty_code` AS STRING))) = '0' THEN NULL ELSE `specialty_code` END AS `specialty_code`",
    "`specialty_display` AS `specialty_display`",
    "`current_location_id` AS `current_location_id`",
    "`organization_id` AS `organization_id`",
    "`service_provider_organization_id` AS `service_provider_organization_id`",
    "`reason_for_visit` AS `reason_for_visit`",
    "`attendance_evidence` AS `attendance_evidence`",
    "`attendance_witness_count` AS `attendance_witness_count`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",

    # record_status_effective_to cannot precede record_status_effective_from. The start is
    # the better-attested of the two, so the end is what goes and the row keeps its
    # record_status_effective_from. Hit 1 of 48,222,739 rows (2.07e-06%) when profiled on
    # 2026-08-24.
    "CASE WHEN `record_status_effective_from` IS NOT NULL AND `record_status_effective_to` IS NOT NULL AND `record_status_effective_from` > `record_status_effective_to` THEN NULL ELSE `record_status_effective_to` END AS `record_status_effective_to`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
]

SPINE_ENCOUNTER_COLUMN_COMMENTS = {
    "encounter_id": "Deterministic encounter primary key.",
    "subject_key": "Always-populated subject join key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier.",
    "parent_encounter_id": "Governed containment parent when supplied by source evidence.",
    "parentage_status": "Provenance state for encounter containment.",
    "encounter_level": "Rules-light source-classified encounter level.",
    "class_code": "Source encounter class code.",
    "class_display": "Source encounter class display.",
    "type_code": "Source encounter type code.",
    "type_display": "Source encounter type display.",
    "type_class_code": "Source encounter type-class code.",
    "type_class_display": "Source encounter type-class display.",
    "status_code": "Native encounter status code.",
    "status_display": "Native encounter status display.",
    "period_start": "Best observed encounter start.",
    "period_end": "Best observed encounter end.",
    "arrival_method": "Method selecting period_start.",
    "arrival_confidence": "Source-derived confidence for period_start.",
    "departure_method": "Method selecting period_end.",
    "departure_confidence": "Source-derived confidence for period_end.",
    "length_of_stay_minutes": "Source-productised encounter duration.",
    "scheduled_start": "Scheduled arrival timestamp.",
    "scheduled_end": "Scheduled departure timestamp.",
    "registration_datetime":
        "Registration timestamp retained as source evidence. Gold QC transform rules: gold.spine.encounter.registration_datetime.beyond_2100_below_9999_owner, gold.spine.encounter.registration_datetime.d1900_01_01_owner, gold.spine.encounter.registration_datetime.pre1901_other_owner.",
    "inpatient_admit_datetime": "Inpatient admission timestamp.",
    "discharge_datetime":
        "Discharge timestamp. Gold QC transform rules: gold.spine.encounter.discharge_datetime.beyond_2100_below_9999_owner.",
    "workflow_complete_datetime": "Administrative workflow completion timestamp.",
    "raw_arrival_datetime": "Raw ARRIVE_DT_TM retained without asserting observability.",
    "raw_departure_datetime":
        "Raw DEPART_DT_TM retained without asserting clinical meaning.",
    "admission_source_code": "Admission source code.",
    "admission_source_display": "Admission source display.",
    "discharge_destination_code": "Discharge destination code.",
    "discharge_destination_display": "Discharge destination display.",
    "responsible_service_code": "Responsible service code.",
    "responsible_service_display": "Responsible service display.",
    "specialty_code": "Source specialty-unit code.",
    "specialty_display": "Source specialty-unit display.",
    "current_location_id": "Current source nurse-unit location reference.",
    "organization_id": "Source encounter organization reference.",
    "service_provider_organization_id": "Source service-provider organization reference.",
    "reason_for_visit": "Verbatim source reason for visit.",
    "attendance_evidence": "Productised attendance evidence classification.",
    "attendance_witness_count": "Number of attendance witnesses in the source product.",
    "confidentiality_code": "Source confidentiality code.",
    "vip_ind": "Source VIP indicator.",
    "record_status": "Normalized source lifecycle status.",
    "record_status_effective_from": "Source active-status timestamp.",
    "record_status_effective_to": "Source effective end when superseded.",
    "load_batch_id": "Deterministic bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
}

@dp.materialized_view(
    name=_n("gold_spine.encounter"),
    comment=(
        "One source encounter with evidence-based arrival and departure semantics; "
        "containment remains explicit when unavailable. Gold QC twin of the silver product: "
        "13 columns are repaired or nulled, 0 check(s) are advisory. Each rule states its "
        "reason in the pipeline notebook, and Lakeflow expectation metrics report what every "
        "rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
def gold_spine_encounter():
    """Quality-controlled twin of journey_spine.encounter."""
    # 18 rows point at a person_id the spine does not have. The pointer is nulled so it
    # cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    df = _qc(
        "spine_encounter",
        SPINE_ENCOUNTER_SELECT,
        fk_columns=["person_id"],
    )
    return _with_comments(df, SPINE_ENCOUNTER_COLUMN_COMMENTS)

In [0]:
# ==== journey_spine.encounter_identifier ====

SPINE_ENCOUNTER_IDENTIFIER_SELECT = [
    "`encounter_identifier_id` AS `encounter_identifier_id`",
    "`encounter_id` AS `encounter_id`",
    "`identifier_system` AS `identifier_system`",
    "`identifier_type_code` AS `identifier_type_code`",
    "`identifier_value` AS `identifier_value`",
    "`status` AS `status`",
    "`current_ind` AS `current_ind`",
    "`multi_active_ind` AS `multi_active_ind`",

    # 1800-01-01 is a placeholder low date, not a record from 1800. Hit 41 of 142,054,710
    # rows (2.89e-05%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`valid_from` AS DATE) = DATE'1800-01-01' THEN NULL ELSE `valid_from` END AS `valid_from`",
    "`valid_to` AS `valid_to`",
    "`source_table` AS `source_table`",
    "`source_row_id` AS `source_row_id`",
    "`load_batch_id` AS `load_batch_id`",
    "`loaded_at` AS `loaded_at`",
]

SPINE_ENCOUNTER_IDENTIFIER_MANDATORY_RULES = {
    # The flow nulls encounter_id when it names a parent the spine does not have, and this
    # rule then drops the row, because an identifier for an encounter that is not in the
    # spine identifies nothing a researcher can join to. That was 318,195 orphaned rows plus
    # 0 that already had no encounter_id, out of 142,054,710.
    "gold.spine.encounter_identifier.encounter_id.fk_containment":
        "`encounter_id` IS NOT NULL",
}

SPINE_ENCOUNTER_IDENTIFIER_COLUMN_COMMENTS = {
    "encounter_identifier_id": "Deterministic id minted from the source alias row key.",
    "encounter_id": "Encounter reference (minted).",
    "identifier_system":
        "Type-scoped identifier system URI (urn:cerner:encntr_alias:<normalized alias type>) — 23.17M values live under multiple alias types",
    "identifier_type_code": "Source alias type verbatim.",
    "identifier_value": "Identifier value; published and IG-governed at serve time",
    "status": "Alias lifecycle status.",
    "current_ind": "Bronze current-alias indicator.",
    "multi_active_ind":
        "Bronze multiple-active-aliases indicator for this encounter and type.",
    "valid_from":
        "Alias effective-from timestamp. Gold QC transform rules: gold.spine.encounter_identifier.valid_from.d1800_01_01_owner.",
    "valid_to": "Alias effective-to timestamp when supplied.",
    "source_table": "Registered bronze source table.",
    "source_row_id": "Native source row identifier.",
    "load_batch_id": "Deterministic batch token derived from bronze load time.",
    "loaded_at": "Bronze pipeline write timestamp (this feed family has no ADC_UPDT).",
}

@dp.materialized_view(
    name=_n("gold_spine.encounter_identifier"),
    comment=(
        "One source encounter identifier assignment (FIN and sibling alias types) from "
        "map_encounter_identifier; reference surface parallel to person_identifier. Gold QC "
        "twin of the silver product: 2 columns are repaired or nulled, 1 rule(s) drop rows, 0 "
        "check(s) are advisory. Each rule states its reason in the pipeline notebook, and "
        "Lakeflow expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(SPINE_ENCOUNTER_IDENTIFIER_MANDATORY_RULES)
def gold_spine_encounter_identifier():
    """Quality-controlled twin of journey_spine.encounter_identifier."""
    df = _qc(
        "spine_encounter_identifier",
        SPINE_ENCOUNTER_IDENTIFIER_SELECT,
        fk_columns=["encounter_id"],
    )
    return _with_comments(df, SPINE_ENCOUNTER_IDENTIFIER_COLUMN_COMMENTS)

In [0]:
# ==== journey_spine.episode ====

SPINE_EPISODE_SELECT = [
    "`episode_id` AS `episode_id`",
    "`person_id` AS `person_id`",
    "`subject_key` AS `subject_key`",
    "`episode_display` AS `episode_display`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 2,794 of 7,799,236 rows
    # (0.0358%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`episode_type_code` AS STRING))) = '0' THEN NULL ELSE `episode_type_code` END AS `episode_type_code`",
    "`episode_type_display` AS `episode_type_display`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 945,739 of 7,799,236 rows
    # (12.1%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`status_code` AS STRING))) = '0' THEN NULL ELSE `status_code` END AS `status_code`",
    "`status_display` AS `status_display`",
    "`period_start` AS `period_start`",

    # period_end cannot precede period_start. The start is the better-attested of the two,
    # so the end is what goes and the row keeps its period_start. Hit 222 of 7,799,236 rows
    # (0.00285%) when profiled on 2026-08-24.
    "CASE WHEN `period_start` IS NOT NULL AND `period_end` IS NOT NULL AND `period_start` > `period_end` THEN NULL ELSE `period_end` END AS `period_end`",
    "`breach_datetime` AS `breach_datetime`",
    "`pause_days` AS `pause_days`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 7,799,236 of 7,799,236 rows
    # (100%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`close_reason_code` AS STRING))) = '0' THEN NULL ELSE `close_reason_code` END AS `close_reason_code`",
    "`close_reason_display` AS `close_reason_display`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 1,293,188 of 7,799,236 rows
    # (16.6%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`service_category_code` AS STRING))) = '0' THEN NULL ELSE `service_category_code` END AS `service_category_code`",
    "`service_category_display` AS `service_category_display`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 745,773 of 7,799,236 rows
    # (9.56%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`referring_facility_code` AS STRING))) = '0' THEN NULL ELSE `referring_facility_code` END AS `referring_facility_code`",
    "`referring_facility_display` AS `referring_facility_display`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 1 of 7,799,236 rows
    # (1.28e-05%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`contributor_system_code` AS STRING))) = '0' THEN NULL ELSE `contributor_system_code` END AS `contributor_system_code`",
    "`contributor_system_display` AS `contributor_system_display`",
    "`direct_encounter_id` AS `direct_encounter_id`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",

    # A date before 1901 that is not one of the known placeholders. Nothing in this estate
    # predates the twentieth century, so these are mistyped or mis-scaled rather than early.
    # Hit 12 of 7,799,236 rows (0.000154%) when profiled on 2026-08-24.
    #
    # record_status_effective_to cannot precede record_status_effective_from. The start is
    # the better-attested of the two, so the end is what goes and the row keeps its
    # record_status_effective_from. Hit 12 of 7,799,236 rows (0.000154%) when profiled on
    # 2026-08-24.
    "CASE WHEN `record_status_effective_from` IS NOT NULL AND `record_status_effective_to` IS NOT NULL AND `record_status_effective_from` > `record_status_effective_to` THEN NULL ELSE CASE WHEN CAST(`record_status_effective_to` AS DATE) < DATE'1901-01-01' AND CAST(`record_status_effective_to` AS DATE) NOT IN (DATE'1800-01-01', DATE'1899-12-30', DATE'1900-01-01') THEN NULL ELSE `record_status_effective_to` END END AS `record_status_effective_to`",
    "`source_table` AS `source_table`",
    "`source_row_id` AS `source_row_id`",
    "`load_batch_id` AS `load_batch_id`",
    "`loaded_at` AS `loaded_at`",
]

SPINE_EPISODE_ADVISORY_RULES = {
    # This bounds a period of validity, and a future end is exactly how the source says a
    # record is still current -- nulling it would assert the record is valid forever, which
    # is a stronger and worse claim than the one being corrected. Seen on 1 of 7,799,236
    # rows (1.28e-05%) when profiled on 2026-08-24.
    "gold.spine.episode.record_status_effective_to.future_owner":
        "NOT COALESCE((CAST(`record_status_effective_to` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",
}

SPINE_EPISODE_COLUMN_COMMENTS = {
    "episode_id": "Deterministic id minted from the Millennium episode identifier.",
    "person_id": "Millennium person identifier.",
    "subject_key": "Peppered subject key.",
    "episode_display": "Source episode display name.",
    "episode_type_code": "Source episode type code.",
    "episode_type_display": "Source episode type display.",
    "status_code": "Source episode status code.",
    "status_display": "Source episode status display.",
    "period_start": "Sentinel-cleaned episode begin timestamp (bronze DQ triplet).",
    "period_end": "Sentinel-cleaned episode end timestamp when supplied.",
    "breach_datetime": "Sentinel-cleaned episode breach timestamp when supplied.",
    "pause_days": "Source pause-day count.",
    "close_reason_code": "Source close-reason code.",
    "close_reason_display": "Source close-reason display.",
    "service_category_code": "Source service-category code.",
    "service_category_display": "Source service-category display.",
    "referring_facility_code": "Source referring-facility code.",
    "referring_facility_display": "Source referring-facility display.",
    "contributor_system_code": "Source contributor-system code.",
    "contributor_system_display": "Source contributor-system display.",
    "direct_encounter_id":
        "Agreement-gated direct encounter reference published by bronze alongside the N-to-M relation; evidence only",
    "record_status": "Normalized source lifecycle status.",
    "record_status_effective_from": "Source status effective timestamp.",
    "record_status_effective_to":
        "Source status end timestamp when superseded. Gold QC transform rules: gold.spine.episode.record_status_effective_to.pre1901_other_owner.",
    "source_table": "Registered bronze source table.",
    "source_row_id": "Native source row identifier.",
    "load_batch_id": "Deterministic batch token derived from bronze load time.",
    "loaded_at": "Bronze load timestamp; never pipeline wall-clock time.",
}

@dp.materialized_view(
    name=_n("gold_spine.episode"),
    comment=(
        "One Millennium episode container populated from the governed feed map_episode; "
        "source-backed only, never heuristic. Gold QC twin of the silver product: 9 columns "
        "are repaired or nulled, 1 check(s) are advisory. Each rule states its reason in the "
        "pipeline notebook, and Lakeflow expectation metrics report what every rule matched "
        "on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all(SPINE_EPISODE_ADVISORY_RULES)
def gold_spine_episode():
    """Quality-controlled twin of journey_spine.episode."""
    # 618 rows point at a person_id the spine does not have. The pointer is nulled so it
    # cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    df = _qc(
        "spine_episode",
        SPINE_EPISODE_SELECT,
        fk_columns=["person_id"],
    )
    return _with_comments(df, SPINE_EPISODE_COLUMN_COMMENTS)

In [0]:
# ==== journey_spine.episode_encounter ====

SPINE_EPISODE_ENCOUNTER_SELECT = [
    "`episode_encounter_id` AS `episode_encounter_id`",
    "`episode_id` AS `episode_id`",
    "`encounter_id` AS `encounter_id`",
    "`relation_status_code` AS `relation_status_code`",
    "`source_duplicate_count` AS `source_duplicate_count`",
    "`valid_from` AS `valid_from`",
    "`valid_to` AS `valid_to`",
    "`record_status` AS `record_status`",
    "`source_table` AS `source_table`",
    "`source_row_id` AS `source_row_id`",
    "`load_batch_id` AS `load_batch_id`",
    "`loaded_at` AS `loaded_at`",
]

SPINE_EPISODE_ENCOUNTER_MANDATORY_RULES = {
    # The flow nulls encounter_id when it names a parent the spine does not have, and this
    # rule then drops the row, because this table is nothing but the pair of keys, so a link
    # with an end missing links nothing. That was 71,166 orphaned rows plus 0 that already
    # had no encounter_id, out of 26,812,360.
    "gold.spine.episode_encounter.encounter_id.fk_containment":
        "`encounter_id` IS NOT NULL",

    # The flow nulls episode_id when it names a parent the spine does not have, and this
    # rule then drops the row, because this table is nothing but the pair of keys, so a link
    # with an end missing links nothing. That was 20,686 orphaned rows plus 0 that already
    # had no episode_id, out of 26,812,360.
    "gold.spine.episode_encounter.episode_id.fk_containment": "`episode_id` IS NOT NULL",
}

SPINE_EPISODE_ENCOUNTER_COLUMN_COMMENTS = {
    "episode_encounter_id": "Deterministic id minted from the source relation row.",
    "episode_id":
        "Episode container reference (minted). Rows naming an episode absent from the spine are dropped, so every value here resolves.",
    "encounter_id": "Member encounter reference (minted).",
    "relation_status_code":
        "Source relation classification. The orphan marker for episode IDs absent from mill_episode does not appear: those rows are dropped.",
    "source_duplicate_count":
        "Raw byte-identical duplicate rows collapsed into this canonical row by bronze.",
    "valid_from": "Membership effective-from timestamp.",
    "valid_to": "Membership effective-to timestamp when supplied.",
    "record_status": "Normalized source lifecycle status.",
    "source_table": "Registered bronze source table.",
    "source_row_id": "Native source row identifier.",
    "load_batch_id": "Deterministic batch token derived from bronze load time.",
    "loaded_at": "Bronze load timestamp.",
}

@dp.materialized_view(
    name=_n("gold_spine.episode_encounter"),
    comment=(
        "N-to-M episode-to-encounter membership from the governed feed map_episode_encounter; "
        "multi-relation pairs preserved. Relations naming an episode or an encounter the "
        "spine does not have are dropped, so both keys resolve. Gold QC twin of the silver "
        "product: 2 columns are repaired or nulled, 2 rule(s) drop rows, 0 check(s) are "
        "advisory. Each rule states its reason in the pipeline notebook, and Lakeflow "
        "expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(SPINE_EPISODE_ENCOUNTER_MANDATORY_RULES)
def gold_spine_episode_encounter():
    """Quality-controlled twin of journey_spine.episode_encounter."""
    df = _qc(
        "spine_episode_encounter",
        SPINE_EPISODE_ENCOUNTER_SELECT,
        fk_columns=["encounter_id", "episode_id"],
    )
    return _with_comments(df, SPINE_EPISODE_ENCOUNTER_COLUMN_COMMENTS)

In [0]:
# ==== journey_spine.journey ====

SPINE_JOURNEY_SELECT = [
    "`journey_id` AS `journey_id`",
    "`parent_journey_id` AS `parent_journey_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`journey_type_code` AS `journey_type_code`",
    "`journey_type_display` AS `journey_type_display`",
    "`period_start` AS `period_start`",

    # 2100-12-31 is the far-future marker the source writes to mean 'no end yet'. The
    # absence is what the row means, and NULL states it without putting a fictional date
    # into a range comparison. Hit 1,202 of 824,072 rows (0.146%) when profiled on
    # 2026-08-24.
    #
    # period_end cannot precede period_start. The start is the better-attested of the two,
    # so the end is what goes and the row keeps its period_start. Hit 472 of 824,072 rows
    # (0.0573%) when profiled on 2026-08-24.
    "CASE WHEN `period_start` IS NOT NULL AND `period_end` IS NOT NULL AND `period_start` > `period_end` THEN NULL ELSE CASE WHEN CAST(`period_end` AS DATE) = DATE'2100-12-31' THEN NULL ELSE `period_end` END END AS `period_end`",
    "`status_code` AS `status_code`",
    "`defining_coding_system` AS `defining_coding_system`",
    "`defining_code` AS `defining_code`",
    "`defining_display` AS `defining_display`",
    "`outcome_code` AS `outcome_code`",
    "`outcome_display` AS `outcome_display`",
    "`source_journey_identifier` AS `source_journey_identifier`",
    "`construction_rule` AS `construction_rule`",
    "`construction_version` AS `construction_version`",
    "`source_system` AS `source_system`",
    "`source_table` AS `source_table`",
    "`source_row_id` AS `source_row_id`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
]

SPINE_JOURNEY_ADVISORY_RULES = {
    # This bounds a period of validity, and a future end is exactly how the source says a
    # record is still current -- nulling it would assert the record is valid forever, which
    # is a stronger and worse claim than the one being corrected. Seen on 25,297 of 824,072
    # rows (3.07%) when profiled on 2026-08-24.
    "gold.spine.journey.period_end.future_owner":
        "NOT COALESCE((CAST(`period_end` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",

    # This bounds a period of validity, and a future end is exactly how the source says a
    # record is still current -- nulling it would assert the record is valid forever, which
    # is a stronger and worse claim than the one being corrected. Seen on 6,633 of 824,072
    # rows (0.805%) when profiled on 2026-08-24.
    "gold.spine.journey.period_start.future_owner":
        "NOT COALESCE((CAST(`period_start` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",
}

SPINE_JOURNEY_COLUMN_COMMENTS = {
    "journey_id": "Deterministic journey primary key.",
    "parent_journey_id": "Parent journey for governed nesting.",
    "subject_key": "Always-populated primary-subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved primary-subject person identifier when available.",
    "journey_type_code": "Controlled journey type.",
    "journey_type_display": "Human-readable journey type.",
    "period_start": "Earliest source-supported journey boundary.",
    "period_end":
        "Latest source-supported journey boundary. Gold QC transform rules: gold.spine.journey.period_end.open_sentinel.",
    "status_code": "Normalized journey lifecycle state.",
    "defining_coding_system": "Coding system for the constructor-defining concept.",
    "defining_code": "Constructor-defining concept code.",
    "defining_display": "Constructor-defining concept display.",
    "outcome_code": "Source pregnancy outcome code where supplied.",
    "outcome_display": "Source pregnancy outcome display where supplied.",
    "source_journey_identifier": "Verbatim source pregnancy identifier.",
    "construction_rule": "Governed constructor name.",
    "construction_version": "Governed constructor version.",
    "source_system": "Source-system identifier.",
    "source_table": "Fully qualified configured pregnancy source table.",
    "source_row_id": "Stable constructor source-row identifier.",
    "load_batch_id": "Deterministic bronze batch token.",
    "source_update_timestamp":
        "Latest native source update timestamp supporting construction.",
    "loaded_at": "Latest bronze load timestamp supporting construction.",
}

@dp.materialized_view(
    name=_n("gold_spine.journey"),
    comment=(
        "One deterministic longitudinal episode of care, including nested structural child "
        "journeys. Gold QC twin of the silver product: 2 columns are repaired or nulled, 2 "
        "check(s) are advisory. Each rule states its reason in the pipeline notebook, and "
        "Lakeflow expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all(SPINE_JOURNEY_ADVISORY_RULES)
def gold_spine_journey():
    """Quality-controlled twin of journey_spine.journey."""
    # 4 rows point at a person_id the spine does not have. The pointer is nulled so it
    # cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    df = _qc(
        "spine_journey",
        SPINE_JOURNEY_SELECT,
        fk_columns=["person_id"],
    )
    return _with_comments(df, SPINE_JOURNEY_COLUMN_COMMENTS)

In [0]:
# ==== journey_spine.journey_link ====

SPINE_JOURNEY_LINK_SELECT = [
    "`journey_link_id` AS `journey_link_id`",
    "`journey_id` AS `journey_id`",
    "`member_type` AS `member_type`",
    "`encounter_id` AS `encounter_id`",
    "`patient_event_id` AS `patient_event_id`",
    "`role_code` AS `role_code`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 5 of 646,884,263 rows (7.73e-07%) when profiled on 2026-08-24.
    #
    # 1900-01-01 is a placeholder low date rather than a date in 1900; on a non-birth field
    # it carries no more meaning than it does on a birth one. Hit 10 of 646,884,263 rows
    # (1.55e-06%) when profiled on 2026-08-24.
    #
    # A date before 1901 that is not one of the known placeholders. Nothing in this estate
    # predates the twentieth century, so these are mistyped or mis-scaled rather than early.
    # Hit 353 of 646,884,263 rows (5.46e-05%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`member_start_datetime` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`member_start_datetime` AS DATE)) < 9999 OR CAST(`member_start_datetime` AS DATE) = DATE'1900-01-01' OR CAST(`member_start_datetime` AS DATE) < DATE'1901-01-01' AND CAST(`member_start_datetime` AS DATE) NOT IN (DATE'1800-01-01', DATE'1899-12-30', DATE'1900-01-01') THEN NULL ELSE `member_start_datetime` END AS `member_start_datetime`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 4 of 646,884,263 rows (6.18e-07%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`member_end_datetime` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`member_end_datetime` AS DATE)) < 9999 THEN NULL ELSE `member_end_datetime` END AS `member_end_datetime`",
    "`record_status` AS `record_status`",
    "`construction_rule` AS `construction_rule`",
    "`construction_version` AS `construction_version`",
    "`source_system` AS `source_system`",
    "`source_table` AS `source_table`",
    "`source_row_id` AS `source_row_id`",
    "`load_batch_id` AS `load_batch_id`",
    "`loaded_at` AS `loaded_at`",
]

SPINE_JOURNEY_LINK_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 646,884,263 rows of
    # 646,884,263 that are current and attributable. Superseded versions and rows whose
    # identity was never resolved are not research data, and a consumer who wants them has
    # silver.
    "research_surface": "(record_status = 'active')",
}

SPINE_JOURNEY_LINK_ADVISORY_RULES = {
    # This bounds a period of validity, and a future end is exactly how the source says a
    # record is still current -- nulling it would assert the record is valid forever, which
    # is a stronger and worse claim than the one being corrected. Seen on 55,712 of
    # 646,884,263 rows (0.00861%) when profiled on 2026-08-24.
    "gold.spine.journey_link.member_end_datetime.future_owner":
        "NOT COALESCE((CAST(`member_end_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",

    # This bounds a period of validity, and a future end is exactly how the source says a
    # record is still current -- nulling it would assert the record is valid forever, which
    # is a stronger and worse claim than the one being corrected. Seen on 56,267 of
    # 646,884,263 rows (0.0087%) when profiled on 2026-08-24.
    "gold.spine.journey_link.member_start_datetime.future_owner":
        "NOT COALESCE((CAST(`member_start_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",

    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 0 of 646,884,263 at the profile.
    "gold.spine.journey_link.record_status.default_view_active": "record_status = 'active'",

    # Left as a warning because it fires on 15,386,217 of 646,884,263 rows (2.38%) when
    # profiled on 2026-08-24 -- at that rate the rule's assumption about what
    # member_start_datetime and member_end_datetime mean is the thing in doubt, not the
    # data. The inverted gaps are mostly minutes, which reads as two clocks rather than two
    # events in the wrong order.
    "gold.spine.journey_link.table.ordering_violation_member_start_datetime_member_end_datetime":
        "NOT COALESCE((`member_start_datetime` IS NOT NULL AND `member_end_datetime` IS NOT NULL AND `member_start_datetime` > `member_end_datetime`), FALSE)",
}

SPINE_JOURNEY_LINK_COLUMN_COMMENTS = {
    "journey_link_id": "Deterministic journey-link primary key.",
    "journey_id": "Journey reference.",
    "member_type": "Kind of linked member.",
    "encounter_id": "Encounter member when member_type is encounter.",
    "patient_event_id": "Event member when member_type is patient_event.",
    "role_code": "Controlled link role.",
    "member_start_datetime":
        "Linked member start timestamp used by the constructor. Gold QC transform rules: gold.spine.journey_link.member_start_datetime.beyond_2100_below_9999_owner, gold.spine.journey_link.member_start_datetime.d1900_01_01_owner, gold.spine.journey_link.member_start_datetime.pre1901_other_owner.",
    "member_end_datetime":
        "Linked member end timestamp used by the constructor. Gold QC transform rules: gold.spine.journey_link.member_end_datetime.beyond_2100_below_9999_owner.",
    "record_status": "Normalized link lifecycle state.",
    "construction_rule":
        "Governed overlap-construction rule; episode_membership populated from the governed episode feeds.",
    "construction_version": "Governed overlap-construction version.",
    "source_system": "Constructor source-system identifier.",
    "source_table": "Fully qualified configured journey-constructor source table.",
    "source_row_id": "Stable constructor source-row identifier.",
    "load_batch_id": "Deterministic bronze batch token.",
    "loaded_at": "Latest bronze load timestamp supporting the constructor.",
}

@dp.materialized_view(
    name=_n("gold_spine.journey_link"),
    comment=(
        "One deterministic journey-to-encounter or journey-to-event overlap link. Gold QC "
        "twin of the silver product: 2 columns are repaired or nulled, 1 rule(s) drop rows, 4 "
        "check(s) are advisory. Each rule states its reason in the pipeline notebook, and "
        "Lakeflow expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(SPINE_JOURNEY_LINK_MANDATORY_RULES)
@_expect_all(SPINE_JOURNEY_LINK_ADVISORY_RULES)
def gold_spine_journey_link():
    """Quality-controlled twin of journey_spine.journey_link."""
    df = _qc("spine_journey_link", SPINE_JOURNEY_LINK_SELECT)
    return _with_comments(df, SPINE_JOURNEY_LINK_COLUMN_COMMENTS)

In [0]:
# ==== journey_spine.journey_participant ====

SPINE_JOURNEY_PARTICIPANT_SELECT = [
    "`journey_participant_id` AS `journey_participant_id`",
    "`journey_id` AS `journey_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`role_code` AS `role_code`",
    "`role_display` AS `role_display`",
    "`valid_from` AS `valid_from`",

    # 2100-12-31 is the far-future marker the source writes to mean 'no end yet'. The
    # absence is what the row means, and NULL states it without putting a fictional date
    # into a range comparison. Hit 1,202 of 1,010,584 rows (0.119%) when profiled on
    # 2026-08-24.
    #
    # valid_to cannot precede valid_from. The start is the better-attested of the two, so
    # the end is what goes and the row keeps its valid_from. Hit 472 of 1,010,584 rows
    # (0.0467%) when profiled on 2026-08-24.
    "CASE WHEN `valid_from` IS NOT NULL AND `valid_to` IS NOT NULL AND `valid_from` > `valid_to` THEN NULL ELSE CASE WHEN CAST(`valid_to` AS DATE) = DATE'2100-12-31' THEN NULL ELSE `valid_to` END END AS `valid_to`",
    "`record_status` AS `record_status`",
    "`construction_rule` AS `construction_rule`",
    "`construction_version` AS `construction_version`",
    "`source_system` AS `source_system`",
    "`source_table` AS `source_table`",
    "`source_row_id` AS `source_row_id`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
]

SPINE_JOURNEY_PARTICIPANT_ADVISORY_RULES = {
    # This bounds a period of validity, and a future end is exactly how the source says a
    # record is still current -- nulling it would assert the record is valid forever, which
    # is a stronger and worse claim than the one being corrected. Seen on 7,368 of 1,010,584
    # rows (0.729%) when profiled on 2026-08-24.
    "gold.spine.journey_participant.valid_from.future_owner":
        "NOT COALESCE((CAST(`valid_from` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",

    # This bounds a period of validity, and a future end is exactly how the source says a
    # record is still current -- nulling it would assert the record is valid forever, which
    # is a stronger and worse claim than the one being corrected. Seen on 25,297 of
    # 1,010,584 rows (2.5%) when profiled on 2026-08-24.
    "gold.spine.journey_participant.valid_to.future_owner":
        "NOT COALESCE((CAST(`valid_to` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",
}

SPINE_JOURNEY_PARTICIPANT_COLUMN_COMMENTS = {
    "journey_participant_id": "Deterministic journey-participant primary key.",
    "journey_id": "Journey reference.",
    "subject_key": "Always-populated participant subject key.",
    "subject_id_system": "Identifier system used for participant subject_key.",
    "person_id": "Resolved participant person identifier when available.",
    "role_code": "Controlled role in the journey.",
    "role_display": "Human-readable participant role.",
    "valid_from": "Role validity start.",
    "valid_to":
        "Role validity end. Gold QC transform rules: gold.spine.journey_participant.valid_to.open_sentinel.",
    "record_status": "Normalized source lifecycle status.",
    "construction_rule": "Governed participant-construction rule.",
    "construction_version": "Governed participant-construction version.",
    "source_system": "Source-system identifier.",
    "source_table": "Fully qualified configured bronze source table.",
    "source_row_id": "Stable source-row identifier supporting the role.",
    "load_batch_id": "Deterministic bronze batch token.",
    "source_update_timestamp": "Latest native source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
}

@dp.materialized_view(
    name=_n("gold_spine.journey_participant"),
    comment=(
        "One journey-to-person role interval, retaining unresolved participants through "
        "subject_key. Gold QC twin of the silver product: 2 columns are repaired or nulled, 2 "
        "check(s) are advisory. Each rule states its reason in the pipeline notebook, and "
        "Lakeflow expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all(SPINE_JOURNEY_PARTICIPANT_ADVISORY_RULES)
def gold_spine_journey_participant():
    """Quality-controlled twin of journey_spine.journey_participant."""
    # 106 rows point at a person_id the spine does not have. The pointer is nulled so it
    # cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    df = _qc(
        "spine_journey_participant",
        SPINE_JOURNEY_PARTICIPANT_SELECT,
        fk_columns=["person_id"],
    )
    return _with_comments(df, SPINE_JOURNEY_PARTICIPANT_COLUMN_COMMENTS)

In [0]:
# ==== journey_spine.location_stay ====

SPINE_LOCATION_STAY_SELECT = [
    "`location_stay_id` AS `location_stay_id`",
    "`encounter_id` AS `encounter_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`location_id` AS `location_id`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 1 of 65,583,474 rows (1.52e-06%) when profiled on 2026-08-24.
    #
    # 2100-12-31 appears here on a field that is not an end date, so it cannot be the 'still
    # open' marker it is elsewhere and is a placeholder instead. Hit 8,156 of 65,583,474
    # rows (0.0124%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`period_start` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`period_start` AS DATE)) < 9999 OR CAST(`period_start` AS DATE) = DATE'2100-12-31' THEN NULL ELSE `period_start` END AS `period_start`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 3 of 65,583,474 rows (4.57e-06%) when profiled on 2026-08-24.
    #
    # 2100-12-31 is the far-future marker the source writes to mean 'no end yet'. The
    # absence is what the row means, and NULL states it without putting a fictional date
    # into a range comparison. Hit 2,318 of 65,583,474 rows (0.00353%) when profiled on
    # 2026-08-24.
    "CASE WHEN CAST(`period_end` AS DATE) = DATE'2100-12-31' OR CAST(`period_end` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`period_end` AS DATE)) < 9999 THEN NULL ELSE `period_end` END AS `period_end`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 12,714,506 of 65,583,474 rows
    # (19.4%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`nurse_unit_code` AS STRING))) = '0' THEN NULL ELSE `nurse_unit_code` END AS `nurse_unit_code`",
    "`nurse_unit_display` AS `nurse_unit_display`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 8,237,381 of 65,583,474 rows
    # (12.6%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`building_code` AS STRING))) = '0' THEN NULL ELSE `building_code` END AS `building_code`",
    "`building_display` AS `building_display`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 4,981,884 of 65,583,474 rows
    # (7.6%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`facility_code` AS STRING))) = '0' THEN NULL ELSE `facility_code` END AS `facility_code`",
    "`facility_display` AS `facility_display`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 58,174,875 of 65,583,474 rows
    # (88.7%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`room_code` AS STRING))) = '0' THEN NULL ELSE `room_code` END AS `room_code`",
    "`room_display` AS `room_display`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 58,488,665 of 65,583,474 rows
    # (89.2%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`bed_code` AS STRING))) = '0' THEN NULL ELSE `bed_code` END AS `bed_code`",
    "`bed_display` AS `bed_display`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 4,768,656 of 65,583,474 rows
    # (7.27%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`service_code` AS STRING))) = '0' THEN NULL ELSE `service_code` END AS `service_code`",
    "`service_display` AS `service_display`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 62,075,788 of 65,583,474 rows
    # (94.7%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`transfer_reason_code` AS STRING))) = '0' THEN NULL ELSE `transfer_reason_code` END AS `transfer_reason_code`",
    "`transfer_reason_display` AS `transfer_reason_display`",
    "`source_history_row_count` AS `source_history_row_count`",
    "`first_history_event_sequence` AS `first_history_event_sequence`",
    "`last_history_event_sequence` AS `last_history_event_sequence`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 65,574,001 of 65,583,474 rows
    # (100%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`confidentiality_code` AS STRING))) = '0' THEN NULL ELSE `confidentiality_code` END AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`record_status` AS `record_status`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`source_table` AS `source_table`",
    "`source_row_id` AS `source_row_id`",
    "`load_batch_id` AS `load_batch_id`",
    "`loaded_at` AS `loaded_at`",
]

SPINE_LOCATION_STAY_ADVISORY_RULES = {
    # This bounds a period of validity, and a future end is exactly how the source says a
    # record is still current -- nulling it would assert the record is valid forever, which
    # is a stronger and worse claim than the one being corrected. Seen on 1 of 65,583,474
    # rows (1.52e-06%) when profiled on 2026-08-24.
    "gold.spine.location_stay.period_end.future_owner":
        "NOT COALESCE((CAST(`period_end` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",

    # This bounds a period of validity, and a future end is exactly how the source says a
    # record is still current -- nulling it would assert the record is valid forever, which
    # is a stronger and worse claim than the one being corrected. Seen on 1 of 65,583,474
    # rows (1.52e-06%) when profiled on 2026-08-24.
    "gold.spine.location_stay.period_start.future_owner":
        "NOT COALESCE((CAST(`period_start` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",

    # This bounds a period of validity, and a future end is exactly how the source says a
    # record is still current -- nulling it would assert the record is valid forever, which
    # is a stronger and worse claim than the one being corrected. Seen on 1 of 65,583,474
    # rows (1.52e-06%) when profiled on 2026-08-24.
    "gold.spine.location_stay.record_status_effective_to.future_owner":
        "NOT COALESCE((CAST(`record_status_effective_to` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",
}

SPINE_LOCATION_STAY_COLUMN_COMMENTS = {
    "location_stay_id": "Deterministic location-stop primary key.",
    "encounter_id": "Encounter reference for this stop.",
    "subject_key": "Always-populated subject join key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier.",
    "location_id": "Governed current location-dimension FK when resolvable.",
    "period_start":
        "Location-stop start. Gold QC transform rules: gold.spine.location_stay.period_start.beyond_2100_below_9999_owner, gold.spine.location_stay.period_start.d2100_12_31_owner.",
    "period_end":
        "Location-stop end. Gold QC transform rules: gold.spine.location_stay.period_end.beyond_2100_below_9999_owner, gold.spine.location_stay.period_end.open_sentinel.",
    "nurse_unit_code": "Historical nurse-unit code.",
    "nurse_unit_display": "Historical nurse-unit display.",
    "building_code": "Historical building code.",
    "building_display": "Historical building display.",
    "facility_code": "Historical facility code.",
    "facility_display": "Historical facility display.",
    "room_code": "Historical room code.",
    "room_display": "Historical room display.",
    "bed_code": "Historical bed code.",
    "bed_display": "Historical bed display.",
    "service_code": "Service code during the stop.",
    "service_display": "Service display during the stop.",
    "transfer_reason_code": "Source transfer-reason code.",
    "transfer_reason_display": "Source transfer-reason display.",
    "source_history_row_count": "Number of history rows grouped into the stop.",
    "first_history_event_sequence": "First source history sequence in the stop.",
    "last_history_event_sequence": "Last source history sequence in the stop.",
    "confidentiality_code": "Source confidentiality code.",
    "vip_ind": "Source VIP indicator.",
    "record_status": "Normalized source lifecycle status.",
    "record_status_effective_to": "Stop end used as lifecycle end when present.",
    "source_table": "Fully qualified bronze source table.",
    "source_row_id": "Deterministic grouped source-row identity.",
    "load_batch_id": "Deterministic bronze batch token.",
    "loaded_at": "Latest bronze load timestamp in the grouped stop.",
}

@dp.materialized_view(
    name=_n("gold_spine.location_stay"),
    comment=(
        "One grouped physical location stop per encounter, retaining raw historical location "
        "evidence even when the current dimension cannot resolve it. Gold QC twin of the "
        "silver product: 12 columns are repaired or nulled, 3 check(s) are advisory. Each "
        "rule states its reason in the pipeline notebook, and Lakeflow expectation metrics "
        "report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all(SPINE_LOCATION_STAY_ADVISORY_RULES)
def gold_spine_location_stay():
    """Quality-controlled twin of journey_spine.location_stay."""
    # 9,473 rows point at a encounter_id the spine does not have. The pointer is nulled so
    # it cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    # 24 rows point at a person_id the spine does not have. The pointer is nulled so it
    # cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    df = _qc(
        "spine_location_stay",
        SPINE_LOCATION_STAY_SELECT,
        fk_columns=["encounter_id", "person_id"],
    )
    return _with_comments(df, SPINE_LOCATION_STAY_COLUMN_COMMENTS)

In [0]:
# ==== journey_spine.person ====

SPINE_PERSON_SELECT = [
    "`person_id` AS `person_id`",
    "`active` AS `active`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 390,007 of 6,961,795 rows
    # (5.6%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`gender_code` AS STRING))) = '0' THEN NULL ELSE `gender_code` END AS `gender_code`",
    "`gender_display` AS `gender_display`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 555,945 of 6,961,795 rows
    # (7.99%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`ethnicity_code` AS STRING))) = '0' THEN NULL ELSE `ethnicity_code` END AS `ethnicity_code`",
    "`ethnicity_display` AS `ethnicity_display`",

    # 2100-12-31 appears here on a field that is not an end date, so it cannot be the 'still
    # open' marker it is elsewhere and is a placeholder instead. Hit 1 of 6,961,795 rows
    # (1.44e-05%) when profiled on 2026-08-24.
    #
    # 1899-12-30 is the zero point of the OLE/Excel date scale, so it is what a spreadsheet
    # or a COM layer writes when the date was left blank. It is not a date anyone recorded.
    # Hit 4 of 6,961,795 rows (5.75e-05%) when profiled on 2026-08-24.
    #
    # A date before 1901 that is not one of the known placeholders. Nothing in this estate
    # predates the twentieth century, so these are mistyped or mis-scaled rather than early.
    # Hit 4,817 of 6,961,795 rows (0.0692%) when profiled on 2026-08-24.
    #
    # 1900-01-01 on a birth field is the source's 'date of birth unknown' encoding, not a
    # birth in 1900. Left in place it would make thousands of people the same improbable
    # age. Hit 13,163 of 6,961,795 rows (0.189%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`birth_date` AS DATE) = DATE'1899-12-30' OR CAST(`birth_date` AS DATE) = DATE'1900-01-01' OR CAST(`birth_date` AS DATE) = DATE'2100-12-31' OR CAST(`birth_date` AS DATE) < DATE'1901-01-01' AND CAST(`birth_date` AS DATE) NOT IN (DATE'1800-01-01', DATE'1899-12-30', DATE'1900-01-01') THEN NULL ELSE `birth_date` END AS `birth_date`",

    # 2100-12-31 appears here on a field that is not an end date, so it cannot be the 'still
    # open' marker it is elsewhere and is a placeholder instead. Hit 1 of 6,961,795 rows
    # (1.44e-05%) when profiled on 2026-08-24.
    #
    # 1899-12-30 is the zero point of the OLE/Excel date scale, so it is what a spreadsheet
    # or a COM layer writes when the date was left blank. It is not a date anyone recorded.
    # Hit 4 of 6,961,795 rows (5.75e-05%) when profiled on 2026-08-24.
    #
    # A date before 1901 that is not one of the known placeholders. Nothing in this estate
    # predates the twentieth century, so these are mistyped or mis-scaled rather than early.
    # Hit 4,817 of 6,961,795 rows (0.0692%) when profiled on 2026-08-24.
    #
    # 1900-01-01 on a birth field is the source's 'date of birth unknown' encoding, not a
    # birth in 1900. Left in place it would make thousands of people the same improbable
    # age. Hit 13,163 of 6,961,795 rows (0.189%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`birth_datetime` AS DATE) = DATE'1899-12-30' OR CAST(`birth_datetime` AS DATE) = DATE'1900-01-01' OR CAST(`birth_datetime` AS DATE) = DATE'2100-12-31' OR CAST(`birth_datetime` AS DATE) < DATE'1901-01-01' AND CAST(`birth_datetime` AS DATE) NOT IN (DATE'1800-01-01', DATE'1899-12-30', DATE'1900-01-01') THEN NULL ELSE `birth_datetime` END AS `birth_datetime`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 6,571,856 of 6,961,795 rows
    # (94.4%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`birth_precision_code` AS STRING))) = '0' THEN NULL ELSE `birth_precision_code` END AS `birth_precision_code`",
    "`birth_precision_display` AS `birth_precision_display`",
    "`birth_precision_flag` AS `birth_precision_flag`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 4,111,484 of 6,961,795 rows
    # (59.1%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`language_code` AS STRING))) = '0' THEN NULL ELSE `language_code` END AS `language_code`",
    "`language_display` AS `language_display`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 2,706,320 of 6,961,795 rows
    # (38.9%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`marital_status_code` AS STRING))) = '0' THEN NULL ELSE `marital_status_code` END AS `marital_status_code`",
    "`marital_status_display` AS `marital_status_display`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 3,597,316 of 6,961,795 rows
    # (51.7%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`religion_code` AS STRING))) = '0' THEN NULL ELSE `religion_code` END AS `religion_code`",
    "`religion_display` AS `religion_display`",
    "`deceased_ind` AS `deceased_ind`",

    # 2,051 people carry a death recorded before their own birth. One of the two is wrong;
    # the birth is the better attested -- present for 93.7% of people and sourced from
    # demographics rather than an activity feed -- so the death stamp is the one removed.
    "CASE WHEN `deceased_datetime` < `birth_datetime` THEN NULL ELSE `deceased_datetime` END AS `deceased_datetime`",
    "`deceased_datetime_precision` AS `deceased_datetime_precision`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 6,960,935 of 6,961,795 rows
    # (100%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`confidentiality_code` AS STRING))) = '0' THEN NULL ELSE `confidentiality_code` END AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`current_address_id` AS `current_address_id`",
    "`latest_known_address_id` AS `latest_known_address_id`",
    "`address_selection_status` AS `address_selection_status`",
    "`current_mrn` AS `current_mrn`",
    "`current_mrn_status` AS `current_mrn_status`",
    "`mrn_selection_status` AS `mrn_selection_status`",
    "`nhs_number` AS `nhs_number`",
    "`nhs_number_status` AS `nhs_number_status`",
    "`nhs_number_selection_status` AS `nhs_number_selection_status`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",

    # record_status_effective_to cannot precede record_status_effective_from. The start is
    # the better-attested of the two, so the end is what goes and the row keeps its
    # record_status_effective_from. Hit 27,876 of 6,961,795 rows (0.4%) when profiled on
    # 2026-08-24.
    "CASE WHEN `record_status_effective_from` IS NOT NULL AND `record_status_effective_to` IS NOT NULL AND `record_status_effective_from` > `record_status_effective_to` THEN NULL ELSE `record_status_effective_to` END AS `record_status_effective_to`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`load_batch_id` AS `load_batch_id`",
    "`loaded_at` AS `loaded_at`",
]

SPINE_PERSON_COLUMN_COMMENTS = {
    "person_id": "Millennium person identifier and table primary key.",
    "active": "Source active indicator.",
    "gender_code": "Source administrative gender code.",
    "gender_display": "Source administrative gender display.",
    "ethnicity_code": "Source ethnicity code.",
    "ethnicity_display": "Source ethnicity display.",
    "birth_date":
        "Calendar birth date. Gold QC transform rules: gold.spine.person.birth_date.d2100_12_31_owner, gold.spine.person.birth_date.ole_zero_date, gold.spine.person.birth_date.pre1901_other_owner, gold.spine.person.birth_date.unknown_birth_sentinel.",
    "birth_datetime":
        "Source-compatible birth timestamp. Gold QC transform rules: gold.spine.person.birth_datetime.d2100_12_31_owner, gold.spine.person.birth_datetime.ole_zero_date, gold.spine.person.birth_datetime.pre1901_other_owner, gold.spine.person.birth_datetime.unknown_birth_sentinel.",
    "birth_precision_code": "Source code describing birth-date precision.",
    "birth_precision_display": "Display for the source birth precision.",
    "birth_precision_flag": "Raw source birth precision flag.",
    "language_code": "Preferred language source code.",
    "language_display": "Preferred language display.",
    "marital_status_code": "Source marital-status code.",
    "marital_status_display": "Source marital-status display.",
    "religion_code": "Source religion code.",
    "religion_display": "Source religion display.",
    "deceased_ind": "Whether the source indicates the person is deceased.",
    "deceased_datetime": "Source death timestamp.",
    "deceased_datetime_precision": "Raw source death-time precision.",
    "confidentiality_code":
        "Source confidentiality level carried for downstream access control.",
    "vip_ind": "Source VIP indicator carried as data.",
    "current_address_id": "Current source address reference when available.",
    "latest_known_address_id": "Latest-known source address reference.",
    "address_selection_status":
        "Provenance for current versus latest-known address selection.",
    "current_mrn":
        "Current hospital MRN selected from map_patient_identifier: active alias first, then latest valid end-effective, NULLS LAST, deterministic SOURCE_PK tiebreak; NULL when selection is ambiguous or no alias exists.",
    "current_mrn_status":
        "Lifecycle status of the selected hospital MRN alias; NULL when selection is ambiguous or no alias exists.",
    "mrn_selection_status":
        "Selection outcome for the governed hospital MRN alias pool; ambiguous = multiple active aliases",
    "nhs_number":
        "Current NHS number selected from map_patient_identifier: active alias first, then latest valid end-effective, NULLS LAST, deterministic SOURCE_PK tiebreak; NULL when selection is ambiguous or no alias exists.",
    "nhs_number_status":
        "Lifecycle status of the selected NHS-number alias; NULL when selection is ambiguous or no alias exists.",
    "nhs_number_selection_status":
        "Selection outcome for the governed NHS-number alias pool; ambiguous = multiple active aliases",
    "record_status": "Normalized source lifecycle status.",
    "record_status_effective_from": "Source status effective timestamp.",
    "record_status_effective_to": "Source status end timestamp when supplied.",
    "source_update_timestamp": "Native source update timestamp.",
    "load_batch_id": "Deterministic batch token derived from bronze load time.",
    "loaded_at": "Bronze load/update timestamp; never pipeline wall-clock time.",
}

@dp.materialized_view(
    name=_n("gold_spine.person"),
    comment=(
        "One resolved Millennium person with FHIR-aligned demographics and retained source "
        "lifecycle. Gold QC twin of the silver product: 11 columns are repaired or nulled, 0 "
        "check(s) are advisory. Each rule states its reason in the pipeline notebook, and "
        "Lakeflow expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
def gold_spine_person():
    """Quality-controlled twin of journey_spine.person."""
    df = _qc("spine_person", SPINE_PERSON_SELECT)
    return _with_comments(df, SPINE_PERSON_COLUMN_COMMENTS)

In [0]:
# ==== journey_spine.person_identifier ====

SPINE_PERSON_IDENTIFIER_SELECT = [
    "`person_identifier_id` AS `person_identifier_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identifier_system` AS `identifier_system`",

    # An empty or whitespace-only string is how the source writes 'nothing here'. It reads
    # as a value in a query and is not one, so it is nulled. Hit 2 of 27,522,731 rows
    # (7.27e-06%) when profiled on 2026-08-24.
    "CASE WHEN TRIM(CAST(`identifier_value` AS STRING)) = '' THEN NULL ELSE `identifier_value` END AS `identifier_value`",
    "`identifier_type_code` AS `identifier_type_code`",
    "`status` AS `status`",

    # 1900-01-01 is a placeholder low date rather than a date in 1900; on a non-birth field
    # it carries no more meaning than it does on a birth one. Hit 17 of 27,522,731 rows
    # (6.18e-05%) when profiled on 2026-08-24.
    #
    # A date before 1901 that is not one of the known placeholders. Nothing in this estate
    # predates the twentieth century, so these are mistyped or mis-scaled rather than early.
    # Hit 235,303 of 27,522,731 rows (0.855%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`valid_from` AS DATE) = DATE'1900-01-01' OR CAST(`valid_from` AS DATE) < DATE'1901-01-01' AND CAST(`valid_from` AS DATE) NOT IN (DATE'1800-01-01', DATE'1899-12-30', DATE'1900-01-01') THEN NULL ELSE `valid_from` END AS `valid_from`",

    # 1900-01-01 is a placeholder low date rather than a date in 1900; on a non-birth field
    # it carries no more meaning than it does on a birth one. Hit 3 of 27,522,731 rows
    # (1.09e-05%) when profiled on 2026-08-24.
    #
    # 2100-12-31 is the far-future marker the source writes to mean 'no end yet'. The
    # absence is what the row means, and NULL states it without putting a fictional date
    # into a range comparison. Hit 6,923,735 of 27,522,731 rows (25.2%) when profiled on
    # 2026-08-24.
    #
    # A date before 1901 that is not one of the known placeholders. Nothing in this estate
    # predates the twentieth century, so these are mistyped or mis-scaled rather than early.
    # Hit 18 of 27,522,731 rows (6.54e-05%) when profiled on 2026-08-24.
    #
    # valid_to cannot precede valid_from. The start is the better-attested of the two, so
    # the end is what goes and the row keeps its valid_from. Hit 993 of 27,522,731 rows
    # (0.00361%) when profiled on 2026-08-24.
    "CASE WHEN `valid_from` IS NOT NULL AND `valid_to` IS NOT NULL AND `valid_from` > `valid_to` THEN NULL ELSE CASE WHEN CAST(`valid_to` AS DATE) = DATE'2100-12-31' OR CAST(`valid_to` AS DATE) = DATE'1900-01-01' OR CAST(`valid_to` AS DATE) < DATE'1901-01-01' AND CAST(`valid_to` AS DATE) NOT IN (DATE'1800-01-01', DATE'1899-12-30', DATE'1900-01-01') THEN NULL ELSE `valid_to` END END AS `valid_to`",
    "`source_system` AS `source_system`",
    "`source_table` AS `source_table`",
    "`source_row_id` AS `source_row_id`",
    "`identity_status` AS `identity_status`",
    "`load_batch_id` AS `load_batch_id`",
    "`loaded_at` AS `loaded_at`",
]

SPINE_PERSON_IDENTIFIER_MANDATORY_RULES = {
    # The research surface. identity_status = 'resolved' keeps the 27,018,061 rows of
    # 27,522,731 that are current and attributable. Superseded versions and rows whose
    # identity was never resolved are not research data, and a consumer who wants them has
    # silver.
    "research_surface": "(identity_status = 'resolved')",

    # The flow nulls person_id when it names a parent the spine does not have, and this rule
    # then drops the row, because an identifier belonging to no person identifies nothing a
    # researcher can join to. That was 1,383 orphaned rows plus 504,671 that already had no
    # person_id, out of 27,522,731.
    "gold.spine.person_identifier.person_id.fk_containment": "`person_id` IS NOT NULL",
}

SPINE_PERSON_IDENTIFIER_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 504,670 of 27,522,731 at the profile.
    "gold.spine.person_identifier.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # This bounds a period of validity, and a future end is exactly how the source says a
    # record is still current -- nulling it would assert the record is valid forever, which
    # is a stronger and worse claim than the one being corrected. Seen on 2 of 27,522,731
    # rows (7.27e-06%) when profiled on 2026-08-24.
    "gold.spine.person_identifier.valid_from.future_owner":
        "NOT COALESCE((CAST(`valid_from` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",

    # This bounds a period of validity, and a future end is exactly how the source says a
    # record is still current -- nulling it would assert the record is valid forever, which
    # is a stronger and worse claim than the one being corrected. Seen on 38 of 27,522,731
    # rows (0.000138%) when profiled on 2026-08-24.
    "gold.spine.person_identifier.valid_to.future_owner":
        "NOT COALESCE((CAST(`valid_to` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",
}

SPINE_PERSON_IDENTIFIER_COLUMN_COMMENTS = {
    "person_identifier_id": "Deterministic identifier-assignment primary key.",
    "subject_key": "Peppered join key over the strongest available subject evidence.",
    "subject_id_system": "Identifier system used to derive subject_key.",
    "person_id":
        "Resolved Millennium person identifier, present on every row: identifiers belonging to no person in the spine are dropped.",
    "identifier_system": "Namespace for the source identifier.",
    "identifier_value":
        "Source identifier value. Gold QC transform rules: gold.spine.person_identifier.identifier_value.empty_string.",
    "identifier_type_code": "Identifier type code.",
    "status": "Source assignment or matching status.",
    "valid_from":
        "Identifier validity start. Gold QC transform rules: gold.spine.person_identifier.valid_from.d1900_01_01_owner, gold.spine.person_identifier.valid_from.pre1901_other_owner.",
    "valid_to":
        "Identifier validity end. Gold QC transform rules: gold.spine.person_identifier.valid_to.d1900_01_01_owner, gold.spine.person_identifier.valid_to.open_sentinel, gold.spine.person_identifier.valid_to.pre1901_other_owner.",
    "source_system": "Source system name.",
    "source_table": "Fully qualified bronze source table.",
    "source_row_id": "Stable source row identifier.",
    "identity_status": "Resolution state.",
    "load_batch_id": "Bronze batch token when the source exposes one.",
    "loaded_at": "Bronze load timestamp when exposed.",
}

@dp.materialized_view(
    name=_n("gold_spine.person_identifier"),
    comment=(
        "One source identifier assignment with an always-populated subject key. Gold QC twin "
        "of the silver product: 4 columns are repaired or nulled, 2 rule(s) drop rows, 3 "
        "check(s) are advisory. Each rule states its reason in the pipeline notebook, and "
        "Lakeflow expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(SPINE_PERSON_IDENTIFIER_MANDATORY_RULES)
@_expect_all(SPINE_PERSON_IDENTIFIER_ADVISORY_RULES)
def gold_spine_person_identifier():
    """Quality-controlled twin of journey_spine.person_identifier."""
    df = _qc(
        "spine_person_identifier",
        SPINE_PERSON_IDENTIFIER_SELECT,
        fk_columns=["person_id"],
    )
    return _with_comments(df, SPINE_PERSON_IDENTIFIER_COLUMN_COMMENTS)

In [0]:
# ==== journey_spine.person_relationship ====

SPINE_PERSON_RELATIONSHIP_SELECT = [
    "`person_relationship_id` AS `person_relationship_id`",
    "`source_subject_key` AS `source_subject_key`",
    "`source_person_id` AS `source_person_id`",
    "`target_subject_key` AS `target_subject_key`",
    "`target_person_id` AS `target_person_id`",
    "`relationship_type_code` AS `relationship_type_code`",
    "`inverse_relationship_type_code` AS `inverse_relationship_type_code`",
    "`valid_from` AS `valid_from`",
    "`valid_to` AS `valid_to`",
    "`record_status` AS `record_status`",
    "`construction_rule` AS `construction_rule`",
    "`construction_version` AS `construction_version`",
    "`source_system` AS `source_system`",
    "`source_table` AS `source_table`",
    "`source_row_id` AS `source_row_id`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
]

SPINE_PERSON_RELATIONSHIP_ADVISORY_RULES = {
    # This bounds a period of validity, and a future end is exactly how the source says a
    # record is still current -- nulling it would assert the record is valid forever, which
    # is a stronger and worse claim than the one being corrected. Seen on 735 of 186,512
    # rows (0.394%) when profiled on 2026-08-24.
    "gold.spine.person_relationship.valid_from.future_owner":
        "NOT COALESCE((CAST(`valid_from` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",
}

SPINE_PERSON_RELATIONSHIP_COLUMN_COMMENTS = {
    "person_relationship_id": "Deterministic relationship primary key.",
    "source_subject_key":
        "Always-populated subject key for the relationship source person.",
    "source_person_id": "Resolved source-side person identifier when available.",
    "target_subject_key":
        "Always-populated subject key for the relationship target person.",
    "target_person_id": "Resolved target-side person identifier when available.",
    "relationship_type_code": "Controlled source-to-target relationship type.",
    "inverse_relationship_type_code": "Controlled inverse relationship type.",
    "valid_from": "Relationship validity start from source evidence.",
    "valid_to": "Relationship validity end when source evidence supplies one.",
    "record_status": "Normalized source lifecycle status.",
    "construction_rule": "Governed relationship-construction rule.",
    "construction_version": "Governed relationship-construction version.",
    "source_system": "Source-system identifier.",
    "source_table": "Fully qualified configured bronze source table.",
    "source_row_id": "Stable source birth-row identifier.",
    "load_batch_id": "Deterministic bronze batch token.",
    "source_update_timestamp": "Latest native source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
}

@dp.materialized_view(
    name=_n("gold_spine.person_relationship"),
    comment=(
        "One source-backed relationship between two people; maternity currently supplies "
        "mother-to-baby links. Gold QC twin of the silver product: 0 columns are repaired or "
        "nulled, 1 check(s) are advisory. Each rule states its reason in the pipeline "
        "notebook, and Lakeflow expectation metrics report what every rule matched on each "
        "update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all(SPINE_PERSON_RELATIONSHIP_ADVISORY_RULES)
def gold_spine_person_relationship():
    """Quality-controlled twin of journey_spine.person_relationship."""
    df = _qc("spine_person_relationship", SPINE_PERSON_RELATIONSHIP_SELECT)
    return _with_comments(df, SPINE_PERSON_RELATIONSHIP_COLUMN_COMMENTS)

In [0]:
# ==== journey_spine.request_thread ====

SPINE_REQUEST_THREAD_SELECT = [
    "`request_thread_id` AS `request_thread_id`",
    "`source_patient_event_id` AS `source_patient_event_id`",
    "`target_patient_event_id` AS `target_patient_event_id`",
    "`link_type_code` AS `link_type_code`",
    "`subject_key` AS `subject_key`",
    "`person_id` AS `person_id`",
    "`encounter_id` AS `encounter_id`",
    "`request_identifier` AS `request_identifier`",
    "`response_identifier` AS `response_identifier`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 1 of 439,351,360 rows (2.28e-07%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`requested_datetime` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`requested_datetime` AS DATE)) < 9999 THEN NULL ELSE `requested_datetime` END AS `requested_datetime`",

    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 61,563 of 439,351,360 rows (0.014%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`responded_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `responded_datetime` END AS `responded_datetime`",
    "`source_history_row_count` AS `source_history_row_count`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`source_system` AS `source_system`",
    "`source_table` AS `source_table`",
    "`source_row_id` AS `source_row_id`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
]

SPINE_REQUEST_THREAD_COLUMN_COMMENTS = {
    "request_thread_id": "Deterministic request-edge primary key.",
    "source_patient_event_id": "Request-side patient event identifier.",
    "target_patient_event_id": "Response-side patient event identifier.",
    "link_type_code": "Controlled directed relationship type.",
    "subject_key": "Always-populated subject key copied from request evidence.",
    "person_id": "Resolved person identifier when available.",
    "encounter_id": "Encounter context when supplied by source evidence.",
    "request_identifier": "Verbatim source order identifier.",
    "response_identifier": "Verbatim source report-parent identifier.",
    "requested_datetime":
        "Source request timestamp. Gold QC transform rules: gold.spine.request_thread.requested_datetime.beyond_2100_below_9999_owner.",
    "responded_datetime": "Source report or measurement timestamp.",
    "source_history_row_count": "Number of source rows supporting the edge.",
    "record_status": "Normalized edge lifecycle from source validity.",
    "record_status_effective_from": "Earliest supporting source validity start.",
    "record_status_effective_to": "Source validity end when superseded.",
    "source_system": "Source system identifier.",
    "source_table": "Fully qualified configured source table.",
    "source_row_id": "Representative stable source row identifier.",
    "load_batch_id": "Deterministic bronze batch token.",
    "source_update_timestamp": "Latest native source update timestamp.",
    "loaded_at": "Latest bronze load timestamp supporting the edge.",
}

@dp.materialized_view(
    name=_n("gold_spine.request_thread"),
    comment="Directed indexed-event edges for pathology order-to-report, medication order-to-administration, and waiting-list entry-to-appointment routes. Pathology edges are accession-scoped co-membership (requested test and report series share an accession), not result-level attribution. Gold QC twin of the silver product: 4 columns are repaired or nulled, 0 check(s) are advisory. Each rule states its reason in the pipeline notebook, and Lakeflow expectation metrics report what every rule matched on each update.",
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
def gold_spine_request_thread():
    """Quality-controlled twin of journey_spine.request_thread."""
    # 343,157 rows point at a encounter_id the spine does not have. The pointer is nulled so
    # it cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    # 479 rows point at a person_id the spine does not have. The pointer is nulled so it
    # cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    df = _qc(
        "spine_request_thread",
        SPINE_REQUEST_THREAD_SELECT,
        fk_columns=["encounter_id", "person_id"],
    )
    return _with_comments(df, SPINE_REQUEST_THREAD_COLUMN_COMMENTS)

In [0]:
# ======== Clinical facts ========
#
# What was observed, ordered, prescribed, diagnosed and done.
# 54 products follow.

In [0]:
# ==== journey_clinical.allergy_intolerance ====

CLINICAL_ALLERGY_INTOLERANCE_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",

    # 1 rows point at a person_id the spine does not have. The pointer is nulled so it
    # cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    "CASE WHEN NOT `person_id_resolved` THEN NULL ELSE `person_id` END AS `person_id`",
    "`identity_status` AS `identity_status`",

    # 5,049 rows point at a encounter_id the spine does not have. The pointer is nulled so
    # it cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    "CASE WHEN NOT `encounter_id_resolved` THEN NULL ELSE `encounter_id` END AS `encounter_id`",
    "`event_datetime` AS `event_datetime`",

    # event_end_datetime cannot precede event_datetime. The start is the better-attested of
    # the two, so the end is what goes and the row keeps its event_datetime. Hit 1,019 of
    # 518,748 rows (0.196%) when profiled on 2026-08-24.
    "CASE WHEN `event_datetime` IS NOT NULL AND `event_end_datetime` IS NOT NULL AND `event_datetime` > `event_end_datetime` THEN NULL ELSE `event_end_datetime` END AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",

    # 'UNKNOWN' is a placeholder the source writes when the value was not recorded; it is
    # not a code, so it is nulled rather than passed on as one. Hit 204 of 518,748 rows
    # (0.0393%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`source_code` AS STRING))) = 'UNKNOWN' THEN NULL ELSE `source_code` END AS `source_code`",
    "`source_display` AS `source_display`",
    "`substance_code` AS `substance_code`",
    "`substance_type_code` AS `substance_type_code`",
    "`substance_type_display` AS `substance_type_display`",
    "`reaction_class_code` AS `reaction_class_code`",
    "`reaction_class_display` AS `reaction_class_display`",
    "`reaction_status_code` AS `reaction_status_code`",
    "`reaction_status_display` AS `reaction_status_display`",
    "`severity_code` AS `severity_code`",
    "`severity_display` AS `severity_display`",
    "`absence_assertion_ind` AS `absence_assertion_ind`",
    "`onset_precision_display` AS `onset_precision_display`",
    "`source_of_info_display` AS `source_of_info_display`",
    "`verified_status_flag` AS `verified_status_flag`",
    "`reviewed_datetime` AS `reviewed_datetime`",
    "`cancel_reason_display` AS `cancel_reason_display`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",

    # record_status_effective_to cannot precede record_status_effective_from. The start is
    # the better-attested of the two, so the end is what goes and the row keeps its
    # record_status_effective_from. Hit 17 of 518,748 rows (0.00328%) when profiled on
    # 2026-08-24.
    "CASE WHEN `record_status_effective_from` IS NOT NULL AND `record_status_effective_to` IS NOT NULL AND `record_status_effective_from` > `record_status_effective_to` THEN NULL ELSE `record_status_effective_to` END AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
    "`event_before_birth` AS `event_before_birth`",
    "`event_after_death_30d` AS `event_after_death_30d`",
]

CLINICAL_ALLERGY_INTOLERANCE_MANDATORY_RULES = {
    # The research surface. identity_status = 'resolved' keeps the 518,748 rows of 518,748
    # that are current and attributable. Superseded versions and rows whose identity was
    # never resolved are not research data, and a consumer who wants them has silver.
    "research_surface": "(identity_status = 'resolved')",
}

CLINICAL_ALLERGY_INTOLERANCE_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 0 of 518,748 at the profile.
    "gold.clinical.allergy_intolerance.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 762 of 518,748 rows (0.147%) when profiled on 2026-08-24.
    "gold.clinical.allergy_intolerance.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 109 of 518,748 rows (0.021%) when profiled on 2026-08-24.
    "gold.clinical.allergy_intolerance.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",
}

CLINICAL_ALLERGY_INTOLERANCE_COLUMN_COMMENTS = {
    "patient_event_id": "Stable product-wide event identifier.",
    "fact_row_id": "Storage-row identifier; equal to patient_event_id for this fact.",
    "subject_key": "Always-populated peppered subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier when available.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Recording encounter reference when supplied.",
    "event_datetime": "Sentinel-cleaned onset",
    "event_end_datetime": "Reaction-status end when resolved or cancelled.",
    "source_coding_system": "Verbatim substance code system.",
    "source_code": "Substance code (SNOMED then source identifier then nomenclature id)",
    "source_display": "Verbatim substance display text.",
    "substance_code":
        "Source and mapped substance codings as a one-level CodeableConcept VARIANT.",
    "substance_type_code": "Source substance type code.",
    "substance_type_display": "Substance type (Drug/Food/Environment/...).",
    "reaction_class_code": "Source reaction class code.",
    "reaction_class_display": "Reaction class (Allergy/Intolerance/Side Effect/...).",
    "reaction_status_code": "Source reaction status code.",
    "reaction_status_display": "Reaction status (Active/Cancelled/Resolved/Proposed).",
    "severity_code": "Source severity code.",
    "severity_display": "Severity (Mild/Moderate/Severe).",
    "absence_assertion_ind":
        "True when the row asserts ABSENCE of allergy (e.g. no known allergies) rather than a positive assertion.",
    "onset_precision_display": "Source onset precision.",
    "source_of_info_display": "Source-of-information display.",
    "verified_status_flag": "Pharmacy-verified flag verbatim.",
    "reviewed_datetime": "Sentinel-cleaned last review timestamp.",
    "cancel_reason_display": "Cancel reason when cancelled.",
    "record_status": "Normalized silver lifecycle status; Cancelled maps to retracted.",
    "record_status_effective_from": "Source status effective start.",
    "record_status_effective_to": "Source status effective end.",
    "confidentiality_code": "Security classification when supplied.",
    "vip_ind": "VIP indicator when supplied.",
    "withheld_identity_ind": "Withheld-identity indicator when supplied.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Bronze source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.allergy_intolerance"),
    comment=(
        "One Millennium allergy/intolerance assertion (map_allergy) with reaction class, "
        "severity, absence assertions, and cancel/review lifecycle. Gold QC twin of the "
        "silver product: 5 columns are repaired or nulled, 1 rule(s) drop rows, 3 check(s) "
        "are advisory. Each rule states its reason in the pipeline notebook, and Lakeflow "
        "expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_ALLERGY_INTOLERANCE_MANDATORY_RULES)
@_expect_all(CLINICAL_ALLERGY_INTOLERANCE_ADVISORY_RULES)
def gold_clinical_allergy_intolerance():
    """Quality-controlled twin of journey_clinical.allergy_intolerance."""
    df = _qc("clinical_allergy_intolerance", CLINICAL_ALLERGY_INTOLERANCE_SELECT)
    return _with_comments(df, CLINICAL_ALLERGY_INTOLERANCE_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.appointment ====

CLINICAL_APPOINTMENT_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",

    # 5 rows point at a person_id the spine does not have. The pointer is nulled so it
    # cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    "CASE WHEN NOT `person_id_resolved` THEN NULL ELSE `person_id` END AS `person_id`",
    "`identity_status` AS `identity_status`",

    # 74,180 rows point at a encounter_id the spine does not have. The pointer is nulled so
    # it cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    "CASE WHEN NOT `encounter_id_resolved` THEN NULL ELSE `encounter_id` END AS `encounter_id`",
    "`event_datetime` AS `event_datetime`",
    "`event_end_datetime` AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 1 of 32,293,081 rows
    # (3.1e-06%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`source_code` AS STRING))) = '0' THEN NULL ELSE `source_code` END AS `source_code`",
    "`source_display` AS `source_display`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 1 of 32,293,081 rows
    # (3.1e-06%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`appointment_type_code` AS STRING))) = '0' THEN NULL ELSE `appointment_type_code` END AS `appointment_type_code`",
    "`appointment_type_display` AS `appointment_type_display`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 1 of 32,293,081 rows
    # (3.1e-06%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`status_code` AS STRING))) = '0' THEN NULL ELSE `status_code` END AS `status_code`",
    "`status_display` AS `status_display`",
    "`status_meaning` AS `status_meaning`",
    "`referral_identifier` AS `referral_identifier`",
    "`requested_datetime` AS `requested_datetime`",
    "`original_requested_start` AS `original_requested_start`",

    # 2100-12-31 is the far-future marker the source writes to mean 'no end yet'. The
    # absence is what the row means, and NULL states it without putting a fictional date
    # into a range comparison. Hit 86 of 32,293,081 rows (0.000266%) when profiled on
    # 2026-08-24.
    #
    # original_requested_end cannot precede original_requested_start. The start is the
    # better-attested of the two, so the end is what goes and the row keeps its
    # original_requested_start. Hit 713 of 32,293,081 rows (0.00221%) when profiled on
    # 2026-08-24.
    "CASE WHEN `original_requested_start` IS NOT NULL AND `original_requested_end` IS NOT NULL AND `original_requested_start` > `original_requested_end` THEN NULL ELSE CASE WHEN CAST(`original_requested_end` AS DATE) = DATE'2100-12-31' THEN NULL ELSE `original_requested_end` END END AS `original_requested_end`",
    "`first_booked_datetime` AS `first_booked_datetime`",
    "`booking_iterations` AS `booking_iterations`",
    "`resource_history` AS `resource_history`",
    "`requested_practitioner_id` AS `requested_practitioner_id`",
    "`allocated_practitioner_id` AS `allocated_practitioner_id`",
    "`location_id` AS `location_id`",
    "`organization_id` AS `organization_id`",
    "`recurrence_parent_id` AS `recurrence_parent_id`",
    "`recurrence_type_flag` AS `recurrence_type_flag`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",

    # 2100-12-31 is the far-future marker the source writes to mean 'no end yet'. The
    # absence is what the row means, and NULL states it without putting a fictional date
    # into a range comparison. Hit 2 of 32,293,081 rows (6.19e-06%) when profiled on
    # 2026-08-24.
    "CASE WHEN CAST(`record_status_effective_to` AS DATE) = DATE'2100-12-31' THEN NULL ELSE `record_status_effective_to` END AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`fact_category` AS `fact_category`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
    "`event_before_birth` AS `event_before_birth`",
    "`event_after_death_30d` AS `event_after_death_30d`",
]

CLINICAL_APPOINTMENT_MANDATORY_RULES = {
    # The research surface. identity_status = 'resolved' keeps the 31,446,236 rows of
    # 32,293,081 that are current and attributable. Superseded versions and rows whose
    # identity was never resolved are not research data, and a consumer who wants them has
    # silver.
    "research_surface": "(identity_status = 'resolved')",
}

CLINICAL_APPOINTMENT_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 846,845 of 32,293,081 at the profile.
    "gold.clinical.appointment.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 64,860 of 32,293,081 rows (0.201%) when profiled on 2026-08-24.
    "gold.clinical.appointment.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 451 of 32,293,081 rows (0.0014%) when profiled on 2026-08-24.
    "gold.clinical.appointment.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",
}

CLINICAL_APPOINTMENT_COLUMN_COMMENTS = {
    "patient_event_id": "Stable product-wide appointment identifier.",
    "fact_row_id": "Storage-row identifier equal to patient_event_id.",
    "subject_key": "Always-populated peppered subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier when available.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Best-available encounter reference supplied by scheduling.",
    "event_datetime": "Booked slot start where available",
    "event_end_datetime": "Booked slot end where available.",
    "source_coding_system": "Verbatim appointment-type coding system.",
    "source_code": "Verbatim appointment-type code.",
    "source_display": "Verbatim appointment-type display.",
    "appointment_type_code": "Source appointment-type code.",
    "appointment_type_display": "Source appointment-type display.",
    "status_code": "Source scheduling-state code.",
    "status_display": "Source scheduling-state display.",
    "status_meaning": "Source scheduling-state meaning.",
    "referral_identifier":
        "UBRN or other referral alias retained as linkage evidence only.",
    "requested_datetime": "Source referral/request timestamp.",
    "original_requested_start": "Original requested slot start.",
    "original_requested_end":
        "Original requested slot end. Gold QC transform rules: gold.clinical.appointment.original_requested_end.open_sentinel.",
    "first_booked_datetime": "First booking timestamp supplied by scheduling.",
    "booking_iterations": "Deterministically ordered booking and location iterations.",
    "resource_history": "Deterministically ordered scheduling-resource and slot history.",
    "requested_practitioner_id": "Requested practitioner reference when supplied.",
    "allocated_practitioner_id":
        "Deterministic representative allocated practitioner; all allocations remain in resource_history.",
    "location_id": "Deterministic location reference from scheduling history.",
    "organization_id": "Scheduling organization reference.",
    "recurrence_parent_id": "Parent appointment identifier for recurring schedules.",
    "recurrence_type_flag": "Raw source recurrence flag.",
    "record_status": "Normalized source lifecycle status.",
    "record_status_effective_from": "Source current-row effective start.",
    "record_status_effective_to":
        "Source lifecycle end or source-absence detection timestamp. Gold QC transform rules: gold.clinical.appointment.record_status_effective_to.open_sentinel.",
    "confidentiality_code": "Source confidentiality code when supplied.",
    "vip_ind": "Source VIP indicator when supplied.",
    "withheld_identity_ind": "Withheld-identity indicator when supplied.",
    "fact_category":
        "Whether this fact is clinical or administrative in the v2 plane merge.",
    "source_feed": "Registered feed owning the fact.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Latest native source update timestamp.",
    "loaded_at": "Latest bronze load timestamp.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.appointment"),
    comment=(
        "One scheduling appointment with current state and ordered booking/resource history. "
        "Gold QC twin of the silver product: 7 columns are repaired or nulled, 1 rule(s) drop "
        "rows, 3 check(s) are advisory. Each rule states its reason in the pipeline notebook, "
        "and Lakeflow expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_APPOINTMENT_MANDATORY_RULES)
@_expect_all(CLINICAL_APPOINTMENT_ADVISORY_RULES)
def gold_clinical_appointment():
    """Quality-controlled twin of journey_clinical.appointment."""
    df = _qc("clinical_appointment", CLINICAL_APPOINTMENT_SELECT)
    return _with_comments(df, CLINICAL_APPOINTMENT_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.baby_delivery ====

CLINICAL_BABY_DELIVERY_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",
    "`event_datetime` AS `event_datetime`",

    # event_end_datetime cannot precede event_datetime. The start is the better-attested of
    # the two, so the end is what goes and the row keeps its event_datetime. Hit 3 of 99,299
    # rows (0.00302%) when profiled on 2026-08-24.
    "CASE WHEN `event_datetime` IS NOT NULL AND `event_end_datetime` IS NOT NULL AND `event_datetime` > `event_end_datetime` THEN NULL ELSE `event_end_datetime` END AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 53,647 of 99,299 rows (54%)
    # when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`source_code` AS STRING))) = '0' THEN NULL ELSE `source_code` END AS `source_code`",
    "`source_display` AS `source_display`",
    "`baby_person_id` AS `baby_person_id`",
    "`birth_order` AS `birth_order`",
    "`pregnancy_outcome` AS `pregnancy_outcome`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 53,647 of 99,299 rows (54%)
    # when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`delivery_method_code` AS STRING))) = '0' THEN NULL ELSE `delivery_method_code` END AS `delivery_method_code`",
    "`phenotypic_sex` AS `phenotypic_sex`",
    "`gestation_length_birth` AS `gestation_length_birth`",
    "`birthweight` AS `birthweight`",
    "`apgar_5` AS `apgar_5`",
    "`baby_death_datetime` AS `baby_death_datetime`",
    "`first_feed_datetime` AS `first_feed_datetime`",
    "`first_feed_code` AS `first_feed_code`",
    "`first_feed_breast_milk_status` AS `first_feed_breast_milk_status`",
    "`breast_milk_status_discharge` AS `breast_milk_status_discharge`",
    "`skin_to_skin_ind` AS `skin_to_skin_ind`",
    "`baby_discharge_datetime` AS `baby_discharge_datetime`",
    "`delivery_org_site` AS `delivery_org_site`",
    "`birth_setting` AS `birth_setting`",
    "`birth_place_type` AS `birth_place_type`",
    "`midwifery_place_type` AS `midwifery_place_type`",
    "`labour_delivery_id` AS `labour_delivery_id`",
    "`labour_delivery_ref` AS `labour_delivery_ref`",
    "`pregnancy_id` AS `pregnancy_id`",
    "`journey_pregnancy_id` AS `journey_pregnancy_id`",
    "`source_link_status` AS `source_link_status`",
    "`pregnancy_orphan_ind` AS `pregnancy_orphan_ind`",
    "`spine_person_mismatch_ind` AS `spine_person_mismatch_ind`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`fact_category` AS `fact_category`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
]

CLINICAL_BABY_DELIVERY_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 99,299 rows of 99,299 that
    # are current and attributable; identity_status = 'resolved' keeps the 98,912 rows of
    # 99,299 that are current and attributable. Superseded versions and rows whose identity
    # was never resolved are not research data, and a consumer who wants them has silver.
    "research_surface": "(identity_status = 'resolved') AND (record_status = 'active')",
}

CLINICAL_BABY_DELIVERY_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 387 of 99,299 at the profile.
    "gold.clinical.baby_delivery.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 0 of 99,299 at the profile.
    "gold.clinical.baby_delivery.record_status.default_view_active":
        "record_status = 'active'",
}

CLINICAL_BABY_DELIVERY_COLUMN_COMMENTS = {
    "patient_event_id": "Published field patient_event_id.",
    "fact_row_id": "Published field fact_row_id.",
    "subject_key": "Published field subject_key.",
    "subject_id_system": "Published field subject_id_system.",
    "person_id": "Published field person_id.",
    "identity_status": "Published field identity_status.",
    "encounter_id": "Published field encounter_id.",
    "event_datetime": "Published field event_datetime.",
    "event_end_datetime": "Published field event_end_datetime.",
    "source_coding_system": "Published field source_coding_system.",
    "source_code": "Published field source_code.",
    "source_display": "Published field source_display.",
    "baby_person_id": "Published field baby_person_id.",
    "birth_order": "Published field birth_order.",
    "pregnancy_outcome": "Published field pregnancy_outcome.",
    "delivery_method_code": "Published field delivery_method_code.",
    "phenotypic_sex": "Published field phenotypic_sex.",
    "gestation_length_birth": "Published field gestation_length_birth.",
    "birthweight": "Published field birthweight.",
    "apgar_5": "Published field apgar_5.",
    "baby_death_datetime": "Published field baby_death_datetime.",
    "first_feed_datetime": "Published field first_feed_datetime.",
    "first_feed_code": "Published field first_feed_code.",
    "first_feed_breast_milk_status": "Published field first_feed_breast_milk_status.",
    "breast_milk_status_discharge": "Published field breast_milk_status_discharge.",
    "skin_to_skin_ind": "Published field skin_to_skin_ind.",
    "baby_discharge_datetime": "Published field baby_discharge_datetime.",
    "delivery_org_site": "Published field delivery_org_site.",
    "birth_setting": "Published field birth_setting.",
    "birth_place_type": "Published field birth_place_type.",
    "midwifery_place_type": "Published field midwifery_place_type.",
    "labour_delivery_id": "Published field labour_delivery_id.",
    "labour_delivery_ref": "Published field labour_delivery_ref.",
    "pregnancy_id": "Published field pregnancy_id.",
    "journey_pregnancy_id": "Published field journey_pregnancy_id.",
    "source_link_status": "Published field source_link_status.",
    "pregnancy_orphan_ind": "Published field pregnancy_orphan_ind.",
    "spine_person_mismatch_ind": "Published field spine_person_mismatch_ind.",
    "record_status": "Published field record_status.",
    "record_status_effective_from": "Published field record_status_effective_from.",
    "record_status_effective_to": "Published field record_status_effective_to.",
    "confidentiality_code": "Published field confidentiality_code.",
    "vip_ind": "Published field vip_ind.",
    "withheld_identity_ind": "Published field withheld_identity_ind.",
    "fact_category": "Published field fact_category.",
    "source_feed": "Published field source_feed.",
    "load_batch_id": "Published field load_batch_id.",
    "source_update_timestamp": "Published field source_update_timestamp.",
    "loaded_at": "Published field loaded_at.",
}

@dp.materialized_view(
    name=_n("gold_clinical.baby_delivery"),
    comment=(
        "One MSDS baby-delivery row with mother identity recovered through the pregnancy "
        "spine. Gold QC twin of the silver product: 3 columns are repaired or nulled, 1 "
        "rule(s) drop rows, 2 check(s) are advisory. Each rule states its reason in the "
        "pipeline notebook, and Lakeflow expectation metrics report what every rule matched "
        "on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_BABY_DELIVERY_MANDATORY_RULES)
@_expect_all(CLINICAL_BABY_DELIVERY_ADVISORY_RULES)
def gold_clinical_baby_delivery():
    """Quality-controlled twin of journey_clinical.baby_delivery."""
    df = _qc("clinical_baby_delivery", CLINICAL_BABY_DELIVERY_SELECT)
    return _with_comments(df, CLINICAL_BABY_DELIVERY_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.cancer_treatment ====

CLINICAL_CANCER_TREATMENT_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",

    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 141 of 2,789,249 rows (0.00506%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `event_datetime` END AS `event_datetime`",

    # event_end_datetime cannot precede event_datetime. The start is the better-attested of
    # the two, so the end is what goes and the row keeps its event_datetime. Hit 1,381 of
    # 2,789,249 rows (0.0495%) when profiled on 2026-08-24.
    "CASE WHEN `event_datetime` IS NOT NULL AND `event_end_datetime` IS NOT NULL AND `event_datetime` > `event_end_datetime` THEN NULL ELSE `event_end_datetime` END AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`drug_code` AS `drug_code`",
    "`treatment_plan` AS `treatment_plan`",
    "`regimen_name` AS `regimen_name`",
    "`indication` AS `indication`",
    "`record_type` AS `record_type`",

    # A negative amount is not possible for a dose, quantity or score, and nothing in the
    # row says what the intended magnitude was. Hit 217 of 2,789,249 rows (0.00778%) when
    # profiled on 2026-08-24.
    "CASE WHEN `dose_value` < 0 THEN NULL ELSE `dose_value` END AS `dose_value`",
    "`dose_total` AS `dose_total`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 70,434 of 2,789,249 rows
    # (2.53%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`dose_unit_code` AS STRING))) = '0' THEN NULL ELSE `dose_unit_code` END AS `dose_unit_code`",
    "`route_code` AS `route_code`",

    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 141 of 2,789,249 rows (0.00506%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`start_date` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `start_date` END AS `start_date`",

    # end_date cannot precede start_date. The start is the better-attested of the two, so
    # the end is what goes and the row keeps its start_date. Hit 1,381 of 2,789,249 rows
    # (0.0495%) when profiled on 2026-08-24.
    "CASE WHEN `start_date` IS NOT NULL AND `end_date` IS NOT NULL AND `start_date` > `end_date` THEN NULL ELSE `end_date` END AS `end_date`",
    "`final_treatment_date` AS `final_treatment_date`",
    "`course_finished` AS `course_finished`",
    "`planned_cycles` AS `planned_cycles`",
    "`default_cycles` AS `default_cycles`",
    "`chemo_radiation` AS `chemo_radiation`",
    "`procurement_opcs_code` AS `procurement_opcs_code`",
    "`delivery_opcs_code` AS `delivery_opcs_code`",
    "`drug_similarity` AS `drug_similarity`",
    "`iqemo_course_id` AS `iqemo_course_id`",
    "`aria_rx_key` AS `aria_rx_key`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",

    # Silver derived this flag by comparing event_datetime with the person's own dates,
    # before the rules above corrected event_datetime. On the rows where event_datetime
    # changed, the flag describes a timestamp gold no longer publishes. This product carries
    # a VARIANT column, which rules out the spine join gold would need to recompute the
    # flag, so it is nulled where the value beneath it moved rather than left asserting
    # something stale.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `event_before_birth` END AS `event_before_birth`",

    # Silver derived this flag by comparing event_datetime with the person's own dates,
    # before the rules above corrected event_datetime. On the rows where event_datetime
    # changed, the flag describes a timestamp gold no longer publishes. This product carries
    # a VARIANT column, which rules out the spine join gold would need to recompute the
    # flag, so it is nulled where the value beneath it moved rather than left asserting
    # something stale.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `event_after_death_30d` END AS `event_after_death_30d`",
]

CLINICAL_CANCER_TREATMENT_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 2,789,249 rows of 2,789,249
    # that are current and attributable; identity_status = 'resolved' keeps the 2,716,669
    # rows of 2,789,249 that are current and attributable. Superseded versions and rows
    # whose identity was never resolved are not research data, and a consumer who wants them
    # has silver.
    "research_surface": "(identity_status = 'resolved') AND (record_status = 'active')",
}

CLINICAL_CANCER_TREATMENT_ADVISORY_RULES = {
    # Counted rather than nulled because a treatment course that is still running has a
    # planned end date, which is in the future by construction. Seen on 5,115 of 2,789,249
    # rows (0.183%) when profiled on 2026-08-24.
    "gold.clinical.cancer_treatment.end_date.future_owner":
        "NOT COALESCE((CAST(`end_date` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",

    # Counted rather than nulled because a treatment course that is still running has a
    # planned end date, which is in the future by construction. Seen on 5,115 of 2,789,249
    # rows (0.183%) when profiled on 2026-08-24.
    "gold.clinical.cancer_treatment.event_end_datetime.future_owner":
        "NOT COALESCE((CAST(`event_end_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",

    # Counted rather than nulled because a treatment course that is still running has a
    # planned end date, which is in the future by construction. Seen on 6,921 of 2,789,249
    # rows (0.248%) when profiled on 2026-08-24.
    "gold.clinical.cancer_treatment.final_treatment_date.future_owner":
        "NOT COALESCE((CAST(`final_treatment_date` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",

    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 72,580 of 2,789,249 at the profile.
    "gold.clinical.cancer_treatment.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 0 of 2,789,249 at the profile.
    "gold.clinical.cancer_treatment.record_status.default_view_active":
        "record_status = 'active'",

    # This bounds a period of validity, and a future end is exactly how the source says a
    # record is still current -- nulling it would assert the record is valid forever, which
    # is a stronger and worse claim than the one being corrected. Seen on 141 of 2,789,249
    # rows (0.00506%) when profiled on 2026-08-24.
    "gold.clinical.cancer_treatment.record_status_effective_from.future_owner":
        "NOT COALESCE((CAST(`record_status_effective_from` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 6,565 of 2,789,249 rows (0.235%) when profiled on 2026-08-24.
    "gold.clinical.cancer_treatment.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 350 of 2,789,249 rows (0.0125%) when profiled on 2026-08-24.
    "gold.clinical.cancer_treatment.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",
}

CLINICAL_CANCER_TREATMENT_COLUMN_COMMENTS = {
    "patient_event_id": "Stable treatment event identifier.",
    "fact_row_id": "Storage-row identifier equal to patient_event_id.",
    "subject_key": "Always-populated peppered subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference when available.",
    "event_datetime": "Treatment start timestamp.",
    "event_end_datetime": "Treatment end or final-treatment timestamp.",
    "source_coding_system": "SACT drug-token coding system.",
    "source_code": "Normalized drug token with agent-name display fallback.",
    "source_display": "Source agent name.",
    "drug_code": "Source and mapped drug codings.",
    "treatment_plan": "Source treatment plan.",
    "regimen_name": "Treatment regimen name.",
    "indication": "Treatment indication.",
    "record_type": "ARIA/iQemo linkage class.",
    "dose_value": "Constituent dose value.",
    "dose_total": "Total planned or delivered dose.",
    "dose_unit_code": "Dose unit code.",
    "route_code": "Administration route code.",
    "start_date": "Source start date.",
    "end_date": "Source end date.",
    "final_treatment_date": "Final treatment date.",
    "course_finished": "Course-finished indicator.",
    "planned_cycles": "Planned cycle count.",
    "default_cycles": "Default regimen cycle count.",
    "chemo_radiation": "Concurrent chemo-radiation indicator.",
    "procurement_opcs_code": "OPCS procurement code.",
    "delivery_opcs_code": "OPCS delivery code.",
    "drug_similarity": "Drug-link similarity score.",
    "iqemo_course_id": "iQemo course identifier.",
    "aria_rx_key": "ARIA prescription key.",
    "record_status": "Normalized source-record lifecycle.",
    "record_status_effective_from": "Lifecycle start.",
    "record_status_effective_to": "Source absence timestamp.",
    "confidentiality_code": "Security classification when supplied.",
    "vip_ind": "VIP indicator when supplied.",
    "withheld_identity_ind": "Withheld-identity indicator.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.cancer_treatment"),
    comment=(
        "One SACT constituent-drug treatment fact, kept separate from encounter-bound "
        "Millennium medication administration. Gold QC twin of the silver product: 8 columns "
        "are repaired or nulled, 1 rule(s) drop rows, 8 check(s) are advisory. Each rule "
        "states its reason in the pipeline notebook, and Lakeflow expectation metrics report "
        "what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_CANCER_TREATMENT_MANDATORY_RULES)
@_expect_all(CLINICAL_CANCER_TREATMENT_ADVISORY_RULES)
def gold_clinical_cancer_treatment():
    """Quality-controlled twin of journey_clinical.cancer_treatment."""
    df = _qc("clinical_cancer_treatment", CLINICAL_CANCER_TREATMENT_SELECT)
    return _with_comments(df, CLINICAL_CANCER_TREATMENT_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.cancer_treatment_cycle ====

CLINICAL_CANCER_TREATMENT_CYCLE_SELECT = [
    "`cancer_treatment_cycle_id` AS `cancer_treatment_cycle_id`",
    "`iqemo_course_id` AS `iqemo_course_id`",
    "`cycle_sequence_id` AS `cycle_sequence_id`",
    "`regimen_cycle_id` AS `regimen_cycle_id`",
    "`cycle_code` AS `cycle_code`",
    "`prescribed_datetime` AS `prescribed_datetime`",
    "`pharmacy_confirmed_datetime` AS `pharmacy_confirmed_datetime`",
    "`cycle_start_datetime` AS `cycle_start_datetime`",
    "`cancellation_datetime` AS `cancellation_datetime`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 1,813 of 294,917 rows (0.615%)
    # when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`cycle_status_code` AS STRING))) = '0' THEN NULL ELSE `cycle_status_code` END AS `cycle_status_code`",
    "`treatment_response_id` AS `treatment_response_id`",
    "`line_of_treatment` AS `line_of_treatment`",
    "`regimen_number` AS `regimen_number`",
    "`course_link_status` AS `course_link_status`",
    "`outcome_comments` AS `outcome_comments`",
    "`record_status` AS `record_status`",
    "`source_table` AS `source_table`",
    "`source_row_id` AS `source_row_id`",
    "`load_batch_id` AS `load_batch_id`",
    "`loaded_at` AS `loaded_at`",
]

CLINICAL_CANCER_TREATMENT_CYCLE_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 294,917 rows of 294,917 that
    # are current and attributable. Superseded versions and rows whose identity was never
    # resolved are not research data, and a consumer who wants them has silver.
    "research_surface": "(record_status = 'active')",
}

CLINICAL_CANCER_TREATMENT_CYCLE_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 0 of 294,917 at the profile.
    "gold.clinical.cancer_treatment_cycle.record_status.default_view_active":
        "record_status = 'active'",
}

CLINICAL_CANCER_TREATMENT_CYCLE_COLUMN_COMMENTS = {
    "cancer_treatment_cycle_id": "Stable composite cycle identifier.",
    "iqemo_course_id": "iQemo chemotherapy course identifier.",
    "cycle_sequence_id": "Verbatim treatment-cycle sequence token.",
    "regimen_cycle_id": "Regimen cycle identifier.",
    "cycle_code": "Source cycle code.",
    "prescribed_datetime": "Sentinel-cleaned prescribed timestamp.",
    "pharmacy_confirmed_datetime": "Sentinel-cleaned pharmacy-confirmed timestamp.",
    "cycle_start_datetime": "Sentinel-cleaned cycle start timestamp.",
    "cancellation_datetime": "Sentinel-cleaned cancellation timestamp.",
    "cycle_status_code": "Source cycle status code.",
    "treatment_response_id": "Treatment-response identifier.",
    "line_of_treatment": "Line of treatment.",
    "regimen_number": "Regimen number.",
    "course_link_status": "Parent course-link status.",
    "outcome_comments": "Source outcome comments.",
    "record_status": "Reference-row lifecycle.",
    "source_table": "Registered source table.",
    "source_row_id": "Composite source row identifier.",
    "load_batch_id": "Bronze batch token.",
    "loaded_at": "Bronze load timestamp.",
}

@dp.materialized_view(
    name=_n("gold_clinical.cancer_treatment_cycle"),
    comment=(
        "One iQemo chemotherapy cycle child reference at composite course/cycle grain. Gold "
        "QC twin of the silver product: 1 columns are repaired or nulled, 1 rule(s) drop "
        "rows, 1 check(s) are advisory. Each rule states its reason in the pipeline notebook, "
        "and Lakeflow expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_CANCER_TREATMENT_CYCLE_MANDATORY_RULES)
@_expect_all(CLINICAL_CANCER_TREATMENT_CYCLE_ADVISORY_RULES)
def gold_clinical_cancer_treatment_cycle():
    """Quality-controlled twin of journey_clinical.cancer_treatment_cycle."""
    df = _qc("clinical_cancer_treatment_cycle", CLINICAL_CANCER_TREATMENT_CYCLE_SELECT)
    return _with_comments(df, CLINICAL_CANCER_TREATMENT_CYCLE_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.clinical_finding ====

CLINICAL_CLINICAL_FINDING_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",

    # 2,617 rows point at a person_id the spine does not have. The pointer is nulled so it
    # cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    "CASE WHEN NOT `person_id_resolved` THEN NULL ELSE `person_id` END AS `person_id`",
    "`identity_status` AS `identity_status`",

    # 2,302,162 rows point at a encounter_id the spine does not have. The pointer is nulled
    # so it cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    "CASE WHEN NOT `encounter_id_resolved` THEN NULL ELSE `encounter_id` END AS `encounter_id`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 588 of 1,591,253,116 rows (3.7e-05%) when profiled on 2026-08-24.
    #
    # 1900-01-01 is a placeholder low date rather than a date in 1900; on a non-birth field
    # it carries no more meaning than it does on a birth one. Hit 19 of 1,591,253,116 rows
    # (1.19e-06%) when profiled on 2026-08-24.
    #
    # 1899-12-30 is the zero point of the OLE/Excel date scale, so it is what a spreadsheet
    # or a COM layer writes when the date was left blank. It is not a date anyone recorded.
    # Hit 1 of 1,591,253,116 rows (6.28e-08%) when profiled on 2026-08-24.
    #
    # A date before 1901 that is not one of the known placeholders. Nothing in this estate
    # predates the twentieth century, so these are mistyped or mis-scaled rather than early.
    # Hit 257 of 1,591,253,116 rows (1.62e-05%) when profiled on 2026-08-24.
    #
    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 165,534 of 1,591,253,116 rows (0.0104%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE CASE WHEN CAST(`event_datetime` AS DATE) = DATE'1899-12-30' OR CAST(`event_datetime` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`event_datetime` AS DATE)) < 9999 OR CAST(`event_datetime` AS DATE) = DATE'1900-01-01' OR CAST(`event_datetime` AS DATE) < DATE'1901-01-01' AND CAST(`event_datetime` AS DATE) NOT IN (DATE'1800-01-01', DATE'1899-12-30', DATE'1900-01-01') THEN NULL ELSE `event_datetime` END END AS `event_datetime`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 460 of 1,591,253,116 rows (2.89e-05%) when profiled on 2026-08-24.
    #
    # 1899-12-30 is the zero point of the OLE/Excel date scale, so it is what a spreadsheet
    # or a COM layer writes when the date was left blank. It is not a date anyone recorded.
    # Hit 1 of 1,591,253,116 rows (6.28e-08%) when profiled on 2026-08-24.
    #
    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 2,311 of 1,591,253,116 rows (0.000145%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`event_end_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE CASE WHEN CAST(`event_end_datetime` AS DATE) = DATE'1899-12-30' OR CAST(`event_end_datetime` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`event_end_datetime` AS DATE)) < 9999 THEN NULL ELSE `event_end_datetime` END END AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`finding_code` AS `finding_code`",
    "`finding_kind` AS `finding_kind`",

    # An empty or whitespace-only string is how the source writes 'nothing here'. It reads
    # as a value in a query and is not one, so it is nulled. Hit 78,553,897 of 1,591,253,116
    # rows (4.94%) when profiled on 2026-08-24.
    "CASE WHEN TRIM(CAST(`value_text` AS STRING)) = '' THEN NULL ELSE `value_text` END AS `value_text`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 128 of 1,591,253,116 rows (8.04e-06%) when profiled on 2026-08-24.
    #
    # 1900-01-01 is a placeholder low date rather than a date in 1900; on a non-birth field
    # it carries no more meaning than it does on a birth one. Hit 19 of 1,591,253,116 rows
    # (1.19e-06%) when profiled on 2026-08-24.
    #
    # A date before 1901 that is not one of the known placeholders. Nothing in this estate
    # predates the twentieth century, so these are mistyped or mis-scaled rather than early.
    # Hit 257 of 1,591,253,116 rows (1.62e-05%) when profiled on 2026-08-24.
    #
    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 163,319 of 1,591,253,116 rows (0.0103%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`value_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE CASE WHEN CAST(`value_datetime` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`value_datetime` AS DATE)) < 9999 OR CAST(`value_datetime` AS DATE) = DATE'1900-01-01' OR CAST(`value_datetime` AS DATE) < DATE'1901-01-01' AND CAST(`value_datetime` AS DATE) NOT IN (DATE'1800-01-01', DATE'1899-12-30', DATE'1900-01-01') THEN NULL ELSE `value_datetime` END END AS `value_datetime`",
    "`value_code` AS `value_code`",
    "`value_display` AS `value_display`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 1,589,467,349 of 1,591,253,116
    # rows (99.9%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`normalcy_code` AS STRING))) = '0' THEN NULL ELSE `normalcy_code` END AS `normalcy_code`",
    "`normalcy_display` AS `normalcy_display`",
    "`result_status_code` AS `result_status_code`",
    "`result_status_display` AS `result_status_display`",
    "`order_id` AS `order_id`",
    "`parent_event_id` AS `parent_event_id`",
    "`performer_practitioner_id` AS `performer_practitioner_id`",
    "`verifier_practitioner_id` AS `verifier_practitioner_id`",
    "`organization_id` AS `organization_id`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",

    # 2100-12-31 is the far-future marker the source writes to mean 'no end yet'. The
    # absence is what the row means, and NULL states it without putting a fictional date
    # into a range comparison. Hit 7,784,512 of 1,591,253,116 rows (0.489%) when profiled on
    # 2026-08-24.
    #
    # record_status_effective_to cannot precede record_status_effective_from. The start is
    # the better-attested of the two, so the end is what goes and the row keeps its
    # record_status_effective_from. Hit 355,227 of 1,591,253,116 rows (0.0223%) when
    # profiled on 2026-08-24.
    "CASE WHEN `record_status_effective_from` IS NOT NULL AND `record_status_effective_to` IS NOT NULL AND `record_status_effective_from` > `record_status_effective_to` THEN NULL ELSE CASE WHEN CAST(`record_status_effective_to` AS DATE) = DATE'2100-12-31' THEN NULL ELSE `record_status_effective_to` END END AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",

    # Silver derived this flag by comparing event_datetime with the person's own dates,
    # before the rules above corrected event_datetime. On the rows where event_datetime
    # changed, the flag describes a timestamp gold no longer publishes. This product carries
    # a VARIANT column, which rules out the spine join gold would need to recompute the
    # flag, so it is nulled where the value beneath it moved rather than left asserting
    # something stale.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `event_before_birth` END AS `event_before_birth`",

    # Silver derived this flag by comparing event_datetime with the person's own dates,
    # before the rules above corrected event_datetime. On the rows where event_datetime
    # changed, the flag describes a timestamp gold no longer publishes. This product carries
    # a VARIANT column, which rules out the spine join gold would need to recompute the
    # flag, so it is nulled where the value beneath it moved rather than left asserting
    # something stale.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `event_after_death_30d` END AS `event_after_death_30d`",
]

CLINICAL_CLINICAL_FINDING_MANDATORY_RULES = {
    # The research surface. identity_status = 'resolved' keeps the 1,591,253,116 rows of
    # 1,591,253,116 that are current and attributable. Superseded versions and rows whose
    # identity was never resolved are not research data, and a consumer who wants them has
    # silver.
    "research_surface": "(identity_status = 'resolved')",
}

CLINICAL_CLINICAL_FINDING_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 0 of 1,591,253,116 at the profile.
    "gold.clinical.clinical_finding.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 250,156 of 1,591,253,116 rows (0.0157%) when profiled on
    # 2026-08-24.
    "gold.clinical.clinical_finding.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 29,903,501 of 1,591,253,116 rows (1.88%) when profiled on
    # 2026-08-24.
    "gold.clinical.clinical_finding.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",

    # Left as a warning because it fires on 42,016,359 of 1,591,253,116 rows (2.64%) when
    # profiled on 2026-08-24 -- at that rate the rule's assumption about what event_datetime
    # and event_end_datetime mean is the thing in doubt, not the data. The inverted gaps are
    # mostly minutes, which reads as two clocks rather than two events in the wrong order.
    "gold.clinical.clinical_finding.table.ordering_violation_event_datetime_event_end_datetime":
        "NOT COALESCE((`event_datetime` IS NOT NULL AND `event_end_datetime` IS NOT NULL AND `event_datetime` > `event_end_datetime`), FALSE)",
}

CLINICAL_CLINICAL_FINDING_COLUMN_COMMENTS = {
    "patient_event_id": "Stable product-wide clinical-finding identifier.",
    "fact_row_id": "Storage-row identifier equal to patient_event_id.",
    "subject_key": "Always-populated peppered subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Best-available encounter reference.",
    "event_datetime":
        "Clinically relevant finding timestamp. Gold QC transform rules: gold.clinical.clinical_finding.event_datetime.beyond_2100_below_9999_owner, gold.clinical.clinical_finding.event_datetime.d1900_01_01_owner, gold.clinical.clinical_finding.event_datetime.ole_zero_date, gold.clinical.clinical_finding.event_datetime.pre1901_other_owner.",
    "event_end_datetime":
        "Source effective end timestamp. Gold QC transform rules: gold.clinical.clinical_finding.event_end_datetime.beyond_2100_below_9999_owner, gold.clinical.clinical_finding.event_end_datetime.ole_zero_date.",
    "source_coding_system": "Verbatim source event coding system.",
    "source_code": "Verbatim source event code.",
    "source_display": "Verbatim source event display.",
    "finding_code": "Source finding CodeableConcept.",
    "finding_kind": "Generic result representation carried by the source feed.",
    "value_text":
        "Verbatim text result or descriptor. Gold QC transform rules: gold.clinical.clinical_finding.value_text.empty_string.",
    "value_datetime":
        "Date or date-time result value. Gold QC transform rules: gold.clinical.clinical_finding.value_datetime.beyond_2100_below_9999_owner, gold.clinical.clinical_finding.value_datetime.d1900_01_01_owner, gold.clinical.clinical_finding.value_datetime.pre1901_other_owner.",
    "value_code": "Source coded result value.",
    "value_display": "Source coded result display.",
    "normalcy_code": "Source normalcy or interpretation code.",
    "normalcy_display": "Source normalcy or interpretation display.",
    "result_status_code": "Source result-status code.",
    "result_status_display": "Source result-status display.",
    "order_id": "Source order identifier when supplied.",
    "parent_event_id": "Source parent-event identifier.",
    "performer_practitioner_id": "Performing practitioner reference.",
    "verifier_practitioner_id": "Verifying practitioner reference.",
    "organization_id": "Source organization reference.",
    "record_status": "Normalized source-record lifecycle.",
    "record_status_effective_from": "Source status effective start.",
    "record_status_effective_to":
        "Source status effective end. Gold QC transform rules: gold.clinical.clinical_finding.record_status_effective_to.open_sentinel.",
    "confidentiality_code": "Source confidentiality code.",
    "vip_ind": "Source VIP indicator.",
    "withheld_identity_ind": "Withheld-identity indicator.",
    "source_feed": "Registered generic-event feed.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.clinical_finding"),
    comment=(
        "One governed residual clinical observation from a generic Millennium event feed, "
        "without diagnosis inference. Gold QC twin of the silver product: 10 columns are "
        "repaired or nulled, 1 rule(s) drop rows, 4 check(s) are advisory. Each rule states "
        "its reason in the pipeline notebook, and Lakeflow expectation metrics report what "
        "every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_CLINICAL_FINDING_MANDATORY_RULES)
@_expect_all(CLINICAL_CLINICAL_FINDING_ADVISORY_RULES)
def gold_clinical_clinical_finding():
    """Quality-controlled twin of journey_clinical.clinical_finding."""
    df = _qc("clinical_clinical_finding", CLINICAL_CLINICAL_FINDING_SELECT)
    return _with_comments(df, CLINICAL_CLINICAL_FINDING_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.clinical_score ====

CLINICAL_CLINICAL_SCORE_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",

    # 327 rows point at a person_id the spine does not have. The pointer is nulled so it
    # cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    "CASE WHEN NOT `person_id_resolved` THEN NULL ELSE `person_id` END AS `person_id`",
    "`identity_status` AS `identity_status`",

    # 3,298 rows point at a encounter_id the spine does not have. The pointer is nulled so
    # it cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    "CASE WHEN NOT `encounter_id_resolved` THEN NULL ELSE `encounter_id` END AS `encounter_id`",

    # A block of these stamps was written after being divided by a thousand -- a millisecond
    # epoch stored as seconds -- which lands the whole block in January 1970. Multiplying
    # back up recovers the real time to within about seventeen minutes, and 97.4% of the
    # 12,655,764 recovered stamps land inside the encounter the row already points at. The
    # rescaled value is only accepted when it lands between 1990 and the moment the row was
    # loaded; anything else was a different defect and is nulled by the rule below instead
    # of being invented.
    #
    # The remaining 1970 stamps did not rescale into a plausible date, so the value is known
    # to be wrong and the truth is not known.
    "CASE WHEN YEAR(CAST(`event_datetime` AS TIMESTAMP)) = 1970 AND timestamp_seconds(CAST(unix_timestamp(`event_datetime`) AS BIGINT) * 1000) BETWEEN TIMESTAMP'1990-01-01 00:00:00' AND `loaded_at` THEN timestamp_seconds(CAST(unix_timestamp(`event_datetime`) AS BIGINT) * 1000) WHEN YEAR(CAST(`event_datetime` AS TIMESTAMP)) = 1970 THEN NULL ELSE `event_datetime` END AS `event_datetime`",

    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 321 of 303,073,548 rows (0.000106%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`event_end_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `event_end_datetime` END AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`score_code` AS `score_code`",
    "`score_name` AS `score_name`",

    # A magnitude above 1e12 is outside any scale this field is measured on, so the number
    # carries no meaning even though something was recorded. Hit 3 of 303,073,548 rows
    # (9.9e-07%) when profiled on 2026-08-24.
    #
    # A negative amount is not possible for a dose, quantity or score, and nothing in the
    # row says what the intended magnitude was. Hit 2,652 of 303,073,548 rows (0.000875%)
    # when profiled on 2026-08-24.
    "CASE WHEN ABS(CAST(`value_number` AS DOUBLE)) > 1e12 OR `value_number` < 0 THEN NULL ELSE `value_number` END AS `value_number`",
    "`value_text` AS `value_text`",
    "`unit_source_value` AS `unit_source_value`",
    "`unit_concept_id` AS `unit_concept_id`",
    "`components` AS `components`",
    "`component_count` AS `component_count`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 260,387,120 of 303,073,548
    # rows (85.9%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`interpretation_code` AS STRING))) = '0' THEN NULL ELSE `interpretation_code` END AS `interpretation_code`",
    "`interpretation_display` AS `interpretation_display`",
    "`result_status_code` AS `result_status_code`",
    "`result_status_display` AS `result_status_display`",
    "`performer_practitioner_id` AS `performer_practitioner_id`",
    "`source_form_id` AS `source_form_id`",
    "`promotion_rule_id` AS `promotion_rule_id`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",

    # Silver derived this flag by comparing event_datetime with the person's own dates,
    # before the rules above corrected event_datetime. On the rows where event_datetime
    # changed, the flag describes a timestamp gold no longer publishes. This product carries
    # a VARIANT column, which rules out the spine join gold would need to recompute the
    # flag, so it is nulled where the value beneath it moved rather than left asserting
    # something stale.
    "CASE WHEN YEAR(CAST(`event_datetime` AS TIMESTAMP)) = 1970 THEN NULL ELSE `event_before_birth` END AS `event_before_birth`",

    # Silver derived this flag by comparing event_datetime with the person's own dates,
    # before the rules above corrected event_datetime. On the rows where event_datetime
    # changed, the flag describes a timestamp gold no longer publishes. This product carries
    # a VARIANT column, which rules out the spine join gold would need to recompute the
    # flag, so it is nulled where the value beneath it moved rather than left asserting
    # something stale.
    "CASE WHEN YEAR(CAST(`event_datetime` AS TIMESTAMP)) = 1970 THEN NULL ELSE `event_after_death_30d` END AS `event_after_death_30d`",
]

CLINICAL_CLINICAL_SCORE_MANDATORY_RULES = {
    # The research surface. identity_status = 'resolved' keeps the 303,073,419 rows of
    # 303,073,548 that are current and attributable. Superseded versions and rows whose
    # identity was never resolved are not research data, and a consumer who wants them has
    # silver.
    "research_surface": "(identity_status = 'resolved')",
}

CLINICAL_CLINICAL_SCORE_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 129 of 303,073,548 at the profile.
    "gold.clinical.clinical_score.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 35,015 of 303,073,548 rows (0.0116%) when profiled on
    # 2026-08-24.
    "gold.clinical.clinical_score.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 6,147,121 of 303,073,548 rows (2.03%) when profiled on
    # 2026-08-24.
    "gold.clinical.clinical_score.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",

    # Left as a warning because it fires on 247,483,238 of 303,073,548 rows (81.7%) when
    # profiled on 2026-08-24 -- at that rate the rule's assumption about what event_datetime
    # and event_end_datetime mean is the thing in doubt, not the data. The inverted gaps are
    # mostly minutes, which reads as two clocks rather than two events in the wrong order.
    "gold.clinical.clinical_score.table.ordering_violation_event_datetime_event_end_datetime":
        "NOT COALESCE((`event_datetime` IS NOT NULL AND `event_end_datetime` IS NOT NULL AND `event_datetime` > `event_end_datetime`), FALSE)",
}

CLINICAL_CLINICAL_SCORE_COLUMN_COMMENTS = {
    "patient_event_id": "Stable clinical-score event identifier.",
    "fact_row_id": "Storage-row identifier.",
    "subject_key": "Always-populated subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference.",
    "event_datetime": "Score timestamp.",
    "event_end_datetime": "Score end timestamp.",
    "source_coding_system": "Source coding system.",
    "source_code": "Source score code.",
    "source_display": "Source score display.",
    "score_code": "Source and mapped score CodeableConcept.",
    "score_name": "Human-readable score or scale name.",
    "value_number": "Numeric score value.",
    "value_text": "Verbatim score text.",
    "unit_source_value": "Source unit when supplied.",
    "unit_concept_id": "Mapped unit concept identifier.",
    "components": "Ordered component values retained for the score result.",
    "component_count": "Number of ordered component objects.",
    "interpretation_code": "Source interpretation code.",
    "interpretation_display": "Source interpretation display.",
    "result_status_code": "Source result status code.",
    "result_status_display": "Source result status display.",
    "performer_practitioner_id": "Performing practitioner reference.",
    "source_form_id": "Source form event for promoted scores.",
    "promotion_rule_id": "Governed promotion-rule identifier.",
    "record_status": "Normalized lifecycle.",
    "record_status_effective_from": "Lifecycle start.",
    "record_status_effective_to": "Lifecycle end when retained inactive.",
    "confidentiality_code": "Security classification when supplied.",
    "vip_ind": "VIP indicator when supplied.",
    "withheld_identity_ind": "Withheld identity indicator.",
    "source_feed": "Registered source route.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.clinical_score"),
    comment=(
        "One native or form-promoted clinical score with ordered component evidence. Gold QC "
        "twin of the silver product: 8 columns are repaired or nulled, 1 rule(s) drop rows, 4 "
        "check(s) are advisory. Each rule states its reason in the pipeline notebook, and "
        "Lakeflow expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_CLINICAL_SCORE_MANDATORY_RULES)
@_expect_all(CLINICAL_CLINICAL_SCORE_ADVISORY_RULES)
def gold_clinical_clinical_score():
    """Quality-controlled twin of journey_clinical.clinical_score."""
    df = _qc("clinical_clinical_score", CLINICAL_CLINICAL_SCORE_SELECT)
    return _with_comments(df, CLINICAL_CLINICAL_SCORE_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.community_care_activity ====

CLINICAL_COMMUNITY_CARE_ACTIVITY_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",

    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 2 of 15,483,317 rows (1.29e-05%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `event_datetime` END AS `event_datetime`",
    "`event_end_datetime` AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 3 of 15,483,317 rows (1.94e-05%) when profiled on 2026-08-24.
    #
    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 2 of 15,483,317 rows (1.29e-05%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`care_activity_date` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE CASE WHEN CAST(`care_activity_date` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`care_activity_date` AS DATE)) < 9999 THEN NULL ELSE `care_activity_date` END END AS `care_activity_date`",
    "`care_activity_date_quality_status` AS `care_activity_date_quality_status`",
    "`community_contact_id` AS `community_contact_id`",
    "`contact_match_status` AS `contact_match_status`",
    "`contact_candidate_count` AS `contact_candidate_count`",
    "`same_date_contact_candidate_count` AS `same_date_contact_candidate_count`",
    "`community_database_id` AS `community_database_id`",
    "`care_activity_id` AS `care_activity_id`",
    "`community_patient_key` AS `community_patient_key`",
    "`service_id` AS `service_id`",
    "`service_name` AS `service_name`",
    "`care_professional_local_id` AS `care_professional_local_id`",
    "`duration_minutes` AS `duration_minutes`",
    "`duration_quality_status` AS `duration_quality_status`",
    "`source_clinical_term` AS `source_clinical_term`",
    "`source_clinical_term_key` AS `source_clinical_term_key`",
    "`observation_type_id` AS `observation_type_id`",
    "`code_category_id` AS `code_category_id`",
    "`observation_value_raw` AS `observation_value_raw`",
    "`observation_value_numeric` AS `observation_value_numeric`",
    "`unit_source_value` AS `unit_source_value`",
    "`normalized_ucum_code` AS `normalized_ucum_code`",
    "`unit_concept_id` AS `unit_concept_id`",
    "`unit_concept_name` AS `unit_concept_name`",
    "`unit_mapping_status` AS `unit_mapping_status`",
    "`unit_mapping_method` AS `unit_mapping_method`",
    "`snomed_candidate_count` AS `snomed_candidate_count`",
    "`snomed_candidate_concept_id` AS `snomed_candidate_concept_id`",
    "`snomed_candidate_code` AS `snomed_candidate_code`",
    "`snomed_candidate_name` AS `snomed_candidate_name`",
    "`snomed_candidate_domain` AS `snomed_candidate_domain`",
    "`snomed_candidate_class` AS `snomed_candidate_class`",
    "`snomed_candidate_method` AS `snomed_candidate_method`",
    "`snomed_candidate_status` AS `snomed_candidate_status`",
    "`person_match_status` AS `person_match_status`",
    "`record_status` AS `record_status`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 3 of 15,483,317 rows (1.94e-05%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`record_status_effective_from` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`record_status_effective_from` AS DATE)) < 9999 THEN NULL ELSE `record_status_effective_from` END AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`fact_category` AS `fact_category`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
]

CLINICAL_COMMUNITY_CARE_ACTIVITY_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 15,436,295 rows of 15,483,317
    # that are current and attributable; identity_status = 'resolved' keeps the 15,293,141
    # rows of 15,483,317 that are current and attributable. Superseded versions and rows
    # whose identity was never resolved are not research data, and a consumer who wants them
    # has silver.
    "research_surface": "(identity_status = 'resolved') AND (record_status = 'active')",
}

CLINICAL_COMMUNITY_CARE_ACTIVITY_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 190,176 of 15,483,317 at the profile.
    "gold.clinical.community_care_activity.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 47,022 of 15,483,317 at the profile.
    "gold.clinical.community_care_activity.record_status.default_view_active":
        "record_status = 'active'",

    # This bounds a period of validity, and a future end is exactly how the source says a
    # record is still current -- nulling it would assert the record is valid forever, which
    # is a stronger and worse claim than the one being corrected. Seen on 2 of 15,483,317
    # rows (1.29e-05%) when profiled on 2026-08-24.
    "gold.clinical.community_care_activity.record_status_effective_from.future_owner":
        "NOT COALESCE((CAST(`record_status_effective_from` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 7,362 of 15,483,317 rows (0.0475%) when profiled on 2026-08-24.
    "gold.clinical.community_care_activity.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 99 of 15,483,317 rows (0.000639%) when profiled on 2026-08-24.
    "gold.clinical.community_care_activity.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",
}

CLINICAL_COMMUNITY_CARE_ACTIVITY_COLUMN_COMMENTS = {
    "patient_event_id": "Stable row identifier.",
    "fact_row_id": "Stable row identifier.",
    "subject_key": "Published field.",
    "subject_id_system": "Published field.",
    "person_id": "Published field.",
    "identity_status": "Published field.",
    "encounter_id": "Published field.",
    "event_datetime": "Published field.",
    "event_end_datetime": "Published field.",
    "source_coding_system": "Published field.",
    "source_code": "Published field.",
    "source_display": "Published field.",
    "care_activity_date":
        "Published field. Gold QC transform rules: gold.clinical.community_care_activity.care_activity_date.beyond_2100_below_9999_owner.",
    "care_activity_date_quality_status": "Published field.",
    "community_contact_id": "Published field.",
    "contact_match_status": "Published field.",
    "contact_candidate_count": "Published field.",
    "same_date_contact_candidate_count": "Published field.",
    "community_database_id": "Published field.",
    "care_activity_id": "Published field.",
    "community_patient_key": "Published field.",
    "service_id": "Published field.",
    "service_name": "Published field.",
    "care_professional_local_id": "Published field.",
    "duration_minutes": "Published field.",
    "duration_quality_status": "Published field.",
    "source_clinical_term": "Published field.",
    "source_clinical_term_key": "Published field.",
    "observation_type_id": "Published field.",
    "code_category_id": "Published field.",
    "observation_value_raw": "Published field.",
    "observation_value_numeric": "Published field.",
    "unit_source_value": "Published field.",
    "normalized_ucum_code": "Published field.",
    "unit_concept_id": "Published field.",
    "unit_concept_name": "Published field.",
    "unit_mapping_status": "Published field.",
    "unit_mapping_method": "Published field.",
    "snomed_candidate_count": "Published field.",
    "snomed_candidate_concept_id": "Published field.",
    "snomed_candidate_code": "Published field.",
    "snomed_candidate_name": "Published field.",
    "snomed_candidate_domain": "Published field.",
    "snomed_candidate_class": "Published field.",
    "snomed_candidate_method": "Published field.",
    "snomed_candidate_status": "Published field.",
    "person_match_status": "Published field.",
    "record_status": "Published field.",
    "record_status_effective_from":
        "Published field. Gold QC transform rules: gold.clinical.community_care_activity.record_status_effective_from.beyond_2100_below_9999_owner.",
    "record_status_effective_to": "Published field.",
    "confidentiality_code": "Published field.",
    "vip_ind": "Published field.",
    "withheld_identity_ind": "Published field.",
    "fact_category": "Published field.",
    "source_feed": "Published field.",
    "load_batch_id": "Published field.",
    "source_update_timestamp": "Published field.",
    "loaded_at": "Published field.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.community_care_activity"),
    comment=(
        "One admitted CSDS community-care activity. Gold QC twin of the silver product: 3 "
        "columns are repaired or nulled, 1 rule(s) drop rows, 5 check(s) are advisory. Each "
        "rule states its reason in the pipeline notebook, and Lakeflow expectation metrics "
        "report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_COMMUNITY_CARE_ACTIVITY_MANDATORY_RULES)
@_expect_all(CLINICAL_COMMUNITY_CARE_ACTIVITY_ADVISORY_RULES)
def gold_clinical_community_care_activity():
    """Quality-controlled twin of journey_clinical.community_care_activity."""
    df = _qc(
        "clinical_community_care_activity",
        CLINICAL_COMMUNITY_CARE_ACTIVITY_SELECT,
        date_flags=["event_after_death_30d", "event_before_birth"],
    )
    return _with_comments(df, CLINICAL_COMMUNITY_CARE_ACTIVITY_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.community_care_contact ====

CLINICAL_COMMUNITY_CARE_CONTACT_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",
    "`event_datetime` AS `event_datetime`",
    "`event_end_datetime` AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`care_contact_date` AS `care_contact_date`",
    "`community_database_id` AS `community_database_id`",
    "`care_contact_id` AS `care_contact_id`",
    "`community_patient_key` AS `community_patient_key`",
    "`service_request_id` AS `service_request_id`",
    "`service_id` AS `service_id`",
    "`service_name` AS `service_name`",
    "`team_id` AS `team_id`",
    "`care_contact_id_variant_count` AS `care_contact_id_variant_count`",
    "`person_match_status` AS `person_match_status`",
    "`duration_minutes` AS `duration_minutes`",
    "`duration_quality_status` AS `duration_quality_status`",
    "`earliest_reasonable_offer_date` AS `earliest_reasonable_offer_date`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 1 of 2,978,381 rows (3.36e-05%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`earliest_clinically_appropriate_date` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`earliest_clinically_appropriate_date` AS DATE)) < 9999 THEN NULL ELSE `earliest_clinically_appropriate_date` END AS `earliest_clinically_appropriate_date`",
    "`commissioner_ods_code` AS `commissioner_ods_code`",
    "`commissioner_organization_id` AS `commissioner_organization_id`",
    "`commissioner_organization_name` AS `commissioner_organization_name`",
    "`consultation_mechanism_code` AS `consultation_mechanism_code`",
    "`consultation_mechanism_display` AS `consultation_mechanism_display`",
    "`location_type_code` AS `location_type_code`",
    "`location_type_display` AS `location_type_display`",
    "`service_team_type_code` AS `service_team_type_code`",
    "`service_team_type_display` AS `service_team_type_display`",
    "`service_team_type_mapping_status` AS `service_team_type_mapping_status`",
    "`source_consultation_term` AS `source_consultation_term`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`fact_category` AS `fact_category`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
]

CLINICAL_COMMUNITY_CARE_CONTACT_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 2,978,082 rows of 2,978,381
    # that are current and attributable; identity_status = 'resolved' keeps the 2,942,152
    # rows of 2,978,381 that are current and attributable. Superseded versions and rows
    # whose identity was never resolved are not research data, and a consumer who wants them
    # has silver.
    "research_surface": "(identity_status = 'resolved') AND (record_status = 'active')",
}

CLINICAL_COMMUNITY_CARE_CONTACT_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 36,229 of 2,978,381 at the profile.
    "gold.clinical.community_care_contact.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 299 of 2,978,381 at the profile.
    "gold.clinical.community_care_contact.record_status.default_view_active":
        "record_status = 'active'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 1,700 of 2,978,381 rows (0.0571%) when profiled on 2026-08-24.
    "gold.clinical.community_care_contact.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 13 of 2,978,381 rows (0.000436%) when profiled on 2026-08-24.
    "gold.clinical.community_care_contact.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",
}

CLINICAL_COMMUNITY_CARE_CONTACT_COLUMN_COMMENTS = {
    "patient_event_id": "Stable row identifier.",
    "fact_row_id": "Stable row identifier.",
    "subject_key": "Published field.",
    "subject_id_system": "Published field.",
    "person_id": "Published field.",
    "identity_status": "Published field.",
    "encounter_id": "Published field.",
    "event_datetime": "Published field.",
    "event_end_datetime": "Published field.",
    "source_coding_system": "Published field.",
    "source_code": "Published field.",
    "source_display": "Published field.",
    "care_contact_date": "Published field.",
    "community_database_id": "Published field.",
    "care_contact_id": "Published field.",
    "community_patient_key": "Published field.",
    "service_request_id": "Published field.",
    "service_id": "Published field.",
    "service_name": "Published field.",
    "team_id": "Published field.",
    "care_contact_id_variant_count": "Published field.",
    "person_match_status": "Published field.",
    "duration_minutes": "Published field.",
    "duration_quality_status": "Published field.",
    "earliest_reasonable_offer_date": "Published field.",
    "earliest_clinically_appropriate_date":
        "Published field. Gold QC transform rules: gold.clinical.community_care_contact.earliest_clinically_appropriate_date.beyond_2100_below_9999_owner.",
    "commissioner_ods_code": "Published field.",
    "commissioner_organization_id": "Published field.",
    "commissioner_organization_name": "Published field.",
    "consultation_mechanism_code": "Published field.",
    "consultation_mechanism_display": "Published field.",
    "location_type_code": "Published field.",
    "location_type_display": "Published field.",
    "service_team_type_code": "Published field.",
    "service_team_type_display": "Published field.",
    "service_team_type_mapping_status": "Published field.",
    "source_consultation_term": "Published field.",
    "record_status": "Published field.",
    "record_status_effective_from": "Published field.",
    "record_status_effective_to": "Published field.",
    "confidentiality_code": "Published field.",
    "vip_ind": "Published field.",
    "withheld_identity_ind": "Published field.",
    "fact_category": "Published field.",
    "source_feed": "Published field.",
    "load_batch_id": "Published field.",
    "source_update_timestamp": "Published field.",
    "loaded_at": "Published field.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.community_care_contact"),
    comment=(
        "One CSDS community-care contact. Gold QC twin of the silver product: 1 columns are "
        "repaired or nulled, 1 rule(s) drop rows, 4 check(s) are advisory. Each rule states "
        "its reason in the pipeline notebook, and Lakeflow expectation metrics report what "
        "every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_COMMUNITY_CARE_CONTACT_MANDATORY_RULES)
@_expect_all(CLINICAL_COMMUNITY_CARE_CONTACT_ADVISORY_RULES)
def gold_clinical_community_care_contact():
    """Quality-controlled twin of journey_clinical.community_care_contact."""
    df = _qc(
        "clinical_community_care_contact",
        CLINICAL_COMMUNITY_CARE_CONTACT_SELECT,
        date_flags=["event_after_death_30d", "event_before_birth"],
    )
    return _with_comments(df, CLINICAL_COMMUNITY_CARE_CONTACT_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.condition ====

CLINICAL_CONDITION_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",

    # 27 rows point at a person_id the spine does not have. The pointer is nulled so it
    # cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    "CASE WHEN NOT `person_id_resolved` THEN NULL ELSE `person_id` END AS `person_id`",
    "`identity_status` AS `identity_status`",

    # 44,025 rows point at a encounter_id the spine does not have. The pointer is nulled so
    # it cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    "CASE WHEN NOT `encounter_id_resolved` THEN NULL ELSE `encounter_id` END AS `encounter_id`",

    # 1900-01-01 is a placeholder low date rather than a date in 1900; on a non-birth field
    # it carries no more meaning than it does on a birth one. Hit 5 of 49,615,774 rows
    # (1.01e-05%) when profiled on 2026-08-24.
    #
    # A date before 1901 that is not one of the known placeholders. Nothing in this estate
    # predates the twentieth century, so these are mistyped or mis-scaled rather than early.
    # Hit 2 of 49,615,774 rows (4.03e-06%) when profiled on 2026-08-24.
    #
    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 8 of 49,615,774 rows (1.61e-05%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE CASE WHEN CAST(`event_datetime` AS DATE) = DATE'1900-01-01' OR CAST(`event_datetime` AS DATE) < DATE'1901-01-01' AND CAST(`event_datetime` AS DATE) NOT IN (DATE'1800-01-01', DATE'1899-12-30', DATE'1900-01-01') THEN NULL ELSE `event_datetime` END END AS `event_datetime`",

    # event_end_datetime cannot precede event_datetime. The start is the better-attested of
    # the two, so the end is what goes and the row keeps its event_datetime. Hit 9,847 of
    # 49,615,774 rows (0.0198%) when profiled on 2026-08-24.
    "CASE WHEN `event_datetime` IS NOT NULL AND `event_end_datetime` IS NOT NULL AND `event_datetime` > `event_end_datetime` THEN NULL ELSE `event_end_datetime` END AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",

    # 'UNKNOWN' is a placeholder the source writes when the value was not recorded; it is
    # not a code, so it is nulled rather than passed on as one. Hit 2 of 49,615,774 rows
    # (4.03e-06%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`source_code` AS STRING))) = 'UNKNOWN' THEN NULL ELSE `source_code` END AS `source_code`",
    "`source_display` AS `source_display`",
    "`condition_code` AS `condition_code`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 718,283 of 49,615,774 rows
    # (1.45%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`category_code` AS STRING))) = '0' THEN NULL ELSE `category_code` END AS `category_code`",
    "`category_display` AS `category_display`",
    "`clinical_status_code` AS `clinical_status_code`",
    "`clinical_status_display` AS `clinical_status_display`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 37,260,590 of 49,615,774 rows
    # (75.1%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`verification_status_code` AS STRING))) = '0' THEN NULL ELSE `verification_status_code` END AS `verification_status_code`",
    "`verification_status_display` AS `verification_status_display`",

    # 1900-01-01 is a placeholder low date rather than a date in 1900; on a non-birth field
    # it carries no more meaning than it does on a birth one. Hit 5 of 49,615,774 rows
    # (1.01e-05%) when profiled on 2026-08-24.
    #
    # A date before 1901 that is not one of the known placeholders. Nothing in this estate
    # predates the twentieth century, so these are mistyped or mis-scaled rather than early.
    # Hit 2 of 49,615,774 rows (4.03e-06%) when profiled on 2026-08-24.
    #
    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 8 of 49,615,774 rows (1.61e-05%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`onset_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE CASE WHEN CAST(`onset_datetime` AS DATE) = DATE'1900-01-01' OR CAST(`onset_datetime` AS DATE) < DATE'1901-01-01' AND CAST(`onset_datetime` AS DATE) NOT IN (DATE'1800-01-01', DATE'1899-12-30', DATE'1900-01-01') THEN NULL ELSE `onset_datetime` END END AS `onset_datetime`",
    "`abatement_datetime` AS `abatement_datetime`",
    "`body_site_code` AS `body_site_code`",
    "`body_site_display` AS `body_site_display`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 49,482,167 of 49,615,774 rows
    # (99.7%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`severity_code` AS STRING))) = '0' THEN NULL ELSE `severity_code` END AS `severity_code`",
    "`severity_display` AS `severity_display`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 48,966,635 of 49,615,774 rows
    # (98.7%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`laterality_code` AS STRING))) = '0' THEN NULL ELSE `laterality_code` END AS `laterality_code`",
    "`laterality_display` AS `laterality_display`",
    "`asserted_datetime` AS `asserted_datetime`",
    "`asserter_practitioner_id` AS `asserter_practitioner_id`",
    "`recorder_practitioner_id` AS `recorder_practitioner_id`",
    "`revision_history` AS `revision_history`",
    "`revision_history_count` AS `revision_history_count`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",

    # record_status_effective_to cannot precede record_status_effective_from. The start is
    # the better-attested of the two, so the end is what goes and the row keeps its
    # record_status_effective_from. Hit 108,547 of 49,615,774 rows (0.219%) when profiled on
    # 2026-08-24.
    "CASE WHEN `record_status_effective_from` IS NOT NULL AND `record_status_effective_to` IS NOT NULL AND `record_status_effective_from` > `record_status_effective_to` THEN NULL ELSE `record_status_effective_to` END AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",

    # Silver derived this flag by comparing event_datetime with the person's own dates,
    # before the rules above corrected event_datetime. On the rows where event_datetime
    # changed, the flag describes a timestamp gold no longer publishes. This product carries
    # a VARIANT column, which rules out the spine join gold would need to recompute the
    # flag, so it is nulled where the value beneath it moved rather than left asserting
    # something stale.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `event_before_birth` END AS `event_before_birth`",

    # Silver derived this flag by comparing event_datetime with the person's own dates,
    # before the rules above corrected event_datetime. On the rows where event_datetime
    # changed, the flag describes a timestamp gold no longer publishes. This product carries
    # a VARIANT column, which rules out the spine join gold would need to recompute the
    # flag, so it is nulled where the value beneath it moved rather than left asserting
    # something stale.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `event_after_death_30d` END AS `event_after_death_30d`",
]

CLINICAL_CONDITION_MANDATORY_RULES = {
    # The research surface. identity_status = 'resolved' keeps the 49,615,752 rows of
    # 49,615,774 that are current and attributable. Superseded versions and rows whose
    # identity was never resolved are not research data, and a consumer who wants them has
    # silver.
    "research_surface": "(identity_status = 'resolved')",
}

CLINICAL_CONDITION_ADVISORY_RULES = {
    # Kept as a regression guard on the accepted set ('resolved', 'provisional',
    # 'unresolved'): the research-surface filter already removes every row that fails it, so
    # this expectation should read zero forever and is worth watching for the day it does
    # not. Measured at 20 of 49,615,774 rows (4.03e-05%) when profiled on 2026-08-24.
    "gold.clinical.condition.identity_status.accepted_values":
        "NOT COALESCE((`identity_status` IS NOT NULL AND `identity_status` NOT IN ('resolved', 'provisional', 'unresolved')), FALSE)",

    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 22 of 49,615,774 at the profile.
    "gold.clinical.condition.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 56,771 of 49,615,774 rows (0.114%) when profiled on 2026-08-24.
    "gold.clinical.condition.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 358,178 of 49,615,774 rows (0.722%) when profiled on 2026-08-24.
    "gold.clinical.condition.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",
}

CLINICAL_CONDITION_COLUMN_COMMENTS = {
    "patient_event_id": "Stable product-wide condition identifier.",
    "fact_row_id": "Storage-row identifier equal to patient_event_id.",
    "subject_key": "Always-populated peppered subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Best-available encounter reference.",
    "event_datetime":
        "Primary assertion timestamp. Gold QC transform rules: gold.clinical.condition.event_datetime.d1900_01_01_owner, gold.clinical.condition.event_datetime.pre1901_other_owner.",
    "event_end_datetime": "Source effective end where supplied.",
    "source_coding_system": "Verbatim source coding system.",
    "source_code": "Source condition code",
    "source_display": "Verbatim source condition display.",
    "condition_code": "Source and mapped condition codings.",
    "category_code": "Source condition category code.",
    "category_display": "Source condition category display.",
    "clinical_status_code": "Source clinical-status code.",
    "clinical_status_display": "Source clinical-status display.",
    "verification_status_code": "Source verification-status code.",
    "verification_status_display": "Source verification-status display.",
    "onset_datetime":
        "Source onset timestamp. Gold QC transform rules: gold.clinical.condition.onset_datetime.d1900_01_01_owner, gold.clinical.condition.onset_datetime.pre1901_other_owner.",
    "abatement_datetime": "Source abatement or effective-end timestamp.",
    "body_site_code": "Body-site code where supplied.",
    "body_site_display": "Body-site display where supplied.",
    "severity_code": "Source severity code.",
    "severity_display": "Source severity display.",
    "laterality_code": "Source laterality code.",
    "laterality_display": "Source laterality display.",
    "asserted_datetime": "Source assertion timestamp.",
    "asserter_practitioner_id": "Asserting practitioner reference.",
    "recorder_practitioner_id": "Recording or source-status practitioner reference.",
    "revision_history":
        "Ordered JSON revision history from map_problem_history; tombstoned revisions carry source_tombstone_ind.",
    "revision_history_count": "Number of retained problem revision rows.",
    "record_status": "Normalized source-record lifecycle.",
    "record_status_effective_from": "Source status effective start.",
    "record_status_effective_to": "Source status effective end.",
    "confidentiality_code": "Source confidentiality code.",
    "vip_ind": "Source VIP indicator.",
    "withheld_identity_ind": "Withheld-identity indicator.",
    "source_feed": "Registered source feed owning the assertion.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at":
        "Latest bronze load time across the fact row and its folded evidence rows.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.condition"),
    comment=(
        "One diagnosis or problem assertion from a registered source feed, without cross-feed "
        "deduplication. Gold QC twin of the silver product: 13 columns are repaired or "
        "nulled, 1 rule(s) drop rows, 4 check(s) are advisory. Each rule states its reason in "
        "the pipeline notebook, and Lakeflow expectation metrics report what every rule "
        "matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_CONDITION_MANDATORY_RULES)
@_expect_all(CLINICAL_CONDITION_ADVISORY_RULES)
def gold_clinical_condition():
    """Quality-controlled twin of journey_clinical.condition."""
    df = _qc("clinical_condition", CLINICAL_CONDITION_SELECT)
    return _with_comments(df, CLINICAL_CONDITION_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.condition_stage ====

CLINICAL_CONDITION_STAGE_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",
    "`event_datetime` AS `event_datetime`",

    # event_end_datetime cannot precede event_datetime. The start is the better-attested of
    # the two, so the end is what goes and the row keeps its event_datetime. Hit 499 of
    # 133,086 rows (0.375%) when profiled on 2026-08-24.
    "CASE WHEN `event_datetime` IS NOT NULL AND `event_end_datetime` IS NOT NULL AND `event_datetime` > `event_end_datetime` THEN NULL ELSE `event_end_datetime` END AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`stage_code` AS `stage_code`",
    "`stage_of_disease` AS `stage_of_disease`",
    "`stage_criteria` AS `stage_criteria`",
    "`dx_type` AS `dx_type`",
    "`dx_confirmed` AS `dx_confirmed`",
    "`dx_method` AS `dx_method`",
    "`history_ind` AS `history_ind`",
    "`current_entry_ind` AS `current_entry_ind`",
    "`cause_of_death_ind` AS `cause_of_death_ind`",
    "`onset_datetime` AS `onset_datetime`",
    "`resolution_datetime` AS `resolution_datetime`",
    "`clinical_description` AS `clinical_description`",

    # An empty or whitespace-only string is how the source writes 'nothing here'. It reads
    # as a value in a query and is not one, so it is nulled. Hit 28,718 of 133,086 rows
    # (21.6%) when profiled on 2026-08-24.
    "CASE WHEN TRIM(CAST(`dx_comment` AS STRING)) = '' THEN NULL ELSE `dx_comment` END AS `dx_comment`",
    "`person_link_status` AS `person_link_status`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
    "`event_before_birth` AS `event_before_birth`",
    "`event_after_death_30d` AS `event_after_death_30d`",
]

CLINICAL_CONDITION_STAGE_MANDATORY_RULES = {
    # The research surface. identity_status = 'resolved' keeps the 130,441 rows of 133,086
    # that are current and attributable. Superseded versions and rows whose identity was
    # never resolved are not research data, and a consumer who wants them has silver.
    "research_surface": "(identity_status = 'resolved')",
}

CLINICAL_CONDITION_STAGE_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 2,645 of 133,086 at the profile.
    "gold.clinical.condition_stage.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 41 of 133,086 rows (0.0308%) when profiled on 2026-08-24.
    "gold.clinical.condition_stage.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 26 of 133,086 rows (0.0195%) when profiled on 2026-08-24.
    "gold.clinical.condition_stage.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",
}

CLINICAL_CONDITION_STAGE_COLUMN_COMMENTS = {
    "patient_event_id": "Stable condition-stage event identifier.",
    "fact_row_id": "Storage-row identifier equal to patient_event_id.",
    "subject_key": "Always-populated peppered subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference when available.",
    "event_datetime": "Sentinel-cleaned diagnosis onset timestamp.",
    "event_end_datetime": "Sentinel-cleaned resolution timestamp.",
    "source_coding_system": "ICD-10 coding system.",
    "source_code": "ICD diagnosis code with diagnosis-name fallback.",
    "source_display": "Diagnosis name or description.",
    "stage_code": "Source ICD and mapped OMOP diagnosis codings.",
    "stage_of_disease": "Source disease stage.",
    "stage_criteria": "Staging criteria description.",
    "dx_type": "Diagnosis type.",
    "dx_confirmed": "Diagnosis confirmation state.",
    "dx_method": "Diagnosis method.",
    "history_ind": "History indicator.",
    "current_entry_ind": "Current-entry indicator.",
    "cause_of_death_ind": "Cause-of-death indicator.",
    "onset_datetime": "Source onset timestamp.",
    "resolution_datetime": "Source resolution timestamp.",
    "clinical_description": "Clinical diagnosis description.",
    "dx_comment":
        "Diagnosis comment. Gold QC transform rules: gold.clinical.condition_stage.dx_comment.empty_string.",
    "person_link_status": "Bronze person-link status.",
    "record_status": "Normalized source-record lifecycle.",
    "record_status_effective_from": "Lifecycle start.",
    "record_status_effective_to": "Source absence timestamp.",
    "confidentiality_code": "Security classification when supplied.",
    "vip_ind": "VIP indicator when supplied.",
    "withheld_identity_ind": "Withheld-identity indicator.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.condition_stage"),
    comment=(
        "One frozen ARIA diagnosis-and-staging assertion with ICD source coding and mapped "
        "OMOP diagnosis evidence. Gold QC twin of the silver product: 2 columns are repaired "
        "or nulled, 1 rule(s) drop rows, 3 check(s) are advisory. Each rule states its reason "
        "in the pipeline notebook, and Lakeflow expectation metrics report what every rule "
        "matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_CONDITION_STAGE_MANDATORY_RULES)
@_expect_all(CLINICAL_CONDITION_STAGE_ADVISORY_RULES)
def gold_clinical_condition_stage():
    """Quality-controlled twin of journey_clinical.condition_stage."""
    df = _qc("clinical_condition_stage", CLINICAL_CONDITION_STAGE_SELECT)
    return _with_comments(df, CLINICAL_CONDITION_STAGE_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.costed_activity ====

CLINICAL_COSTED_ACTIVITY_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",
    "`event_datetime` AS `event_datetime`",

    # event_end_datetime cannot precede event_datetime. The start is the better-attested of
    # the two, so the end is what goes and the row keeps its event_datetime. Hit 26 of
    # 5,562,087 rows (0.000467%) when profiled on 2026-08-24.
    "CASE WHEN `event_datetime` IS NOT NULL AND `event_end_datetime` IS NOT NULL AND `event_datetime` > `event_end_datetime` THEN NULL ELSE `event_end_datetime` END AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`extract_cd` AS `extract_cd`",
    "`activity_record_id` AS `activity_record_id`",
    "`feed_type` AS `feed_type`",
    "`plemi` AS `plemi`",
    "`nhs_number_status_cd` AS `nhs_number_status_cd`",
    "`cds_id` AS `cds_id`",
    "`attendance_id` AS `attendance_id`",
    "`arrival_date` AS `arrival_date`",
    "`arrival_time` AS `arrival_time`",
    "`departure_date` AS `departure_date`",
    "`departure_time` AS `departure_time`",
    "`departure_type_cd` AS `departure_type_cd`",
    "`provider_org_cd` AS `provider_org_cd`",
    "`patient_org_cd` AS `patient_org_cd`",
    "`pathway_id` AS `pathway_id`",
    "`pod_cd` AS `pod_cd`",
    "`treatment_function_cd` AS `treatment_function_cd`",
    "`source_los` AS `source_los`",
    "`cf_band_cd` AS `cf_band_cd`",
    "`episode_number` AS `episode_number`",
    "`episode_start_datetime` AS `episode_start_datetime`",

    # episode_end_datetime cannot precede episode_start_datetime. The start is the better-
    # attested of the two, so the end is what goes and the row keeps its
    # episode_start_datetime. Hit 87 of 5,562,087 rows (0.00156%) when profiled on
    # 2026-08-24.
    "CASE WHEN `episode_start_datetime` IS NOT NULL AND `episode_end_datetime` IS NOT NULL AND `episode_start_datetime` > `episode_end_datetime` THEN NULL ELSE `episode_end_datetime` END AS `episode_end_datetime`",
    "`episode_type_cd` AS `episode_type_cd`",
    "`hosp_spell_id` AS `hosp_spell_id`",
    "`hrg_cd` AS `hrg_cd`",
    "`hrg_desc` AS `hrg_desc`",
    "`fce_hrg_cd` AS `fce_hrg_cd`",
    "`fce_hrg_desc` AS `fce_hrg_desc`",
    "`spell_hrg_cd` AS `spell_hrg_cd`",
    "`spell_hrg_desc` AS `spell_hrg_desc`",
    "`appointment_date` AS `appointment_date`",
    "`appointment_time` AS `appointment_time`",
    "`critical_care_unit_function_cd` AS `critical_care_unit_function_cd`",
    "`organs_supported` AS `organs_supported`",
    "`critical_care_period_type_cd` AS `critical_care_period_type_cd`",
    "`critical_care_level_ind` AS `critical_care_level_ind`",
    "`unbundled_activity_datetime` AS `unbundled_activity_datetime`",
    "`unbundled_activity_cd` AS `unbundled_activity_cd`",
    "`unbundled_hrg_cd` AS `unbundled_hrg_cd`",
    "`unbundled_hrg_desc` AS `unbundled_hrg_desc`",
    "`partial_costing_ind` AS `partial_costing_ind`",
    "`care_datetime` AS `care_datetime`",
    "`care_id` AS `care_id`",
    "`clinical_contact_duration` AS `clinical_contact_duration`",
    "`chs_currency_cd` AS `chs_currency_cd`",
    "`team_type_cd` AS `team_type_cd`",
    "`contact_subject_cd` AS `contact_subject_cd`",
    "`consult_type_cd` AS `consult_type_cd`",
    "`consult_medium_cd` AS `consult_medium_cd`",
    "`location_cd` AS `location_cd`",
    "`gp_therapy_ind` AS `gp_therapy_ind`",
    "`service_request_id` AS `service_request_id`",
    "`cost_line_count` AS `cost_line_count`",
    "`total_cost_sum` AS `total_cost_sum`",
    "`total_o_cost_sum` AS `total_o_cost_sum`",
    "`person_link_method` AS `person_link_method`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`fact_category` AS `fact_category`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
]

CLINICAL_COSTED_ACTIVITY_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 5,562,087 rows of 5,562,087
    # that are current and attributable; identity_status = 'resolved' keeps the 5,380,935
    # rows of 5,562,087 that are current and attributable. Superseded versions and rows
    # whose identity was never resolved are not research data, and a consumer who wants them
    # has silver.
    "research_surface": "(identity_status = 'resolved') AND (record_status = 'active')",
}

CLINICAL_COSTED_ACTIVITY_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 181,152 of 5,562,087 at the profile.
    "gold.clinical.costed_activity.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 0 of 5,562,087 at the profile.
    "gold.clinical.costed_activity.record_status.default_view_active":
        "record_status = 'active'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 630 of 5,562,087 rows (0.0113%) when profiled on 2026-08-24.
    "gold.clinical.costed_activity.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 5,201 of 5,562,087 rows (0.0935%) when profiled on 2026-08-24.
    "gold.clinical.costed_activity.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",
}

CLINICAL_COSTED_ACTIVITY_COLUMN_COMMENTS = {
    "patient_event_id": "Stable row identifier.",
    "fact_row_id": "Stable row identifier.",
    "subject_key": "Published field.",
    "subject_id_system": "Published field.",
    "person_id": "Published field.",
    "identity_status": "Published field.",
    "encounter_id": "Published field.",
    "event_datetime": "Published field.",
    "event_end_datetime": "Published field.",
    "source_coding_system": "Published field.",
    "source_code": "Published field.",
    "source_display": "Published field.",
    "extract_cd": "Published field.",
    "activity_record_id": "Published field.",
    "feed_type": "Published field.",
    "plemi": "Published field.",
    "nhs_number_status_cd": "Published field.",
    "cds_id": "Published field.",
    "attendance_id": "Published field.",
    "arrival_date": "Published field.",
    "arrival_time": "Published field.",
    "departure_date": "Published field.",
    "departure_time": "Published field.",
    "departure_type_cd": "Published field.",
    "provider_org_cd": "Published field.",
    "patient_org_cd": "Published field.",
    "pathway_id": "Published field.",
    "pod_cd": "Published field.",
    "treatment_function_cd": "Published field.",
    "source_los": "Published field.",
    "cf_band_cd": "Published field.",
    "episode_number": "Published field.",
    "episode_start_datetime": "Published field.",
    "episode_end_datetime": "Published field.",
    "episode_type_cd": "Published field.",
    "hosp_spell_id": "Published field.",
    "hrg_cd": "Published field.",
    "hrg_desc": "Published field.",
    "fce_hrg_cd": "Published field.",
    "fce_hrg_desc": "Published field.",
    "spell_hrg_cd": "Published field.",
    "spell_hrg_desc": "Published field.",
    "appointment_date": "Published field.",
    "appointment_time": "Published field.",
    "critical_care_unit_function_cd": "Published field.",
    "organs_supported": "Published field.",
    "critical_care_period_type_cd": "Published field.",
    "critical_care_level_ind": "Published field.",
    "unbundled_activity_datetime": "Published field.",
    "unbundled_activity_cd": "Published field.",
    "unbundled_hrg_cd": "Published field.",
    "unbundled_hrg_desc": "Published field.",
    "partial_costing_ind": "Published field.",
    "care_datetime": "Published field.",
    "care_id": "Published field.",
    "clinical_contact_duration": "Published field.",
    "chs_currency_cd": "Published field.",
    "team_type_cd": "Published field.",
    "contact_subject_cd": "Published field.",
    "consult_type_cd": "Published field.",
    "consult_medium_cd": "Published field.",
    "location_cd": "Published field.",
    "gp_therapy_ind": "Published field.",
    "service_request_id": "Published field.",
    "cost_line_count": "Published field.",
    "total_cost_sum": "Published field.",
    "total_o_cost_sum": "Published field.",
    "person_link_method": "Published field.",
    "record_status": "Published field.",
    "record_status_effective_from": "Published field.",
    "record_status_effective_to": "Published field.",
    "confidentiality_code": "Published field.",
    "vip_ind": "Published field.",
    "withheld_identity_ind": "Published field.",
    "fact_category": "Published field.",
    "source_feed": "Published field.",
    "load_batch_id": "Published field.",
    "source_update_timestamp": "Published field.",
    "loaded_at": "Published field.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.costed_activity"),
    comment=(
        "One frozen PLICS costed activity. Gold QC twin of the silver product: 3 columns are "
        "repaired or nulled, 1 rule(s) drop rows, 4 check(s) are advisory. Each rule states "
        "its reason in the pipeline notebook, and Lakeflow expectation metrics report what "
        "every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_COSTED_ACTIVITY_MANDATORY_RULES)
@_expect_all(CLINICAL_COSTED_ACTIVITY_ADVISORY_RULES)
def gold_clinical_costed_activity():
    """Quality-controlled twin of journey_clinical.costed_activity."""
    # 20 rows point at a person_id the spine does not have. The pointer is nulled so it
    # cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    df = _qc(
        "clinical_costed_activity",
        CLINICAL_COSTED_ACTIVITY_SELECT,
        fk_columns=["person_id"],
        date_flags=["event_after_death_30d", "event_before_birth"],
    )
    return _with_comments(df, CLINICAL_COSTED_ACTIVITY_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.critical_care_activity ====

CLINICAL_CRITICAL_CARE_ACTIVITY_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",
    "`event_datetime` AS `event_datetime`",
    "`event_end_datetime` AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`period_link_status` AS `period_link_status`",
    "`period_business_key` AS `period_business_key`",
    "`parent_period_id` AS `parent_period_id`",
    "`cds_apc_id` AS `cds_apc_id`",
    "`cc_type` AS `cc_type`",
    "`source_duplicate_count` AS `source_duplicate_count`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",

    # record_status_effective_to cannot precede record_status_effective_from. The start is
    # the better-attested of the two, so the end is what goes and the row keeps its
    # record_status_effective_from. Hit 1 of 341,038 rows (0.000293%) when profiled on
    # 2026-08-24.
    "CASE WHEN `record_status_effective_from` IS NOT NULL AND `record_status_effective_to` IS NOT NULL AND `record_status_effective_from` > `record_status_effective_to` THEN NULL ELSE `record_status_effective_to` END AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`fact_category` AS `fact_category`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
]

CLINICAL_CRITICAL_CARE_ACTIVITY_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 336,263 rows of 341,038 that
    # are current and attributable; identity_status = 'resolved' keeps the 331,191 rows of
    # 341,038 that are current and attributable. Superseded versions and rows whose identity
    # was never resolved are not research data, and a consumer who wants them has silver.
    "research_surface": "(identity_status = 'resolved') AND (record_status = 'active')",
}

CLINICAL_CRITICAL_CARE_ACTIVITY_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 9,847 of 341,038 at the profile.
    "gold.clinical.critical_care_activity.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 4,775 of 341,038 at the profile.
    "gold.clinical.critical_care_activity.record_status.default_view_active":
        "record_status = 'active'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 7,892 of 341,038 rows (2.31%) when profiled on 2026-08-24.
    "gold.clinical.critical_care_activity.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",
}

CLINICAL_CRITICAL_CARE_ACTIVITY_COLUMN_COMMENTS = {
    "patient_event_id": "Published field patient_event_id.",
    "fact_row_id": "Published field fact_row_id.",
    "subject_key": "Published field subject_key.",
    "subject_id_system": "Published field subject_id_system.",
    "person_id": "Published field person_id.",
    "identity_status": "Published field identity_status.",
    "encounter_id": "Published field encounter_id.",
    "event_datetime": "Published field event_datetime.",
    "event_end_datetime": "Published field event_end_datetime.",
    "source_coding_system": "Published field source_coding_system.",
    "source_code": "Published field source_code.",
    "source_display": "Published field source_display.",
    "period_link_status": "Published field period_link_status.",
    "period_business_key": "Published field period_business_key.",
    "parent_period_id": "Published field parent_period_id.",
    "cds_apc_id": "Published field cds_apc_id.",
    "cc_type": "Published field cc_type.",
    "source_duplicate_count": "Published field source_duplicate_count.",
    "record_status": "Published field record_status.",
    "record_status_effective_from": "Published field record_status_effective_from.",
    "record_status_effective_to": "Published field record_status_effective_to.",
    "confidentiality_code": "Published field confidentiality_code.",
    "vip_ind": "Published field vip_ind.",
    "withheld_identity_ind": "Published field withheld_identity_ind.",
    "fact_category": "Published field fact_category.",
    "source_feed": "Published field source_feed.",
    "load_batch_id": "Published field load_batch_id.",
    "source_update_timestamp": "Published field source_update_timestamp.",
    "loaded_at": "Published field loaded_at.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.critical_care_activity"),
    comment=(
        "One admitted CCMDS critical-care activity with parent-link evidence. Gold QC twin of "
        "the silver product: 1 columns are repaired or nulled, 1 rule(s) drop rows, 3 "
        "check(s) are advisory. Each rule states its reason in the pipeline notebook, and "
        "Lakeflow expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_CRITICAL_CARE_ACTIVITY_MANDATORY_RULES)
@_expect_all(CLINICAL_CRITICAL_CARE_ACTIVITY_ADVISORY_RULES)
def gold_clinical_critical_care_activity():
    """Quality-controlled twin of journey_clinical.critical_care_activity."""
    df = _qc(
        "clinical_critical_care_activity",
        CLINICAL_CRITICAL_CARE_ACTIVITY_SELECT,
        date_flags=["event_before_birth"],
    )
    return _with_comments(df, CLINICAL_CRITICAL_CARE_ACTIVITY_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.critical_care_admission ====

CLINICAL_CRITICAL_CARE_ADMISSION_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",
    "`event_datetime` AS `event_datetime`",

    # event_end_datetime cannot precede event_datetime. The start is the better-attested of
    # the two, so the end is what goes and the row keeps its event_datetime. Hit 6 of
    # 141,391 rows (0.00424%) when profiled on 2026-08-24.
    "CASE WHEN `event_datetime` IS NOT NULL AND `event_end_datetime` IS NOT NULL AND `event_datetime` > `event_end_datetime` THEN NULL ELSE `event_end_datetime` END AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`admission_key` AS `admission_key`",
    "`source_unit` AS `source_unit`",
    "`unit_raw` AS `unit_raw`",
    "`source_site` AS `source_site`",
    "`hospital_admission_datetime` AS `hospital_admission_datetime`",
    "`unit_admission_datetime` AS `unit_admission_datetime`",
    "`unit_discharge_datetime` AS `unit_discharge_datetime`",
    "`hospital_discharge_datetime` AS `hospital_discharge_datetime`",
    "`admission_diagnosis_code` AS `admission_diagnosis_code`",
    "`max_organ_support` AS `max_organ_support`",
    "`cause_of_death` AS `cause_of_death`",
    "`date_of_death` AS `date_of_death`",
    "`source_updated_at` AS `source_updated_at`",
    "`person_link_status` AS `person_link_status`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`fact_category` AS `fact_category`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
]

CLINICAL_CRITICAL_CARE_ADMISSION_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 141,391 rows of 141,391 that
    # are current and attributable; identity_status = 'resolved' keeps the 135,937 rows of
    # 141,391 that are current and attributable. Superseded versions and rows whose identity
    # was never resolved are not research data, and a consumer who wants them has silver.
    "research_surface": "(identity_status = 'resolved') AND (record_status = 'active')",
}

CLINICAL_CRITICAL_CARE_ADMISSION_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 5,454 of 141,391 at the profile.
    "gold.clinical.critical_care_admission.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 0 of 141,391 at the profile.
    "gold.clinical.critical_care_admission.record_status.default_view_active":
        "record_status = 'active'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 12 of 141,391 rows (0.00849%) when profiled on 2026-08-24.
    "gold.clinical.critical_care_admission.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 1 of 141,391 rows (0.000707%) when profiled on 2026-08-24.
    "gold.clinical.critical_care_admission.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",
}

CLINICAL_CRITICAL_CARE_ADMISSION_COLUMN_COMMENTS = {
    "patient_event_id": "Published field patient_event_id.",
    "fact_row_id": "Published field fact_row_id.",
    "subject_key": "Published field subject_key.",
    "subject_id_system": "Published field subject_id_system.",
    "person_id": "Published field person_id.",
    "identity_status": "Published field identity_status.",
    "encounter_id": "Published field encounter_id.",
    "event_datetime": "Published field event_datetime.",
    "event_end_datetime": "Published field event_end_datetime.",
    "source_coding_system": "Published field source_coding_system.",
    "source_code": "Published field source_code.",
    "source_display": "Published field source_display.",
    "admission_key": "Published field admission_key.",
    "source_unit": "Published field source_unit.",
    "unit_raw": "Published field unit_raw.",
    "source_site": "Published field source_site.",
    "hospital_admission_datetime": "Published field hospital_admission_datetime.",
    "unit_admission_datetime": "Published field unit_admission_datetime.",
    "unit_discharge_datetime": "Published field unit_discharge_datetime.",
    "hospital_discharge_datetime": "Published field hospital_discharge_datetime.",
    "admission_diagnosis_code": "Published field admission_diagnosis_code.",
    "max_organ_support": "Published field max_organ_support.",
    "cause_of_death": "Published field cause_of_death.",
    "date_of_death": "Published field date_of_death.",
    "source_updated_at": "Published field source_updated_at.",
    "person_link_status": "Published field person_link_status.",
    "record_status": "Published field record_status.",
    "record_status_effective_from": "Published field record_status_effective_from.",
    "record_status_effective_to": "Published field record_status_effective_to.",
    "confidentiality_code": "Published field confidentiality_code.",
    "vip_ind": "Published field vip_ind.",
    "withheld_identity_ind": "Published field withheld_identity_ind.",
    "fact_category": "Published field fact_category.",
    "source_feed": "Published field source_feed.",
    "load_batch_id": "Published field load_batch_id.",
    "source_update_timestamp": "Published field source_update_timestamp.",
    "loaded_at": "Published field loaded_at.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.critical_care_admission"),
    comment=(
        "One Medicus critical-care admission per source admission key. Gold QC twin of the "
        "silver product: 1 columns are repaired or nulled, 1 rule(s) drop rows, 4 check(s) "
        "are advisory. Each rule states its reason in the pipeline notebook, and Lakeflow "
        "expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_CRITICAL_CARE_ADMISSION_MANDATORY_RULES)
@_expect_all(CLINICAL_CRITICAL_CARE_ADMISSION_ADVISORY_RULES)
def gold_clinical_critical_care_admission():
    """Quality-controlled twin of journey_clinical.critical_care_admission."""
    df = _qc(
        "clinical_critical_care_admission",
        CLINICAL_CRITICAL_CARE_ADMISSION_SELECT,
        date_flags=["event_after_death_30d", "event_before_birth"],
    )
    return _with_comments(df, CLINICAL_CRITICAL_CARE_ADMISSION_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.critical_care_daily_score ====

CLINICAL_CRITICAL_CARE_DAILY_SCORE_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",
    "`event_datetime` AS `event_datetime`",
    "`event_end_datetime` AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",

    # A negative amount is not possible for a dose, quantity or score, and nothing in the
    # row says what the intended magnitude was. Hit 55,833 of 3,989,576 rows (1.4%) when
    # profiled on 2026-08-24.
    "CASE WHEN `score_value` < 0 THEN NULL ELSE `score_value` END AS `score_value`",
    "`score_value_raw` AS `score_value_raw`",
    "`score_date` AS `score_date`",
    "`score_calc_date` AS `score_calc_date`",
    "`admission_key` AS `admission_key`",
    "`critical_care_admission_id` AS `critical_care_admission_id`",
    "`day_latest_ind` AS `day_latest_ind`",
    "`row_class` AS `row_class`",
    "`person_link_status` AS `person_link_status`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`fact_category` AS `fact_category`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
]

CLINICAL_CRITICAL_CARE_DAILY_SCORE_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 3,989,576 rows of 3,989,576
    # that are current and attributable; identity_status = 'resolved' keeps the 3,798,086
    # rows of 3,989,576 that are current and attributable. Superseded versions and rows
    # whose identity was never resolved are not research data, and a consumer who wants them
    # has silver.
    "research_surface": "(identity_status = 'resolved') AND (record_status = 'active')",
}

CLINICAL_CRITICAL_CARE_DAILY_SCORE_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 191,490 of 3,989,576 at the profile.
    "gold.clinical.critical_care_daily_score.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 0 of 3,989,576 at the profile.
    "gold.clinical.critical_care_daily_score.record_status.default_view_active":
        "record_status = 'active'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 156 of 3,989,576 rows (0.00391%) when profiled on 2026-08-24.
    "gold.clinical.critical_care_daily_score.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",
}

CLINICAL_CRITICAL_CARE_DAILY_SCORE_COLUMN_COMMENTS = {
    "patient_event_id": "Published field patient_event_id.",
    "fact_row_id": "Published field fact_row_id.",
    "subject_key": "Published field subject_key.",
    "subject_id_system": "Published field subject_id_system.",
    "person_id": "Published field person_id.",
    "identity_status": "Published field identity_status.",
    "encounter_id": "Published field encounter_id.",
    "event_datetime": "Published field event_datetime.",
    "event_end_datetime": "Published field event_end_datetime.",
    "source_coding_system": "Published field source_coding_system.",
    "source_code": "Published field source_code.",
    "source_display": "Published field source_display.",
    "score_value": "Published field score_value.",
    "score_value_raw": "Published field score_value_raw.",
    "score_date": "Published field score_date.",
    "score_calc_date": "Published field score_calc_date.",
    "admission_key": "Published field admission_key.",
    "critical_care_admission_id": "Published field critical_care_admission_id.",
    "day_latest_ind": "Published field day_latest_ind.",
    "row_class": "Published field row_class.",
    "person_link_status": "Published field person_link_status.",
    "record_status": "Published field record_status.",
    "record_status_effective_from": "Published field record_status_effective_from.",
    "record_status_effective_to": "Published field record_status_effective_to.",
    "confidentiality_code": "Published field confidentiality_code.",
    "vip_ind": "Published field vip_ind.",
    "withheld_identity_ind": "Published field withheld_identity_ind.",
    "fact_category": "Published field fact_category.",
    "source_feed": "Published field source_feed.",
    "load_batch_id": "Published field load_batch_id.",
    "source_update_timestamp": "Published field source_update_timestamp.",
    "loaded_at": "Published field loaded_at.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.critical_care_daily_score"),
    comment=(
        "One admitted Medicus daily critical-care score row. Gold QC twin of the silver "
        "product: 1 columns are repaired or nulled, 1 rule(s) drop rows, 3 check(s) are "
        "advisory. Each rule states its reason in the pipeline notebook, and Lakeflow "
        "expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_CRITICAL_CARE_DAILY_SCORE_MANDATORY_RULES)
@_expect_all(CLINICAL_CRITICAL_CARE_DAILY_SCORE_ADVISORY_RULES)
def gold_clinical_critical_care_daily_score():
    """Quality-controlled twin of journey_clinical.critical_care_daily_score."""
    df = _qc(
        "clinical_critical_care_daily_score",
        CLINICAL_CRITICAL_CARE_DAILY_SCORE_SELECT,
        date_flags=["event_after_death_30d"],
    )
    return _with_comments(df, CLINICAL_CRITICAL_CARE_DAILY_SCORE_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.critical_care_period ====

CLINICAL_CRITICAL_CARE_PERIOD_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",
    "`event_datetime` AS `event_datetime`",

    # event_end_datetime cannot precede event_datetime. The start is the better-attested of
    # the two, so the end is what goes and the row keeps its event_datetime. Hit 19 of
    # 176,886 rows (0.0107%) when profiled on 2026-08-24.
    "CASE WHEN `event_datetime` IS NOT NULL AND `event_end_datetime` IS NOT NULL AND `event_datetime` > `event_end_datetime` THEN NULL ELSE `event_end_datetime` END AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`period_end_datetime` AS `period_end_datetime`",
    "`period_business_key` AS `period_business_key`",
    "`no_current_version_ind` AS `no_current_version_ind`",
    "`business_key_status` AS `business_key_status`",
    "`source_valid_ind` AS `source_valid_ind`",
    "`care_type` AS `care_type`",
    "`unit_function` AS `unit_function`",
    "`unit_id` AS `unit_id`",
    "`cds_source_system` AS `cds_source_system`",
    "`level2_days` AS `level2_days`",
    "`level3_days` AS `level3_days`",
    "`organ_systems_supported` AS `organ_systems_supported`",
    "`gestation_length` AS `gestation_length`",
    "`discharge_status` AS `discharge_status`",
    "`discharge_destination` AS `discharge_destination`",
    "`source_encounter_id` AS `source_encounter_id`",
    "`cc_encounter_id` AS `cc_encounter_id`",
    "`cds_apc_id` AS `cds_apc_id`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`fact_category` AS `fact_category`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
]

CLINICAL_CRITICAL_CARE_PERIOD_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 176,886 rows of 176,886 that
    # are current and attributable; identity_status = 'resolved' keeps the 169,705 rows of
    # 176,886 that are current and attributable. Superseded versions and rows whose identity
    # was never resolved are not research data, and a consumer who wants them has silver.
    "research_surface": "(identity_status = 'resolved') AND (record_status = 'active')",
}

CLINICAL_CRITICAL_CARE_PERIOD_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 7,181 of 176,886 at the profile.
    "gold.clinical.critical_care_period.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 0 of 176,886 at the profile.
    "gold.clinical.critical_care_period.record_status.default_view_active":
        "record_status = 'active'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 308 of 176,886 rows (0.174%) when profiled on 2026-08-24.
    "gold.clinical.critical_care_period.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 64 of 176,886 rows (0.0362%) when profiled on 2026-08-24.
    "gold.clinical.critical_care_period.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",
}

CLINICAL_CRITICAL_CARE_PERIOD_COLUMN_COMMENTS = {
    "patient_event_id": "Published field patient_event_id.",
    "fact_row_id": "Published field fact_row_id.",
    "subject_key": "Published field subject_key.",
    "subject_id_system": "Published field subject_id_system.",
    "person_id": "Published field person_id.",
    "identity_status": "Published field identity_status.",
    "encounter_id": "Published field encounter_id.",
    "event_datetime": "Published field event_datetime.",
    "event_end_datetime": "Published field event_end_datetime.",
    "source_coding_system": "Published field source_coding_system.",
    "source_code": "Published field source_code.",
    "source_display": "Published field source_display.",
    "period_end_datetime": "Published field period_end_datetime.",
    "period_business_key": "Published field period_business_key.",
    "no_current_version_ind": "Published field no_current_version_ind.",
    "business_key_status": "Published field business_key_status.",
    "source_valid_ind": "Published field source_valid_ind.",
    "care_type": "Published field care_type.",
    "unit_function": "Published field unit_function.",
    "unit_id": "Published field unit_id.",
    "cds_source_system": "Published field cds_source_system.",
    "level2_days": "Published field level2_days.",
    "level3_days": "Published field level3_days.",
    "organ_systems_supported": "Published field organ_systems_supported.",
    "gestation_length": "Published field gestation_length.",
    "discharge_status": "Published field discharge_status.",
    "discharge_destination": "Published field discharge_destination.",
    "source_encounter_id": "Published field source_encounter_id.",
    "cc_encounter_id": "Published field cc_encounter_id.",
    "cds_apc_id": "Published field cds_apc_id.",
    "record_status": "Published field record_status.",
    "record_status_effective_from": "Published field record_status_effective_from.",
    "record_status_effective_to": "Published field record_status_effective_to.",
    "confidentiality_code": "Published field confidentiality_code.",
    "vip_ind": "Published field vip_ind.",
    "withheld_identity_ind": "Published field withheld_identity_ind.",
    "fact_category": "Published field fact_category.",
    "source_feed": "Published field source_feed.",
    "load_batch_id": "Published field load_batch_id.",
    "source_update_timestamp": "Published field source_update_timestamp.",
    "loaded_at": "Published field loaded_at.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.critical_care_period"),
    comment=(
        "One CCMDS critical-care period per business key with deterministic version "
        "selection. Gold QC twin of the silver product: 1 columns are repaired or nulled, 1 "
        "rule(s) drop rows, 4 check(s) are advisory. Each rule states its reason in the "
        "pipeline notebook, and Lakeflow expectation metrics report what every rule matched "
        "on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_CRITICAL_CARE_PERIOD_MANDATORY_RULES)
@_expect_all(CLINICAL_CRITICAL_CARE_PERIOD_ADVISORY_RULES)
def gold_clinical_critical_care_period():
    """Quality-controlled twin of journey_clinical.critical_care_period."""
    df = _qc(
        "clinical_critical_care_period",
        CLINICAL_CRITICAL_CARE_PERIOD_SELECT,
        date_flags=["event_after_death_30d", "event_before_birth"],
    )
    return _with_comments(df, CLINICAL_CRITICAL_CARE_PERIOD_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.device ====

CLINICAL_DEVICE_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",
    "`event_datetime` AS `event_datetime`",
    "`event_end_datetime` AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`device_code` AS `device_code`",
    "`device_role` AS `device_role`",
    "`model_name` AS `model_name`",

    # 'UNKNOWN' is a placeholder the source writes when the value was not recorded; it is
    # not a code, so it is nulled rather than passed on as one. Hit 128 of 37,089 rows
    # (0.345%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`model_code` AS STRING))) = 'UNKNOWN' THEN NULL ELSE `model_code` END AS `model_code`",
    "`manufacturer` AS `manufacturer`",
    "`manufacturer_parent` AS `manufacturer_parent`",
    "`serial_number` AS `serial_number`",
    "`implanted_datetime` AS `implanted_datetime`",
    "`implant_date_quality` AS `implant_date_quality`",
    "`explanted_ind` AS `explanted_ind`",
    "`lead_chamber` AS `lead_chamber`",
    "`lead_location` AS `lead_location`",
    "`pocket_site` AS `pocket_site`",
    "`status_display` AS `status_display`",
    "`device_mapping_status` AS `device_mapping_status`",

    # An empty or whitespace-only string is how the source writes 'nothing here'. It reads
    # as a value in a query and is not one, so it is nulled. Hit 2,241 of 37,089 rows
    # (6.04%) when profiled on 2026-08-24.
    "CASE WHEN TRIM(CAST(`comment_text` AS STRING)) = '' THEN NULL ELSE `comment_text` END AS `comment_text`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
    "`event_before_birth` AS `event_before_birth`",
    "`event_after_death_30d` AS `event_after_death_30d`",
]

CLINICAL_DEVICE_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 37,089 rows of 37,089 that
    # are current and attributable; identity_status = 'resolved' keeps the 37,063 rows of
    # 37,089 that are current and attributable. Superseded versions and rows whose identity
    # was never resolved are not research data, and a consumer who wants them has silver.
    "research_surface": "(identity_status = 'resolved') AND (record_status = 'active')",
}

CLINICAL_DEVICE_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 26 of 37,089 at the profile.
    "gold.clinical.device.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 0 of 37,089 at the profile.
    "gold.clinical.device.record_status.default_view_active": "record_status = 'active'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 364 of 37,089 rows (0.981%) when profiled on 2026-08-24.
    "gold.clinical.device.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 79 of 37,089 rows (0.213%) when profiled on 2026-08-24.
    "gold.clinical.device.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",
}

CLINICAL_DEVICE_COLUMN_COMMENTS = {
    "patient_event_id": "Stable device event identifier.",
    "fact_row_id": "Storage-row identifier equal to patient_event_id.",
    "subject_key": "Always-populated peppered subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference when available.",
    "event_datetime": "Sentinel-cleaned implantation timestamp.",
    "event_end_datetime": "Device lifecycle end timestamp when available.",
    "source_coding_system": "MediConnect device-type coding system.",
    "source_code": "Device type with type/model display fallback for unknown type zero.",
    "source_display": "Device type or model display.",
    "device_code": "Source and mapped device codings.",
    "device_role": "Device role.",
    "model_name": "Device model name.",
    "model_code": "Device model code.",
    "manufacturer": "Device manufacturer.",
    "manufacturer_parent": "Parent manufacturer.",
    "serial_number": "Device serial number.",
    "implanted_datetime": "Source implantation timestamp.",
    "implant_date_quality": "Implant-date quality class.",
    "explanted_ind": "Inferred explant indicator; not independently confirmed.",
    "lead_chamber": "Lead chamber.",
    "lead_location": "Lead location.",
    "pocket_site": "Device pocket site.",
    "status_display": "Source device status display.",
    "device_mapping_status": "Device terminology mapping status.",
    "comment_text":
        "Source device comment. Gold QC transform rules: gold.clinical.device.comment_text.empty_string.",
    "record_status": "Normalized source-record lifecycle.",
    "record_status_effective_from": "Lifecycle start.",
    "record_status_effective_to": "Source absence timestamp.",
    "confidentiality_code": "Security classification when supplied.",
    "vip_ind": "VIP indicator when supplied.",
    "withheld_identity_ind": "Withheld-identity indicator.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.device"),
    comment=(
        "One frozen MediConnect implanted-device registry row with source-type and mapped "
        "SNOMED device coding. Gold QC twin of the silver product: 2 columns are repaired or "
        "nulled, 1 rule(s) drop rows, 4 check(s) are advisory. Each rule states its reason in "
        "the pipeline notebook, and Lakeflow expectation metrics report what every rule "
        "matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_DEVICE_MANDATORY_RULES)
@_expect_all(CLINICAL_DEVICE_ADVISORY_RULES)
def gold_clinical_device():
    """Quality-controlled twin of journey_clinical.device."""
    df = _qc("clinical_device", CLINICAL_DEVICE_SELECT)
    return _with_comments(df, CLINICAL_DEVICE_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.drug_expenditure ====

CLINICAL_DRUG_EXPENDITURE_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",
    "`event_datetime` AS `event_datetime`",
    "`event_end_datetime` AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`transaction_id` AS `transaction_id`",
    "`financial_year` AS `financial_year`",
    "`financial_month` AS `financial_month`",
    "`reporting_year` AS `reporting_year`",
    "`reporting_month` AS `reporting_month`",
    "`provider_org_cd` AS `provider_org_cd`",
    "`site_cd` AS `site_cd`",
    "`site_name` AS `site_name`",
    "`specialty_cd` AS `specialty_cd`",
    "`consultant_cd` AS `consultant_cd`",
    "`patient_type` AS `patient_type`",
    "`pod_cd` AS `pod_cd`",
    "`chargeable_item` AS `chargeable_item`",
    "`additional_info` AS `additional_info`",
    "`dmd_raw` AS `dmd_raw`",
    "`dmd_code` AS `dmd_code`",
    "`dmd_concept_id` AS `dmd_concept_id`",
    "`dmd_concept_name` AS `dmd_concept_name`",
    "`drug_standard_concept_id` AS `drug_standard_concept_id`",
    "`drug_standard_concept_name` AS `drug_standard_concept_name`",
    "`dmd_mapping_status` AS `dmd_mapping_status`",
    "`dmd_taxonomy_cd` AS `dmd_taxonomy_cd`",
    "`route_of_administration` AS `route_of_administration`",
    "`strength` AS `strength`",
    "`volume` AS `volume`",
    "`pack_size` AS `pack_size`",

    # A negative amount is not possible for a dose, quantity or score, and nothing in the
    # row says what the intended magnitude was. Hit 5 of 1,167,749 rows (0.000428%) when
    # profiled on 2026-08-24.
    "CASE WHEN `quantity` < 0 THEN NULL ELSE `quantity` END AS `quantity`",
    "`unit_of_measure` AS `unit_of_measure`",
    "`dispensing_route` AS `dispensing_route`",
    "`dispensing_location` AS `dispensing_location`",
    "`indication` AS `indication`",
    "`funding_reference` AS `funding_reference`",
    "`hcdr_category_cd` AS `hcdr_category_cd`",
    "`hcdr_category_desc` AS `hcdr_category_desc`",
    "`ccg_residence_cd` AS `ccg_residence_cd`",
    "`ccg_gp_cd` AS `ccg_gp_cd`",
    "`commissioner_cd` AS `commissioner_cd`",
    "`commissioner_type` AS `commissioner_type`",
    "`service_line` AS `service_line`",
    "`service_category_cd` AS `service_category_cd`",
    "`unit_price_supplier` AS `unit_price_supplier`",
    "`unit_price_commissioner` AS `unit_price_commissioner`",
    "`vat` AS `vat`",
    "`vat_cd` AS `vat_cd`",
    "`income` AS `income`",
    "`cost` AS `cost`",
    "`margin` AS `margin`",
    "`lloyds_dispensing_fee` AS `lloyds_dispensing_fee`",
    "`production_fee` AS `production_fee`",
    "`fixed_patient_income` AS `fixed_patient_income`",
    "`cost_centre_desc` AS `cost_centre_desc`",
    "`drug_feed` AS `drug_feed`",
    "`data_set` AS `data_set`",
    "`drug_category` AS `drug_category`",
    "`ledger_cd` AS `ledger_cd`",
    "`exclusion_flag` AS `exclusion_flag`",
    "`exclusion_reason` AS `exclusion_reason`",
    "`ledger_lv3_cd` AS `ledger_lv3_cd`",
    "`ledger_lv3_desc` AS `ledger_lv3_desc`",
    "`ledger_lv6_cd` AS `ledger_lv6_cd`",
    "`ledger_lv6_desc` AS `ledger_lv6_desc`",
    "`ledger_lv7_cd` AS `ledger_lv7_cd`",
    "`ledger_lv7_desc` AS `ledger_lv7_desc`",
    "`ledger_lv9_cd` AS `ledger_lv9_cd`",
    "`ledger_lv9_desc` AS `ledger_lv9_desc`",
    "`slr_cd` AS `slr_cd`",
    "`diabetic_flag` AS `diabetic_flag`",
    "`imcoe_flag` AS `imcoe_flag`",
    "`source_duplicate_count` AS `source_duplicate_count`",
    "`person_link_method` AS `person_link_method`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`fact_category` AS `fact_category`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
]

CLINICAL_DRUG_EXPENDITURE_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 1,167,749 rows of 1,167,749
    # that are current and attributable; identity_status = 'resolved' keeps the 1,167,731
    # rows of 1,167,749 that are current and attributable. Superseded versions and rows
    # whose identity was never resolved are not research data, and a consumer who wants them
    # has silver.
    "research_surface": "(identity_status = 'resolved') AND (record_status = 'active')",
}

CLINICAL_DRUG_EXPENDITURE_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 18 of 1,167,749 at the profile.
    "gold.clinical.drug_expenditure.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 0 of 1,167,749 at the profile.
    "gold.clinical.drug_expenditure.record_status.default_view_active":
        "record_status = 'active'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 3,486 of 1,167,749 rows (0.299%) when profiled on 2026-08-24.
    "gold.clinical.drug_expenditure.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 12 of 1,167,749 rows (0.00103%) when profiled on 2026-08-24.
    "gold.clinical.drug_expenditure.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",
}

CLINICAL_DRUG_EXPENDITURE_COLUMN_COMMENTS = {
    "patient_event_id": "Stable row identifier.",
    "fact_row_id": "Stable row identifier.",
    "subject_key": "Published field.",
    "subject_id_system": "Published field.",
    "person_id": "Published field.",
    "identity_status": "Published field.",
    "encounter_id": "Published field.",
    "event_datetime": "Published field.",
    "event_end_datetime": "Published field.",
    "source_coding_system": "Published field.",
    "source_code": "Published field.",
    "source_display": "Published field.",
    "transaction_id": "Published field.",
    "financial_year": "Published field.",
    "financial_month": "Published field.",
    "reporting_year": "Published field.",
    "reporting_month": "Published field.",
    "provider_org_cd": "Published field.",
    "site_cd": "Published field.",
    "site_name": "Published field.",
    "specialty_cd": "Published field.",
    "consultant_cd": "Published field.",
    "patient_type": "Published field.",
    "pod_cd": "Published field.",
    "chargeable_item": "Published field.",
    "additional_info": "Published field.",
    "dmd_raw": "Published field.",
    "dmd_code": "Published field.",
    "dmd_concept_id": "Published field.",
    "dmd_concept_name": "Published field.",
    "drug_standard_concept_id": "Published field.",
    "drug_standard_concept_name": "Published field.",
    "dmd_mapping_status": "Published field.",
    "dmd_taxonomy_cd": "Published field.",
    "route_of_administration": "Published field.",
    "strength": "Published field.",
    "volume": "Published field.",
    "pack_size": "Published field.",
    "quantity": "Published field.",
    "unit_of_measure": "Published field.",
    "dispensing_route": "Published field.",
    "dispensing_location": "Published field.",
    "indication": "Published field.",
    "funding_reference": "Published field.",
    "hcdr_category_cd": "Published field.",
    "hcdr_category_desc": "Published field.",
    "ccg_residence_cd": "Published field.",
    "ccg_gp_cd": "Published field.",
    "commissioner_cd": "Published field.",
    "commissioner_type": "Published field.",
    "service_line": "Published field.",
    "service_category_cd": "Published field.",
    "unit_price_supplier": "Published field.",
    "unit_price_commissioner": "Published field.",
    "vat": "Published field.",
    "vat_cd": "Published field.",
    "income": "Published field.",
    "cost": "Published field.",
    "margin": "Published field.",
    "lloyds_dispensing_fee": "Published field.",
    "production_fee": "Published field.",
    "fixed_patient_income": "Published field.",
    "cost_centre_desc": "Published field.",
    "drug_feed": "Published field.",
    "data_set": "Published field.",
    "drug_category": "Published field.",
    "ledger_cd": "Published field.",
    "exclusion_flag": "Published field.",
    "exclusion_reason": "Published field.",
    "ledger_lv3_cd": "Published field.",
    "ledger_lv3_desc": "Published field.",
    "ledger_lv6_cd": "Published field.",
    "ledger_lv6_desc": "Published field.",
    "ledger_lv7_cd": "Published field.",
    "ledger_lv7_desc": "Published field.",
    "ledger_lv9_cd": "Published field.",
    "ledger_lv9_desc": "Published field.",
    "slr_cd": "Published field.",
    "diabetic_flag": "Published field.",
    "imcoe_flag": "Published field.",
    "source_duplicate_count": "Published field.",
    "person_link_method": "Published field.",
    "record_status": "Published field.",
    "record_status_effective_from": "Published field.",
    "record_status_effective_to": "Published field.",
    "confidentiality_code": "Published field.",
    "vip_ind": "Published field.",
    "withheld_identity_ind": "Published field.",
    "fact_category": "Published field.",
    "source_feed": "Published field.",
    "load_batch_id": "Published field.",
    "source_update_timestamp": "Published field.",
    "loaded_at": "Published field.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.drug_expenditure"),
    comment=(
        "One HCD drug-expenditure row keyed by ROW_HASH. Gold QC twin of the silver product: "
        "1 columns are repaired or nulled, 1 rule(s) drop rows, 4 check(s) are advisory. Each "
        "rule states its reason in the pipeline notebook, and Lakeflow expectation metrics "
        "report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_DRUG_EXPENDITURE_MANDATORY_RULES)
@_expect_all(CLINICAL_DRUG_EXPENDITURE_ADVISORY_RULES)
def gold_clinical_drug_expenditure():
    """Quality-controlled twin of journey_clinical.drug_expenditure."""
    df = _qc(
        "clinical_drug_expenditure",
        CLINICAL_DRUG_EXPENDITURE_SELECT,
        date_flags=["event_after_death_30d", "event_before_birth"],
    )
    return _with_comments(df, CLINICAL_DRUG_EXPENDITURE_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.elective_access_entry ====

CLINICAL_ELECTIVE_ACCESS_ENTRY_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",
    "`event_datetime` AS `event_datetime`",
    "`event_end_datetime` AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`waiting_list_oid` AS `waiting_list_oid`",
    "`source_system_oid` AS `source_system_oid`",
    "`pathway_oid` AS `pathway_oid`",
    "`referral_oid` AS `referral_oid`",
    "`patient_oid` AS `patient_oid`",
    "`waiting_list_id` AS `waiting_list_id`",
    "`legacy_waiting_list_id` AS `legacy_waiting_list_id`",
    "`waiting_list_name` AS `waiting_list_name`",
    "`waiting_list_code` AS `waiting_list_code`",
    "`department` AS `department`",
    "`division` AS `division`",
    "`business_unit` AS `business_unit`",
    "`clinical_priority_code` AS `clinical_priority_code`",
    "`clinical_priority_recorded_datetime` AS `clinical_priority_recorded_datetime`",
    "`site_rvid` AS `site_rvid`",
    "`treatment_function_rvid` AS `treatment_function_rvid`",
    "`admin_category_rvid` AS `admin_category_rvid`",
    "`intended_management_rvid` AS `intended_management_rvid`",
    "`admit_method_rvid` AS `admit_method_rvid`",
    "`priority_rvid` AS `priority_rvid`",
    "`status_rvid` AS `status_rvid`",
    "`elective_admission_type_rvid` AS `elective_admission_type_rvid`",
    "`encounter_type_rvid` AS `encounter_type_rvid`",
    "`removal_reason_rvid` AS `removal_reason_rvid`",
    "`division_rvid` AS `division_rvid`",
    "`tci_location_rvid` AS `tci_location_rvid`",
    "`admit_offer_outcome_rvid` AS `admit_offer_outcome_rvid`",
    "`lead_clinician_prid` AS `lead_clinician_prid`",
    "`status_reason` AS `status_reason`",
    "`status_change_datetime` AS `status_change_datetime`",
    "`decided_to_admit_datetime` AS `decided_to_admit_datetime`",
    "`tci_datetime` AS `tci_datetime`",
    "`tci_future_ind` AS `tci_future_ind`",
    "`tci_created_datetime` AS `tci_created_datetime`",
    "`guaranteed_activity_datetime` AS `guaranteed_activity_datetime`",
    "`actual_guaranteed_activity_datetime` AS `actual_guaranteed_activity_datetime`",
    "`planned_datetime` AS `planned_datetime`",
    "`earliest_reasonable_offer_datetime` AS `earliest_reasonable_offer_datetime`",
    "`admit_datetime` AS `admit_datetime`",
    "`comments` AS `comments`",
    "`active_ind` AS `active_ind`",
    "`created_datetime` AS `created_datetime`",
    "`created_by_prid` AS `created_by_prid`",
    "`modified_datetime` AS `modified_datetime`",
    "`modified_by_prid` AS `modified_by_prid`",
    "`person_link_status` AS `person_link_status`",
    "`person_link_method` AS `person_link_method`",
    "`identifier_link_status` AS `identifier_link_status`",
    "`linkage_historical_fallback_ind` AS `linkage_historical_fallback_ind`",
    "`linkage_fallback_conflict_ind` AS `linkage_fallback_conflict_ind`",
    "`nhs_number_valid_ind` AS `nhs_number_valid_ind`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`fact_category` AS `fact_category`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
]

CLINICAL_ELECTIVE_ACCESS_ENTRY_MANDATORY_RULES = {
    # The research surface. identity_status = 'resolved' keeps the 3,149,964 rows of
    # 3,150,053 that are current and attributable. Superseded versions and rows whose
    # identity was never resolved are not research data, and a consumer who wants them has
    # silver.
    "research_surface": "(identity_status = 'resolved')",
}

CLINICAL_ELECTIVE_ACCESS_ENTRY_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 89 of 3,150,053 at the profile.
    "gold.clinical.elective_access_entry.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 20,927 of 3,150,053 rows (0.664%) when profiled on 2026-08-24.
    "gold.clinical.elective_access_entry.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 4 of 3,150,053 rows (0.000127%) when profiled on 2026-08-24.
    "gold.clinical.elective_access_entry.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",

    # Left as a warning because it fires on 488,498 of 3,150,053 rows (15.5%) when profiled
    # on 2026-08-24 -- at that rate the rule's assumption about what event_datetime and
    # event_end_datetime mean is the thing in doubt, not the data. The inverted gaps are
    # mostly minutes, which reads as two clocks rather than two events in the wrong order.
    "gold.clinical.elective_access_entry.table.ordering_violation_event_datetime_event_end_datetime":
        "NOT COALESCE((`event_datetime` IS NOT NULL AND `event_end_datetime` IS NOT NULL AND `event_datetime` > `event_end_datetime`), FALSE)",
}

CLINICAL_ELECTIVE_ACCESS_ENTRY_COLUMN_COMMENTS = {
    "patient_event_id": "Stable row identifier.",
    "fact_row_id": "Stable row identifier.",
    "subject_key": "Published field.",
    "subject_id_system": "Published field.",
    "person_id": "Published field.",
    "identity_status": "Published field.",
    "encounter_id": "Published field.",
    "event_datetime": "Published field.",
    "event_end_datetime": "Published field.",
    "source_coding_system": "Published field.",
    "source_code": "Published field.",
    "source_display": "Published field.",
    "waiting_list_oid": "Published field.",
    "source_system_oid": "Published field.",
    "pathway_oid": "Published field.",
    "referral_oid": "Published field.",
    "patient_oid": "Published field.",
    "waiting_list_id": "Published field.",
    "legacy_waiting_list_id": "Published field.",
    "waiting_list_name": "Published field.",
    "waiting_list_code": "Published field.",
    "department": "Published field.",
    "division": "Published field.",
    "business_unit": "Published field.",
    "clinical_priority_code": "Published field.",
    "clinical_priority_recorded_datetime": "Published field.",
    "site_rvid": "Published field.",
    "treatment_function_rvid": "Published field.",
    "admin_category_rvid": "Published field.",
    "intended_management_rvid": "Published field.",
    "admit_method_rvid": "Published field.",
    "priority_rvid": "Published field.",
    "status_rvid": "Published field.",
    "elective_admission_type_rvid": "Published field.",
    "encounter_type_rvid": "Published field.",
    "removal_reason_rvid": "Published field.",
    "division_rvid": "Published field.",
    "tci_location_rvid": "Published field.",
    "admit_offer_outcome_rvid": "Published field.",
    "lead_clinician_prid": "Published field.",
    "status_reason": "Published field.",
    "status_change_datetime": "Published field.",
    "decided_to_admit_datetime": "Published field.",
    "tci_datetime": "Published field.",
    "tci_future_ind": "Published field.",
    "tci_created_datetime": "Published field.",
    "guaranteed_activity_datetime": "Published field.",
    "actual_guaranteed_activity_datetime": "Published field.",
    "planned_datetime": "Published field.",
    "earliest_reasonable_offer_datetime": "Published field.",
    "admit_datetime": "Published field.",
    "comments": "Direct identifier published and IG-governed at serve time",
    "active_ind": "Published field.",
    "created_datetime": "Published field.",
    "created_by_prid": "Published field.",
    "modified_datetime": "Published field.",
    "modified_by_prid": "Published field.",
    "person_link_status": "Published field.",
    "person_link_method": "Published field.",
    "identifier_link_status": "Published field.",
    "linkage_historical_fallback_ind": "Published field.",
    "linkage_fallback_conflict_ind": "Published field.",
    "nhs_number_valid_ind": "Published field.",
    "record_status": "Published field.",
    "record_status_effective_from": "Published field.",
    "record_status_effective_to": "Published field.",
    "confidentiality_code": "Published field.",
    "vip_ind": "Published field.",
    "withheld_identity_ind": "Published field.",
    "fact_category": "Published field.",
    "source_feed": "Published field.",
    "load_batch_id": "Published field.",
    "source_update_timestamp": "Published field.",
    "loaded_at": "Published field.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.elective_access_entry"),
    comment=(
        "One LUNA elective-access entry. Gold QC twin of the silver product: 1 columns are "
        "repaired or nulled, 1 rule(s) drop rows, 4 check(s) are advisory. Each rule states "
        "its reason in the pipeline notebook, and Lakeflow expectation metrics report what "
        "every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_ELECTIVE_ACCESS_ENTRY_MANDATORY_RULES)
@_expect_all(CLINICAL_ELECTIVE_ACCESS_ENTRY_ADVISORY_RULES)
def gold_clinical_elective_access_entry():
    """Quality-controlled twin of journey_clinical.elective_access_entry."""
    # 1 rows point at a person_id the spine does not have. The pointer is nulled so it
    # cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    df = _qc(
        "clinical_elective_access_entry",
        CLINICAL_ELECTIVE_ACCESS_ENTRY_SELECT,
        fk_columns=["person_id"],
        date_flags=["event_after_death_30d", "event_before_birth"],
    )
    return _with_comments(df, CLINICAL_ELECTIVE_ACCESS_ENTRY_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.endoscopy_finding ====

CLINICAL_ENDOSCOPY_FINDING_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",
    "`event_datetime` AS `event_datetime`",
    "`event_end_datetime` AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`finding_code` AS `finding_code`",
    "`endoscopy_exam_event_id` AS `endoscopy_exam_event_id`",
    "`section_id` AS `section_id`",
    "`subsection_id` AS `subsection_id`",
    "`parent_term_id` AS `parent_term_id`",
    "`display_order` AS `display_order`",
    "`confirmed_ind` AS `confirmed_ind`",
    "`text_changed_ind` AS `text_changed_ind`",
    "`free_text_ind` AS `free_text_ind`",
    "`term_mapping_status` AS `term_mapping_status`",
    "`person_link_status` AS `person_link_status`",
    "`authored_datetime` AS `authored_datetime`",
    "`event_time_source` AS `event_time_source`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
]

CLINICAL_ENDOSCOPY_FINDING_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 9,220,706 rows of 9,220,706
    # that are current and attributable. Superseded versions and rows whose identity was
    # never resolved are not research data, and a consumer who wants them has silver.
    "research_surface": "(record_status = 'active')",
}

CLINICAL_ENDOSCOPY_FINDING_ADVISORY_RULES = {
    # Warned rather than filtered on identity_status = 'resolved': Endobase has no sound
    # person linkage; the mandatory predicate would empty the 9,220,706-row product.
    "gold.clinical.endoscopy_finding.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 0 of 9,220,706 at the profile.
    "gold.clinical.endoscopy_finding.record_status.default_view_active":
        "record_status = 'active'",
}

CLINICAL_ENDOSCOPY_FINDING_COLUMN_COMMENTS = {
    "patient_event_id": "Stable endoscopy-finding identifier.",
    "fact_row_id": "Storage-row identifier equal to patient_event_id.",
    "subject_key": "Always-populated peppered subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Parent exam encounter reference.",
    "event_datetime": "Parent exam time",
    "event_end_datetime": "Finding end timestamp when supplied.",
    "source_coding_system": "DGVS term coding system.",
    "source_code": "DGVS term identifier with term-text fallback.",
    "source_display": "Endobase term text.",
    "finding_code": "Source DGVS and mapped SNOMED/OMOP codings.",
    "endoscopy_exam_event_id": "Parent Endobase procedure event identifier.",
    "section_id": "Report section identifier.",
    "subsection_id": "Report subsection identifier.",
    "parent_term_id": "Parent term identifier.",
    "display_order": "Source display order.",
    "confirmed_ind": "Finding confirmation indicator.",
    "text_changed_ind": "Source text-changed indicator.",
    "free_text_ind": "Free-text route indicator.",
    "term_mapping_status": "C4 term-mapping status.",
    "person_link_status": "Bronze person-link status.",
    "authored_datetime": "Report-authoring timestamp.",
    "event_time_source": "Whether event time came from the exam or authored fallback.",
    "record_status": "Normalized source-record lifecycle.",
    "record_status_effective_from": "Lifecycle start.",
    "record_status_effective_to": "Source absence timestamp.",
    "confidentiality_code": "Security classification when supplied.",
    "vip_ind": "VIP indicator when supplied.",
    "withheld_identity_ind": "Withheld-identity indicator.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
}

@dp.materialized_view(
    name=_n("gold_clinical.endoscopy_finding"),
    comment=(
        "One coded Endobase exam term with parent-exam clinical time and C4 mapping columns "
        "wired for later application. Gold QC twin of the silver product: 0 columns are "
        "repaired or nulled, 1 rule(s) drop rows, 2 check(s) are advisory. Each rule states "
        "its reason in the pipeline notebook, and Lakeflow expectation metrics report what "
        "every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_ENDOSCOPY_FINDING_MANDATORY_RULES)
@_expect_all(CLINICAL_ENDOSCOPY_FINDING_ADVISORY_RULES)
def gold_clinical_endoscopy_finding():
    """Quality-controlled twin of journey_clinical.endoscopy_finding."""
    df = _qc("clinical_endoscopy_finding", CLINICAL_ENDOSCOPY_FINDING_SELECT)
    return _with_comments(df, CLINICAL_ENDOSCOPY_FINDING_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.family_history ====

CLINICAL_FAMILY_HISTORY_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",
    "`event_datetime` AS `event_datetime`",
    "`event_end_datetime` AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`condition_code` AS `condition_code`",
    "`relationship_code` AS `relationship_code`",
    "`relationship_display` AS `relationship_display`",
    "`relationship_type_code` AS `relationship_type_code`",
    "`relationship_type_display` AS `relationship_type_display`",
    "`onset_age` AS `onset_age`",
    "`onset_age_unit` AS `onset_age_unit`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 10,578 of 10,640 rows (99.4%)
    # when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`severity_code` AS STRING))) = '0' THEN NULL ELSE `severity_code` END AS `severity_code`",
    "`severity_display` AS `severity_display`",
    "`source_lifecycle_status` AS `source_lifecycle_status`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",

    # record_status_effective_to cannot precede record_status_effective_from. The start is
    # the better-attested of the two, so the end is what goes and the row keeps its
    # record_status_effective_from. Hit 55 of 10,640 rows (0.517%) when profiled on
    # 2026-08-24.
    "CASE WHEN `record_status_effective_from` IS NOT NULL AND `record_status_effective_to` IS NOT NULL AND `record_status_effective_from` > `record_status_effective_to` THEN NULL ELSE `record_status_effective_to` END AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`asserter_practitioner_id` AS `asserter_practitioner_id`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
    "`event_after_death_30d` AS `event_after_death_30d`",
]

CLINICAL_FAMILY_HISTORY_MANDATORY_RULES = {
    # The research surface. identity_status = 'resolved' keeps the 10,640 rows of 10,640
    # that are current and attributable. Superseded versions and rows whose identity was
    # never resolved are not research data, and a consumer who wants them has silver.
    "research_surface": "(identity_status = 'resolved')",
}

CLINICAL_FAMILY_HISTORY_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 0 of 10,640 at the profile.
    "gold.clinical.family_history.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 21 of 10,640 rows (0.197%) when profiled on 2026-08-24.
    "gold.clinical.family_history.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",
}

CLINICAL_FAMILY_HISTORY_COLUMN_COMMENTS = {
    "patient_event_id": "Stable product-wide event identifier.",
    "fact_row_id": "Storage-row identifier; equal to patient_event_id for this fact.",
    "subject_key": "Always-populated peppered subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier when available.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Best-available encounter reference.",
    "event_datetime": "Source family-history effective start.",
    "event_end_datetime": "Source effective end when the assertion is no longer current.",
    "source_coding_system": "Verbatim source code system.",
    "source_code": "Source condition code",
    "source_display": "Verbatim source condition display.",
    "condition_code":
        "Source and mapped condition codings as a one-level CodeableConcept VARIANT.",
    "relationship_code": "Source family relationship code.",
    "relationship_display": "Source family relationship display.",
    "relationship_type_code": "Source relationship record type code.",
    "relationship_type_display": "Source relationship record type display.",
    "onset_age": "Source recorded age at onset.",
    "onset_age_unit": "Unit for recorded onset age.",
    "severity_code": "Source severity code.",
    "severity_display": "Source severity display.",
    "source_lifecycle_status": "Verbatim source life-cycle status.",
    "record_status": "Normalized silver lifecycle status.",
    "record_status_effective_from": "Source status effective start.",
    "record_status_effective_to": "Source status effective end.",
    "confidentiality_code": "Security classification when supplied.",
    "vip_ind": "VIP indicator when supplied.",
    "withheld_identity_ind": "Withheld-identity indicator when supplied.",
    "asserter_practitioner_id": "Asserting practitioner reference when supplied.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Bronze source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.family_history"),
    comment=(
        "One family-history assertion with the standard event block and retained source "
        "lifecycle. Gold QC twin of the silver product: 2 columns are repaired or nulled, 1 "
        "rule(s) drop rows, 2 check(s) are advisory. Each rule states its reason in the "
        "pipeline notebook, and Lakeflow expectation metrics report what every rule matched "
        "on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_FAMILY_HISTORY_MANDATORY_RULES)
@_expect_all(CLINICAL_FAMILY_HISTORY_ADVISORY_RULES)
def gold_clinical_family_history():
    """Quality-controlled twin of journey_clinical.family_history."""
    df = _qc("clinical_family_history", CLINICAL_FAMILY_HISTORY_SELECT)
    return _with_comments(df, CLINICAL_FAMILY_HISTORY_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.form ====

CLINICAL_FORM_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",

    # 983 rows point at a person_id the spine does not have. The pointer is nulled so it
    # cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    "CASE WHEN NOT `person_id_resolved` THEN NULL ELSE `person_id` END AS `person_id`",
    "`identity_status` AS `identity_status`",

    # 13,944 rows point at a encounter_id the spine does not have. The pointer is nulled so
    # it cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    "CASE WHEN NOT `encounter_id_resolved` THEN NULL ELSE `encounter_id` END AS `encounter_id`",

    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 45 of 55,393,916 rows (8.12e-05%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `event_datetime` END AS `event_datetime`",

    # event_end_datetime cannot precede event_datetime. The start is the better-attested of
    # the two, so the end is what goes and the row keeps its event_datetime. Hit 18,824 of
    # 55,393,916 rows (0.034%) when profiled on 2026-08-24.
    "CASE WHEN `event_datetime` IS NOT NULL AND `event_end_datetime` IS NOT NULL AND `event_datetime` > `event_end_datetime` THEN NULL ELSE `event_end_datetime` END AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`form_type_code` AS `form_type_code`",
    "`form_type_display` AS `form_type_display`",
    "`form_status_code` AS `form_status_code`",
    "`form_status_display` AS `form_status_display`",

    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 45 of 55,393,916 rows (8.12e-05%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`authored_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `authored_datetime` END AS `authored_datetime`",
    "`completed_datetime` AS `completed_datetime`",
    "`performed_practitioner_id` AS `performed_practitioner_id`",
    "`organization_id` AS `organization_id`",
    "`responses` AS `responses`",
    "`response_row_count` AS `response_row_count`",
    "`active_response_row_count` AS `active_response_row_count`",
    "`empty_response_row_count` AS `empty_response_row_count`",
    "`invalid_response_row_count` AS `invalid_response_row_count`",
    "`matched_response_row_count` AS `matched_response_row_count`",
    "`unmatched_response_row_count` AS `unmatched_response_row_count`",
    "`context_conflict_ind` AS `context_conflict_ind`",
    "`context_quarantined_ind` AS `context_quarantined_ind`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",

    # record_status_effective_to cannot precede record_status_effective_from. The start is
    # the better-attested of the two, so the end is what goes and the row keeps its
    # record_status_effective_from. Hit 34 of 55,393,916 rows (6.14e-05%) when profiled on
    # 2026-08-24.
    "CASE WHEN `record_status_effective_from` IS NOT NULL AND `record_status_effective_to` IS NOT NULL AND `record_status_effective_from` > `record_status_effective_to` THEN NULL ELSE `record_status_effective_to` END AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",

    # Silver derived this flag by comparing event_datetime with the person's own dates,
    # before the rules above corrected event_datetime. On the rows where event_datetime
    # changed, the flag describes a timestamp gold no longer publishes. This product carries
    # a VARIANT column, which rules out the spine join gold would need to recompute the
    # flag, so it is nulled where the value beneath it moved rather than left asserting
    # something stale.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `event_before_birth` END AS `event_before_birth`",

    # Silver derived this flag by comparing event_datetime with the person's own dates,
    # before the rules above corrected event_datetime. On the rows where event_datetime
    # changed, the flag describes a timestamp gold no longer publishes. This product carries
    # a VARIANT column, which rules out the spine join gold would need to recompute the
    # flag, so it is nulled where the value beneath it moved rather than left asserting
    # something stale.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `event_after_death_30d` END AS `event_after_death_30d`",
]

CLINICAL_FORM_MANDATORY_RULES = {
    # The research surface. identity_status = 'resolved' keeps the 55,387,704 rows of
    # 55,393,916 that are current and attributable. Superseded versions and rows whose
    # identity was never resolved are not research data, and a consumer who wants them has
    # silver.
    "research_surface": "(identity_status = 'resolved')",
}

CLINICAL_FORM_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 6,212 of 55,393,916 at the profile.
    "gold.clinical.form.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # This bounds a period of validity, and a future end is exactly how the source says a
    # record is still current -- nulling it would assert the record is valid forever, which
    # is a stronger and worse claim than the one being corrected. Seen on 45 of 55,393,916
    # rows (8.12e-05%) when profiled on 2026-08-24.
    "gold.clinical.form.record_status_effective_from.future_owner":
        "NOT COALESCE((CAST(`record_status_effective_from` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 8,310 of 55,393,916 rows (0.015%) when profiled on 2026-08-24.
    "gold.clinical.form.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 12,285 of 55,393,916 rows (0.0222%) when profiled on 2026-08-24.
    "gold.clinical.form.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",
}

CLINICAL_FORM_COLUMN_COMMENTS = {
    "patient_event_id": "Stable form event identifier.",
    "fact_row_id": "Storage-row identifier equal to patient_event_id.",
    "subject_key": "Always-populated subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference.",
    "event_datetime": "First documented timestamp.",
    "event_end_datetime": "Last documented or performed timestamp.",
    "source_coding_system": "Source form coding system.",
    "source_code": "Source form reference identifier.",
    "source_display": "Source form description.",
    "form_type_code": "Source form type code.",
    "form_type_display": "Source form type display.",
    "form_status_code": "Source form status code.",
    "form_status_display": "Source form status display.",
    "authored_datetime": "Earliest form documentation timestamp.",
    "completed_datetime": "Latest documentation or performed timestamp.",
    "performed_practitioner_id": "Performing practitioner reference.",
    "organization_id": "Source organization reference.",
    "responses": "Ordered lossless form responses with typed values and mapping evidence.",
    "response_row_count": "Number of response rows grouped into the form.",
    "active_response_row_count": "Number of active response rows.",
    "empty_response_row_count": "Number of retained empty response rows.",
    "invalid_response_row_count": "Number of source-classified invalid response rows.",
    "matched_response_row_count": "Number of canonical matched response rows.",
    "unmatched_response_row_count": "Number of canonical unmatched response rows.",
    "context_conflict_ind": "Whether source form context conflicts were detected.",
    "context_quarantined_ind": "Whether source form context was quarantined.",
    "record_status": "Normalized form lifecycle.",
    "record_status_effective_from": "Form lifecycle start.",
    "record_status_effective_to": "Form lifecycle end when retained inactive.",
    "confidentiality_code": "Security classification when supplied.",
    "vip_ind": "VIP indicator when supplied.",
    "withheld_identity_ind": "Withheld identity indicator.",
    "source_feed": "Registered owning feed.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Latest native source update timestamp.",
    "loaded_at": "Latest contributing bronze load timestamp.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.form"),
    comment=(
        "One completed or retained PowerForm instance with ordered response content. Gold QC "
        "twin of the silver product: 8 columns are repaired or nulled, 1 rule(s) drop rows, 4 "
        "check(s) are advisory. Each rule states its reason in the pipeline notebook, and "
        "Lakeflow expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_FORM_MANDATORY_RULES)
@_expect_all(CLINICAL_FORM_ADVISORY_RULES)
def gold_clinical_form():
    """Quality-controlled twin of journey_clinical.form."""
    df = _qc("clinical_form", CLINICAL_FORM_SELECT)
    return _with_comments(df, CLINICAL_FORM_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.genomic_result ====

CLINICAL_GENOMIC_RESULT_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",
    "`event_datetime` AS `event_datetime`",
    "`event_end_datetime` AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`genomic_test_id` AS `genomic_test_id`",
    "`report_version_id` AS `report_version_id`",
    "`hgnc_id` AS `hgnc_id`",
    "`reported_gene_symbol` AS `reported_gene_symbol`",
    "`normalized_gene_symbol` AS `normalized_gene_symbol`",
    "`partner_hgnc_id` AS `partner_hgnc_id`",
    "`partner_gene_symbol` AS `partner_gene_symbol`",
    "`alteration_type` AS `alteration_type`",
    "`detection_status` AS `detection_status`",
    "`hgvs_c_raw` AS `hgvs_c_raw`",
    "`hgvs_c_parsed` AS `hgvs_c_parsed`",
    "`hgvs_p_raw` AS `hgvs_p_raw`",
    "`hgvs_p_parsed` AS `hgvs_p_parsed`",
    "`transcript` AS `transcript`",
    "`hgvs_validation_status` AS `hgvs_validation_status`",
    "`genome_build` AS `genome_build`",
    "`chromosome` AS `chromosome`",
    "`position_start` AS `position_start`",
    "`position_end` AS `position_end`",
    "`vaf_raw` AS `vaf_raw`",
    "`vaf` AS `vaf`",
    "`zygosity` AS `zygosity`",
    "`reported_classification` AS `reported_classification`",
    "`reported_tier` AS `reported_tier`",
    "`copy_number` AS `copy_number`",
    "`ratio_raw` AS `ratio_raw`",
    "`iscn_raw` AS `iscn_raw`",
    "`clinvar_concept_id` AS `clinvar_concept_id`",
    "`omop_genomic_concept_id` AS `omop_genomic_concept_id`",
    "`snomed_code` AS `snomed_code`",
    "`evidence_text` AS `evidence_text`",
    "`evidence_start` AS `evidence_start`",
    "`evidence_end` AS `evidence_end`",
    "`parser_profile_id` AS `parser_profile_id`",
    "`parser_version` AS `parser_version`",
    "`review_status` AS `review_status`",
    "`lifecycle_status` AS `lifecycle_status`",
    "`is_current` AS `is_current`",
    "`research_qi_only` AS `research_qi_only`",
    "`specimen_id` AS `specimen_id`",
    "`accession_identifier` AS `accession_identifier`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
]

CLINICAL_GENOMIC_RESULT_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 0 rows of 0 that are current
    # and attributable; identity_status = 'resolved' keeps the 0 rows of 0 that are current
    # and attributable. Superseded versions and rows whose identity was never resolved are
    # not research data, and a consumer who wants them has silver.
    "research_surface": "(identity_status = 'resolved') AND (record_status = 'active')",
}

CLINICAL_GENOMIC_RESULT_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 0 of 0 at the profile.
    "gold.clinical.genomic_result.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 0 of 0 at the profile.
    "gold.clinical.genomic_result.record_status.default_view_active":
        "record_status = 'active'",
}

CLINICAL_GENOMIC_RESULT_COLUMN_COMMENTS = {
    "patient_event_id": "Stable genomic-result event identifier.",
    "fact_row_id": "Storage-row identifier.",
    "subject_key": "Best available subject key.",
    "subject_id_system": "Subject-key system.",
    "person_id": "Resolved person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference; unavailable at this grain.",
    "event_datetime": "Report issue time with accession fallback.",
    "event_end_datetime": "Event end time.",
    "source_coding_system": "Gene-symbol or alteration coding namespace.",
    "source_code": "Gene",
    "source_display": "Finding display.",
    "genomic_test_id": "Parent genomic-test fact reference.",
    "report_version_id": "Source report-version identifier.",
    "hgnc_id": "Primary HGNC identifier.",
    "reported_gene_symbol": "Gene symbol as reported.",
    "normalized_gene_symbol": "Approved HGNC symbol.",
    "partner_hgnc_id": "Partner HGNC identifier.",
    "partner_gene_symbol": "Partner gene symbol.",
    "alteration_type": "Reported alteration type.",
    "detection_status": "Finding detection status.",
    "hgvs_c_raw": "Raw coding HGVS.",
    "hgvs_c_parsed": "Parsed coding HGVS.",
    "hgvs_p_raw": "Raw protein HGVS.",
    "hgvs_p_parsed": "Parsed protein HGVS.",
    "transcript": "Reported transcript.",
    "hgvs_validation_status": "HGVS validation state.",
    "genome_build": "Explicit genome build.",
    "chromosome": "Explicit chromosome.",
    "position_start": "Genomic start coordinate.",
    "position_end": "Genomic end coordinate.",
    "vaf_raw": "Raw variant allele frequency.",
    "vaf": "Parsed variant allele fraction.",
    "zygosity": "Reported zygosity.",
    "reported_classification": "Reported classification.",
    "reported_tier": "Reported tier.",
    "copy_number": "Reported copy number.",
    "ratio_raw": "Raw molecular ratio.",
    "iscn_raw": "Raw ISCN notation.",
    "clinvar_concept_id": "ClinVar concept identifier.",
    "omop_genomic_concept_id": "OMOP Genomic concept identifier.",
    "snomed_code": "SNOMED finding code.",
    "evidence_text": "Minimal narrative evidence span.",
    "evidence_start": "Evidence start offset.",
    "evidence_end": "Evidence end offset.",
    "parser_profile_id": "Parser profile.",
    "parser_version": "Parser implementation version.",
    "review_status": "Source finding review status.",
    "lifecycle_status": "Inherited report lifecycle.",
    "is_current": "Whether the source report version is current.",
    "research_qi_only": "Research/QI release flag.",
    "specimen_id": "Alias-resolved accession specimen reference.",
    "accession_identifier": "Alias-resolved accession identifier.",
    "record_status": "Normalized finding lifecycle.",
    "record_status_effective_from": "Status validity start.",
    "record_status_effective_to": "Status validity end.",
    "confidentiality_code": "Security code.",
    "vip_ind": "VIP indicator.",
    "withheld_identity_ind": "Withheld identity indicator.",
    "source_feed": "Owning feed.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze material-change timestamp.",
}

@dp.materialized_view(
    name=_n("gold_clinical.genomic_result"),
    comment=(
        "One reportable molecular or cytogenetic finding. Schema-first and empty_by_design on "
        "2026-08-18 because no detected assays exist upstream. Evidence text quotes report "
        "narrative and is ig_risk 4, ig_severity 2 with serve-time stripping only. Gold QC "
        "twin of the silver product: 0 columns are repaired or nulled, 1 rule(s) drop rows, 2 "
        "check(s) are advisory. Each rule states its reason in the pipeline notebook, and "
        "Lakeflow expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_GENOMIC_RESULT_MANDATORY_RULES)
@_expect_all(CLINICAL_GENOMIC_RESULT_ADVISORY_RULES)
def gold_clinical_genomic_result():
    """Quality-controlled twin of journey_clinical.genomic_result."""
    df = _qc("clinical_genomic_result", CLINICAL_GENOMIC_RESULT_SELECT)
    return _with_comments(df, CLINICAL_GENOMIC_RESULT_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.genomic_test ====

CLINICAL_GENOMIC_TEST_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",
    "`event_datetime` AS `event_datetime`",
    "`event_end_datetime` AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`assay_code` AS `assay_code`",
    "`assay_name` AS `assay_name`",
    "`method` AS `method`",
    "`analysis_context` AS `analysis_context`",
    "`overall_result_status` AS `overall_result_status`",
    "`panel_code` AS `panel_code`",
    "`panel_version` AS `panel_version`",
    "`panel_version_inferred` AS `panel_version_inferred`",
    "`parser_profile_id` AS `parser_profile_id`",
    "`report_version_id` AS `report_version_id`",
    "`pathology_report_id` AS `pathology_report_id`",
    "`specimen_id` AS `specimen_id`",
    "`accession_identifier` AS `accession_identifier`",
    "`test_snomed_code` AS `test_snomed_code`",
    "`test_loinc_code` AS `test_loinc_code`",
    "`test_omop_concept_id` AS `test_omop_concept_id`",
    "`is_current` AS `is_current`",
    "`research_qi_only` AS `research_qi_only`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
]

CLINICAL_GENOMIC_TEST_MANDATORY_RULES = {
    # The research surface. identity_status = 'resolved' keeps the 40,540 rows of 51,845
    # that are current and attributable. Superseded versions and rows whose identity was
    # never resolved are not research data, and a consumer who wants them has silver.
    "research_surface": "(identity_status = 'resolved')",
}

CLINICAL_GENOMIC_TEST_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 11,305 of 51,845 at the profile.
    "gold.clinical.genomic_test.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 282 of 51,845 rows (0.544%) when profiled on 2026-08-24.
    "gold.clinical.genomic_test.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",
}

CLINICAL_GENOMIC_TEST_COLUMN_COMMENTS = {
    "patient_event_id": "Stable genomic-test event identifier.",
    "fact_row_id": "Storage-row identifier.",
    "subject_key": "Best available subject key.",
    "subject_id_system": "Subject-key system.",
    "person_id": "Resolved person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference; unavailable at this grain.",
    "event_datetime": "Report issue time with accession fallback.",
    "event_end_datetime": "Event end time.",
    "source_coding_system": "Assay coding namespace.",
    "source_code": "Source assay code.",
    "source_display": "Source assay display.",
    "assay_code": "Source assay or report code.",
    "assay_name": "Resolved assay name.",
    "method": "Reported assay method.",
    "analysis_context": "Somatic or germline analysis context.",
    "overall_result_status": "Source assay-level result status.",
    "panel_code": "Governed panel code.",
    "panel_version": "Explicit or inferred panel version.",
    "panel_version_inferred": "Whether panel version was inferred.",
    "parser_profile_id": "Deterministic parser profile.",
    "report_version_id": "Parent report-version identifier.",
    "pathology_report_id": "Report-series fact reference.",
    "specimen_id": "Alias-resolved accession specimen reference.",
    "accession_identifier": "Alias-resolved accession identifier.",
    "test_snomed_code": "Approved SNOMED assay code.",
    "test_loinc_code": "Approved LOINC assay code.",
    "test_omop_concept_id": "Standard OMOP assay concept.",
    "is_current": "Whether the source report version is current.",
    "research_qi_only": "Research/QI release flag.",
    "record_status": "Normalized report-version status.",
    "record_status_effective_from": "Status validity start.",
    "record_status_effective_to": "Status validity end.",
    "confidentiality_code": "Security code.",
    "vip_ind": "VIP indicator.",
    "withheld_identity_ind": "Withheld identity indicator.",
    "source_feed": "Owning feed.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze material-change timestamp.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.genomic_test"),
    comment=(
        "One molecular or cytogenetic assay per report version; all versions publish and "
        "is_current determines active versus superseded. Panel and vocabulary columns were "
        "all NULL on 2026-08-18, and ADC_UPDT was a single static-build instant. Gold QC twin "
        "of the silver product: 0 columns are repaired or nulled, 1 rule(s) drop rows, 2 "
        "check(s) are advisory. Each rule states its reason in the pipeline notebook, and "
        "Lakeflow expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_GENOMIC_TEST_MANDATORY_RULES)
@_expect_all(CLINICAL_GENOMIC_TEST_ADVISORY_RULES)
def gold_clinical_genomic_test():
    """Quality-controlled twin of journey_clinical.genomic_test."""
    df = _qc(
        "clinical_genomic_test",
        CLINICAL_GENOMIC_TEST_SELECT,
        date_flags=["event_after_death_30d"],
    )
    return _with_comments(df, CLINICAL_GENOMIC_TEST_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.hrg_grouping ====

CLINICAL_HRG_GROUPING_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",

    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 13 of 47,137,282 rows (2.76e-05%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `event_datetime` END AS `event_datetime`",

    # event_end_datetime cannot precede event_datetime. The start is the better-attested of
    # the two, so the end is what goes and the row keeps its event_datetime. Hit 1,584 of
    # 47,137,282 rows (0.00336%) when profiled on 2026-08-24.
    "CASE WHEN `event_datetime` IS NOT NULL AND `event_end_datetime` IS NOT NULL AND `event_datetime` > `event_end_datetime` THEN NULL ELSE `event_end_datetime` END AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`cds_record_id` AS `cds_record_id`",
    "`person_link_method` AS `person_link_method`",

    # 1800-01-01 is a placeholder low date, not a record from 1800. Hit 49 of 47,137,282
    # rows (0.000104%) when profiled on 2026-08-24.
    #
    # A date before 1901 that is not one of the known placeholders. Nothing in this estate
    # predates the twentieth century, so these are mistyped or mis-scaled rather than early.
    # Hit 1 of 47,137,282 rows (2.12e-06%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`admission_datetime` AS DATE) = DATE'1800-01-01' OR CAST(`admission_datetime` AS DATE) < DATE'1901-01-01' AND CAST(`admission_datetime` AS DATE) NOT IN (DATE'1800-01-01', DATE'1899-12-30', DATE'1900-01-01') THEN NULL ELSE `admission_datetime` END AS `admission_datetime`",
    "`discharge_datetime` AS `discharge_datetime`",
    "`episode_start_datetime` AS `episode_start_datetime`",

    # episode_end_datetime cannot precede episode_start_datetime. The start is the better-
    # attested of the two, so the end is what goes and the row keeps its
    # episode_start_datetime. Hit 1,584 of 47,137,282 rows (0.00336%) when profiled on
    # 2026-08-24.
    "CASE WHEN `episode_start_datetime` IS NOT NULL AND `episode_end_datetime` IS NOT NULL AND `episode_start_datetime` > `episode_end_datetime` THEN NULL ELSE `episode_end_datetime` END AS `episode_end_datetime`",
    "`hosp_prov_spell_num` AS `hosp_prov_spell_num`",
    "`provider_org_cd` AS `provider_org_cd`",
    "`episode_order` AS `episode_order`",
    "`episode_duration_days` AS `episode_duration_days`",
    "`main_specialty_cd` AS `main_specialty_cd`",
    "`treatment_function_cd` AS `treatment_function_cd`",
    "`admission_method_cd` AS `admission_method_cd`",
    "`admission_source_cd` AS `admission_source_cd`",
    "`admission_source_desc` AS `admission_source_desc`",
    "`discharge_method_cd` AS `discharge_method_cd`",
    "`discharge_dest_cd` AS `discharge_dest_cd`",
    "`discharge_dest_desc` AS `discharge_dest_desc`",
    "`patient_class_cd` AS `patient_class_cd`",
    "`patient_class_desc` AS `patient_class_desc`",
    "`source_age` AS `source_age`",
    "`source_sex_cd` AS `source_sex_cd`",
    "`neonatal_care_level_cd` AS `neonatal_care_level_cd`",
    "`critical_care_days` AS `critical_care_days`",
    "`rehab_days` AS `rehab_days`",
    "`icd_diagnosis_codes_json` AS `icd_diagnosis_codes_json`",
    "`opcs_procedure_codes_json` AS `opcs_procedure_codes_json`",
    "`fce_hrg_cd` AS `fce_hrg_cd`",
    "`fce_hrg_desc` AS `fce_hrg_desc`",
    "`fce_grouping_method_flag` AS `fce_grouping_method_flag`",
    "`fce_dominant_proc_cd` AS `fce_dominant_proc_cd`",
    "`fce_dominant_proc_desc` AS `fce_dominant_proc_desc`",
    "`fce_pbc_cd` AS `fce_pbc_cd`",
    "`fce_calc_episode_duration` AS `fce_calc_episode_duration`",
    "`fce_reporting_episode_duration` AS `fce_reporting_episode_duration`",
    "`dominant_episode_flag` AS `dominant_episode_flag`",
    "`spell_hrg_cd` AS `spell_hrg_cd`",
    "`spell_hrg_desc` AS `spell_hrg_desc`",
    "`spell_grouping_method_flag` AS `spell_grouping_method_flag`",
    "`spell_dominant_proc_cd` AS `spell_dominant_proc_cd`",
    "`spell_dominant_proc_desc` AS `spell_dominant_proc_desc`",
    "`spell_primary_diag_cd` AS `spell_primary_diag_cd`",
    "`spell_primary_diag_desc` AS `spell_primary_diag_desc`",
    "`spell_secondary_diag_cd` AS `spell_secondary_diag_cd`",
    "`spell_secondary_diag_desc` AS `spell_secondary_diag_desc`",
    "`spell_episode_count` AS `spell_episode_count`",
    "`spell_los` AS `spell_los`",
    "`spell_reporting_los` AS `spell_reporting_los`",
    "`spell_critical_care_days` AS `spell_critical_care_days`",
    "`spell_ssc_cd` AS `spell_ssc_cd`",
    "`spell_best_practice_cd` AS `spell_best_practice_cd`",
    "`first_attend_cd` AS `first_attend_cd`",
    "`first_attend_desc` AS `first_attend_desc`",
    "`grouping_method_flag` AS `grouping_method_flag`",
    "`dominant_proc_cd` AS `dominant_proc_cd`",
    "`dominant_proc_desc` AS `dominant_proc_desc`",
    "`grouper_errors` AS `grouper_errors`",
    "`record_status` AS `record_status`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 61,616 of 47,137,282 rows (0.131%) when profiled on 2026-08-24.
    #
    # 1800-01-01 is a placeholder low date, not a record from 1800. Hit 3,398 of 47,137,282
    # rows (0.00721%) when profiled on 2026-08-24.
    #
    # 2100-12-31 appears here on a field that is not an end date, so it cannot be the 'still
    # open' marker it is elsewhere and is a placeholder instead. Hit 1,558 of 47,137,282
    # rows (0.00331%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`record_status_effective_from` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`record_status_effective_from` AS DATE)) < 9999 OR CAST(`record_status_effective_from` AS DATE) = DATE'1800-01-01' OR CAST(`record_status_effective_from` AS DATE) = DATE'2100-12-31' THEN NULL ELSE `record_status_effective_from` END AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`fact_category` AS `fact_category`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
]

CLINICAL_HRG_GROUPING_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 47,137,282 rows of 47,137,282
    # that are current and attributable; identity_status = 'resolved' keeps the 43,616,250
    # rows of 47,137,282 that are current and attributable. Superseded versions and rows
    # whose identity was never resolved are not research data, and a consumer who wants them
    # has silver.
    "research_surface": "(identity_status = 'resolved') AND (record_status = 'active')",
}

CLINICAL_HRG_GROUPING_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 3,521,032 of 47,137,282 at the profile.
    "gold.clinical.hrg_grouping.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 0 of 47,137,282 at the profile.
    "gold.clinical.hrg_grouping.record_status.default_view_active":
        "record_status = 'active'",

    # This bounds a period of validity, and a future end is exactly how the source says a
    # record is still current -- nulling it would assert the record is valid forever, which
    # is a stronger and worse claim than the one being corrected. Seen on 13 of 47,137,282
    # rows (2.76e-05%) when profiled on 2026-08-24.
    "gold.clinical.hrg_grouping.record_status_effective_from.future_owner":
        "NOT COALESCE((CAST(`record_status_effective_from` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 147,553 of 47,137,282 rows (0.313%) when profiled on 2026-08-24.
    "gold.clinical.hrg_grouping.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 43,005 of 47,137,282 rows (0.0912%) when profiled on 2026-08-24.
    "gold.clinical.hrg_grouping.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",
}

CLINICAL_HRG_GROUPING_COLUMN_COMMENTS = {
    "patient_event_id": "Stable row identifier.",
    "fact_row_id": "Stable row identifier.",
    "subject_key": "Published field.",
    "subject_id_system": "Published field.",
    "person_id": "Published field.",
    "identity_status": "Published field.",
    "encounter_id": "Published field.",
    "event_datetime": "Published field.",
    "event_end_datetime": "Published field.",
    "source_coding_system": "Published field.",
    "source_code": "Published field.",
    "source_display": "Published field.",
    "cds_record_id": "Published field.",
    "person_link_method": "Published field.",
    "admission_datetime":
        "Published field. Gold QC transform rules: gold.clinical.hrg_grouping.admission_datetime.d1800_01_01_owner, gold.clinical.hrg_grouping.admission_datetime.pre1901_other_owner.",
    "discharge_datetime": "Published field.",
    "episode_start_datetime": "Published field.",
    "episode_end_datetime": "Published field.",
    "hosp_prov_spell_num": "Published field.",
    "provider_org_cd": "Published field.",
    "episode_order": "Published field.",
    "episode_duration_days": "Published field.",
    "main_specialty_cd": "Published field.",
    "treatment_function_cd": "Published field.",
    "admission_method_cd": "Published field.",
    "admission_source_cd": "Published field.",
    "admission_source_desc": "Published field.",
    "discharge_method_cd": "Published field.",
    "discharge_dest_cd": "Published field.",
    "discharge_dest_desc": "Published field.",
    "patient_class_cd": "Published field.",
    "patient_class_desc": "Published field.",
    "source_age": "Published field.",
    "source_sex_cd": "Published field.",
    "neonatal_care_level_cd": "Published field.",
    "critical_care_days": "Published field.",
    "rehab_days": "Published field.",
    "icd_diagnosis_codes_json": "Published field.",
    "opcs_procedure_codes_json": "Published field.",
    "fce_hrg_cd": "Published field.",
    "fce_hrg_desc": "Published field.",
    "fce_grouping_method_flag": "Published field.",
    "fce_dominant_proc_cd": "Published field.",
    "fce_dominant_proc_desc": "Published field.",
    "fce_pbc_cd": "Published field.",
    "fce_calc_episode_duration": "Published field.",
    "fce_reporting_episode_duration": "Published field.",
    "dominant_episode_flag": "Published field.",
    "spell_hrg_cd": "Published field.",
    "spell_hrg_desc": "Published field.",
    "spell_grouping_method_flag": "Published field.",
    "spell_dominant_proc_cd": "Published field.",
    "spell_dominant_proc_desc": "Published field.",
    "spell_primary_diag_cd": "Published field.",
    "spell_primary_diag_desc": "Published field.",
    "spell_secondary_diag_cd": "Published field.",
    "spell_secondary_diag_desc": "Published field.",
    "spell_episode_count": "Published field.",
    "spell_los": "Published field.",
    "spell_reporting_los": "Published field.",
    "spell_critical_care_days": "Published field.",
    "spell_ssc_cd": "Published field.",
    "spell_best_practice_cd": "Published field.",
    "first_attend_cd": "Published field.",
    "first_attend_desc": "Published field.",
    "grouping_method_flag": "Published field.",
    "dominant_proc_cd": "Published field.",
    "dominant_proc_desc": "Published field.",
    "grouper_errors": "Published field.",
    "record_status": "Published field.",
    "record_status_effective_from":
        "Published field. Gold QC transform rules: gold.clinical.hrg_grouping.record_status_effective_from.beyond_2100_below_9999_owner, gold.clinical.hrg_grouping.record_status_effective_from.d1800_01_01_owner, gold.clinical.hrg_grouping.record_status_effective_from.d2100_12_31_owner.",
    "record_status_effective_to": "Published field.",
    "confidentiality_code": "Published field.",
    "vip_ind": "Published field.",
    "withheld_identity_ind": "Published field.",
    "fact_category": "Published field.",
    "source_feed": "Published field.",
    "load_batch_id": "Published field.",
    "source_update_timestamp": "Published field.",
    "loaded_at": "Published field.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.hrg_grouping"),
    comment=(
        "One SLAM APC or outpatient HRG grouping row. Gold QC twin of the silver product: 6 "
        "columns are repaired or nulled, 1 rule(s) drop rows, 5 check(s) are advisory. Each "
        "rule states its reason in the pipeline notebook, and Lakeflow expectation metrics "
        "report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_HRG_GROUPING_MANDATORY_RULES)
@_expect_all(CLINICAL_HRG_GROUPING_ADVISORY_RULES)
def gold_clinical_hrg_grouping():
    """Quality-controlled twin of journey_clinical.hrg_grouping."""
    # 5 rows point at a person_id the spine does not have. The pointer is nulled so it
    # cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    df = _qc(
        "clinical_hrg_grouping",
        CLINICAL_HRG_GROUPING_SELECT,
        fk_columns=["person_id"],
        date_flags=["event_after_death_30d", "event_before_birth"],
    )
    return _with_comments(df, CLINICAL_HRG_GROUPING_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.imaging_exam ====

CLINICAL_IMAGING_EXAM_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",

    # 22 rows point at a person_id the spine does not have. The pointer is nulled so it
    # cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    "CASE WHEN NOT `person_id_resolved` THEN NULL ELSE `person_id` END AS `person_id`",
    "`identity_status` AS `identity_status`",

    # 46,874 rows point at a encounter_id the spine does not have. The pointer is nulled so
    # it cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    "CASE WHEN NOT `encounter_id_resolved` THEN NULL ELSE `encounter_id` END AS `encounter_id`",

    # 1899-12-30 is the zero point of the OLE/Excel date scale, so it is what a spreadsheet
    # or a COM layer writes when the date was left blank. It is not a date anyone recorded.
    # Hit 1 of 50,008,147 rows (2e-06%) when profiled on 2026-08-24.
    #
    # A date before 1901 that is not one of the known placeholders. Nothing in this estate
    # predates the twentieth century, so these are mistyped or mis-scaled rather than early.
    # Hit 1 of 50,008,147 rows (2e-06%) when profiled on 2026-08-24.
    #
    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 30 of 50,008,147 rows (6e-05%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE CASE WHEN CAST(`event_datetime` AS DATE) = DATE'1899-12-30' OR CAST(`event_datetime` AS DATE) < DATE'1901-01-01' AND CAST(`event_datetime` AS DATE) NOT IN (DATE'1800-01-01', DATE'1899-12-30', DATE'1900-01-01') THEN NULL ELSE `event_datetime` END END AS `event_datetime`",

    # Stamps landing in January 1970 carry the same millisecond-epoch defect found on
    # clinical_score and vital_sign, but nothing here corroborates a rescaled value the way
    # the encounter window does there, so the stamp is nulled rather than reconstructed. It
    # affected 1,177,239 rows. The catalogue's date rules stop at 1901 and do not reach
    # this.
    "CASE WHEN YEAR(CAST(`event_end_datetime` AS TIMESTAMP)) = 1970 THEN NULL ELSE `event_end_datetime` END AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",

    # 'UNKNOWN' is a placeholder the source writes when the value was not recorded; it is
    # not a code, so it is nulled rather than passed on as one. Hit 65 of 50,008,147 rows
    # (0.00013%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`source_code` AS STRING))) = 'UNKNOWN' THEN NULL ELSE `source_code` END AS `source_code`",
    "`source_display` AS `source_display`",
    "`exam_code` AS `exam_code`",
    "`status_code` AS `status_code`",
    "`accession_identifier` AS `accession_identifier`",
    "`study_instance_uid` AS `study_instance_uid`",
    "`modality_code` AS `modality_code`",

    # 'UNKNOWN' is a placeholder the source writes when the value was not recorded; it is
    # not a code, so it is nulled rather than passed on as one. Hit 321 of 50,008,147 rows
    # (0.000642%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`body_site_code` AS STRING))) = 'UNKNOWN' THEN NULL ELSE `body_site_code` END AS `body_site_code`",
    "`report_patient_event_id` AS `report_patient_event_id`",
    "`requester_practitioner_id` AS `requester_practitioner_id`",
    "`performer_practitioner_id` AS `performer_practitioner_id`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",

    # Silver derived this flag by comparing event_datetime with the person's own dates,
    # before the rules above corrected event_datetime. On the rows where event_datetime
    # changed, the flag describes a timestamp gold no longer publishes. This product carries
    # a VARIANT column, which rules out the spine join gold would need to recompute the
    # flag, so it is nulled where the value beneath it moved rather than left asserting
    # something stale.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `event_before_birth` END AS `event_before_birth`",

    # Silver derived this flag by comparing event_datetime with the person's own dates,
    # before the rules above corrected event_datetime. On the rows where event_datetime
    # changed, the flag describes a timestamp gold no longer publishes. This product carries
    # a VARIANT column, which rules out the spine join gold would need to recompute the
    # flag, so it is nulled where the value beneath it moved rather than left asserting
    # something stale.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `event_after_death_30d` END AS `event_after_death_30d`",
]

CLINICAL_IMAGING_EXAM_MANDATORY_RULES = {
    # The research surface. identity_status = 'resolved' keeps the 48,943,391 rows of
    # 50,008,147 that are current and attributable. Superseded versions and rows whose
    # identity was never resolved are not research data, and a consumer who wants them has
    # silver.
    "research_surface": "(identity_status = 'resolved')",
}

CLINICAL_IMAGING_EXAM_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 1,064,756 of 50,008,147 at the profile.
    "gold.clinical.imaging_exam.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 36,156 of 50,008,147 rows (0.0723%) when profiled on 2026-08-24.
    "gold.clinical.imaging_exam.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 1,247,681 of 50,008,147 rows (2.49%) when profiled on
    # 2026-08-24.
    "gold.clinical.imaging_exam.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",

    # Left as a warning because it fires on 2,567,329 of 50,008,147 rows (5.13%) when
    # profiled on 2026-08-24 -- at that rate the rule's assumption about what event_datetime
    # and event_end_datetime mean is the thing in doubt, not the data. The inverted gaps are
    # mostly minutes, which reads as two clocks rather than two events in the wrong order.
    "gold.clinical.imaging_exam.table.ordering_violation_event_datetime_event_end_datetime":
        "NOT COALESCE((`event_datetime` IS NOT NULL AND `event_end_datetime` IS NOT NULL AND `event_datetime` > `event_end_datetime`), FALSE)",
}

CLINICAL_IMAGING_EXAM_COLUMN_COMMENTS = {
    "patient_event_id": "Stable imaging-exam identifier.",
    "fact_row_id": "Storage-row identifier.",
    "subject_key": "Always-populated peppered subject key when populated.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved person reference.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference.",
    "event_datetime":
        "Imaging study start timestamp. Gold QC transform rules: gold.clinical.imaging_exam.event_datetime.ole_zero_date, gold.clinical.imaging_exam.event_datetime.pre1901_other_owner.",
    "event_end_datetime": "Imaging study end timestamp.",
    "source_coding_system": "Source exam coding system.",
    "source_code": "Source exam code",
    "source_display": "Source exam display.",
    "exam_code": "Source and mapped imaging-exam CodeableConcept.",
    "status_code": "Imaging study status.",
    "accession_identifier": "Imaging accession identifier.",
    "study_instance_uid": "DICOM study instance UID.",
    "modality_code": "Imaging modality code.",
    "body_site_code": "Body-site code.",
    "report_patient_event_id": "Linked diagnostic-report event when available.",
    "requester_practitioner_id": "Requesting practitioner reference.",
    "performer_practitioner_id": "Performing practitioner reference.",
    "record_status": "Normalized lifecycle.",
    "record_status_effective_from": "Lifecycle start.",
    "record_status_effective_to": "Lifecycle end.",
    "confidentiality_code": "Security classification.",
    "vip_ind": "VIP indicator.",
    "withheld_identity_ind": "Withheld-identity indicator.",
    "source_feed": "Registered PACS or Millennium radiology source feed.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.imaging_exam"),
    comment=(
        "One PACS or Millennium radiology examination/study; the Millennium report-link field "
        "is schema-only until it first populates. Gold QC twin of the silver product: 8 "
        "columns are repaired or nulled, 1 rule(s) drop rows, 4 check(s) are advisory. Each "
        "rule states its reason in the pipeline notebook, and Lakeflow expectation metrics "
        "report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_IMAGING_EXAM_MANDATORY_RULES)
@_expect_all(CLINICAL_IMAGING_EXAM_ADVISORY_RULES)
def gold_clinical_imaging_exam():
    """Quality-controlled twin of journey_clinical.imaging_exam."""
    df = _qc("clinical_imaging_exam", CLINICAL_IMAGING_EXAM_SELECT)
    return _with_comments(df, CLINICAL_IMAGING_EXAM_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.indication ====

CLINICAL_INDICATION_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",

    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 150 of 3,282,008,054 rows (4.57e-06%) when profiled on 2026-08-24.
    #
    # Stamps landing in January 1970 carry the same millisecond-epoch defect found on
    # clinical_score and vital_sign, but nothing here corroborates a rescaled value the way
    # the encounter window does there, so the stamp is nulled rather than reconstructed. The
    # catalogue's date rules stop at 1901 and do not reach this.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS OR YEAR(CAST(`event_datetime` AS TIMESTAMP)) = 1970 THEN NULL ELSE `event_datetime` END AS `event_datetime`",
    "`event_end_datetime` AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",

    # 'UNKNOWN' is a placeholder the source writes when the value was not recorded; it is
    # not a code, so it is nulled rather than passed on as one. Hit 3,300 of 3,282,008,054
    # rows (0.000101%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`source_code` AS STRING))) = 'UNKNOWN' THEN NULL ELSE `source_code` END AS `source_code`",
    "`source_display` AS `source_display`",
    "`relation_type` AS `relation_type`",
    "`source_field` AS `source_field`",
    "`source_text` AS `source_text`",
    "`evidence_text` AS `evidence_text`",
    "`evidence_start` AS `evidence_start`",
    "`evidence_end` AS `evidence_end`",
    "`snomed_code` AS `snomed_code`",
    "`snomed_term` AS `snomed_term`",
    "`omop_concept_id` AS `omop_concept_id`",
    "`assertion` AS `assertion`",
    "`temporality` AS `temporality`",
    "`experiencer` AS `experiencer`",
    "`rule_id` AS `rule_id`",
    "`rule_version` AS `rule_version`",
    "`confidence` AS `confidence`",
    "`mapping_status` AS `mapping_status`",
    "`ig_release_status` AS `ig_release_status`",
    "`is_current` AS `is_current`",
    "`research_qi_only` AS `research_qi_only`",
    "`specimen_id` AS `specimen_id`",
    "`accession_identifier` AS `accession_identifier`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
]

CLINICAL_INDICATION_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 3,282,008,054 rows of
    # 3,282,008,054 that are current and attributable; identity_status = 'resolved' keeps
    # the 3,282,007,974 rows of 3,282,008,054 that are current and attributable. Superseded
    # versions and rows whose identity was never resolved are not research data, and a
    # consumer who wants them has silver.
    "research_surface": "(identity_status = 'resolved') AND (record_status = 'active')",
}

CLINICAL_INDICATION_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 80 of 3,282,008,054 at the profile.
    "gold.clinical.indication.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 0 of 3,282,008,054 at the profile.
    "gold.clinical.indication.record_status.default_view_active":
        "record_status = 'active'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 4,323,749 of 3,282,008,054 rows (0.132%) when profiled on
    # 2026-08-24.
    "gold.clinical.indication.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 268,150 of 3,282,008,054 rows (0.00817%) when profiled on
    # 2026-08-24.
    "gold.clinical.indication.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",
}

CLINICAL_INDICATION_COLUMN_COMMENTS = {
    "patient_event_id": "Stable indication event identifier.",
    "fact_row_id": "Storage-row identifier.",
    "subject_key": "Best available subject key.",
    "subject_id_system": "Subject-key system.",
    "person_id": "Resolved person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference; unavailable at this grain.",
    "event_datetime": "Accession report/sample/request fallback time.",
    "event_end_datetime": "Event end time.",
    "source_coding_system": "Indication text namespace.",
    "source_code": "Coded diagnosis display string.",
    "source_display": "Coded diagnosis display string.",
    "relation_type": "Evidence relation type.",
    "source_field": "Source evidence field.",
    "source_text": "Unmodified coded diagnosis display string.",
    "evidence_text": "Exact evidence span.",
    "evidence_start": "Evidence start offset.",
    "evidence_end": "Evidence end offset.",
    "snomed_code": "SNOMED condition code.",
    "snomed_term": "SNOMED display term.",
    "omop_concept_id": "Standard OMOP condition concept.",
    "assertion": "Evidence assertion.",
    "temporality": "Evidence temporality.",
    "experiencer": "Evidence experiencer.",
    "rule_id": "Deterministic rule identifier.",
    "rule_version": "Deterministic rule version.",
    "confidence": "Rule-specific confidence.",
    "mapping_status": "Source mapping status.",
    "ig_release_status": "IG release status published verbatim.",
    "is_current": "Whether evidence remains current.",
    "research_qi_only": "Research/QI release flag.",
    "specimen_id": "Alias-resolved accession specimen reference.",
    "accession_identifier": "Alias-resolved accession identifier.",
    "record_status": "Normalized evidence lifecycle.",
    "record_status_effective_from": "Status validity start.",
    "record_status_effective_to": "Status validity end.",
    "confidentiality_code": "Security code.",
    "vip_ind": "VIP indicator.",
    "withheld_identity_ind": "Withheld identity indicator.",
    "source_feed": "Owning feed.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze material-change timestamp.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.indication"),
    comment=(
        "Accession-scoped diagnosis-context evidence from diagnosis_context_window_v1. "
        "Evidence spans are the complete coded diagnosis display string; event time uses "
        "accession report_dt, sample_dt, then request_dt. Text is ig_risk 4, ig_severity 2; "
        "ig_release_status is describe-only and prod activation is gated. Gold QC twin of the "
        "silver product: 3 columns are repaired or nulled, 1 rule(s) drop rows, 4 check(s) "
        "are advisory. Each rule states its reason in the pipeline notebook, and Lakeflow "
        "expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_INDICATION_MANDATORY_RULES)
@_expect_all(CLINICAL_INDICATION_ADVISORY_RULES)
def gold_clinical_indication():
    """Quality-controlled twin of journey_clinical.indication."""
    # 84 rows point at a person_id the spine does not have. The pointer is nulled so it
    # cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    df = _qc(
        "clinical_indication",
        CLINICAL_INDICATION_SELECT,
        fk_columns=["person_id"],
        date_flags=["event_after_death_30d", "event_before_birth"],
    )
    return _with_comments(df, CLINICAL_INDICATION_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.labour_delivery ====

CLINICAL_LABOUR_DELIVERY_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",
    "`event_datetime` AS `event_datetime`",

    # event_end_datetime cannot precede event_datetime. The start is the better-attested of
    # the two, so the end is what goes and the row keeps its event_datetime. Hit 283 of
    # 101,238 rows (0.28%) when profiled on 2026-08-24.
    "CASE WHEN `event_datetime` IS NOT NULL AND `event_end_datetime` IS NOT NULL AND `event_datetime` > `event_end_datetime` THEN NULL ELSE `event_end_datetime` END AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`labour_delivery_id` AS `labour_delivery_id`",
    "`labour_onset_method` AS `labour_onset_method`",
    "`labour_onset_presentation` AS `labour_onset_presentation`",
    "`caesarean_datetime` AS `caesarean_datetime`",
    "`decision_to_deliver_datetime` AS `decision_to_deliver_datetime`",
    "`rom_datetime` AS `rom_datetime`",
    "`rom_method` AS `rom_method`",
    "`rom_reason` AS `rom_reason`",
    "`second_stage_datetime` AS `second_stage_datetime`",
    "`third_stage_end_datetime` AS `third_stage_end_datetime`",
    "`episiotomy_reason` AS `episiotomy_reason`",
    "`placenta_delivery_method` AS `placenta_delivery_method`",
    "`mother_admission_method` AS `mother_admission_method`",
    "`mother_discharge_datetime` AS `mother_discharge_datetime`",
    "`mother_discharge_method` AS `mother_discharge_method`",
    "`mother_discharge_destination` AS `mother_discharge_destination`",
    "`intrapartum_org_site` AS `intrapartum_org_site`",
    "`intrapartum_setting` AS `intrapartum_setting`",
    "`postnatal_lead_provider` AS `postnatal_lead_provider`",
    "`pregnancy_id` AS `pregnancy_id`",
    "`journey_pregnancy_id` AS `journey_pregnancy_id`",
    "`source_link_status` AS `source_link_status`",
    "`pregnancy_orphan_ind` AS `pregnancy_orphan_ind`",
    "`spine_person_mismatch_ind` AS `spine_person_mismatch_ind`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`fact_category` AS `fact_category`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
]

CLINICAL_LABOUR_DELIVERY_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 101,238 rows of 101,238 that
    # are current and attributable; identity_status = 'resolved' keeps the 100,859 rows of
    # 101,238 that are current and attributable. Superseded versions and rows whose identity
    # was never resolved are not research data, and a consumer who wants them has silver.
    "research_surface": "(identity_status = 'resolved') AND (record_status = 'active')",
}

CLINICAL_LABOUR_DELIVERY_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 379 of 101,238 at the profile.
    "gold.clinical.labour_delivery.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 0 of 101,238 at the profile.
    "gold.clinical.labour_delivery.record_status.default_view_active":
        "record_status = 'active'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 1 of 101,238 rows (0.000988%) when profiled on 2026-08-24.
    "gold.clinical.labour_delivery.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",
}

CLINICAL_LABOUR_DELIVERY_COLUMN_COMMENTS = {
    "patient_event_id": "Published field patient_event_id.",
    "fact_row_id": "Published field fact_row_id.",
    "subject_key": "Published field subject_key.",
    "subject_id_system": "Published field subject_id_system.",
    "person_id": "Published field person_id.",
    "identity_status": "Published field identity_status.",
    "encounter_id": "Published field encounter_id.",
    "event_datetime": "Published field event_datetime.",
    "event_end_datetime": "Published field event_end_datetime.",
    "source_coding_system": "Published field source_coding_system.",
    "source_code": "Published field source_code.",
    "source_display": "Published field source_display.",
    "labour_delivery_id": "Published field labour_delivery_id.",
    "labour_onset_method": "Published field labour_onset_method.",
    "labour_onset_presentation": "Published field labour_onset_presentation.",
    "caesarean_datetime": "Published field caesarean_datetime.",
    "decision_to_deliver_datetime": "Published field decision_to_deliver_datetime.",
    "rom_datetime": "Published field rom_datetime.",
    "rom_method": "Published field rom_method.",
    "rom_reason": "Published field rom_reason.",
    "second_stage_datetime": "Published field second_stage_datetime.",
    "third_stage_end_datetime": "Published field third_stage_end_datetime.",
    "episiotomy_reason": "Published field episiotomy_reason.",
    "placenta_delivery_method": "Published field placenta_delivery_method.",
    "mother_admission_method": "Published field mother_admission_method.",
    "mother_discharge_datetime": "Published field mother_discharge_datetime.",
    "mother_discharge_method": "Published field mother_discharge_method.",
    "mother_discharge_destination": "Published field mother_discharge_destination.",
    "intrapartum_org_site": "Published field intrapartum_org_site.",
    "intrapartum_setting": "Published field intrapartum_setting.",
    "postnatal_lead_provider": "Published field postnatal_lead_provider.",
    "pregnancy_id": "Published field pregnancy_id.",
    "journey_pregnancy_id": "Published field journey_pregnancy_id.",
    "source_link_status": "Published field source_link_status.",
    "pregnancy_orphan_ind": "Published field pregnancy_orphan_ind.",
    "spine_person_mismatch_ind": "Published field spine_person_mismatch_ind.",
    "record_status": "Published field record_status.",
    "record_status_effective_from": "Published field record_status_effective_from.",
    "record_status_effective_to": "Published field record_status_effective_to.",
    "confidentiality_code": "Published field confidentiality_code.",
    "vip_ind": "Published field vip_ind.",
    "withheld_identity_ind": "Published field withheld_identity_ind.",
    "fact_category": "Published field fact_category.",
    "source_feed": "Published field source_feed.",
    "load_batch_id": "Published field load_batch_id.",
    "source_update_timestamp": "Published field source_update_timestamp.",
    "loaded_at": "Published field loaded_at.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.labour_delivery"),
    comment=(
        "One MSDS labour-delivery row with pregnancy-spine identity evidence. Gold QC twin of "
        "the silver product: 2 columns are repaired or nulled, 1 rule(s) drop rows, 3 "
        "check(s) are advisory. Each rule states its reason in the pipeline notebook, and "
        "Lakeflow expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_LABOUR_DELIVERY_MANDATORY_RULES)
@_expect_all(CLINICAL_LABOUR_DELIVERY_ADVISORY_RULES)
def gold_clinical_labour_delivery():
    """Quality-controlled twin of journey_clinical.labour_delivery."""
    # 1 rows point at a person_id the spine does not have. The pointer is nulled so it
    # cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    df = _qc(
        "clinical_labour_delivery",
        CLINICAL_LABOUR_DELIVERY_SELECT,
        fk_columns=["person_id"],
        date_flags=["event_after_death_30d"],
    )
    return _with_comments(df, CLINICAL_LABOUR_DELIVERY_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.maternity_care_contact ====

CLINICAL_MATERNITY_CARE_CONTACT_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",
    "`event_datetime` AS `event_datetime`",
    "`event_end_datetime` AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`care_contact_id` AS `care_contact_id`",
    "`attend_code` AS `attend_code`",
    "`consult_type` AS `consult_type`",
    "`contact_subject` AS `contact_subject`",
    "`medium` AS `medium`",
    "`duration` AS `duration`",
    "`admin_category` AS `admin_category`",
    "`gp_therapy_ind` AS `gp_therapy_ind`",
    "`cancel_datetime` AS `cancel_datetime`",
    "`cancel_reason` AS `cancel_reason`",
    "`replacement_offer_datetime` AS `replacement_offer_datetime`",
    "`replacement_appointment_datetime` AS `replacement_appointment_datetime`",
    "`organization_id` AS `organization_id`",
    "`site_id` AS `site_id`",
    "`location_code` AS `location_code`",
    "`pregnancy_id` AS `pregnancy_id`",
    "`journey_pregnancy_id` AS `journey_pregnancy_id`",
    "`source_link_status` AS `source_link_status`",
    "`pregnancy_orphan_ind` AS `pregnancy_orphan_ind`",
    "`spine_person_mismatch_ind` AS `spine_person_mismatch_ind`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`fact_category` AS `fact_category`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
]

CLINICAL_MATERNITY_CARE_CONTACT_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 1,052,832 rows of 1,052,832
    # that are current and attributable; identity_status = 'resolved' keeps the 1,052,592
    # rows of 1,052,832 that are current and attributable. Superseded versions and rows
    # whose identity was never resolved are not research data, and a consumer who wants them
    # has silver.
    "research_surface": "(identity_status = 'resolved') AND (record_status = 'active')",
}

CLINICAL_MATERNITY_CARE_CONTACT_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 240 of 1,052,832 at the profile.
    "gold.clinical.maternity_care_contact.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 0 of 1,052,832 at the profile.
    "gold.clinical.maternity_care_contact.record_status.default_view_active":
        "record_status = 'active'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 1 of 1,052,832 rows (9.5e-05%) when profiled on 2026-08-24.
    "gold.clinical.maternity_care_contact.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 7 of 1,052,832 rows (0.000665%) when profiled on 2026-08-24.
    "gold.clinical.maternity_care_contact.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",
}

CLINICAL_MATERNITY_CARE_CONTACT_COLUMN_COMMENTS = {
    "patient_event_id": "Published field patient_event_id.",
    "fact_row_id": "Published field fact_row_id.",
    "subject_key": "Published field subject_key.",
    "subject_id_system": "Published field subject_id_system.",
    "person_id": "Published field person_id.",
    "identity_status": "Published field identity_status.",
    "encounter_id": "Published field encounter_id.",
    "event_datetime": "Published field event_datetime.",
    "event_end_datetime": "Published field event_end_datetime.",
    "source_coding_system": "Published field source_coding_system.",
    "source_code": "Published field source_code.",
    "source_display": "Published field source_display.",
    "care_contact_id": "Published field care_contact_id.",
    "attend_code": "Published field attend_code.",
    "consult_type": "Published field consult_type.",
    "contact_subject": "Published field contact_subject.",
    "medium": "Published field medium.",
    "duration": "Published field duration.",
    "admin_category": "Published field admin_category.",
    "gp_therapy_ind": "Published field gp_therapy_ind.",
    "cancel_datetime": "Published field cancel_datetime.",
    "cancel_reason": "Published field cancel_reason.",
    "replacement_offer_datetime": "Published field replacement_offer_datetime.",
    "replacement_appointment_datetime": "Published field replacement_appointment_datetime.",
    "organization_id": "Published field organization_id.",
    "site_id": "Published field site_id.",
    "location_code": "Published field location_code.",
    "pregnancy_id": "Published field pregnancy_id.",
    "journey_pregnancy_id": "Published field journey_pregnancy_id.",
    "source_link_status": "Published field source_link_status.",
    "pregnancy_orphan_ind": "Published field pregnancy_orphan_ind.",
    "spine_person_mismatch_ind": "Published field spine_person_mismatch_ind.",
    "record_status": "Published field record_status.",
    "record_status_effective_from": "Published field record_status_effective_from.",
    "record_status_effective_to": "Published field record_status_effective_to.",
    "confidentiality_code": "Published field confidentiality_code.",
    "vip_ind": "Published field vip_ind.",
    "withheld_identity_ind": "Published field withheld_identity_ind.",
    "fact_category": "Published field fact_category.",
    "source_feed": "Published field source_feed.",
    "load_batch_id": "Published field load_batch_id.",
    "source_update_timestamp": "Published field source_update_timestamp.",
    "loaded_at": "Published field loaded_at.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.maternity_care_contact"),
    comment=(
        "One MSDS maternity care contact with pregnancy-spine identity evidence. Gold QC twin "
        "of the silver product: 0 columns are repaired or nulled, 1 rule(s) drop rows, 4 "
        "check(s) are advisory. Each rule states its reason in the pipeline notebook, and "
        "Lakeflow expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_MATERNITY_CARE_CONTACT_MANDATORY_RULES)
@_expect_all(CLINICAL_MATERNITY_CARE_CONTACT_ADVISORY_RULES)
def gold_clinical_maternity_care_contact():
    """Quality-controlled twin of journey_clinical.maternity_care_contact."""
    df = _qc(
        "clinical_maternity_care_contact",
        CLINICAL_MATERNITY_CARE_CONTACT_SELECT,
        date_flags=["event_after_death_30d", "event_before_birth"],
    )
    return _with_comments(df, CLINICAL_MATERNITY_CARE_CONTACT_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.medication_admin ====

CLINICAL_MEDICATION_ADMIN_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",

    # 1 rows point at a person_id the spine does not have. The pointer is nulled so it
    # cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    "CASE WHEN NOT `person_id_resolved` THEN NULL ELSE `person_id` END AS `person_id`",
    "`identity_status` AS `identity_status`",

    # 360,093 rows point at a encounter_id the spine does not have. The pointer is nulled so
    # it cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    "CASE WHEN NOT `encounter_id_resolved` THEN NULL ELSE `encounter_id` END AS `encounter_id`",
    "`event_datetime` AS `event_datetime`",
    "`event_end_datetime` AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`medication_code` AS `medication_code`",
    "`medication_order_id` AS `medication_order_id`",
    "`administration_status_code` AS `administration_status_code`",
    "`administration_status_display` AS `administration_status_display`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 2,644 of 60,094,539 rows
    # (0.0044%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`source_event_type_code` AS STRING))) = '0' THEN NULL ELSE `source_event_type_code` END AS `source_event_type_code`",
    "`source_event_type_display` AS `source_event_type_display`",
    "`status_history` AS `status_history`",
    "`status_history_count` AS `status_history_count`",

    # A magnitude above 1e12 is outside any scale this field is measured on, so the number
    # carries no meaning even though something was recorded. Hit 2 of 60,094,539 rows
    # (3.33e-06%) when profiled on 2026-08-24.
    #
    # A negative amount is not possible for a dose, quantity or score, and nothing in the
    # row says what the intended magnitude was. Hit 5 of 60,094,539 rows (8.32e-06%) when
    # profiled on 2026-08-24.
    "CASE WHEN ABS(CAST(`dose_value` AS DOUBLE)) > 1e12 OR `dose_value` < 0 THEN NULL ELSE `dose_value` END AS `dose_value`",
    "`dose_unit` AS `dose_unit`",
    "`initial_dose_value` AS `initial_dose_value`",
    "`initial_dose_unit` AS `initial_dose_unit`",
    "`dose_in_mg` AS `dose_in_mg`",
    "`dose_in_ml` AS `dose_in_ml`",
    "`dose_standardization_status` AS `dose_standardization_status`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 2,511 of 60,094,539 rows
    # (0.00418%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`route_code` AS STRING))) = '0' THEN NULL ELSE `route_code` END AS `route_code`",
    "`route_display` AS `route_display`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 44,278,106 of 60,094,539 rows
    # (73.7%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`site_code` AS STRING))) = '0' THEN NULL ELSE `site_code` END AS `site_code`",
    "`site_display` AS `site_display`",
    "`infused_volume` AS `infused_volume`",
    "`infused_volume_unit` AS `infused_volume_unit`",
    "`infusion_rate` AS `infusion_rate`",
    "`infusion_rate_unit` AS `infusion_rate_unit`",
    "`ingredients` AS `ingredients`",
    "`ingredient_count` AS `ingredient_count`",
    "`performer_practitioner_id` AS `performer_practitioner_id`",
    "`verifier_practitioner_id` AS `verifier_practitioner_id`",
    "`location_id` AS `location_id`",
    "`organization_id` AS `organization_id`",
    "`scheduled_datetime` AS `scheduled_datetime`",

    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 2 of 60,094,539 rows (3.33e-06%) when profiled on 2026-08-24.
    #
    # Stamps landing in January 1970 carry the same millisecond-epoch defect found on
    # clinical_score and vital_sign, but nothing here corroborates a rescaled value the way
    # the encounter window does there, so the stamp is nulled rather than reconstructed. It
    # affected 19,076 rows. The catalogue's date rules stop at 1901 and do not reach this.
    "CASE WHEN CAST(`performed_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS OR YEAR(CAST(`performed_datetime` AS TIMESTAMP)) = 1970 THEN NULL ELSE `performed_datetime` END AS `performed_datetime`",
    "`verified_datetime` AS `verified_datetime`",
    "`order_status_code` AS `order_status_code`",
    "`order_status_display` AS `order_status_display`",
    "`prn_ind` AS `prn_ind`",
    "`iv_ind` AS `iv_ind`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",

    # 2100-12-31 is the far-future marker the source writes to mean 'no end yet'. The
    # absence is what the row means, and NULL states it without putting a fictional date
    # into a range comparison. Hit 8,419 of 60,094,539 rows (0.014%) when profiled on
    # 2026-08-24.
    "CASE WHEN CAST(`record_status_effective_to` AS DATE) = DATE'2100-12-31' THEN NULL ELSE `record_status_effective_to` END AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
    "`event_before_birth` AS `event_before_birth`",
    "`event_after_death_30d` AS `event_after_death_30d`",
]

CLINICAL_MEDICATION_ADMIN_MANDATORY_RULES = {
    # The research surface. identity_status = 'resolved' keeps the 60,094,539 rows of
    # 60,094,539 that are current and attributable. Superseded versions and rows whose
    # identity was never resolved are not research data, and a consumer who wants them has
    # silver.
    "research_surface": "(identity_status = 'resolved')",
}

CLINICAL_MEDICATION_ADMIN_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 0 of 60,094,539 at the profile.
    "gold.clinical.medication_admin.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 9,527 of 60,094,539 rows (0.0159%) when profiled on 2026-08-24.
    "gold.clinical.medication_admin.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 1,772 of 60,094,539 rows (0.00295%) when profiled on 2026-08-24.
    "gold.clinical.medication_admin.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",
}

CLINICAL_MEDICATION_ADMIN_COLUMN_COMMENTS = {
    "patient_event_id": "Stable administration event identifier.",
    "fact_row_id": "Storage row identifier.",
    "subject_key": "Always-populated subject key.",
    "subject_id_system": "Subject-key identifier system.",
    "person_id": "Resolved person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference.",
    "event_datetime": "Administration start or performed time.",
    "event_end_datetime": "Administration end time.",
    "source_coding_system": "Source medication coding system.",
    "source_code": "Source order-synonym identifier.",
    "source_display": "Source medication display.",
    "medication_code": "Source and mapped medication codings.",
    "medication_order_id": "Linked governed medication order when present.",
    "administration_status_code": "Current source result status code.",
    "administration_status_display": "Current source result status display.",
    "source_event_type_code": "Source medication event type code.",
    "source_event_type_display": "Source medication event type display.",
    "status_history": "Ordered administration and order-status milestones.",
    "status_history_count": "Number of retained status milestones.",
    "dose_value": "Effective administered dose.",
    "dose_unit": "Administered dose unit.",
    "initial_dose_value": "Initially documented dose.",
    "initial_dose_unit": "Initially documented dose unit.",
    "dose_in_mg": "Bronze-standardized milligram dose.",
    "dose_in_ml": "Bronze-standardized millilitre dose.",
    "dose_standardization_status": "Bronze dose-standardization status.",
    "route_code": "Administration route code.",
    "route_display": "Administration route display.",
    "site_code": "Administration site code.",
    "site_display": "Administration site display.",
    "infused_volume": "Infused volume.",
    "infused_volume_unit": "Infused volume unit.",
    "infusion_rate": "Infusion rate.",
    "infusion_rate_unit": "Infusion rate unit.",
    "ingredients": "Ordered ingredient-component evidence.",
    "ingredient_count": "Number of retained ingredient rows.",
    "performer_practitioner_id": "Administration performer.",
    "verifier_practitioner_id": "Administration verifier.",
    "location_id": "Administering nurse-unit location.",
    "organization_id": "Source organization reference.",
    "scheduled_datetime": "Scheduled administration time.",
    "performed_datetime": "Performed administration time.",
    "verified_datetime": "Verification time.",
    "order_status_code": "Linked source order status code.",
    "order_status_display": "Linked source order status display.",
    "prn_ind": "As-needed indicator.",
    "iv_ind": "Intravenous indicator.",
    "record_status": "Normalized source-record lifecycle.",
    "record_status_effective_from": "Lifecycle start.",
    "record_status_effective_to":
        "Lifecycle end. Gold QC transform rules: gold.clinical.medication_admin.record_status_effective_to.open_sentinel.",
    "confidentiality_code": "Security classification.",
    "vip_ind": "VIP indicator.",
    "withheld_identity_ind": "Withheld identity indicator.",
    "source_feed": "Registered source feed.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Latest native update timestamp.",
    "loaded_at": "Latest bronze load timestamp.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.medication_admin"),
    comment=(
        "One Millennium medication-administration event with current state and ordered "
        "lifecycle evidence. Gold QC twin of the silver product: 8 columns are repaired or "
        "nulled, 1 rule(s) drop rows, 3 check(s) are advisory. Each rule states its reason in "
        "the pipeline notebook, and Lakeflow expectation metrics report what every rule "
        "matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_MEDICATION_ADMIN_MANDATORY_RULES)
@_expect_all(CLINICAL_MEDICATION_ADMIN_ADVISORY_RULES)
def gold_clinical_medication_admin():
    """Quality-controlled twin of journey_clinical.medication_admin."""
    df = _qc("clinical_medication_admin", CLINICAL_MEDICATION_ADMIN_SELECT)
    return _with_comments(df, CLINICAL_MEDICATION_ADMIN_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.medication_dispense ====

CLINICAL_MEDICATION_DISPENSE_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",

    # Nulled because all 7,673,150 rows carry 1970-01-01 with only a time of day, so the
    # date was lost at source rather than mangled and cannot be recovered.
    "CASE WHEN CAST(`event_datetime` AS DATE) = DATE'1970-01-01' THEN NULL ELSE `event_datetime` END AS `event_datetime`",
    "`event_end_datetime` AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`medication_code` AS `medication_code`",
    "`status_code` AS `status_code`",
    "`issue_type` AS `issue_type`",
    "`issue_category` AS `issue_category`",
    "`quantity` AS `quantity`",
    "`quantity_unit` AS `quantity_unit`",
    "`issued_containers` AS `issued_containers`",
    "`units_per_container` AS `units_per_container`",
    "`drug_form` AS `drug_form`",
    "`drug_strength` AS `drug_strength`",
    "`location_id` AS `location_id`",
    "`issue_value_gbp` AS `issue_value_gbp`",
    "`source_transaction_identifier` AS `source_transaction_identifier`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",

    # Silver derived this flag by comparing event_datetime with the person's own dates,
    # before the rules above corrected event_datetime. On the rows where event_datetime
    # changed, the flag describes a timestamp gold no longer publishes. This product carries
    # a VARIANT column, which rules out the spine join gold would need to recompute the
    # flag, so it is nulled where the value beneath it moved rather than left asserting
    # something stale.
    "CASE WHEN CAST(`event_datetime` AS DATE) = DATE'1970-01-01' THEN NULL ELSE `event_before_birth` END AS `event_before_birth`",
]

CLINICAL_MEDICATION_DISPENSE_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 7,616,956 rows of 7,673,150
    # that are current and attributable; identity_status = 'resolved' keeps the 7,146,842
    # rows of 7,673,150 that are current and attributable. Superseded versions and rows
    # whose identity was never resolved are not research data, and a consumer who wants them
    # has silver.
    "research_surface": "(identity_status = 'resolved') AND (record_status = 'active')",
}

CLINICAL_MEDICATION_DISPENSE_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 526,308 of 7,673,150 at the profile.
    "gold.clinical.medication_dispense.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 56,194 of 7,673,150 at the profile.
    "gold.clinical.medication_dispense.record_status.default_view_active":
        "record_status = 'active'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 3,063,192 of 7,673,150 rows (39.9%) when profiled on 2026-08-24.
    "gold.clinical.medication_dispense.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",
}

CLINICAL_MEDICATION_DISPENSE_COLUMN_COMMENTS = {
    "patient_event_id": "Stable promotion-safe event identifier.",
    "fact_row_id": "Storage-row identifier equal to patient_event_id.",
    "subject_key": "Always-populated peppered subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved person identifier when available.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter context when a source supplies one.",
    "event_datetime": "Source issue timestamp.",
    "event_end_datetime": "Dispense end timestamp when supplied.",
    "source_coding_system": "Verbatim JAC drug coding system.",
    "source_code": "Verbatim JAC drug identifier.",
    "source_display": "Verbatim JAC drug description.",
    "medication_code": "Source JAC and mapped dm+d medication codings.",
    "status_code": "Source-presence-derived dispense status.",
    "issue_type": "Raw JAC issue type.",
    "issue_category": "Governed JAC transaction category; only ISSUE is typed here.",
    "quantity": "Best-effort parsed total units.",
    "quantity_unit": "Source dose-unit description.",
    "issued_containers": "Parsed source container count.",
    "units_per_container": "Parsed source units per container.",
    "drug_form": "Source drug form.",
    "drug_strength": "Source drug strength.",
    "location_id":
        "Resolved care-site reference when a unique source location match exists.",
    "issue_value_gbp":
        "Source-recorded issue value including legitimate negative values outside this route.",
    "source_transaction_identifier": "Raw JAC dailyissues traceability key.",
    "record_status": "Normalized source lifecycle status.",
    "record_status_effective_from": "Upstream source update timestamp.",
    "record_status_effective_to": "Retraction timestamp when source presence is false.",
    "confidentiality_code": "Source confidentiality code when supplied.",
    "vip_ind": "VIP indicator when supplied.",
    "withheld_identity_ind": "Withheld-identity indicator when supplied.",
    "source_feed": "Registered source feed owning the typed fact.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.medication_dispense"),
    comment=(
        "One JAC issue/dispense transaction admitted directly by the pharmacy ISSUE route. "
        "Gold QC twin of the silver product: 2 columns are repaired or nulled, 1 rule(s) drop "
        "rows, 3 check(s) are advisory. Each rule states its reason in the pipeline notebook, "
        "and Lakeflow expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_MEDICATION_DISPENSE_MANDATORY_RULES)
@_expect_all(CLINICAL_MEDICATION_DISPENSE_ADVISORY_RULES)
def gold_clinical_medication_dispense():
    """Quality-controlled twin of journey_clinical.medication_dispense."""
    df = _qc("clinical_medication_dispense", CLINICAL_MEDICATION_DISPENSE_SELECT)
    return _with_comments(df, CLINICAL_MEDICATION_DISPENSE_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.medication_order ====

CLINICAL_MEDICATION_ORDER_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",

    # 2 rows point at a person_id the spine does not have. The pointer is nulled so it
    # cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    "CASE WHEN NOT `person_id_resolved` THEN NULL ELSE `person_id` END AS `person_id`",
    "`identity_status` AS `identity_status`",

    # 1,134,853 rows point at a encounter_id the spine does not have. The pointer is nulled
    # so it cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    "CASE WHEN NOT `encounter_id_resolved` THEN NULL ELSE `encounter_id` END AS `encounter_id`",
    "`event_datetime` AS `event_datetime`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 34 of 365,848,704 rows (9.29e-06%) when profiled on 2026-08-24.
    #
    # A year of 9999 or beyond is the 'never expires' marker. It is a flag written in a date
    # column, so it is nulled rather than compared as a date. Hit 27 of 365,848,704 rows
    # (7.38e-06%) when profiled on 2026-08-24.
    #
    # event_end_datetime cannot precede event_datetime. The start is the better-attested of
    # the two, so the end is what goes and the row keeps its event_datetime. Hit 100,626 of
    # 365,848,704 rows (0.0275%) when profiled on 2026-08-24.
    "CASE WHEN `event_datetime` IS NOT NULL AND `event_end_datetime` IS NOT NULL AND `event_datetime` > `event_end_datetime` THEN NULL ELSE CASE WHEN YEAR(CAST(`event_end_datetime` AS DATE)) >= 9999 OR CAST(`event_end_datetime` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`event_end_datetime` AS DATE)) < 9999 THEN NULL ELSE `event_end_datetime` END END AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`medication_code` AS `medication_code`",
    "`order_status_code` AS `order_status_code`",
    "`order_status_display` AS `order_status_display`",
    "`department_status_code` AS `department_status_code`",
    "`department_status_display` AS `department_status_display`",
    "`active_status_code` AS `active_status_code`",
    "`active_status_display` AS `active_status_display`",
    "`intent_code` AS `intent_code`",
    "`medication_order_type_code` AS `medication_order_type_code`",
    "`medication_order_type_display` AS `medication_order_type_display`",
    "`authored_datetime` AS `authored_datetime`",
    "`effective_start_datetime` AS `effective_start_datetime`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 34 of 365,848,704 rows (9.29e-06%) when profiled on 2026-08-24.
    #
    # A year of 9999 or beyond is the 'never expires' marker. It is a flag written in a date
    # column, so it is nulled rather than compared as a date. Hit 27 of 365,848,704 rows
    # (7.38e-06%) when profiled on 2026-08-24.
    "CASE WHEN YEAR(CAST(`projected_stop_datetime` AS DATE)) >= 9999 OR CAST(`projected_stop_datetime` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`projected_stop_datetime` AS DATE)) < 9999 THEN NULL ELSE `projected_stop_datetime` END AS `projected_stop_datetime`",
    "`discontinued_datetime` AS `discontinued_datetime`",
    "`frequency_id` AS `frequency_id`",
    "`prn_ind` AS `prn_ind`",
    "`iv_ind` AS `iv_ind`",
    "`suspend_ind` AS `suspend_ind`",
    "`resume_ind` AS `resume_ind`",
    "`discontinue_ind` AS `discontinue_ind`",
    "`requester_practitioner_id` AS `requester_practitioner_id`",
    "`organization_id` AS `organization_id`",
    "`clinical_display_line` AS `clinical_display_line`",
    "`order_detail_display_line` AS `order_detail_display_line`",
    "`status_history` AS `status_history`",
    "`status_history_count` AS `status_history_count`",
    "`ingredients` AS `ingredients`",
    "`ingredient_count` AS `ingredient_count`",
    "`order_details` AS `order_details`",
    "`order_detail_count` AS `order_detail_count`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
    "`event_before_birth` AS `event_before_birth`",
    "`event_after_death_30d` AS `event_after_death_30d`",
]

CLINICAL_MEDICATION_ORDER_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 365,848,704 rows of
    # 365,848,704 that are current and attributable; identity_status = 'resolved' keeps the
    # 365,848,704 rows of 365,848,704 that are current and attributable. Superseded versions
    # and rows whose identity was never resolved are not research data, and a consumer who
    # wants them has silver.
    "research_surface": "(identity_status = 'resolved') AND (record_status = 'active')",
}

CLINICAL_MEDICATION_ORDER_ADVISORY_RULES = {
    # This bounds a period of validity, and a future end is exactly how the source says a
    # record is still current -- nulling it would assert the record is valid forever, which
    # is a stronger and worse claim than the one being corrected. Seen on 464 of 365,848,704
    # rows (0.000127%) when profiled on 2026-08-24.
    "gold.clinical.medication_order.effective_start_datetime.future_owner":
        "NOT COALESCE((CAST(`effective_start_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",

    # Counted rather than nulled because an order that is still active has a planned stop
    # date, which is in the future by construction. Seen on 7,281 of 365,848,704 rows
    # (0.00199%) when profiled on 2026-08-24.
    "gold.clinical.medication_order.event_end_datetime.future_owner":
        "NOT COALESCE((CAST(`event_end_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",

    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 0 of 365,848,704 at the profile.
    "gold.clinical.medication_order.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # This bounds a period of validity, and a future end is exactly how the source says a
    # record is still current -- nulling it would assert the record is valid forever, which
    # is a stronger and worse claim than the one being corrected. Seen on 7,281 of
    # 365,848,704 rows (0.00199%) when profiled on 2026-08-24.
    "gold.clinical.medication_order.projected_stop_datetime.future_owner":
        "NOT COALESCE((CAST(`projected_stop_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",

    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 0 of 365,848,704 at the profile.
    "gold.clinical.medication_order.record_status.default_view_active":
        "record_status = 'active'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 180,555 of 365,848,704 rows (0.0494%) when profiled on
    # 2026-08-24.
    "gold.clinical.medication_order.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 27,575 of 365,848,704 rows (0.00754%) when profiled on
    # 2026-08-24.
    "gold.clinical.medication_order.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",
}

CLINICAL_MEDICATION_ORDER_COLUMN_COMMENTS = {
    "patient_event_id": "Stable medication-order event identifier.",
    "fact_row_id": "Storage row identifier.",
    "subject_key": "Always-populated subject key.",
    "subject_id_system": "Subject-key identifier system.",
    "person_id": "Resolved person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference.",
    "event_datetime": "Order authored or start time.",
    "event_end_datetime":
        "Projected stop or discontinue time. Gold QC transform rules: gold.clinical.medication_order.event_end_datetime.beyond_2100_below_9999_owner, gold.clinical.medication_order.event_end_datetime.never_stop_sentinel.",
    "source_coding_system": "Source medication coding system.",
    "source_code": "Source medication synonym identifier.",
    "source_display": "Source medication display.",
    "medication_code": "Source medication CodeableConcept.",
    "order_status_code": "Current source order status code.",
    "order_status_display": "Current source order status display.",
    "department_status_code": "Department workflow status code.",
    "department_status_display": "Department workflow status display.",
    "active_status_code": "Source active-status code.",
    "active_status_display": "Source active-status display.",
    "intent_code": "Medication request intent.",
    "medication_order_type_code": "Source medication-order type code.",
    "medication_order_type_display": "Source medication-order type display.",
    "authored_datetime": "Original order time.",
    "effective_start_datetime": "Current effective start.",
    "projected_stop_datetime":
        "Projected stop time. Gold QC transform rules: gold.clinical.medication_order.projected_stop_datetime.beyond_2100_below_9999_owner, gold.clinical.medication_order.projected_stop_datetime.never_stop_sentinel.",
    "discontinued_datetime": "Discontinue effective time.",
    "frequency_id": "Source frequency identifier.",
    "prn_ind": "As-needed indicator.",
    "iv_ind": "Intravenous indicator.",
    "suspend_ind": "Source suspended indicator.",
    "resume_ind": "Source resumed indicator.",
    "discontinue_ind": "Source discontinued indicator.",
    "requester_practitioner_id": "Last updating provider reference.",
    "organization_id": "Source organization reference.",
    "clinical_display_line": "Source clinical display line.",
    "order_detail_display_line": "Source order-detail display line.",
    "status_history": "Ordered source action/status history.",
    "status_history_count": "Number of retained action rows.",
    "ingredients": "Ordered medication-order ingredients.",
    "ingredient_count": "Number of retained ingredient rows.",
    "order_details": "Ordered latest-action detail rows.",
    "order_detail_count": "Number of retained detail rows.",
    "record_status": "Normalized source-record lifecycle.",
    "record_status_effective_from": "Lifecycle start.",
    "record_status_effective_to": "Lifecycle end.",
    "confidentiality_code": "Security classification.",
    "vip_ind": "VIP indicator.",
    "withheld_identity_ind": "Withheld identity indicator.",
    "source_feed": "Registered source feed.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Latest native update timestamp.",
    "loaded_at": "Latest bronze load timestamp.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.medication_order"),
    comment=(
        "One Millennium medication order with current state plus ordered action and detail "
        "evidence. Gold QC twin of the silver product: 4 columns are repaired or nulled, 1 "
        "rule(s) drop rows, 7 check(s) are advisory. Each rule states its reason in the "
        "pipeline notebook, and Lakeflow expectation metrics report what every rule matched "
        "on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_MEDICATION_ORDER_MANDATORY_RULES)
@_expect_all(CLINICAL_MEDICATION_ORDER_ADVISORY_RULES)
def gold_clinical_medication_order():
    """Quality-controlled twin of journey_clinical.medication_order."""
    df = _qc("clinical_medication_order", CLINICAL_MEDICATION_ORDER_SELECT)
    return _with_comments(df, CLINICAL_MEDICATION_ORDER_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.medication_supply ====

CLINICAL_MEDICATION_SUPPLY_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",
    "`event_datetime` AS `event_datetime`",
    "`event_end_datetime` AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`request_key` AS `request_key`",
    "`item_seq` AS `item_seq`",
    "`request_status_code` AS `request_status_code`",
    "`request_status_display` AS `request_status_display`",
    "`item_status_code` AS `item_status_code`",
    "`item_status_display` AS `item_status_display`",
    "`item_type` AS `item_type`",
    "`item_description` AS `item_description`",
    "`pack_description` AS `pack_description`",
    "`quantity_requested` AS `quantity_requested`",
    "`quantity_original` AS `quantity_original`",
    "`quantity_delivered` AS `quantity_delivered`",
    "`order_unit` AS `order_unit`",
    "`label_directions` AS `label_directions`",
    "`nfd_reason` AS `nfd_reason`",
    "`supply_start_date` AS `supply_start_date`",
    "`supply_interval` AS `supply_interval`",
    "`supply_period` AS `supply_period`",
    "`request_item_count` AS `request_item_count`",
    "`request_complete_item_count` AS `request_complete_item_count`",
    "`request_released_date` AS `request_released_date`",
    "`location_name` AS `location_name`",
    "`cost_centre_name` AS `cost_centre_name`",
    "`indication` AS `indication`",
    "`clinic` AS `clinic`",
    "`dmd_vtm_concept_id` AS `dmd_vtm_concept_id`",
    "`dmd_vtm_code` AS `dmd_vtm_code`",
    "`dmd_vtm_name` AS `dmd_vtm_name`",
    "`drug_mapping_method` AS `drug_mapping_method`",
    "`care_site_cd` AS `care_site_cd`",
    "`care_site_match_method` AS `care_site_match_method`",
    "`lnkpid` AS `lnkpid`",
    "`name_key` AS `name_key`",
    "`mrn_candidates` AS `mrn_candidates`",
    "`nhs_candidates` AS `nhs_candidates`",
    "`person_match_method` AS `person_match_method`",
    "`person_match_status` AS `person_match_status`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`fact_category` AS `fact_category`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
]

CLINICAL_MEDICATION_SUPPLY_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 271,725 rows of 271,752 that
    # are current and attributable; identity_status = 'resolved' keeps the 269,250 rows of
    # 271,752 that are current and attributable. Superseded versions and rows whose identity
    # was never resolved are not research data, and a consumer who wants them has silver.
    "research_surface": "(identity_status = 'resolved') AND (record_status = 'active')",
}

CLINICAL_MEDICATION_SUPPLY_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 2,502 of 271,752 at the profile.
    "gold.clinical.medication_supply.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 27 of 271,752 at the profile.
    "gold.clinical.medication_supply.record_status.default_view_active":
        "record_status = 'active'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 489 of 271,752 rows (0.18%) when profiled on 2026-08-24.
    "gold.clinical.medication_supply.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",
}

CLINICAL_MEDICATION_SUPPLY_COLUMN_COMMENTS = {
    "patient_event_id": "Stable row identifier.",
    "fact_row_id": "Stable row identifier.",
    "subject_key": "Published field.",
    "subject_id_system": "Published field.",
    "person_id": "Published field.",
    "identity_status": "Published field.",
    "encounter_id": "Published field.",
    "event_datetime": "Published field.",
    "event_end_datetime": "Published field.",
    "source_coding_system": "Published field.",
    "source_code": "Published field.",
    "source_display": "Published field.",
    "request_key": "Published field.",
    "item_seq": "Published field.",
    "request_status_code": "Published field.",
    "request_status_display": "Published field.",
    "item_status_code": "Published field.",
    "item_status_display": "Published field.",
    "item_type": "Published field.",
    "item_description": "Published field.",
    "pack_description": "Published field.",
    "quantity_requested": "Published field.",
    "quantity_original": "Published field.",
    "quantity_delivered": "Published field.",
    "order_unit": "Published field.",
    "label_directions": "Published field.",
    "nfd_reason": "Published field.",
    "supply_start_date": "Published field.",
    "supply_interval": "Published field.",
    "supply_period": "Published field.",
    "request_item_count": "Published field.",
    "request_complete_item_count": "Published field.",
    "request_released_date": "Published field.",
    "location_name": "Published field.",
    "cost_centre_name": "Published field.",
    "indication": "Published field.",
    "clinic": "Published field.",
    "dmd_vtm_concept_id": "Published field.",
    "dmd_vtm_code": "Published field.",
    "dmd_vtm_name": "Published field.",
    "drug_mapping_method": "Published field.",
    "care_site_cd": "Published field.",
    "care_site_match_method": "Published field.",
    "lnkpid": "Direct identifier published and IG-governed at serve time",
    "name_key": "Direct identifier published and IG-governed at serve time",
    "mrn_candidates": "Published field.",
    "nhs_candidates": "Published field.",
    "person_match_method": "Published field.",
    "person_match_status": "Published field.",
    "record_status": "Published field.",
    "record_status_effective_from": "Published field.",
    "record_status_effective_to": "Published field.",
    "confidentiality_code": "Published field.",
    "vip_ind": "Published field.",
    "withheld_identity_ind": "Published field.",
    "fact_category": "Published field.",
    "source_feed": "Published field.",
    "load_batch_id": "Published field.",
    "source_update_timestamp": "Published field.",
    "loaded_at": "Published field.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.medication_supply"),
    comment=(
        "One JAC homecare supply-request item; supply, never administration. Gold QC twin of "
        "the silver product: 0 columns are repaired or nulled, 1 rule(s) drop rows, 3 "
        "check(s) are advisory. Each rule states its reason in the pipeline notebook, and "
        "Lakeflow expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_MEDICATION_SUPPLY_MANDATORY_RULES)
@_expect_all(CLINICAL_MEDICATION_SUPPLY_ADVISORY_RULES)
def gold_clinical_medication_supply():
    """Quality-controlled twin of journey_clinical.medication_supply."""
    df = _qc(
        "clinical_medication_supply",
        CLINICAL_MEDICATION_SUPPLY_SELECT,
        date_flags=["event_after_death_30d"],
    )
    return _with_comments(df, CLINICAL_MEDICATION_SUPPLY_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.microbiology_isolate ====

CLINICAL_MICROBIOLOGY_ISOLATE_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",
    "`event_datetime` AS `event_datetime`",
    "`event_end_datetime` AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`pathology_result_id` AS `pathology_result_id`",
    "`report_version_id` AS `report_version_id`",
    "`specimen_type_code` AS `specimen_type_code`",
    "`organism_text` AS `organism_text`",
    "`organism_snomed_code` AS `organism_snomed_code`",
    "`organism_omop_concept_id` AS `organism_omop_concept_id`",
    "`suspected_ind` AS `suspected_ind`",
    "`growth_grade` AS `growth_grade`",
    "`lifecycle_status` AS `lifecycle_status`",
    "`is_current` AS `is_current`",
    "`research_qi_only` AS `research_qi_only`",
    "`specimen_id` AS `specimen_id`",
    "`accession_identifier` AS `accession_identifier`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
]

CLINICAL_MICROBIOLOGY_ISOLATE_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 0 rows of 0 that are current
    # and attributable; identity_status = 'resolved' keeps the 0 rows of 0 that are current
    # and attributable. Superseded versions and rows whose identity was never resolved are
    # not research data, and a consumer who wants them has silver.
    "research_surface": "(identity_status = 'resolved') AND (record_status = 'active')",
}

CLINICAL_MICROBIOLOGY_ISOLATE_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 0 of 0 at the profile.
    "gold.clinical.microbiology_isolate.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 0 of 0 at the profile.
    "gold.clinical.microbiology_isolate.record_status.default_view_active":
        "record_status = 'active'",
}

CLINICAL_MICROBIOLOGY_ISOLATE_COLUMN_COMMENTS = {
    "patient_event_id": "Stable isolate event identifier.",
    "fact_row_id": "Storage-row identifier.",
    "subject_key": "Best available subject key.",
    "subject_id_system": "Subject-key system.",
    "person_id": "Resolved person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference; unavailable at this grain.",
    "event_datetime": "Accession sample/report/request fallback time.",
    "event_end_datetime": "Event end time.",
    "source_coding_system": "Organism coding namespace.",
    "source_code": "Organism text or code.",
    "source_display": "Organism display.",
    "pathology_result_id": "Linked pathology-result fact.",
    "report_version_id": "Related report version.",
    "specimen_type_code": "Source specimen type.",
    "organism_text": "Organism as reported.",
    "organism_snomed_code": "SNOMED organism code.",
    "organism_omop_concept_id": "Standard OMOP organism concept.",
    "suspected_ind": "Whether the organism is hedged or suspected.",
    "growth_grade": "Reported growth grade.",
    "lifecycle_status": "Inherited source lifecycle.",
    "is_current": "Whether the isolate remains current.",
    "research_qi_only": "Research/QI release flag.",
    "specimen_id": "Alias-resolved accession specimen reference.",
    "accession_identifier": "Alias-resolved accession identifier.",
    "record_status": "Normalized isolate lifecycle.",
    "record_status_effective_from": "Status validity start.",
    "record_status_effective_to": "Status validity end.",
    "confidentiality_code": "Security code.",
    "vip_ind": "VIP indicator.",
    "withheld_identity_ind": "Withheld identity indicator.",
    "source_feed": "Owning feed.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze material-change timestamp.",
}

@dp.materialized_view(
    name=_n("gold_clinical.microbiology_isolate"),
    comment=(
        "One organism or isolate finding per accession. Schema-first and empty_by_design in "
        "both catalogs on 2026-08-18. Gold QC twin of the silver product: 0 columns are "
        "repaired or nulled, 1 rule(s) drop rows, 2 check(s) are advisory. Each rule states "
        "its reason in the pipeline notebook, and Lakeflow expectation metrics report what "
        "every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_MICROBIOLOGY_ISOLATE_MANDATORY_RULES)
@_expect_all(CLINICAL_MICROBIOLOGY_ISOLATE_ADVISORY_RULES)
def gold_clinical_microbiology_isolate():
    """Quality-controlled twin of journey_clinical.microbiology_isolate."""
    df = _qc("clinical_microbiology_isolate", CLINICAL_MICROBIOLOGY_ISOLATE_SELECT)
    return _with_comments(df, CLINICAL_MICROBIOLOGY_ISOLATE_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.neonatal_care_day ====

CLINICAL_NEONATAL_CARE_DAY_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",

    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 1 of 507,781 rows (0.000197%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `event_datetime` END AS `event_datetime`",
    "`event_end_datetime` AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`entity_id` AS `entity_id`",

    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 1 of 507,781 rows (0.000197%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`activity_date` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `activity_date` END AS `activity_date`",
    "`ward_location` AS `ward_location`",
    "`unit_function` AS `unit_function`",
    "`critical_care_start` AS `critical_care_start`",
    "`critical_care_discharge` AS `critical_care_discharge`",
    "`activity_codes_json` AS `activity_codes_json`",
    "`high_cost_drugs_json` AS `high_cost_drugs_json`",
    "`episode_link_status` AS `episode_link_status`",
    "`neonatal_episode_id` AS `neonatal_episode_id`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`fact_category` AS `fact_category`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
]

CLINICAL_NEONATAL_CARE_DAY_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 507,781 rows of 507,781 that
    # are current and attributable; identity_status = 'resolved' keeps the 501,933 rows of
    # 507,781 that are current and attributable. Superseded versions and rows whose identity
    # was never resolved are not research data, and a consumer who wants them has silver.
    "research_surface": "(identity_status = 'resolved') AND (record_status = 'active')",
}

CLINICAL_NEONATAL_CARE_DAY_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 5,848 of 507,781 at the profile.
    "gold.clinical.neonatal_care_day.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 0 of 507,781 at the profile.
    "gold.clinical.neonatal_care_day.record_status.default_view_active":
        "record_status = 'active'",

    # This bounds a period of validity, and a future end is exactly how the source says a
    # record is still current -- nulling it would assert the record is valid forever, which
    # is a stronger and worse claim than the one being corrected. Seen on 1 of 507,781 rows
    # (0.000197%) when profiled on 2026-08-24.
    "gold.clinical.neonatal_care_day.record_status_effective_from.future_owner":
        "NOT COALESCE((CAST(`record_status_effective_from` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 48 of 507,781 rows (0.00945%) when profiled on 2026-08-24.
    "gold.clinical.neonatal_care_day.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 17,443 of 507,781 rows (3.44%) when profiled on 2026-08-24.
    "gold.clinical.neonatal_care_day.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",
}

CLINICAL_NEONATAL_CARE_DAY_COLUMN_COMMENTS = {
    "patient_event_id": "Published field patient_event_id.",
    "fact_row_id": "Published field fact_row_id.",
    "subject_key": "Published field subject_key.",
    "subject_id_system": "Published field subject_id_system.",
    "person_id": "Published field person_id.",
    "identity_status": "Published field identity_status.",
    "encounter_id": "Published field encounter_id.",
    "event_datetime": "Published field event_datetime.",
    "event_end_datetime": "Published field event_end_datetime.",
    "source_coding_system": "Published field source_coding_system.",
    "source_code": "Published field source_code.",
    "source_display": "Published field source_display.",
    "entity_id": "Published field entity_id.",
    "activity_date": "Published field activity_date.",
    "ward_location": "Published field ward_location.",
    "unit_function": "Published field unit_function.",
    "critical_care_start": "Published field critical_care_start.",
    "critical_care_discharge": "Published field critical_care_discharge.",
    "activity_codes_json": "Published field activity_codes_json.",
    "high_cost_drugs_json": "Published field high_cost_drugs_json.",
    "episode_link_status": "Published field episode_link_status.",
    "neonatal_episode_id": "Published field neonatal_episode_id.",
    "record_status": "Published field record_status.",
    "record_status_effective_from": "Published field record_status_effective_from.",
    "record_status_effective_to": "Published field record_status_effective_to.",
    "confidentiality_code": "Published field confidentiality_code.",
    "vip_ind": "Published field vip_ind.",
    "withheld_identity_ind": "Published field withheld_identity_ind.",
    "fact_category": "Published field fact_category.",
    "source_feed": "Published field source_feed.",
    "load_batch_id": "Published field load_batch_id.",
    "source_update_timestamp": "Published field source_update_timestamp.",
    "loaded_at": "Published field loaded_at.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.neonatal_care_day"),
    comment=(
        "One admitted BadgerNet neonatal critical-care day. Gold QC twin of the silver "
        "product: 2 columns are repaired or nulled, 1 rule(s) drop rows, 5 check(s) are "
        "advisory. Each rule states its reason in the pipeline notebook, and Lakeflow "
        "expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_NEONATAL_CARE_DAY_MANDATORY_RULES)
@_expect_all(CLINICAL_NEONATAL_CARE_DAY_ADVISORY_RULES)
def gold_clinical_neonatal_care_day():
    """Quality-controlled twin of journey_clinical.neonatal_care_day."""
    df = _qc(
        "clinical_neonatal_care_day",
        CLINICAL_NEONATAL_CARE_DAY_SELECT,
        date_flags=["event_after_death_30d", "event_before_birth"],
    )
    return _with_comments(df, CLINICAL_NEONATAL_CARE_DAY_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.neonatal_episode ====

CLINICAL_NEONATAL_EPISODE_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",
    "`event_datetime` AS `event_datetime`",

    # event_end_datetime cannot precede event_datetime. The start is the better-attested of
    # the two, so the end is what goes and the row keeps its event_datetime. Hit 7 of 36,718
    # rows (0.0191%) when profiled on 2026-08-24.
    "CASE WHEN `event_datetime` IS NOT NULL AND `event_end_datetime` IS NOT NULL AND `event_datetime` > `event_end_datetime` THEN NULL ELSE `event_end_datetime` END AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`entity_id` AS `entity_id`",
    "`badger_unique_id` AS `badger_unique_id`",
    "`mother_person_id` AS `mother_person_id`",
    "`care_location_id` AS `care_location_id`",
    "`care_location_name` AS `care_location_name`",
    "`birth_datetime` AS `birth_datetime`",
    "`birth_datetime_raw` AS `birth_datetime_raw`",
    "`admit_datetime` AS `admit_datetime`",
    "`discharge_datetime` AS `discharge_datetime`",
    "`gestation_weeks` AS `gestation_weeks`",
    "`gestation_days` AS `gestation_days`",
    "`birthweight` AS `birthweight`",
    "`sex` AS `sex`",
    "`final_nnu_outcome` AS `final_nnu_outcome`",
    "`unit_level` AS `unit_level`",
    "`person_link_status` AS `person_link_status`",
    "`record_timestamp` AS `record_timestamp`",
    "`last_update` AS `last_update`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`fact_category` AS `fact_category`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
]

CLINICAL_NEONATAL_EPISODE_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 36,718 rows of 36,718 that
    # are current and attributable; identity_status = 'resolved' keeps the 36,161 rows of
    # 36,718 that are current and attributable. Superseded versions and rows whose identity
    # was never resolved are not research data, and a consumer who wants them has silver.
    "research_surface": "(identity_status = 'resolved') AND (record_status = 'active')",
}

CLINICAL_NEONATAL_EPISODE_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 557 of 36,718 at the profile.
    "gold.clinical.neonatal_episode.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 0 of 36,718 at the profile.
    "gold.clinical.neonatal_episode.record_status.default_view_active":
        "record_status = 'active'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 834 of 36,718 rows (2.27%) when profiled on 2026-08-24.
    "gold.clinical.neonatal_episode.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",
}

CLINICAL_NEONATAL_EPISODE_COLUMN_COMMENTS = {
    "patient_event_id": "Published field patient_event_id.",
    "fact_row_id": "Published field fact_row_id.",
    "subject_key": "Published field subject_key.",
    "subject_id_system": "Published field subject_id_system.",
    "person_id": "Published field person_id.",
    "identity_status": "Published field identity_status.",
    "encounter_id": "Published field encounter_id.",
    "event_datetime": "Published field event_datetime.",
    "event_end_datetime": "Published field event_end_datetime.",
    "source_coding_system": "Published field source_coding_system.",
    "source_code": "Published field source_code.",
    "source_display": "Published field source_display.",
    "entity_id": "Published field entity_id.",
    "badger_unique_id": "Published field badger_unique_id.",
    "mother_person_id": "Published field mother_person_id.",
    "care_location_id": "Published field care_location_id.",
    "care_location_name": "Published field care_location_name.",
    "birth_datetime": "Published field birth_datetime.",
    "birth_datetime_raw": "Published field birth_datetime_raw.",
    "admit_datetime": "Published field admit_datetime.",
    "discharge_datetime": "Published field discharge_datetime.",
    "gestation_weeks": "Published field gestation_weeks.",
    "gestation_days": "Published field gestation_days.",
    "birthweight": "Published field birthweight.",
    "sex": "Published field sex.",
    "final_nnu_outcome": "Published field final_nnu_outcome.",
    "unit_level": "Published field unit_level.",
    "person_link_status": "Published field person_link_status.",
    "record_timestamp": "Published field record_timestamp.",
    "last_update": "Published field last_update.",
    "record_status": "Published field record_status.",
    "record_status_effective_from": "Published field record_status_effective_from.",
    "record_status_effective_to": "Published field record_status_effective_to.",
    "confidentiality_code": "Published field confidentiality_code.",
    "vip_ind": "Published field vip_ind.",
    "withheld_identity_ind": "Published field withheld_identity_ind.",
    "fact_category": "Published field fact_category.",
    "source_feed": "Published field source_feed.",
    "load_batch_id": "Published field load_batch_id.",
    "source_update_timestamp": "Published field source_update_timestamp.",
    "loaded_at": "Published field loaded_at.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.neonatal_episode"),
    comment=(
        "One curated BadgerNet neonatal episode for the baby subject. Gold QC twin of the "
        "silver product: 1 columns are repaired or nulled, 1 rule(s) drop rows, 3 check(s) "
        "are advisory. Each rule states its reason in the pipeline notebook, and Lakeflow "
        "expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_NEONATAL_EPISODE_MANDATORY_RULES)
@_expect_all(CLINICAL_NEONATAL_EPISODE_ADVISORY_RULES)
def gold_clinical_neonatal_episode():
    """Quality-controlled twin of journey_clinical.neonatal_episode."""
    df = _qc(
        "clinical_neonatal_episode",
        CLINICAL_NEONATAL_EPISODE_SELECT,
        date_flags=["event_before_birth"],
    )
    return _with_comments(df, CLINICAL_NEONATAL_EPISODE_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.neonatal_examination ====

CLINICAL_NEONATAL_EXAMINATION_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",
    "`event_datetime` AS `event_datetime`",
    "`event_end_datetime` AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`entity_id` AS `entity_id`",
    "`examination_datetime` AS `examination_datetime`",
    "`exam_date_derived` AS `exam_date_derived`",
    "`include_in_discharge_letter` AS `include_in_discharge_letter`",
    "`head_circumference` AS `head_circumference`",
    "`spine_finding` AS `spine_finding`",
    "`heart_finding` AS `heart_finding`",
    "`genitalia_finding` AS `genitalia_finding`",
    "`hips_finding` AS `hips_finding`",
    "`right_hip_finding` AS `right_hip_finding`",
    "`eyes_finding` AS `eyes_finding`",
    "`spine_comments` AS `spine_comments`",
    "`heart_comments` AS `heart_comments`",
    "`genitalia_comments` AS `genitalia_comments`",
    "`hips_comments` AS `hips_comments`",
    "`eyes_comments` AS `eyes_comments`",
    "`neonatal_episode_id` AS `neonatal_episode_id`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`fact_category` AS `fact_category`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
]

CLINICAL_NEONATAL_EXAMINATION_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 7,528 rows of 7,528 that are
    # current and attributable; identity_status = 'resolved' keeps the 7,416 rows of 7,528
    # that are current and attributable. Superseded versions and rows whose identity was
    # never resolved are not research data, and a consumer who wants them has silver.
    "research_surface": "(identity_status = 'resolved') AND (record_status = 'active')",
}

CLINICAL_NEONATAL_EXAMINATION_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 112 of 7,528 at the profile.
    "gold.clinical.neonatal_examination.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 0 of 7,528 at the profile.
    "gold.clinical.neonatal_examination.record_status.default_view_active":
        "record_status = 'active'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 1 of 7,528 rows (0.0133%) when profiled on 2026-08-24.
    "gold.clinical.neonatal_examination.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 213 of 7,528 rows (2.83%) when profiled on 2026-08-24.
    "gold.clinical.neonatal_examination.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",
}

CLINICAL_NEONATAL_EXAMINATION_COLUMN_COMMENTS = {
    "patient_event_id": "Published field patient_event_id.",
    "fact_row_id": "Published field fact_row_id.",
    "subject_key": "Published field subject_key.",
    "subject_id_system": "Published field subject_id_system.",
    "person_id": "Published field person_id.",
    "identity_status": "Published field identity_status.",
    "encounter_id": "Published field encounter_id.",
    "event_datetime": "Published field event_datetime.",
    "event_end_datetime": "Published field event_end_datetime.",
    "source_coding_system": "Published field source_coding_system.",
    "source_code": "Published field source_code.",
    "source_display": "Published field source_display.",
    "entity_id": "Published field entity_id.",
    "examination_datetime": "Published field examination_datetime.",
    "exam_date_derived": "Published field exam_date_derived.",
    "include_in_discharge_letter": "Published field include_in_discharge_letter.",
    "head_circumference": "Published field head_circumference.",
    "spine_finding": "Published field spine_finding.",
    "heart_finding": "Published field heart_finding.",
    "genitalia_finding": "Published field genitalia_finding.",
    "hips_finding": "Published field hips_finding.",
    "right_hip_finding": "Published field right_hip_finding.",
    "eyes_finding": "Published field eyes_finding.",
    "spine_comments": "Published field spine_comments.",
    "heart_comments": "Published field heart_comments.",
    "genitalia_comments": "Published field genitalia_comments.",
    "hips_comments": "Published field hips_comments.",
    "eyes_comments": "Published field eyes_comments.",
    "neonatal_episode_id": "Published field neonatal_episode_id.",
    "record_status": "Published field record_status.",
    "record_status_effective_from": "Published field record_status_effective_from.",
    "record_status_effective_to": "Published field record_status_effective_to.",
    "confidentiality_code": "Published field confidentiality_code.",
    "vip_ind": "Published field vip_ind.",
    "withheld_identity_ind": "Published field withheld_identity_ind.",
    "fact_category": "Published field fact_category.",
    "source_feed": "Published field source_feed.",
    "load_batch_id": "Published field load_batch_id.",
    "source_update_timestamp": "Published field source_update_timestamp.",
    "loaded_at": "Published field loaded_at.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.neonatal_examination"),
    comment=(
        "One populated BadgerNet neonatal examination. Gold QC twin of the silver product: 0 "
        "columns are repaired or nulled, 1 rule(s) drop rows, 4 check(s) are advisory. Each "
        "rule states its reason in the pipeline notebook, and Lakeflow expectation metrics "
        "report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_NEONATAL_EXAMINATION_MANDATORY_RULES)
@_expect_all(CLINICAL_NEONATAL_EXAMINATION_ADVISORY_RULES)
def gold_clinical_neonatal_examination():
    """Quality-controlled twin of journey_clinical.neonatal_examination."""
    df = _qc(
        "clinical_neonatal_examination",
        CLINICAL_NEONATAL_EXAMINATION_SELECT,
        date_flags=["event_after_death_30d", "event_before_birth"],
    )
    return _with_comments(df, CLINICAL_NEONATAL_EXAMINATION_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.pathology_order ====

CLINICAL_PATHOLOGY_ORDER_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",

    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 16 of 388,597,322 rows (4.12e-06%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `event_datetime` END AS `event_datetime`",
    "`event_end_datetime` AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`source_arm` AS `source_arm`",
    "`wkg_code` AS `wkg_code`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 9 of 388,597,322 rows
    # (2.32e-06%) when profiled on 2026-08-24.
    #
    # 'UNKNOWN' is a placeholder the source writes when the value was not recorded; it is
    # not a code, so it is nulled rather than passed on as one. Hit 325,291 of 388,597,322
    # rows (0.0837%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`tlc_code` AS STRING))) = '0' OR UPPER(TRIM(CAST(`tlc_code` AS STRING))) = 'UNKNOWN' THEN NULL ELSE `tlc_code` END AS `tlc_code`",
    "`order_id` AS `order_id`",
    "`order_mnemonic` AS `order_mnemonic`",
    "`raw_request_text` AS `raw_request_text`",
    "`test_description` AS `test_description`",
    "`test_snomed_code` AS `test_snomed_code`",
    "`test_omop_concept_id` AS `test_omop_concept_id`",
    "`mapping_status` AS `mapping_status`",
    "`request_ordinal` AS `request_ordinal`",
    "`specimen_id` AS `specimen_id`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
]

CLINICAL_PATHOLOGY_ORDER_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 388,597,322 rows of
    # 388,597,322 that are current and attributable; identity_status = 'resolved' keeps the
    # 314,089,167 rows of 388,597,322 that are current and attributable. Superseded versions
    # and rows whose identity was never resolved are not research data, and a consumer who
    # wants them has silver.
    "research_surface": "(identity_status = 'resolved') AND (record_status = 'active')",
}

CLINICAL_PATHOLOGY_ORDER_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 74,508,155 of 388,597,322 at the profile.
    "gold.clinical.pathology_order.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 0 of 388,597,322 at the profile.
    "gold.clinical.pathology_order.record_status.default_view_active":
        "record_status = 'active'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 285,411 of 388,597,322 rows (0.0734%) when profiled on
    # 2026-08-24.
    "gold.clinical.pathology_order.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 1,203,076 of 388,597,322 rows (0.31%) when profiled on
    # 2026-08-24.
    "gold.clinical.pathology_order.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",
}

CLINICAL_PATHOLOGY_ORDER_COLUMN_COMMENTS = {
    "patient_event_id":
        "Stable pathology order event identifier minted from requested_test_occurrence_id.",
    "fact_row_id": "Storage-row identifier equal to patient_event_id.",
    "subject_key": "Always-populated subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference when supplied.",
    "event_datetime": "Requested or source-validity timestamp.",
    "event_end_datetime": "Order event end when supplied.",
    "source_coding_system": "Native order catalogue coding system.",
    "source_code": "Native order catalogue code.",
    "source_display": "Native order description.",
    "source_arm": "Source arm.",
    "wkg_code": "TFC work-group code.",
    "tlc_code": "TFC test-level code.",
    "order_id": "CERNER native order identifier when present.",
    "order_mnemonic": "CERNER order mnemonic.",
    "raw_request_text": "Native request wording. Identifiable free text; ig_risk 4",
    "test_description": "Native requested-test description.",
    "test_snomed_code": "Source-provided requested-test SNOMED code.",
    "test_omop_concept_id": "Source-provided requested-test OMOP concept identifier.",
    "mapping_status":
        "'mapped' means a rule ran, not that codes landed — CERNER arm carries zero baked codes.",
    "request_ordinal": "Source request ordinal within the accession.",
    "specimen_id": "Alias-resolved accession-minted specimen reference.",
    "record_status": "Normalized source-record lifecycle.",
    "record_status_effective_from": "Source validity start.",
    "record_status_effective_to": "Source validity end.",
    "confidentiality_code": "Source confidentiality code.",
    "vip_ind": "VIP indicator.",
    "withheld_identity_ind": "Withheld identity indicator.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.pathology_order"),
    comment=(
        "One requested-test occurrence across TFC_LIMS and CERNER; anchors accession-scoped "
        "order-to-report request threads. Gold QC twin of the silver product: 3 columns are "
        "repaired or nulled, 1 rule(s) drop rows, 4 check(s) are advisory. Each rule states "
        "its reason in the pipeline notebook, and Lakeflow expectation metrics report what "
        "every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_PATHOLOGY_ORDER_MANDATORY_RULES)
@_expect_all(CLINICAL_PATHOLOGY_ORDER_ADVISORY_RULES)
def gold_clinical_pathology_order():
    """Quality-controlled twin of journey_clinical.pathology_order."""
    # 525 rows point at a person_id the spine does not have. The pointer is nulled so it
    # cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    df = _qc(
        "clinical_pathology_order",
        CLINICAL_PATHOLOGY_ORDER_SELECT,
        fk_columns=["person_id"],
        date_flags=["event_after_death_30d", "event_before_birth"],
    )
    return _with_comments(df, CLINICAL_PATHOLOGY_ORDER_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.pathology_report ====

CLINICAL_PATHOLOGY_REPORT_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",

    # 109 rows point at a person_id the spine does not have. The pointer is nulled so it
    # cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    "CASE WHEN NOT `person_id_resolved` THEN NULL ELSE `person_id` END AS `person_id`",
    "`identity_status` AS `identity_status`",

    # 31,471 rows point at a encounter_id the spine does not have. The pointer is nulled so
    # it cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    "CASE WHEN NOT `encounter_id_resolved` THEN NULL ELSE `encounter_id` END AS `encounter_id`",

    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 7 of 91,759,012 rows (7.63e-06%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `event_datetime` END AS `event_datetime`",
    "`event_end_datetime` AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`report_code` AS `report_code`",
    "`report_role` AS `report_role`",
    "`discipline` AS `discipline`",
    "`report_section` AS `report_section`",
    "`lifecycle_status` AS `lifecycle_status`",
    "`version_ordinal` AS `version_ordinal`",
    "`version_count` AS `version_count`",
    "`report_version_id` AS `report_version_id`",
    "`supersedes_report_version_id` AS `supersedes_report_version_id`",
    "`is_current_present` AS `is_current_present`",
    "`document_id` AS `document_id`",

    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 7 of 91,759,012 rows (7.63e-06%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`issued_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `issued_datetime` END AS `issued_datetime`",
    "`specimen_id` AS `specimen_id`",
    "`accession_identifier` AS `accession_identifier`",
    "`report_text_hash` AS `report_text_hash`",
    "`research_qi_only` AS `research_qi_only`",
    "`record_status` AS `record_status`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 32 of 91,759,012 rows (3.49e-05%) when profiled on 2026-08-24.
    #
    # 1899-12-30 is the zero point of the OLE/Excel date scale, so it is what a spreadsheet
    # or a COM layer writes when the date was left blank. It is not a date anyone recorded.
    # Hit 2,727 of 91,759,012 rows (0.00297%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`record_status_effective_from` AS DATE) = DATE'1899-12-30' OR CAST(`record_status_effective_from` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`record_status_effective_from` AS DATE)) < 9999 THEN NULL ELSE `record_status_effective_from` END AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 32 of 91,759,012 rows (3.49e-05%) when profiled on 2026-08-24.
    #
    # 1899-12-30 is the zero point of the OLE/Excel date scale, so it is what a spreadsheet
    # or a COM layer writes when the date was left blank. It is not a date anyone recorded.
    # Hit 2,727 of 91,759,012 rows (0.00297%) when profiled on 2026-08-24.
    #
    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 7 of 91,759,012 rows (7.63e-06%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`source_update_timestamp` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE CASE WHEN CAST(`source_update_timestamp` AS DATE) = DATE'1899-12-30' OR CAST(`source_update_timestamp` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`source_update_timestamp` AS DATE)) < 9999 THEN NULL ELSE `source_update_timestamp` END END AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",

    # Silver derived this flag by comparing event_datetime with the person's own dates,
    # before the rules above corrected event_datetime. On the rows where event_datetime
    # changed, the flag describes a timestamp gold no longer publishes. This product carries
    # a VARIANT column, which rules out the spine join gold would need to recompute the
    # flag, so it is nulled where the value beneath it moved rather than left asserting
    # something stale.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `event_before_birth` END AS `event_before_birth`",

    # Silver derived this flag by comparing event_datetime with the person's own dates,
    # before the rules above corrected event_datetime. On the rows where event_datetime
    # changed, the flag describes a timestamp gold no longer publishes. This product carries
    # a VARIANT column, which rules out the spine join gold would need to recompute the
    # flag, so it is nulled where the value beneath it moved rather than left asserting
    # something stale.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `event_after_death_30d` END AS `event_after_death_30d`",
]

CLINICAL_PATHOLOGY_REPORT_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 91,718,730 rows of 91,759,012
    # that are current and attributable; identity_status = 'resolved' keeps the 71,790,751
    # rows of 91,759,012 that are current and attributable. Superseded versions and rows
    # whose identity was never resolved are not research data, and a consumer who wants them
    # has silver.
    "research_surface": "(identity_status = 'resolved') AND (record_status = 'active')",
}

CLINICAL_PATHOLOGY_REPORT_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 19,968,261 of 91,759,012 at the profile.
    "gold.clinical.pathology_report.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 40,282 of 91,759,012 at the profile.
    "gold.clinical.pathology_report.record_status.default_view_active":
        "record_status = 'active'",

    # This bounds a period of validity, and a future end is exactly how the source says a
    # record is still current -- nulling it would assert the record is valid forever, which
    # is a stronger and worse claim than the one being corrected. Seen on 7 of 91,759,012
    # rows (7.63e-06%) when profiled on 2026-08-24.
    "gold.clinical.pathology_report.record_status_effective_from.future_owner":
        "NOT COALESCE((CAST(`record_status_effective_from` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 167,974 of 91,759,012 rows (0.183%) when profiled on 2026-08-24.
    "gold.clinical.pathology_report.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 6,142 of 91,759,012 rows (0.00669%) when profiled on 2026-08-24.
    "gold.clinical.pathology_report.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",
}

CLINICAL_PATHOLOGY_REPORT_COLUMN_COMMENTS = {
    "patient_event_id": "Stable pathology report identifier minted from report_series_id.",
    "fact_row_id": "Storage-row identifier.",
    "subject_key": "Always-populated subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference.",
    "event_datetime": "Report issue or result timestamp.",
    "event_end_datetime": "Report event end.",
    "source_coding_system": "Source test coding system.",
    "source_code": "Source test code.",
    "source_display": "Source test display.",
    "report_code": "Source and mapped report codings.",
    "report_role": "Latest-version report role.",
    "discipline": "Latest-version pathology discipline.",
    "report_section": "Latest-version report section.",
    "lifecycle_status": "Latest-version lifecycle status.",
    "version_ordinal": "Latest version ordinal in the report series.",
    "version_count": "Number of report versions in the series.",
    "report_version_id": "Latest report-version identifier.",
    "supersedes_report_version_id":
        "Prior report-version identifier superseded by the latest version.",
    "is_current_present": "True when the series has a current version row.",
    "document_id": "Current text-bearing report-version document reference; nullable.",
    "issued_datetime": "Report issue timestamp.",
    "specimen_id": "Linked specimen identifier.",
    "accession_identifier": "Canonical pathology_accession_id.",
    "report_text_hash": "Bronze hash for the latest report version",
    "research_qi_only":
        "Registry doctrine flag; true on 100% of rows today — describe-only",
    "record_status": "Latest-version lifecycle normalized to active or retracted.",
    "record_status_effective_from":
        "Source validity start. Gold QC transform rules: gold.clinical.pathology_report.record_status_effective_from.beyond_2100_below_9999_owner, gold.clinical.pathology_report.record_status_effective_from.ole_zero_date.",
    "record_status_effective_to": "Source validity end.",
    "confidentiality_code": "Security code.",
    "vip_ind": "VIP indicator.",
    "withheld_identity_ind": "Withheld identity indicator.",
    "source_feed": "Registered owning feed.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp":
        "Native source update timestamp. Gold QC transform rules: gold.clinical.pathology_report.source_update_timestamp.beyond_2100_below_9999_owner, gold.clinical.pathology_report.source_update_timestamp.ole_zero_date.",
    "loaded_at": "Bronze load timestamp.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.pathology_report"),
    comment=(
        "One pathology report series keyed by report_series_id using the latest-version "
        "projection; per-version text lives in journey_text.document. Gold QC twin of the "
        "silver product: 8 columns are repaired or nulled, 1 rule(s) drop rows, 5 check(s) "
        "are advisory. Each rule states its reason in the pipeline notebook, and Lakeflow "
        "expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_PATHOLOGY_REPORT_MANDATORY_RULES)
@_expect_all(CLINICAL_PATHOLOGY_REPORT_ADVISORY_RULES)
def gold_clinical_pathology_report():
    """Quality-controlled twin of journey_clinical.pathology_report."""
    df = _qc("clinical_pathology_report", CLINICAL_PATHOLOGY_REPORT_SELECT)
    return _with_comments(df, CLINICAL_PATHOLOGY_REPORT_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.pathology_result ====

CLINICAL_PATHOLOGY_RESULT_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",

    # 2,444 rows point at a person_id the spine does not have. The pointer is nulled so it
    # cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    "CASE WHEN NOT `person_id_resolved` THEN NULL ELSE `person_id` END AS `person_id`",
    "`identity_status` AS `identity_status`",

    # 1,085,677 rows point at a encounter_id the spine does not have. The pointer is nulled
    # so it cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    "CASE WHEN NOT `encounter_id_resolved` THEN NULL ELSE `encounter_id` END AS `encounter_id`",

    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 1,231 of 1,877,832,347 rows (6.56e-05%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `event_datetime` END AS `event_datetime`",

    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 44 of 1,877,832,347 rows (2.34e-06%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`event_end_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `event_end_datetime` END AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`result_code` AS `result_code`",
    "`pathology_report_id` AS `pathology_report_id`",
    "`specimen_id` AS `specimen_id`",
    "`equivalence_group` AS `equivalence_group`",
    "`representation_role` AS `representation_role`",
    "`preferred_result_ind` AS `preferred_result_ind`",
    "`person_projection_status` AS `person_projection_status`",

    # A magnitude above 1e12 is outside any scale this field is measured on, so the number
    # carries no meaning even though something was recorded. Hit 48,580 of 1,877,832,347
    # rows (0.00259%) when profiled on 2026-08-24.
    "CASE WHEN ABS(CAST(`value_number` AS DOUBLE)) > 1e12 THEN NULL ELSE `value_number` END AS `value_number`",

    # An empty or whitespace-only string is how the source writes 'nothing here'. It reads
    # as a value in a query and is not one, so it is nulled. Hit 99,946,150 of 1,877,832,347
    # rows (5.32%) when profiled on 2026-08-24.
    "CASE WHEN TRIM(CAST(`value_text` AS STRING)) = '' THEN NULL ELSE `value_text` END AS `value_text`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 45 of 1,877,832,347 rows (2.4e-06%) when profiled on 2026-08-24.
    #
    # A date before 1901 that is not one of the known placeholders. Nothing in this estate
    # predates the twentieth century, so these are mistyped or mis-scaled rather than early.
    # Hit 25 of 1,877,832,347 rows (1.33e-06%) when profiled on 2026-08-24.
    #
    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 24,769 of 1,877,832,347 rows (0.00132%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`value_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE CASE WHEN CAST(`value_datetime` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`value_datetime` AS DATE)) < 9999 OR CAST(`value_datetime` AS DATE) < DATE'1901-01-01' AND CAST(`value_datetime` AS DATE) NOT IN (DATE'1800-01-01', DATE'1899-12-30', DATE'1900-01-01') THEN NULL ELSE `value_datetime` END END AS `value_datetime`",
    "`value_concept_id` AS `value_concept_id`",
    "`value_concept_display` AS `value_concept_display`",
    "`operator_concept_id` AS `operator_concept_id`",

    # An empty or whitespace-only string is how the source writes 'nothing here'. It reads
    # as a value in a query and is not one, so it is nulled. Hit 13,544,949 of 1,877,832,347
    # rows (0.721%) when profiled on 2026-08-24.
    "CASE WHEN TRIM(CAST(`unit_source_value` AS STRING)) = '' THEN NULL ELSE `unit_source_value` END AS `unit_source_value`",
    "`ucum_code` AS `ucum_code`",
    "`unit_concept_id` AS `unit_concept_id`",
    "`reference_range_low` AS `reference_range_low`",
    "`reference_range_high` AS `reference_range_high`",
    "`interpretation_code` AS `interpretation_code`",
    "`polarity` AS `polarity`",
    "`finding_axis` AS `finding_axis`",
    "`result_status` AS `result_status`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 177 of 1,877,832,347 rows
    # (9.43e-06%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`body_site_code` AS STRING))) = '0' THEN NULL ELSE `body_site_code` END AS `body_site_code`",
    "`clinician_code` AS `clinician_code`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",

    # Silver derived this flag by comparing event_datetime with the person's own dates,
    # before the rules above corrected event_datetime. On the rows where event_datetime
    # changed, the flag describes a timestamp gold no longer publishes. This product carries
    # a VARIANT column, which rules out the spine join gold would need to recompute the
    # flag, so it is nulled where the value beneath it moved rather than left asserting
    # something stale.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `event_before_birth` END AS `event_before_birth`",

    # Silver derived this flag by comparing event_datetime with the person's own dates,
    # before the rules above corrected event_datetime. On the rows where event_datetime
    # changed, the flag describes a timestamp gold no longer publishes. This product carries
    # a VARIANT column, which rules out the spine join gold would need to recompute the
    # flag, so it is nulled where the value beneath it moved rather than left asserting
    # something stale.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `event_after_death_30d` END AS `event_after_death_30d`",
]

CLINICAL_PATHOLOGY_RESULT_MANDATORY_RULES = {
    # The research surface. identity_status = 'resolved' keeps the 1,497,882,424 rows of
    # 1,877,832,347 that are current and attributable; preferred_result_ind = TRUE keeps the
    # 0 rows of 1,877,832,347 that are current and attributable. Superseded versions and
    # rows whose identity was never resolved are not research data, and a consumer who wants
    # them has silver.
    "research_surface": "(identity_status = 'resolved') AND (preferred_result_ind = TRUE)",
}

CLINICAL_PATHOLOGY_RESULT_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 379,949,923 of 1,877,832,347 at the profile.
    "gold.clinical.pathology_result.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Counts what the research surface removed: rows failing preferred_result_ind = TRUE.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 1,877,832,347 of 1,877,832,347 at the profile.
    "gold.clinical.pathology_result.preferred_result_ind.default_view_preferred":
        "preferred_result_ind = TRUE",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 1,240,568 of 1,877,832,347 rows (0.0661%) when profiled on
    # 2026-08-24.
    "gold.clinical.pathology_result.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 5,132,232 of 1,877,832,347 rows (0.273%) when profiled on
    # 2026-08-24.
    "gold.clinical.pathology_result.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",

    # Negative values here are signed by design -- credit lines in the costing feed and
    # below-zero readings on a calibrated scale -- so they are counted, not removed. Seen on
    # 4,908,621 of 1,877,832,347 rows (0.261%) when profiled on 2026-08-24.
    "gold.clinical.pathology_result.value_number.negative_mass_scale_owner":
        "NOT COALESCE((`value_number` < 0), FALSE)",
}

CLINICAL_PATHOLOGY_RESULT_COLUMN_COMMENTS = {
    "patient_event_id": "Stable pathology result identifier.",
    "fact_row_id": "Storage-row identifier.",
    "subject_key": "Always-populated subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference.",
    "event_datetime": "Measurement timestamp.",
    "event_end_datetime": "Measurement end timestamp.",
    "source_coding_system": "Source test coding system.",
    "source_code": "Source test code.",
    "source_display": "Source test display.",
    "result_code": "Source and mapped test codings.",
    "pathology_report_id":
        "Report SERIES reference via the accession parent link; NULL when the parent has no textual report (70% of accessions).",
    "specimen_id": "Accession-minted specimen reference (specimen:accession)",
    "equivalence_group":
        "canonical_result_id from map_pathology_result_equivalence; currently all singleton groups under unique_source_result_v1. NULL means the spine key is not covered by the equivalence layer or source_record_key is NULL.",
    "representation_role": "Result representation role within the equivalence group.",
    "preferred_result_ind": "Whether this is the preferred representation.",
    "person_projection_status": "Equivalence-layer person projection state.",
    "value_number": "Numeric result value.",
    "value_text":
        "Verbatim result value. Gold QC transform rules: gold.clinical.pathology_result.value_text.empty_string.",
    "value_datetime":
        "Datetime result value. Gold QC transform rules: gold.clinical.pathology_result.value_datetime.beyond_2100_below_9999_owner, gold.clinical.pathology_result.value_datetime.pre1901_other_owner.",
    "value_concept_id": "Coded result concept identifier.",
    "value_concept_display": "Coded result concept display.",
    "operator_concept_id": "Result comparison operator concept.",
    "unit_source_value":
        "Verbatim source unit. Gold QC transform rules: gold.clinical.pathology_result.unit_source_value.empty_string.",
    "ucum_code": "UCUM unit code.",
    "unit_concept_id": "OMOP unit concept identifier.",
    "reference_range_low": "Reference-range lower bound.",
    "reference_range_high": "Reference-range upper bound.",
    "interpretation_code": "Source interpretation or normalcy.",
    "polarity": "Source result polarity or growth grade.",
    "finding_axis": "Source finding/result axis.",
    "result_status": "Source result status.",
    "body_site_code": "Source body-site code.",
    "clinician_code": "Source clinician code.",
    "record_status": "Normalized source-record lifecycle.",
    "record_status_effective_from": "Source validity start.",
    "record_status_effective_to": "Source validity end.",
    "confidentiality_code": "Security code.",
    "vip_ind": "VIP indicator.",
    "withheld_identity_ind": "Withheld identity indicator.",
    "source_feed": "Registered owning feed.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.pathology_result"),
    comment=(
        "One pathology result with typed value, units, range, interpretation and mapping "
        "evidence. Gold QC twin of the silver product: 11 columns are repaired or nulled, 1 "
        "rule(s) drop rows, 5 check(s) are advisory. Each rule states its reason in the "
        "pipeline notebook, and Lakeflow expectation metrics report what every rule matched "
        "on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_PATHOLOGY_RESULT_MANDATORY_RULES)
@_expect_all(CLINICAL_PATHOLOGY_RESULT_ADVISORY_RULES)
def gold_clinical_pathology_result():
    """Quality-controlled twin of journey_clinical.pathology_result."""
    df = _qc("clinical_pathology_result", CLINICAL_PATHOLOGY_RESULT_SELECT)
    return _with_comments(df, CLINICAL_PATHOLOGY_RESULT_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.pathway_tracking ====

CLINICAL_PATHWAY_TRACKING_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",
    "`event_datetime` AS `event_datetime`",
    "`event_end_datetime` AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`ptl_unique_id` AS `ptl_unique_id`",
    "`ptl_group_id` AS `ptl_group_id`",
    "`archived_pathway` AS `archived_pathway`",
    "`pathway_oid` AS `pathway_oid`",
    "`referral_oid` AS `referral_oid`",
    "`patient_oid` AS `patient_oid`",
    "`ptl_activity_oid` AS `ptl_activity_oid`",
    "`parent_ptl_unique_id` AS `parent_ptl_unique_id`",
    "`parent_ptl_activity_oid` AS `parent_ptl_activity_oid`",
    "`parent_present_ind` AS `parent_present_ind`",
    "`latest_activity_type` AS `latest_activity_type`",
    "`latest_activity_oid` AS `latest_activity_oid`",
    "`latest_activity_date_future_ind` AS `latest_activity_date_future_ind`",
    "`days_waited` AS `days_waited`",
    "`specialty` AS `specialty`",
    "`treatment_function` AS `treatment_function`",

    # 'UNKNOWN' is a placeholder the source writes when the value was not recorded; it is
    # not a code, so it is nulled rather than passed on as one. Hit 485,692 of 16,305,277
    # rows (2.98%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`treatment_function_code` AS STRING))) = 'UNKNOWN' THEN NULL ELSE `treatment_function_code` END AS `treatment_function_code`",
    "`site` AS `site`",
    "`site_group` AS `site_group`",
    "`division` AS `division`",
    "`lead_clinician` AS `lead_clinician`",
    "`lead_clinician_prid` AS `lead_clinician_prid`",
    "`site_rvid` AS `site_rvid`",
    "`source_key_status` AS `source_key_status`",
    "`patient_spine_ind` AS `patient_spine_ind`",
    "`pathway_spine_ind` AS `pathway_spine_ind`",
    "`referral_spine_ind` AS `referral_spine_ind`",
    "`patient_spine_link_status` AS `patient_spine_link_status`",
    "`pathway_spine_link_status` AS `pathway_spine_link_status`",
    "`referral_spine_link_status` AS `referral_spine_link_status`",
    "`nhs_number_valid_ind` AS `nhs_number_valid_ind`",
    "`linkage_historical_fallback_ind` AS `linkage_historical_fallback_ind`",
    "`linkage_fallback_conflict_ind` AS `linkage_fallback_conflict_ind`",
    "`person_link_status` AS `person_link_status`",
    "`person_link_method` AS `person_link_method`",
    "`identifier_link_status` AS `identifier_link_status`",
    "`source_system_oid` AS `source_system_oid`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`fact_category` AS `fact_category`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
]

CLINICAL_PATHWAY_TRACKING_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 16,277,672 rows of 16,305,277
    # that are current and attributable; identity_status = 'resolved' keeps the 16,253,290
    # rows of 16,305,277 that are current and attributable. Superseded versions and rows
    # whose identity was never resolved are not research data, and a consumer who wants them
    # has silver.
    "research_surface": "(identity_status = 'resolved') AND (record_status = 'active')",
}

CLINICAL_PATHWAY_TRACKING_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 51,987 of 16,305,277 at the profile.
    "gold.clinical.pathway_tracking.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 27,605 of 16,305,277 at the profile.
    "gold.clinical.pathway_tracking.record_status.default_view_active":
        "record_status = 'active'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 85,762 of 16,305,277 rows (0.526%) when profiled on 2026-08-24.
    "gold.clinical.pathway_tracking.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 917 of 16,305,277 rows (0.00562%) when profiled on 2026-08-24.
    "gold.clinical.pathway_tracking.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",
}

CLINICAL_PATHWAY_TRACKING_COLUMN_COMMENTS = {
    "patient_event_id": "Stable row identifier.",
    "fact_row_id": "Stable row identifier.",
    "subject_key": "Published field.",
    "subject_id_system": "Published field.",
    "person_id": "Published field.",
    "identity_status": "Published field.",
    "encounter_id": "Published field.",
    "event_datetime": "Published field.",
    "event_end_datetime": "Published field.",
    "source_coding_system": "Published field.",
    "source_code": "Published field.",
    "source_display": "Published field.",
    "ptl_unique_id": "Published field.",
    "ptl_group_id": "Published field.",
    "archived_pathway": "Published field.",
    "pathway_oid": "Published field.",
    "referral_oid": "Published field.",
    "patient_oid": "Published field.",
    "ptl_activity_oid": "Published field.",
    "parent_ptl_unique_id": "Published field.",
    "parent_ptl_activity_oid": "Published field.",
    "parent_present_ind": "Published field.",
    "latest_activity_type": "Published field.",
    "latest_activity_oid": "Published field.",
    "latest_activity_date_future_ind": "Published field.",
    "days_waited": "Published field.",
    "specialty": "Published field.",
    "treatment_function": "Published field.",
    "treatment_function_code": "Published field.",
    "site": "Published field.",
    "site_group": "Published field.",
    "division": "Published field.",
    "lead_clinician": "Published field.",
    "lead_clinician_prid": "Published field.",
    "site_rvid": "Published field.",
    "source_key_status": "Published field.",
    "patient_spine_ind": "Published field.",
    "pathway_spine_ind": "Published field.",
    "referral_spine_ind": "Published field.",
    "patient_spine_link_status": "Published field.",
    "pathway_spine_link_status": "Published field.",
    "referral_spine_link_status": "Published field.",
    "nhs_number_valid_ind": "Published field.",
    "linkage_historical_fallback_ind": "Published field.",
    "linkage_fallback_conflict_ind": "Published field.",
    "person_link_status": "Published field.",
    "person_link_method": "Published field.",
    "identifier_link_status": "Published field.",
    "source_system_oid": "Published field.",
    "record_status": "Published field.",
    "record_status_effective_from": "Published field.",
    "record_status_effective_to": "Published field.",
    "confidentiality_code": "Published field.",
    "vip_ind": "Published field.",
    "withheld_identity_ind": "Published field.",
    "fact_category": "Published field.",
    "source_feed": "Published field.",
    "load_batch_id": "Published field.",
    "source_update_timestamp": "Published field.",
    "loaded_at": "Published field.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.pathway_tracking"),
    comment=(
        "One whole-trust Pathfinder PTL activity row. Gold QC twin of the silver product: 2 "
        "columns are repaired or nulled, 1 rule(s) drop rows, 4 check(s) are advisory. Each "
        "rule states its reason in the pipeline notebook, and Lakeflow expectation metrics "
        "report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_PATHWAY_TRACKING_MANDATORY_RULES)
@_expect_all(CLINICAL_PATHWAY_TRACKING_ADVISORY_RULES)
def gold_clinical_pathway_tracking():
    """Quality-controlled twin of journey_clinical.pathway_tracking."""
    # 7 rows point at a person_id the spine does not have. The pointer is nulled so it
    # cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    df = _qc(
        "clinical_pathway_tracking",
        CLINICAL_PATHWAY_TRACKING_SELECT,
        fk_columns=["person_id"],
        date_flags=["event_after_death_30d", "event_before_birth"],
    )
    return _with_comments(df, CLINICAL_PATHWAY_TRACKING_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.procedure ====

CLINICAL_PROCEDURE_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",

    # 5 rows point at a person_id the spine does not have. The pointer is nulled so it
    # cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    "CASE WHEN NOT `person_id_resolved` THEN NULL ELSE `person_id` END AS `person_id`",
    "`identity_status` AS `identity_status`",

    # 20,572 rows point at a encounter_id the spine does not have. The pointer is nulled so
    # it cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    "CASE WHEN NOT `encounter_id_resolved` THEN NULL ELSE `encounter_id` END AS `encounter_id`",

    # 1899-12-30 is the zero point of the OLE/Excel date scale, so it is what a spreadsheet
    # or a COM layer writes when the date was left blank. It is not a date anyone recorded.
    # Hit 54 of 16,307,178 rows (0.000331%) when profiled on 2026-08-24.
    #
    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 3,039 of 16,307,178 rows (0.0186%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE CASE WHEN CAST(`event_datetime` AS DATE) = DATE'1899-12-30' THEN NULL ELSE `event_datetime` END END AS `event_datetime`",

    # event_end_datetime cannot precede event_datetime. The start is the better-attested of
    # the two, so the end is what goes and the row keeps its event_datetime. Hit 459 of
    # 16,307,178 rows (0.00281%) when profiled on 2026-08-24.
    #
    # Stamps landing in January 1970 carry the same millisecond-epoch defect found on
    # clinical_score and vital_sign, but nothing here corroborates a rescaled value the way
    # the encounter window does there, so the stamp is nulled rather than reconstructed. The
    # catalogue's date rules stop at 1901 and do not reach this.
    "CASE WHEN `event_datetime` IS NOT NULL AND `event_end_datetime` IS NOT NULL AND `event_datetime` > `event_end_datetime` OR YEAR(CAST(`event_end_datetime` AS TIMESTAMP)) = 1970 THEN NULL ELSE `event_end_datetime` END AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",

    # 'UNKNOWN' is a placeholder the source writes when the value was not recorded; it is
    # not a code, so it is nulled rather than passed on as one. Hit 5 of 16,307,178 rows
    # (3.07e-05%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`source_code` AS STRING))) = 'UNKNOWN' THEN NULL ELSE `source_code` END AS `source_code`",
    "`source_display` AS `source_display`",
    "`procedure_code` AS `procedure_code`",
    "`status_code` AS `status_code`",
    "`status_display` AS `status_display`",

    # 1899-12-30 is the zero point of the OLE/Excel date scale, so it is what a spreadsheet
    # or a COM layer writes when the date was left blank. It is not a date anyone recorded.
    # Hit 54 of 16,307,178 rows (0.000331%) when profiled on 2026-08-24.
    #
    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 3,039 of 16,307,178 rows (0.0186%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`performed_start` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE CASE WHEN CAST(`performed_start` AS DATE) = DATE'1899-12-30' THEN NULL ELSE `performed_start` END END AS `performed_start`",

    # performed_end cannot precede performed_start. The start is the better-attested of the
    # two, so the end is what goes and the row keeps its performed_start. Hit 454 of
    # 16,307,178 rows (0.00278%) when profiled on 2026-08-24.
    #
    # Stamps landing in January 1970 carry the same millisecond-epoch defect found on
    # clinical_score and vital_sign, but nothing here corroborates a rescaled value the way
    # the encounter window does there, so the stamp is nulled rather than reconstructed. It
    # affected 9,337 rows. The catalogue's date rules stop at 1901 and do not reach this.
    "CASE WHEN `performed_start` IS NOT NULL AND `performed_end` IS NOT NULL AND `performed_start` > `performed_end` OR YEAR(CAST(`performed_end` AS TIMESTAMP)) = 1970 THEN NULL ELSE `performed_end` END AS `performed_end`",
    "`body_site_code` AS `body_site_code`",
    "`body_site_display` AS `body_site_display`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 13,837,352 of 16,307,178 rows
    # (84.9%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`laterality_code` AS STRING))) = '0' THEN NULL ELSE `laterality_code` END AS `laterality_code`",
    "`laterality_display` AS `laterality_display`",
    "`performer_practitioner_id` AS `performer_practitioner_id`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 14,015,523 of 16,307,178 rows
    # (85.9%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`procedure_location_code` AS STRING))) = '0' THEN NULL ELSE `procedure_location_code` END AS `procedure_location_code`",
    "`procedure_location_display` AS `procedure_location_display`",
    "`procedure_note` AS `procedure_note`",
    "`implant_description` AS `implant_description`",
    "`device_code` AS `device_code`",
    "`device_display` AS `device_display`",
    "`manufacturer` AS `manufacturer`",
    "`serial_number` AS `serial_number`",
    "`batch_number` AS `batch_number`",
    "`udi_di` AS `udi_di`",
    "`udi_standard` AS `udi_standard`",

    # A magnitude above 1e12 is outside any scale this field is measured on, so the number
    # carries no meaning even though something was recorded. Hit 4 of 16,307,178 rows
    # (2.45e-05%) when profiled on 2026-08-24.
    "CASE WHEN ABS(CAST(`quantity` AS DOUBLE)) > 1e12 THEN NULL ELSE `quantity` END AS `quantity`",
    "`implant_attribute_history` AS `implant_attribute_history`",
    "`implant_attribute_count` AS `implant_attribute_count`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",

    # record_status_effective_to cannot precede record_status_effective_from. The start is
    # the better-attested of the two, so the end is what goes and the row keeps its
    # record_status_effective_from. Hit 424 of 16,307,178 rows (0.0026%) when profiled on
    # 2026-08-24.
    "CASE WHEN `record_status_effective_from` IS NOT NULL AND `record_status_effective_to` IS NOT NULL AND `record_status_effective_from` > `record_status_effective_to` THEN NULL ELSE `record_status_effective_to` END AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",

    # Silver derived this flag by comparing event_datetime with the person's own dates,
    # before the rules above corrected event_datetime. On the rows where event_datetime
    # changed, the flag describes a timestamp gold no longer publishes. This product carries
    # a VARIANT column, which rules out the spine join gold would need to recompute the
    # flag, so it is nulled where the value beneath it moved rather than left asserting
    # something stale.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `event_before_birth` END AS `event_before_birth`",

    # Silver derived this flag by comparing event_datetime with the person's own dates,
    # before the rules above corrected event_datetime. On the rows where event_datetime
    # changed, the flag describes a timestamp gold no longer publishes. This product carries
    # a VARIANT column, which rules out the spine join gold would need to recompute the
    # flag, so it is nulled where the value beneath it moved rather than left asserting
    # something stale.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `event_after_death_30d` END AS `event_after_death_30d`",
]

CLINICAL_PROCEDURE_MANDATORY_RULES = {
    # The research surface. identity_status = 'resolved' keeps the 16,267,993 rows of
    # 16,307,178 that are current and attributable. Superseded versions and rows whose
    # identity was never resolved are not research data, and a consumer who wants them has
    # silver.
    "research_surface": "(identity_status = 'resolved')",
}

CLINICAL_PROCEDURE_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 39,185 of 16,307,178 at the profile.
    "gold.clinical.procedure.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # This bounds a period of validity, and a future end is exactly how the source says a
    # record is still current -- nulling it would assert the record is valid forever, which
    # is a stronger and worse claim than the one being corrected. Seen on 3,038 of
    # 16,307,178 rows (0.0186%) when profiled on 2026-08-24.
    "gold.clinical.procedure.record_status_effective_from.future_owner":
        "NOT COALESCE((CAST(`record_status_effective_from` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 1,009 of 16,307,178 rows (0.00619%) when profiled on 2026-08-24.
    "gold.clinical.procedure.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 41,288 of 16,307,178 rows (0.253%) when profiled on 2026-08-24.
    "gold.clinical.procedure.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",
}

CLINICAL_PROCEDURE_COLUMN_COMMENTS = {
    "patient_event_id": "Stable product-wide procedure identifier.",
    "fact_row_id": "Storage-row identifier equal to patient_event_id.",
    "subject_key": "Always-populated peppered subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Best-available encounter reference.",
    "event_datetime":
        "Primary performed timestamp. Gold QC transform rules: gold.clinical.procedure.event_datetime.ole_zero_date.",
    "event_end_datetime": "Procedure end timestamp.",
    "source_coding_system": "Verbatim source coding system.",
    "source_code": "Source procedure code or implant label",
    "source_display": "Verbatim source procedure display.",
    "procedure_code": "Source and mapped procedure codings.",
    "status_code": "Source-derived FHIR procedure status.",
    "status_display": "Source status display.",
    "performed_start":
        "Procedure performed start. Gold QC transform rules: gold.clinical.procedure.performed_start.ole_zero_date.",
    "performed_end": "Procedure performed end.",
    "body_site_code": "Body-site code where supplied.",
    "body_site_display": "Body-site display where supplied.",
    "laterality_code": "Source laterality code.",
    "laterality_display": "Source laterality display.",
    "performer_practitioner_id": "Performing practitioner reference.",
    "procedure_location_code":
        "Source procedure-location code retained without asserting a location-dimension FK.",
    "procedure_location_display": "Source procedure-location display.",
    "procedure_note": "Source procedure note.",
    "implant_description": "Implant description for implant-placement facts.",
    "device_code": "Device concept identifier for implant-placement facts.",
    "device_display": "Device concept display.",
    "manufacturer": "Implant manufacturer.",
    "serial_number": "Implant serial number.",
    "batch_number": "Implant batch number.",
    "udi_di": "Unique device identifier device identifier.",
    "udi_standard": "UDI issuing standard.",
    "quantity": "Parsed implant quantity.",
    "implant_attribute_history":
        "Ordered JSON implant attribute evidence from map_implant_detail_events.",
    "implant_attribute_count": "Number of retained implant attribute rows.",
    "record_status": "Normalized source-record lifecycle.",
    "record_status_effective_from": "Source status effective start.",
    "record_status_effective_to": "Source status effective end.",
    "confidentiality_code": "Source confidentiality code.",
    "vip_ind": "Source VIP indicator.",
    "withheld_identity_ind": "Withheld-identity indicator.",
    "source_feed": "Registered source feed owning the fact.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at":
        "Latest bronze load time across the fact row and its folded evidence rows.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.procedure"),
    comment=(
        "One performed procedure or implant-placement source fact without cross-feed "
        "deduplication. Gold QC twin of the silver product: 13 columns are repaired or "
        "nulled, 1 rule(s) drop rows, 4 check(s) are advisory. Each rule states its reason in "
        "the pipeline notebook, and Lakeflow expectation metrics report what every rule "
        "matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_PROCEDURE_MANDATORY_RULES)
@_expect_all(CLINICAL_PROCEDURE_ADVISORY_RULES)
def gold_clinical_procedure():
    """Quality-controlled twin of journey_clinical.procedure."""
    df = _qc("clinical_procedure", CLINICAL_PROCEDURE_SELECT)
    return _with_comments(df, CLINICAL_PROCEDURE_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.referral ====

CLINICAL_REFERRAL_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",
    "`event_datetime` AS `event_datetime`",
    "`event_end_datetime` AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`ubrn` AS `ubrn`",
    "`waiting_list_oid` AS `waiting_list_oid`",
    "`pathway_oid` AS `pathway_oid`",
    "`referral_priority_code` AS `referral_priority_code`",
    "`referral_priority_display` AS `referral_priority_display`",
    "`referral_source_code` AS `referral_source_code`",
    "`referral_source_display` AS `referral_source_display`",
    "`status_code` AS `status_code`",
    "`status_display` AS `status_display`",
    "`status_change_reason_code` AS `status_change_reason_code`",
    "`status_change_reason_display` AS `status_change_reason_display`",
    "`status_change_datetime` AS `status_change_datetime`",
    "`encounter_type_code` AS `encounter_type_code`",
    "`encounter_type_display` AS `encounter_type_display`",
    "`suspected_cancer_site_code` AS `suspected_cancer_site_code`",
    "`suspected_cancer_site_display` AS `suspected_cancer_site_display`",
    "`treatment_function_code` AS `treatment_function_code`",
    "`treatment_function_display` AS `treatment_function_display`",
    "`service_type_requested_code` AS `service_type_requested_code`",
    "`service_type_requested_display` AS `service_type_requested_display`",
    "`site_code` AS `site_code`",
    "`site_display` AS `site_display`",
    "`referring_facility_code` AS `referring_facility_code`",
    "`referring_facility_display` AS `referring_facility_display`",
    "`referred_by_org_id` AS `referred_by_org_id`",
    "`booking_type_code` AS `booking_type_code`",
    "`booking_type_display` AS `booking_type_display`",
    "`admin_category_code` AS `admin_category_code`",
    "`admin_category_display` AS `admin_category_display`",
    "`business_unit` AS `business_unit`",
    "`division` AS `division`",

    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 2 of 10,573,076 rows (1.89e-05%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`original_received_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `original_received_datetime` END AS `original_received_datetime`",
    "`ers_ubrn_received` AS `ers_ubrn_received`",
    "`ers_pathway_start` AS `ers_pathway_start`",
    "`ers_service_name` AS `ers_service_name`",
    "`ers_specialty` AS `ers_specialty`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`fact_category` AS `fact_category`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
]

CLINICAL_REFERRAL_MANDATORY_RULES = {
    # The research surface. identity_status = 'resolved' keeps the 10,572,634 rows of
    # 10,573,076 that are current and attributable. Superseded versions and rows whose
    # identity was never resolved are not research data, and a consumer who wants them has
    # silver.
    "research_surface": "(identity_status = 'resolved')",
}

CLINICAL_REFERRAL_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 442 of 10,573,076 at the profile.
    "gold.clinical.referral.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 544 of 10,573,076 rows (0.00515%) when profiled on 2026-08-24.
    "gold.clinical.referral.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 4,649 of 10,573,076 rows (0.044%) when profiled on 2026-08-24.
    "gold.clinical.referral.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",
}

CLINICAL_REFERRAL_COLUMN_COMMENTS = {
    "patient_event_id": "Stable product-wide referral identifier.",
    "fact_row_id": "Storage-row identifier equal to patient_event_id.",
    "subject_key": "Always-populated peppered subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id":
        "Resolved Millennium person identifier from the bronze crosswalk when available.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter context; always null for the LUNA referral feed.",
    "event_datetime": "Referral received timestamp.",
    "event_end_datetime": "Event end timestamp; not supplied by LUNA referrals.",
    "source_coding_system": "Verbatim LUNA treatment-function coding system.",
    "source_code": "Verbatim treatment-function code.",
    "source_display": "Verbatim treatment-function display.",
    "ubrn": "Referral linkage evidence only; never a join key.",
    "waiting_list_oid": "Raw LUNA waiting-list object identifier.",
    "pathway_oid": "Raw LUNA pathway object identifier.",
    "referral_priority_code": "Source referral-priority code.",
    "referral_priority_display": "Source referral-priority display.",
    "referral_source_code": "Source referral-source code.",
    "referral_source_display": "Source referral-source display.",
    "status_code": "Source referral-status code.",
    "status_display": "Source referral-status display.",
    "status_change_reason_code": "Source referral status-change reason code.",
    "status_change_reason_display": "Source referral status-change reason display.",
    "status_change_datetime": "Source referral status-change timestamp.",
    "encounter_type_code": "Source encounter-type code.",
    "encounter_type_display": "Source encounter-type display.",
    "suspected_cancer_site_code": "Source suspected-cancer-site code.",
    "suspected_cancer_site_display": "Source suspected-cancer-site display.",
    "treatment_function_code": "Source treatment-function code.",
    "treatment_function_display": "Source treatment-function display.",
    "service_type_requested_code": "Source requested-service-type code.",
    "service_type_requested_display": "Source requested-service-type display.",
    "site_code": "Source site code.",
    "site_display": "Source site display.",
    "referring_facility_code": "Source referring-facility code.",
    "referring_facility_display": "Source referring-facility display.",
    "referred_by_org_id": "Raw referring organization identifier.",
    "booking_type_code": "Source booking-type code.",
    "booking_type_display": "Source booking-type display.",
    "admin_category_code": "Source administrative-category code.",
    "admin_category_display": "Source administrative-category display.",
    "business_unit": "Source business unit.",
    "division": "Source division.",
    "original_received_datetime": "Original referral received timestamp.",
    "ers_ubrn_received": "e-Referral UBRN received date.",
    "ers_pathway_start": "e-Referral pathway start date.",
    "ers_service_name": "e-Referral service name.",
    "ers_specialty": "e-Referral specialty.",
    "record_status": "Normalized source lifecycle status.",
    "record_status_effective_from": "Source row creation timestamp.",
    "record_status_effective_to":
        "Source lifecycle end or source-absence detection timestamp.",
    "confidentiality_code": "Source confidentiality code when supplied.",
    "vip_ind": "Source VIP indicator when supplied.",
    "withheld_identity_ind": "Withheld-identity indicator when supplied.",
    "fact_category":
        "Whether this fact is clinical or administrative in the v2 plane merge.",
    "source_feed": "Registered feed owning the fact.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Latest native source update timestamp.",
    "loaded_at": "Latest bronze load timestamp.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.referral"),
    comment=(
        "LUNA referral fact activated as a recorded contract change from a zero-row stub with "
        "no consumer break. Gold QC twin of the silver product: 2 columns are repaired or "
        "nulled, 1 rule(s) drop rows, 3 check(s) are advisory. Each rule states its reason in "
        "the pipeline notebook, and Lakeflow expectation metrics report what every rule "
        "matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_REFERRAL_MANDATORY_RULES)
@_expect_all(CLINICAL_REFERRAL_ADVISORY_RULES)
def gold_clinical_referral():
    """Quality-controlled twin of journey_clinical.referral."""
    # 4 rows point at a person_id the spine does not have. The pointer is nulled so it
    # cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    df = _qc(
        "clinical_referral",
        CLINICAL_REFERRAL_SELECT,
        fk_columns=["person_id"],
        date_flags=["event_after_death_30d", "event_before_birth"],
    )
    return _with_comments(df, CLINICAL_REFERRAL_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.registry_entry ====

CLINICAL_REGISTRY_ENTRY_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",
    "`event_datetime` AS `event_datetime`",

    # event_end_datetime cannot precede event_datetime. The start is the better-attested of
    # the two, so the end is what goes and the row keeps its event_datetime. Hit 354 of
    # 236,690 rows (0.15%) when profiled on 2026-08-24.
    "CASE WHEN `event_datetime` IS NOT NULL AND `event_end_datetime` IS NOT NULL AND `event_datetime` > `event_end_datetime` THEN NULL ELSE `event_end_datetime` END AS `event_end_datetime`",
    "`registry_family` AS `registry_family`",
    "`registry_type` AS `registry_type`",
    "`entry_id` AS `entry_id`",
    "`parent_registry_entry_id` AS `parent_registry_entry_id`",
    "`linkage_status` AS `linkage_status`",
    "`mrn` AS `mrn`",
    "`nhs_number` AS `nhs_number`",
    "`date_of_death` AS `date_of_death`",
    "`registry_payload` AS `registry_payload`",
    "`source_edit_datetime` AS `source_edit_datetime`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
    "`event_before_birth` AS `event_before_birth`",
    "`event_after_death_30d` AS `event_after_death_30d`",
]

CLINICAL_REGISTRY_ENTRY_MANDATORY_RULES = {
    # The research surface. identity_status = 'resolved' keeps the 236,653 rows of 236,690
    # that are current and attributable. Superseded versions and rows whose identity was
    # never resolved are not research data, and a consumer who wants them has silver.
    "research_surface": "(identity_status = 'resolved')",
}

CLINICAL_REGISTRY_ENTRY_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 37 of 236,690 at the profile.
    "gold.clinical.registry_entry.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 18 of 236,690 rows (0.0076%) when profiled on 2026-08-24.
    "gold.clinical.registry_entry.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 1 of 236,690 rows (0.000422%) when profiled on 2026-08-24.
    "gold.clinical.registry_entry.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",
}

CLINICAL_REGISTRY_ENTRY_COLUMN_COMMENTS = {
    "patient_event_id": "Stable registry event identifier.",
    "fact_row_id": "Storage-row identifier equal to patient_event_id.",
    "subject_key": "Always-populated peppered subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference when available.",
    "event_datetime":
        "Family-specific clinical timestamp; child rows inherit the parent timestamp.",
    "event_end_datetime": "Family-specific clinical end timestamp.",
    "registry_family": "Parameterized registry family.",
    "registry_type": "Registry subtype for composite-key MDT rows.",
    "entry_id": "Verbatim iWeb entry identifier.",
    "parent_registry_entry_id": "Stable parent registry event identifier for child lanes.",
    "linkage_status": "Bronze person-link status.",
    "mrn": "Source MRN where supplied.",
    "nhs_number": "Source NHS number where supplied.",
    "date_of_death": "Family-specific demographic date of death.",
    "registry_payload": "Full verbatim source row as key/value pairs.",
    "source_edit_datetime": "Source edit timestamp; never treated as clinical time.",
    "record_status": "Normalized source-record lifecycle.",
    "record_status_effective_from": "Lifecycle start.",
    "record_status_effective_to": "Source absence timestamp.",
    "confidentiality_code": "Security classification when supplied.",
    "vip_ind": "VIP indicator when supplied.",
    "withheld_identity_ind": "Withheld-identity indicator.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Source edit timestamp.",
    "loaded_at": "Bronze load timestamp.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.registry_entry"),
    comment=(
        "One iWeb cardiac-registry entry from one of ten registered families, retaining the "
        "full source row as a governed VARIANT payload. Gold QC twin of the silver product: 1 "
        "columns are repaired or nulled, 1 rule(s) drop rows, 3 check(s) are advisory. Each "
        "rule states its reason in the pipeline notebook, and Lakeflow expectation metrics "
        "report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_REGISTRY_ENTRY_MANDATORY_RULES)
@_expect_all(CLINICAL_REGISTRY_ENTRY_ADVISORY_RULES)
def gold_clinical_registry_entry():
    """Quality-controlled twin of journey_clinical.registry_entry."""
    df = _qc("clinical_registry_entry", CLINICAL_REGISTRY_ENTRY_SELECT)
    return _with_comments(df, CLINICAL_REGISTRY_ENTRY_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.research_enrollment ====

CLINICAL_RESEARCH_ENROLLMENT_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",
    "`event_datetime` AS `event_datetime`",
    "`event_end_datetime` AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`research_study_id` AS `research_study_id`",
    "`registration_id` AS `registration_id`",
    "`protocol_accession_number` AS `protocol_accession_number`",
    "`protocol_arm_id` AS `protocol_arm_id`",
    "`status_code` AS `status_code`",
    "`status_desc` AS `status_desc`",
    "`off_study_datetime` AS `off_study_datetime`",
    "`treatment_start_datetime` AS `treatment_start_datetime`",
    "`treatment_completion_datetime` AS `treatment_completion_datetime`",
    "`removal_reason_desc` AS `removal_reason_desc`",

    # An empty or whitespace-only string is how the source writes 'nothing here'. It reads
    # as a value in a query and is not one, so it is nulled. Hit 34,111 of 34,354 rows
    # (99.3%) when profiled on 2026-08-24.
    "CASE WHEN TRIM(CAST(`removal_reason_text` AS STRING)) = '' THEN NULL ELSE `removal_reason_text` END AS `removal_reason_text`",
    "`off_treatment_reason_desc` AS `off_treatment_reason_desc`",

    # An empty or whitespace-only string is how the source writes 'nothing here'. It reads
    # as a value in a query and is not one, so it is nulled. Hit 34,325 of 34,354 rows
    # (99.9%) when profiled on 2026-08-24.
    "CASE WHEN TRIM(CAST(`off_treatment_reason_text` AS STRING)) = '' THEN NULL ELSE `off_treatment_reason_text` END AS `off_treatment_reason_text`",
    "`source_episode_id` AS `source_episode_id`",
    "`enrolling_organization_id` AS `enrolling_organization_id`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`fact_category` AS `fact_category`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
]

CLINICAL_RESEARCH_ENROLLMENT_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 34,354 rows of 34,354 that
    # are current and attributable; identity_status = 'resolved' keeps the 34,354 rows of
    # 34,354 that are current and attributable. Superseded versions and rows whose identity
    # was never resolved are not research data, and a consumer who wants them has silver.
    "research_surface": "(identity_status = 'resolved') AND (record_status = 'active')",
}

CLINICAL_RESEARCH_ENROLLMENT_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 0 of 34,354 at the profile.
    "gold.clinical.research_enrollment.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 0 of 34,354 at the profile.
    "gold.clinical.research_enrollment.record_status.default_view_active":
        "record_status = 'active'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 24 of 34,354 rows (0.0699%) when profiled on 2026-08-24.
    "gold.clinical.research_enrollment.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 2 of 34,354 rows (0.00582%) when profiled on 2026-08-24.
    "gold.clinical.research_enrollment.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",
}

CLINICAL_RESEARCH_ENROLLMENT_COLUMN_COMMENTS = {
    "patient_event_id": "Published field patient_event_id.",
    "fact_row_id": "Published field fact_row_id.",
    "subject_key": "Published field subject_key.",
    "subject_id_system": "Published field subject_id_system.",
    "person_id": "Published field person_id.",
    "identity_status": "Published field identity_status.",
    "encounter_id": "Published field encounter_id.",
    "event_datetime": "Published field event_datetime.",
    "event_end_datetime": "Published field event_end_datetime.",
    "source_coding_system": "Published field source_coding_system.",
    "source_code": "Published field source_code.",
    "source_display": "Published field source_display.",
    "research_study_id": "Published field research_study_id.",
    "registration_id": "Published field registration_id.",
    "protocol_accession_number": "Published field protocol_accession_number.",
    "protocol_arm_id": "Published field protocol_arm_id.",
    "status_code": "Published field status_code.",
    "status_desc": "Published field status_desc.",
    "off_study_datetime": "Published field off_study_datetime.",
    "treatment_start_datetime": "Published field treatment_start_datetime.",
    "treatment_completion_datetime": "Published field treatment_completion_datetime.",
    "removal_reason_desc": "Published field removal_reason_desc.",
    "removal_reason_text":
        "Published field removal_reason_text. Gold QC transform rules: gold.clinical.research_enrollment.removal_reason_text.empty_string.",
    "off_treatment_reason_desc": "Published field off_treatment_reason_desc.",
    "off_treatment_reason_text":
        "Published field off_treatment_reason_text. Gold QC transform rules: gold.clinical.research_enrollment.off_treatment_reason_text.empty_string.",
    "source_episode_id": "Published field source_episode_id.",
    "enrolling_organization_id": "Published field enrolling_organization_id.",
    "record_status": "Published field record_status.",
    "record_status_effective_from": "Published field record_status_effective_from.",
    "record_status_effective_to": "Published field record_status_effective_to.",
    "confidentiality_code": "Published field confidentiality_code.",
    "vip_ind": "Published field vip_ind.",
    "withheld_identity_ind": "Published field withheld_identity_ind.",
    "fact_category": "Published field fact_category.",
    "source_feed": "Published field source_feed.",
    "load_batch_id": "Published field load_batch_id.",
    "source_update_timestamp": "Published field source_update_timestamp.",
    "loaded_at": "Published field loaded_at.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.research_enrollment"),
    comment=(
        "One Cerner research-subject registration using sentinel-safe clean dates. Gold QC "
        "twin of the silver product: 2 columns are repaired or nulled, 1 rule(s) drop rows, 4 "
        "check(s) are advisory. Each rule states its reason in the pipeline notebook, and "
        "Lakeflow expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_RESEARCH_ENROLLMENT_MANDATORY_RULES)
@_expect_all(CLINICAL_RESEARCH_ENROLLMENT_ADVISORY_RULES)
def gold_clinical_research_enrollment():
    """Quality-controlled twin of journey_clinical.research_enrollment."""
    df = _qc(
        "clinical_research_enrollment",
        CLINICAL_RESEARCH_ENROLLMENT_SELECT,
        date_flags=["event_after_death_30d", "event_before_birth"],
    )
    return _with_comments(df, CLINICAL_RESEARCH_ENROLLMENT_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.rtt_activity ====

CLINICAL_RTT_ACTIVITY_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",
    "`event_datetime` AS `event_datetime`",
    "`event_end_datetime` AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`rtt_activity_oid` AS `rtt_activity_oid`",
    "`pathway_oid` AS `pathway_oid`",
    "`referral_id` AS `referral_id`",
    "`appointment_oid` AS `appointment_oid`",
    "`activity_code` AS `activity_code`",
    "`activity_display` AS `activity_display`",
    "`activity_type_code` AS `activity_type_code`",
    "`activity_type_display` AS `activity_type_display`",
    "`status_code` AS `status_code`",
    "`status_display` AS `status_display`",
    "`status_sequence_asc` AS `status_sequence_asc`",
    "`status_sequence_desc` AS `status_sequence_desc`",
    "`activity_sequence_asc` AS `activity_sequence_asc`",
    "`activity_sequence_desc` AS `activity_sequence_desc`",
    "`is_illogical` AS `is_illogical`",
    "`activity_datetime_quality` AS `activity_datetime_quality`",
    "`treatment_function_code` AS `treatment_function_code`",
    "`treatment_function_display` AS `treatment_function_display`",
    "`site_code` AS `site_code`",
    "`site_display` AS `site_display`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`fact_category` AS `fact_category`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
]

CLINICAL_RTT_ACTIVITY_MANDATORY_RULES = {
    # The research surface. identity_status = 'resolved' keeps the 68,818,631 rows of
    # 68,820,429 that are current and attributable. Superseded versions and rows whose
    # identity was never resolved are not research data, and a consumer who wants them has
    # silver.
    "research_surface": "(identity_status = 'resolved')",
}

CLINICAL_RTT_ACTIVITY_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 1,798 of 68,820,429 at the profile.
    "gold.clinical.rtt_activity.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 30,737 of 68,820,429 rows (0.0447%) when profiled on 2026-08-24.
    "gold.clinical.rtt_activity.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 2,170 of 68,820,429 rows (0.00315%) when profiled on 2026-08-24.
    "gold.clinical.rtt_activity.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",
}

CLINICAL_RTT_ACTIVITY_COLUMN_COMMENTS = {
    "patient_event_id": "Stable product-wide RTT activity identifier.",
    "fact_row_id": "Storage-row identifier equal to patient_event_id.",
    "subject_key": "Always-populated peppered subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id":
        "Resolved Millennium person identifier from the bronze crosswalk when available.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter context; not supplied by the LUNA RTT activity feed.",
    "event_datetime": "RTT activity timestamp.",
    "event_end_datetime": "Event end timestamp; not supplied by LUNA RTT activities.",
    "source_coding_system": "Verbatim LUNA RTT-status coding system.",
    "source_code": "Verbatim RTT-status code.",
    "source_display": "Verbatim RTT-status display.",
    "rtt_activity_oid": "Raw LUNA RTT activity object identifier.",
    "pathway_oid": "Raw LUNA pathway object identifier.",
    "referral_id":
        "Minted referral patient_event_id when a source referral object identifier is present.",
    "appointment_oid":
        "Raw appointment linkage evidence; resolution is gated and no edge is emitted.",
    "activity_code": "Source RTT activity code.",
    "activity_display": "Source RTT activity display.",
    "activity_type_code": "Source RTT activity-type code.",
    "activity_type_display": "Source RTT activity-type display.",
    "status_code": "Source RTT-status code.",
    "status_display": "Source RTT-status display.",
    "status_sequence_asc": "Ascending RTT-status sequence.",
    "status_sequence_desc": "Descending RTT-status sequence.",
    "activity_sequence_asc": "Ascending RTT-activity sequence.",
    "activity_sequence_desc": "Descending RTT-activity sequence.",
    "is_illogical": "Bronze quality flag carried as data and never filtered.",
    "activity_datetime_quality":
        "Bronze quality classification for the activity timestamp.",
    "treatment_function_code": "Source treatment-function code.",
    "treatment_function_display": "Source treatment-function display.",
    "site_code": "Source site code.",
    "site_display": "Source site display.",
    "record_status": "Normalized source lifecycle status.",
    "record_status_effective_from": "Source activity creation timestamp.",
    "record_status_effective_to":
        "Source lifecycle end or source-absence detection timestamp.",
    "confidentiality_code": "Source confidentiality code when supplied.",
    "vip_ind": "Source VIP indicator when supplied.",
    "withheld_identity_ind": "Withheld-identity indicator when supplied.",
    "fact_category":
        "Whether this fact is clinical or administrative in the v2 plane merge.",
    "source_feed": "Registered feed owning the fact.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Latest native source update timestamp.",
    "loaded_at": "Latest bronze load timestamp.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.rtt_activity"),
    comment=(
        "LUNA clock-affecting RTT activity and status events with quality flags retained. "
        "Gold QC twin of the silver product: 1 columns are repaired or nulled, 1 rule(s) drop "
        "rows, 3 check(s) are advisory. Each rule states its reason in the pipeline notebook, "
        "and Lakeflow expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_RTT_ACTIVITY_MANDATORY_RULES)
@_expect_all(CLINICAL_RTT_ACTIVITY_ADVISORY_RULES)
def gold_clinical_rtt_activity():
    """Quality-controlled twin of journey_clinical.rtt_activity."""
    # 6 rows point at a person_id the spine does not have. The pointer is nulled so it
    # cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    df = _qc(
        "clinical_rtt_activity",
        CLINICAL_RTT_ACTIVITY_SELECT,
        fk_columns=["person_id"],
        date_flags=["event_after_death_30d", "event_before_birth"],
    )
    return _with_comments(df, CLINICAL_RTT_ACTIVITY_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.rtt_pathway ====

CLINICAL_RTT_PATHWAY_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",

    # 3 rows point at a person_id the spine does not have. The pointer is nulled so it
    # cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    "CASE WHEN NOT `person_id_resolved` THEN NULL ELSE `person_id` END AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",

    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 1 of 7,864,172 rows (1.27e-05%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `event_datetime` END AS `event_datetime`",

    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 1 of 7,864,172 rows (1.27e-05%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`event_end_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `event_end_datetime` END AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`pathway_oid` AS `pathway_oid`",
    "`period_oid` AS `period_oid`",
    "`is_latest_period` AS `is_latest_period`",
    "`start_status_code` AS `start_status_code`",
    "`start_status_display` AS `start_status_display`",
    "`stop_status_code` AS `stop_status_code`",
    "`stop_status_display` AS `stop_status_display`",
    "`current_status_code` AS `current_status_code`",
    "`current_status_display` AS `current_status_display`",
    "`sequence_asc` AS `sequence_asc`",
    "`sequence_desc` AS `sequence_desc`",
    "`clock_discrepant` AS `clock_discrepant`",
    "`core_clock_start` AS `core_clock_start`",
    "`core_clock_stop` AS `core_clock_stop`",
    "`pathway_start_date` AS `pathway_start_date`",
    "`pathway_type_code` AS `pathway_type_code`",
    "`pathway_type_display` AS `pathway_type_display`",
    "`breach_date` AS `breach_date`",
    "`days_waited` AS `days_waited`",
    "`days_waited_active` AS `days_waited_active`",
    "`op_appt_dna_count` AS `op_appt_dna_count`",
    "`treatment_function_code` AS `treatment_function_code`",
    "`treatment_function_display` AS `treatment_function_display`",
    "`site_code` AS `site_code`",
    "`site_display` AS `site_display`",
    "`referring_facility_code` AS `referring_facility_code`",
    "`referring_facility_display` AS `referring_facility_display`",
    "`encounter_types` AS `encounter_types`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",

    # record_status_effective_to cannot precede record_status_effective_from. The start is
    # the better-attested of the two, so the end is what goes and the row keeps its
    # record_status_effective_from. Hit 3 of 7,864,172 rows (3.81e-05%) when profiled on
    # 2026-08-24.
    "CASE WHEN `record_status_effective_from` IS NOT NULL AND `record_status_effective_to` IS NOT NULL AND `record_status_effective_from` > `record_status_effective_to` THEN NULL ELSE `record_status_effective_to` END AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`fact_category` AS `fact_category`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",

    # Silver derived this flag by comparing event_datetime with the person's own dates,
    # before the rules above corrected event_datetime. On the rows where event_datetime
    # changed, the flag describes a timestamp gold no longer publishes. This product carries
    # a VARIANT column, which rules out the spine join gold would need to recompute the
    # flag, so it is nulled where the value beneath it moved rather than left asserting
    # something stale.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `event_before_birth` END AS `event_before_birth`",

    # Silver derived this flag by comparing event_datetime with the person's own dates,
    # before the rules above corrected event_datetime. On the rows where event_datetime
    # changed, the flag describes a timestamp gold no longer publishes. This product carries
    # a VARIANT column, which rules out the spine join gold would need to recompute the
    # flag, so it is nulled where the value beneath it moved rather than left asserting
    # something stale.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `event_after_death_30d` END AS `event_after_death_30d`",
]

CLINICAL_RTT_PATHWAY_MANDATORY_RULES = {
    # The research surface. identity_status = 'resolved' keeps the 7,862,177 rows of
    # 7,864,172 that are current and attributable. Superseded versions and rows whose
    # identity was never resolved are not research data, and a consumer who wants them has
    # silver.
    "research_surface": "(identity_status = 'resolved')",
}

CLINICAL_RTT_PATHWAY_ADVISORY_RULES = {
    # Counted rather than nulled because a breach date is a deadline counted forward from
    # the clock start, so it is in the future for every pathway still inside its target.
    # Seen on 70,601 of 7,864,172 rows (0.898%) when profiled on 2026-08-24.
    "gold.clinical.rtt_pathway.breach_date.future_owner":
        "NOT COALESCE((CAST(`breach_date` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",

    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 1,995 of 7,864,172 at the profile.
    "gold.clinical.rtt_pathway.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 277 of 7,864,172 rows (0.00352%) when profiled on 2026-08-24.
    "gold.clinical.rtt_pathway.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 832 of 7,864,172 rows (0.0106%) when profiled on 2026-08-24.
    "gold.clinical.rtt_pathway.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",
}

CLINICAL_RTT_PATHWAY_COLUMN_COMMENTS = {
    "patient_event_id": "Stable product-wide RTT period identifier.",
    "fact_row_id": "Storage-row identifier equal to patient_event_id.",
    "subject_key": "Always-populated peppered subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id":
        "Resolved Millennium person identifier from the bronze crosswalk when available.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter context; not supplied by the LUNA RTT pathway feed.",
    "event_datetime": "RTT clock-period start timestamp.",
    "event_end_datetime": "RTT clock-period stop timestamp.",
    "source_coding_system": "Verbatim LUNA RTT-status coding system.",
    "source_code": "Verbatim current RTT-status code.",
    "source_display": "Verbatim current RTT-status display.",
    "pathway_oid": "Raw LUNA pathway object identifier.",
    "period_oid":
        "Raw LUNA period object identifier; 0 marks the clockless-pathway sentinel and the column is never null.",
    "is_latest_period": "Whether this is the latest period for the pathway.",
    "start_status_code": "Source clock-start RTT-status code.",
    "start_status_display": "Source clock-start RTT-status display.",
    "stop_status_code": "Source clock-stop RTT-status code.",
    "stop_status_display": "Source clock-stop RTT-status display.",
    "current_status_code": "Source current RTT-status code.",
    "current_status_display": "Source current RTT-status display.",
    "sequence_asc": "Ascending pathway-period sequence.",
    "sequence_desc": "Descending pathway-period sequence.",
    "clock_discrepant":
        "Bronze flag indicating disagreement with the core clock timestamps.",
    "core_clock_start": "Core-system clock start retained as discrepant sidecar data.",
    "core_clock_stop": "Core-system clock stop retained as discrepant sidecar data.",
    "pathway_start_date": "Source pathway start date.",
    "pathway_type_code": "Source pathway-type code.",
    "pathway_type_display": "Source pathway-type display.",
    "breach_date": "Source breach timestamp.",
    "days_waited": "Source total days waited.",
    "days_waited_active": "Source active days waited.",
    "op_appt_dna_count": "Source outpatient did-not-attend count.",
    "treatment_function_code": "Source treatment-function code.",
    "treatment_function_display": "Source treatment-function display.",
    "site_code": "Source site code.",
    "site_display": "Source site display.",
    "referring_facility_code": "Source referring-facility code.",
    "referring_facility_display": "Source referring-facility display.",
    "encounter_types": "Deterministically sorted folded LUNA pathway encounter-type tags.",
    "record_status": "Normalized source lifecycle status.",
    "record_status_effective_from": "Source clock-period creation timestamp.",
    "record_status_effective_to":
        "Source lifecycle end or source-absence detection timestamp.",
    "confidentiality_code": "Source confidentiality code when supplied.",
    "vip_ind": "Source VIP indicator when supplied.",
    "withheld_identity_ind": "Withheld-identity indicator when supplied.",
    "fact_category":
        "Whether this fact is clinical or administrative in the v2 plane merge.",
    "source_feed": "Registered feed owning the fact.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Latest native source update timestamp.",
    "loaded_at": "Latest bronze load timestamp.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.rtt_pathway"),
    comment=(
        "LUNA RTT pathway and clock-period facts including clockless sentinel rows. Gold QC "
        "twin of the silver product: 6 columns are repaired or nulled, 1 rule(s) drop rows, 4 "
        "check(s) are advisory. Each rule states its reason in the pipeline notebook, and "
        "Lakeflow expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_RTT_PATHWAY_MANDATORY_RULES)
@_expect_all(CLINICAL_RTT_PATHWAY_ADVISORY_RULES)
def gold_clinical_rtt_pathway():
    """Quality-controlled twin of journey_clinical.rtt_pathway."""
    df = _qc("clinical_rtt_pathway", CLINICAL_RTT_PATHWAY_SELECT)
    return _with_comments(df, CLINICAL_RTT_PATHWAY_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.specimen ====

CLINICAL_SPECIMEN_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",

    # 135 rows point at a person_id the spine does not have. The pointer is nulled so it
    # cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    "CASE WHEN NOT `person_id_resolved` THEN NULL ELSE `person_id` END AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",

    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 96 of 172,077,158 rows (5.58e-05%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `event_datetime` END AS `event_datetime`",
    "`event_end_datetime` AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 37 of 172,077,158 rows
    # (2.15e-05%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`source_code` AS STRING))) = '0' THEN NULL ELSE `source_code` END AS `source_code`",
    "`source_display` AS `source_display`",
    "`specimen_type` AS `specimen_type`",
    "`accession_identifier` AS `accession_identifier`",
    "`primary_source_accession_id` AS `primary_source_accession_id`",
    "`normalized_lab_no` AS `normalized_lab_no`",
    "`canonical_accession_status` AS `canonical_accession_status`",
    "`person_resolution_status` AS `person_resolution_status`",
    "`lab_series` AS `lab_series`",
    "`discipline` AS `discipline`",
    "`urgent_flag` AS `urgent_flag`",
    "`research_qi_only` AS `research_qi_only`",
    "`clinical_details` AS `clinical_details`",
    "`tlcs_requested` AS `tlcs_requested`",
    "`conditions` AS `conditions`",
    "`reason` AS `reason`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 9 of 172,077,158 rows
    # (5.23e-06%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`body_site_code` AS STRING))) = '0' THEN NULL ELSE `body_site_code` END AS `body_site_code`",
    "`body_site_snomed_code` AS `body_site_snomed_code`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 37 of 172,077,158 rows
    # (2.15e-05%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`specimen_type_code` AS STRING))) = '0' THEN NULL ELSE `specimen_type_code` END AS `specimen_type_code`",
    "`specimen_type_snomed_code` AS `specimen_type_snomed_code`",

    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 96 of 172,077,158 rows (5.58e-05%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`sample_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `sample_datetime` END AS `sample_datetime`",

    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 1 of 172,077,158 rows (5.81e-07%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`request_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `request_datetime` END AS `request_datetime`",

    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 13 of 172,077,158 rows (7.55e-06%) when profiled on 2026-08-24.
    #
    # Stamps landing in January 1970 carry the same millisecond-epoch defect found on
    # clinical_score and vital_sign, but nothing here corroborates a rescaled value the way
    # the encounter window does there, so the stamp is nulled rather than reconstructed. The
    # catalogue's date rules stop at 1901 and do not reach this.
    "CASE WHEN CAST(`report_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS OR YEAR(CAST(`report_datetime` AS TIMESTAMP)) = 1970 THEN NULL ELSE `report_datetime` END AS `report_datetime`",
    "`source_history_row_count` AS `source_history_row_count`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",

    # Silver derived this flag by comparing event_datetime with the person's own dates,
    # before the rules above corrected event_datetime. On the rows where event_datetime
    # changed, the flag describes a timestamp gold no longer publishes. This product carries
    # a VARIANT column, which rules out the spine join gold would need to recompute the
    # flag, so it is nulled where the value beneath it moved rather than left asserting
    # something stale.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `event_before_birth` END AS `event_before_birth`",

    # Silver derived this flag by comparing event_datetime with the person's own dates,
    # before the rules above corrected event_datetime. On the rows where event_datetime
    # changed, the flag describes a timestamp gold no longer publishes. This product carries
    # a VARIANT column, which rules out the spine join gold would need to recompute the
    # flag, so it is nulled where the value beneath it moved rather than left asserting
    # something stale.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `event_after_death_30d` END AS `event_after_death_30d`",
]

CLINICAL_SPECIMEN_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 172,077,158 rows of
    # 172,077,158 that are current and attributable; identity_status = 'resolved' keeps the
    # 150,927,313 rows of 172,077,158 that are current and attributable. Superseded versions
    # and rows whose identity was never resolved are not research data, and a consumer who
    # wants them has silver.
    "research_surface": "(identity_status = 'resolved') AND (record_status = 'active')",
}

CLINICAL_SPECIMEN_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 21,149,845 of 172,077,158 at the profile.
    "gold.clinical.specimen.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 0 of 172,077,158 at the profile.
    "gold.clinical.specimen.record_status.default_view_active": "record_status = 'active'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 88,413 of 172,077,158 rows (0.0514%) when profiled on
    # 2026-08-24.
    "gold.clinical.specimen.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 1,203,382 of 172,077,158 rows (0.699%) when profiled on
    # 2026-08-24.
    "gold.clinical.specimen.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",
}

CLINICAL_SPECIMEN_COLUMN_COMMENTS = {
    "patient_event_id":
        "Stable specimen event identifier minted from pathology_accession_id.",
    "fact_row_id": "Storage-row identifier equal to patient_event_id.",
    "subject_key": "Always-populated subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference when supplied.",
    "event_datetime": "Collection or receipt timestamp.",
    "event_end_datetime": "Event end when supplied.",
    "source_coding_system": "Source specimen type system.",
    "source_code": "Source specimen type code.",
    "source_display": "Source specimen type display.",
    "specimen_type": "Source specimen type CodeableConcept.",
    "accession_identifier":
        "Canonical accession identifier (pathology_accession_id); never LabNo-derived.",
    "primary_source_accession_id":
        "Primary source accession identifier retained as evidence.",
    "normalized_lab_no": "Matching evidence only — reused lab numbers reach 3",
    "canonical_accession_status": "Canonical accession-link state.",
    "person_resolution_status": "Person-projection eligibility state.",
    "lab_series": "Source laboratory series.",
    "discipline": "Source pathology discipline.",
    "urgent_flag": "Source urgent-request indicator code (Y",
    "research_qi_only":
        "Registry doctrine flag; true on 100% of rows today — describe-only",
    "clinical_details": "Source clinical details. Identifiable free text; ig_risk 4",
    "tlcs_requested": "Requested TLC context. Identifiable free text; ig_risk 4",
    "conditions": "Source request conditions. Identifiable free text; ig_risk 4",
    "reason": "Source request reason. Identifiable free text; ig_risk 4",
    "body_site_code": "Source collection body-site code.",
    "body_site_snomed_code": "Source-provided SNOMED body-site code.",
    "specimen_type_code": "Native specimen-type code.",
    "specimen_type_snomed_code": "Source-provided SNOMED specimen-type code.",
    "sample_datetime": "Clamped source sample timestamp.",
    "request_datetime": "Clamped source request timestamp.",
    "report_datetime": "Clamped source report timestamp.",
    "source_history_row_count": "accession_source rows behind this accession.",
    "record_status": "Normalized source-record lifecycle.",
    "record_status_effective_from": "Source validity start.",
    "record_status_effective_to": "Source validity end.",
    "confidentiality_code": "Source confidentiality code.",
    "vip_ind": "VIP indicator.",
    "withheld_identity_ind": "Withheld identity indicator.",
    "source_feed": "Registered owning feed.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.specimen"),
    comment=(
        "One canonical pathology accession with collection and request evidence. Gold QC twin "
        "of the silver product: 10 columns are repaired or nulled, 1 rule(s) drop rows, 4 "
        "check(s) are advisory. Each rule states its reason in the pipeline notebook, and "
        "Lakeflow expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_SPECIMEN_MANDATORY_RULES)
@_expect_all(CLINICAL_SPECIMEN_ADVISORY_RULES)
def gold_clinical_specimen():
    """Quality-controlled twin of journey_clinical.specimen."""
    df = _qc("clinical_specimen", CLINICAL_SPECIMEN_SELECT)
    return _with_comments(df, CLINICAL_SPECIMEN_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.susceptibility_result ====

CLINICAL_SUSCEPTIBILITY_RESULT_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",
    "`event_datetime` AS `event_datetime`",
    "`event_end_datetime` AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`microbiology_isolate_id` AS `microbiology_isolate_id`",
    "`pathology_result_id` AS `pathology_result_id`",
    "`link_status` AS `link_status`",
    "`antimicrobial_text` AS `antimicrobial_text`",
    "`antimicrobial_code` AS `antimicrobial_code`",
    "`antimicrobial_omop_concept_id` AS `antimicrobial_omop_concept_id`",
    "`interpretation_raw` AS `interpretation_raw`",
    "`interpretation` AS `interpretation`",
    "`mic_raw` AS `mic_raw`",
    "`mic` AS `mic`",
    "`unit_source_value` AS `unit_source_value`",
    "`method` AS `method`",
    "`lifecycle_status` AS `lifecycle_status`",
    "`is_current` AS `is_current`",
    "`research_qi_only` AS `research_qi_only`",
    "`specimen_id` AS `specimen_id`",
    "`accession_identifier` AS `accession_identifier`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
]

CLINICAL_SUSCEPTIBILITY_RESULT_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 0 rows of 0 that are current
    # and attributable; identity_status = 'resolved' keeps the 0 rows of 0 that are current
    # and attributable. Superseded versions and rows whose identity was never resolved are
    # not research data, and a consumer who wants them has silver.
    "research_surface": "(identity_status = 'resolved') AND (record_status = 'active')",
}

CLINICAL_SUSCEPTIBILITY_RESULT_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 0 of 0 at the profile.
    "gold.clinical.susceptibility_result.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 0 of 0 at the profile.
    "gold.clinical.susceptibility_result.record_status.default_view_active":
        "record_status = 'active'",
}

CLINICAL_SUSCEPTIBILITY_RESULT_COLUMN_COMMENTS = {
    "patient_event_id": "Stable susceptibility event identifier.",
    "fact_row_id": "Storage-row identifier.",
    "subject_key": "Best available subject key.",
    "subject_id_system": "Subject-key system.",
    "person_id": "Resolved person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference; unavailable at this grain.",
    "event_datetime": "Accession sample/report/request fallback time.",
    "event_end_datetime": "Event end time.",
    "source_coding_system": "Antimicrobial coding namespace.",
    "source_code": "Antimicrobial code or text.",
    "source_display": "Antimicrobial display.",
    "microbiology_isolate_id": "Linked isolate fact when unique.",
    "pathology_result_id": "Linked pathology-result fact.",
    "link_status": "Isolate-link resolution state.",
    "antimicrobial_text": "Antimicrobial as reported.",
    "antimicrobial_code": "Source antimicrobial code.",
    "antimicrobial_omop_concept_id": "Standard OMOP antimicrobial concept.",
    "interpretation_raw": "Raw susceptibility result.",
    "interpretation": "Normalized susceptibility interpretation.",
    "mic_raw": "Raw MIC text.",
    "mic": "Parsed MIC value.",
    "unit_source_value": "Raw MIC unit.",
    "method": "Susceptibility method.",
    "lifecycle_status": "Inherited source lifecycle.",
    "is_current": "Whether the result remains current.",
    "research_qi_only": "Research/QI release flag.",
    "specimen_id": "Alias-resolved accession specimen reference.",
    "accession_identifier": "Alias-resolved accession identifier.",
    "record_status": "Normalized susceptibility lifecycle.",
    "record_status_effective_from": "Status validity start.",
    "record_status_effective_to": "Status validity end.",
    "confidentiality_code": "Security code.",
    "vip_ind": "VIP indicator.",
    "withheld_identity_ind": "Withheld identity indicator.",
    "source_feed": "Owning feed.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze material-change timestamp.",
}

@dp.materialized_view(
    name=_n("gold_clinical.susceptibility_result"),
    comment=(
        "One isolate-antimicrobial susceptibility observation. Schema-first and "
        "empty_by_design in both catalogs on 2026-08-18. Gold QC twin of the silver product: "
        "0 columns are repaired or nulled, 1 rule(s) drop rows, 2 check(s) are advisory. Each "
        "rule states its reason in the pipeline notebook, and Lakeflow expectation metrics "
        "report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_SUSCEPTIBILITY_RESULT_MANDATORY_RULES)
@_expect_all(CLINICAL_SUSCEPTIBILITY_RESULT_ADVISORY_RULES)
def gold_clinical_susceptibility_result():
    """Quality-controlled twin of journey_clinical.susceptibility_result."""
    df = _qc("clinical_susceptibility_result", CLINICAL_SUSCEPTIBILITY_RESULT_SELECT)
    return _with_comments(df, CLINICAL_SUSCEPTIBILITY_RESULT_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.transfusion ====

CLINICAL_TRANSFUSION_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",
    "`event_datetime` AS `event_datetime`",

    # event_end_datetime cannot precede event_datetime. The start is the better-attested of
    # the two, so the end is what goes and the row keeps its event_datetime. Hit 7 of
    # 127,986 rows (0.00547%) when profiled on 2026-08-24.
    "CASE WHEN `event_datetime` IS NOT NULL AND `event_end_datetime` IS NOT NULL AND `event_datetime` > `event_end_datetime` THEN NULL ELSE `event_end_datetime` END AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`begin_datetime` AS `begin_datetime`",

    # end_datetime cannot precede begin_datetime. The start is the better-attested of the
    # two, so the end is what goes and the row keeps its begin_datetime. Hit 7 of 127,986
    # rows (0.00547%) when profiled on 2026-08-24.
    "CASE WHEN `begin_datetime` IS NOT NULL AND `end_datetime` IS NOT NULL AND `begin_datetime` > `end_datetime` THEN NULL ELSE `end_datetime` END AS `end_datetime`",
    "`transfusion_status` AS `transfusion_status`",
    "`elapsed_minutes` AS `elapsed_minutes`",
    "`unit_number` AS `unit_number`",
    "`blood_product_group` AS `blood_product_group`",
    "`blood_unit_group` AS `blood_unit_group`",
    "`patient_blood_group` AS `patient_blood_group`",
    "`quantity_value` AS `quantity_value`",
    "`quantity_raw` AS `quantity_raw`",
    "`begin_location` AS `begin_location`",
    "`end_location` AS `end_location`",
    "`unit_is_irradiated` AS `unit_is_irradiated`",
    "`unit_is_cmv_neg` AS `unit_is_cmv_neg`",
    "`requires_irradiated` AS `requires_irradiated`",
    "`requires_cmv_neg` AS `requires_cmv_neg`",
    "`ambiguity_ind` AS `ambiguity_ind`",
    "`product_concept_id` AS `product_concept_id`",
    "`product_concept_name` AS `product_concept_name`",
    "`unit_group_concept_id` AS `unit_group_concept_id`",
    "`patient_group_concept_id` AS `patient_group_concept_id`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
]

CLINICAL_TRANSFUSION_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 127,986 rows of 127,986 that
    # are current and attributable; identity_status = 'resolved' keeps the 127,955 rows of
    # 127,986 that are current and attributable. Superseded versions and rows whose identity
    # was never resolved are not research data, and a consumer who wants them has silver.
    "research_surface": "(identity_status = 'resolved') AND (record_status = 'active')",
}

CLINICAL_TRANSFUSION_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 31 of 127,986 at the profile.
    "gold.clinical.transfusion.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 0 of 127,986 at the profile.
    "gold.clinical.transfusion.record_status.default_view_active":
        "record_status = 'active'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 1,523 of 127,986 rows (1.19%) when profiled on 2026-08-24.
    "gold.clinical.transfusion.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",
}

CLINICAL_TRANSFUSION_COLUMN_COMMENTS = {
    "patient_event_id": "Stable transfusion event identifier.",
    "fact_row_id": "Storage-row identifier equal to patient_event_id.",
    "subject_key": "Always-populated peppered subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference when available.",
    "event_datetime": "Transfusion begin timestamp falling back to end.",
    "event_end_datetime": "Transfusion end timestamp.",
    "source_coding_system": "BloodTrack product coding system.",
    "source_code": "ISBT product code with product display fallback.",
    "source_display": "Blood product description.",
    "begin_datetime": "Paired transfusion begin timestamp.",
    "end_datetime": "Paired transfusion end timestamp.",
    "transfusion_status": "Pairing and clock-quality outcome.",
    "elapsed_minutes": "Elapsed transfusion minutes when calculable.",
    "unit_number": "Blood unit number.",
    "blood_product_group": "Normalized blood product group.",
    "blood_unit_group": "Blood group recorded on the unit.",
    "patient_blood_group": "Patient blood group at transfusion.",
    "quantity_value": "Parsed transfused quantity.",
    "quantity_raw": "Verbatim quantity text.",
    "begin_location": "Begin workflow location.",
    "end_location": "End workflow location.",
    "unit_is_irradiated": "Unit irradiation flag.",
    "unit_is_cmv_neg": "Unit CMV-negative flag.",
    "requires_irradiated": "Patient requires irradiated product.",
    "requires_cmv_neg": "Patient requires CMV-negative product.",
    "ambiguity_ind": "Pairing ambiguity indicator.",
    "product_concept_id": "Mapped SNOMED device concept identifier.",
    "product_concept_name": "Mapped product concept display.",
    "unit_group_concept_id": "Mapped unit blood-group concept identifier.",
    "patient_group_concept_id": "Mapped patient blood-group concept identifier.",
    "record_status": "Normalized source-record lifecycle.",
    "record_status_effective_from": "Lifecycle start.",
    "record_status_effective_to": "Source absence timestamp.",
    "confidentiality_code": "Security classification when supplied.",
    "vip_ind": "VIP indicator when supplied.",
    "withheld_identity_ind": "Withheld-identity indicator.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.transfusion"),
    comment=(
        "One BloodTrack unit-recipient transfusion episode with paired begin/end evidence and "
        "mapped product and blood-group concepts. Gold QC twin of the silver product: 2 "
        "columns are repaired or nulled, 1 rule(s) drop rows, 3 check(s) are advisory. Each "
        "rule states its reason in the pipeline notebook, and Lakeflow expectation metrics "
        "report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_TRANSFUSION_MANDATORY_RULES)
@_expect_all(CLINICAL_TRANSFUSION_ADVISORY_RULES)
def gold_clinical_transfusion():
    """Quality-controlled twin of journey_clinical.transfusion."""
    df = _qc(
        "clinical_transfusion",
        CLINICAL_TRANSFUSION_SELECT,
        date_flags=["event_after_death_30d"],
    )
    return _with_comments(df, CLINICAL_TRANSFUSION_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.transfusion_event ====

CLINICAL_TRANSFUSION_EVENT_SELECT = [
    "`transfusion_event_id` AS `transfusion_event_id`",
    "`person_id` AS `person_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`identity_status` AS `identity_status`",
    "`event_datetime` AS `event_datetime`",
    "`workflow_step` AS `workflow_step`",
    "`transaction_success_ind` AS `transaction_success_ind`",
    "`unit_number` AS `unit_number`",
    "`product_code` AS `product_code`",
    "`product_description` AS `product_description`",
    "`bloodtrack_unit_id` AS `bloodtrack_unit_id`",
    "`device_name` AS `device_name`",
    "`source_location` AS `source_location`",
    "`linkage_status` AS `linkage_status`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 223,789 of 228,205 rows
    # (98.1%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`response_code` AS STRING))) = '0' THEN NULL ELSE `response_code` END AS `response_code`",
    "`response_text` AS `response_text`",
    "`blood_unit_state` AS `blood_unit_state`",
    "`blood_unit_fate` AS `blood_unit_fate`",
    "`alert_present_ind` AS `alert_present_ind`",
    "`comment_present_ind` AS `comment_present_ind`",
    "`record_status` AS `record_status`",
    "`source_table` AS `source_table`",
    "`source_row_id` AS `source_row_id`",
    "`load_batch_id` AS `load_batch_id`",
    "`loaded_at` AS `loaded_at`",
]

CLINICAL_TRANSFUSION_EVENT_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 228,205 rows of 228,205 that
    # are current and attributable; identity_status = 'resolved' keeps the 227,703 rows of
    # 228,205 that are current and attributable. Superseded versions and rows whose identity
    # was never resolved are not research data, and a consumer who wants them has silver.
    "research_surface": "(identity_status = 'resolved') AND (record_status = 'active')",
}

CLINICAL_TRANSFUSION_EVENT_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 502 of 228,205 at the profile.
    "gold.clinical.transfusion_event.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 0 of 228,205 at the profile.
    "gold.clinical.transfusion_event.record_status.default_view_active":
        "record_status = 'active'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 2,924 of 228,205 rows (1.28%) when profiled on 2026-08-24.
    "gold.clinical.transfusion_event.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",
}

CLINICAL_TRANSFUSION_EVENT_COLUMN_COMMENTS = {
    "transfusion_event_id": "Stable scan-event identifier.",
    "person_id": "Resolved Millennium person identifier.",
    "subject_key": "Always-populated peppered subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "identity_status": "Subject resolution state.",
    "event_datetime": "Effective transaction timestamp.",
    "workflow_step": "BloodTrack workflow step.",
    "transaction_success_ind": "Whether the transaction succeeded.",
    "unit_number": "Blood unit number.",
    "product_code": "Product code scanned.",
    "product_description": "Product display.",
    "bloodtrack_unit_id": "BloodTrack unit identifier.",
    "device_name": "Scanning device name.",
    "source_location": "Source workflow location name.",
    "linkage_status": "Patient linkage status.",
    "response_code": "Device or workflow response code.",
    "response_text": "Device or workflow response text.",
    "blood_unit_state": "Blood unit state.",
    "blood_unit_fate": "Blood unit fate.",
    "alert_present_ind": "Alert presence indicator.",
    "comment_present_ind": "Comment presence indicator.",
    "record_status": "Normalized source-record lifecycle.",
    "source_table": "Registered source table.",
    "source_row_id": "Verbatim bronze row identifier.",
    "load_batch_id": "Bronze batch token.",
    "loaded_at": "Bronze load timestamp.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.transfusion_event"),
    comment=(
        "One BloodTrack scan-grain workflow event, including failed attempts and safety "
        "checks that do not count as transfusions. Gold QC twin of the silver product: 1 "
        "columns are repaired or nulled, 1 rule(s) drop rows, 3 check(s) are advisory. Each "
        "rule states its reason in the pipeline notebook, and Lakeflow expectation metrics "
        "report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_TRANSFUSION_EVENT_MANDATORY_RULES)
@_expect_all(CLINICAL_TRANSFUSION_EVENT_ADVISORY_RULES)
def gold_clinical_transfusion_event():
    """Quality-controlled twin of journey_clinical.transfusion_event."""
    df = _qc(
        "clinical_transfusion_event",
        CLINICAL_TRANSFUSION_EVENT_SELECT,
        date_flags=["event_after_death_30d"],
    )
    return _with_comments(df, CLINICAL_TRANSFUSION_EVENT_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.vital_sign ====

CLINICAL_VITAL_SIGN_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",

    # 1,101 rows point at a person_id the spine does not have. The pointer is nulled so it
    # cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    "CASE WHEN NOT `person_id_resolved` THEN NULL ELSE `person_id` END AS `person_id`",
    "`identity_status` AS `identity_status`",

    # 10,864 rows point at a encounter_id the spine does not have. The pointer is nulled so
    # it cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    "CASE WHEN NOT `encounter_id_resolved` THEN NULL ELSE `encounter_id` END AS `encounter_id`",

    # A block of these stamps was written after being divided by a thousand -- a millisecond
    # epoch stored as seconds -- which lands the whole block in January 1970. Multiplying
    # back up recovers the real time to within about seventeen minutes, and 96.6% of the
    # 11,998,178 recovered stamps land inside the encounter the row already points at. The
    # rescaled value is only accepted when it lands between 1990 and the moment the row was
    # loaded; anything else was a different defect and is nulled by the rule below instead
    # of being invented.
    #
    # The remaining 1970 stamps did not rescale into a plausible date, so the value is known
    # to be wrong and the truth is not known.
    "CASE WHEN YEAR(CAST(`event_datetime` AS TIMESTAMP)) = 1970 AND timestamp_seconds(CAST(unix_timestamp(`event_datetime`) AS BIGINT) * 1000) BETWEEN TIMESTAMP'1990-01-01 00:00:00' AND `loaded_at` THEN timestamp_seconds(CAST(unix_timestamp(`event_datetime`) AS BIGINT) * 1000) WHEN YEAR(CAST(`event_datetime` AS TIMESTAMP)) = 1970 THEN NULL ELSE `event_datetime` END AS `event_datetime`",

    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 254 of 283,505,704 rows (8.96e-05%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`event_end_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `event_end_datetime` END AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`vital_code` AS `vital_code`",

    # A magnitude above 1e12 is outside any scale this field is measured on, so the number
    # carries no meaning even though something was recorded. Hit 24 of 283,505,704 rows
    # (8.47e-06%) when profiled on 2026-08-24.
    #
    # A negative amount is not possible for a dose, quantity or score, and nothing in the
    # row says what the intended magnitude was. Hit 101,207 of 283,505,704 rows (0.0357%)
    # when profiled on 2026-08-24.
    "CASE WHEN ABS(CAST(`value_number` AS DOUBLE)) > 1e12 OR `value_number` < 0 THEN NULL ELSE `value_number` END AS `value_number`",
    "`value_text` AS `value_text`",
    "`unit_source_value` AS `unit_source_value`",
    "`unit_concept_id` AS `unit_concept_id`",
    "`reference_range_low` AS `reference_range_low`",
    "`reference_range_high` AS `reference_range_high`",
    "`method_code` AS `method_code`",
    "`method_display` AS `method_display`",
    "`body_site_code` AS `body_site_code`",
    "`body_site_display` AS `body_site_display`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 132,908,866 of 283,505,704
    # rows (46.9%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`interpretation_code` AS STRING))) = '0' THEN NULL ELSE `interpretation_code` END AS `interpretation_code`",
    "`interpretation_display` AS `interpretation_display`",
    "`result_status_code` AS `result_status_code`",
    "`result_status_display` AS `result_status_display`",
    "`performer_practitioner_id` AS `performer_practitioner_id`",
    "`source_form_id` AS `source_form_id`",
    "`promotion_rule_id` AS `promotion_rule_id`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",
    "`record_status_effective_to` AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",

    # Silver derived this flag by comparing event_datetime with the person's own dates,
    # before the rules above corrected event_datetime. On the rows where event_datetime
    # changed, the flag describes a timestamp gold no longer publishes. This product carries
    # a VARIANT column, which rules out the spine join gold would need to recompute the
    # flag, so it is nulled where the value beneath it moved rather than left asserting
    # something stale.
    "CASE WHEN YEAR(CAST(`event_datetime` AS TIMESTAMP)) = 1970 THEN NULL ELSE `event_before_birth` END AS `event_before_birth`",

    # Silver derived this flag by comparing event_datetime with the person's own dates,
    # before the rules above corrected event_datetime. On the rows where event_datetime
    # changed, the flag describes a timestamp gold no longer publishes. This product carries
    # a VARIANT column, which rules out the spine join gold would need to recompute the
    # flag, so it is nulled where the value beneath it moved rather than left asserting
    # something stale.
    "CASE WHEN YEAR(CAST(`event_datetime` AS TIMESTAMP)) = 1970 THEN NULL ELSE `event_after_death_30d` END AS `event_after_death_30d`",
]

CLINICAL_VITAL_SIGN_MANDATORY_RULES = {
    # The research surface. identity_status = 'resolved' keeps the 283,503,962 rows of
    # 283,505,704 that are current and attributable. Superseded versions and rows whose
    # identity was never resolved are not research data, and a consumer who wants them has
    # silver.
    "research_surface": "(identity_status = 'resolved')",
}

CLINICAL_VITAL_SIGN_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 1,742 of 283,505,704 at the profile.
    "gold.clinical.vital_sign.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 31,497 of 283,505,704 rows (0.0111%) when profiled on
    # 2026-08-24.
    "gold.clinical.vital_sign.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 6,504,311 of 283,505,704 rows (2.29%) when profiled on
    # 2026-08-24.
    "gold.clinical.vital_sign.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",

    # Left as a warning because it fires on 213,158,920 of 283,505,704 rows (75.2%) when
    # profiled on 2026-08-24 -- at that rate the rule's assumption about what event_datetime
    # and event_end_datetime mean is the thing in doubt, not the data. The inverted gaps are
    # mostly minutes, which reads as two clocks rather than two events in the wrong order.
    "gold.clinical.vital_sign.table.ordering_violation_event_datetime_event_end_datetime":
        "NOT COALESCE((`event_datetime` IS NOT NULL AND `event_end_datetime` IS NOT NULL AND `event_datetime` > `event_end_datetime`), FALSE)",
}

CLINICAL_VITAL_SIGN_COLUMN_COMMENTS = {
    "patient_event_id": "Stable vital event identifier.",
    "fact_row_id": "Storage-row identifier.",
    "subject_key": "Always-populated subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved person identifier.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference.",
    "event_datetime": "Measurement timestamp.",
    "event_end_datetime": "Measurement end timestamp.",
    "source_coding_system": "Source coding system.",
    "source_code": "Source vital code.",
    "source_display": "Source vital display.",
    "vital_code": "Source and mapped vital CodeableConcept.",
    "value_number": "Numeric vital value.",
    "value_text": "Verbatim result text.",
    "unit_source_value": "Source result unit.",
    "unit_concept_id": "Mapped unit concept identifier.",
    "reference_range_low": "Reference-range lower bound.",
    "reference_range_high": "Reference-range upper bound.",
    "method_code": "Source measurement method code.",
    "method_display": "Source measurement method display.",
    "body_site_code": "Body-site code when supplied.",
    "body_site_display": "Body-site display when supplied.",
    "interpretation_code": "Source interpretation code.",
    "interpretation_display": "Source interpretation display.",
    "result_status_code": "Source result status code.",
    "result_status_display": "Source result status display.",
    "performer_practitioner_id": "Performing practitioner reference.",
    "source_form_id": "Source form event for promoted measurements.",
    "promotion_rule_id": "Governed promotion-rule identifier.",
    "record_status": "Normalized lifecycle.",
    "record_status_effective_from": "Lifecycle start.",
    "record_status_effective_to": "Lifecycle end when retained inactive.",
    "confidentiality_code": "Security classification when supplied.",
    "vip_ind": "VIP indicator when supplied.",
    "withheld_identity_ind": "Withheld identity indicator.",
    "source_feed": "Registered source route.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Native source update timestamp.",
    "loaded_at": "Bronze load timestamp.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.vital_sign"),
    comment=(
        "One native or form-promoted vital measurement. Gold QC twin of the silver product: 8 "
        "columns are repaired or nulled, 1 rule(s) drop rows, 4 check(s) are advisory. Each "
        "rule states its reason in the pipeline notebook, and Lakeflow expectation metrics "
        "report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_VITAL_SIGN_MANDATORY_RULES)
@_expect_all(CLINICAL_VITAL_SIGN_ADVISORY_RULES)
def gold_clinical_vital_sign():
    """Quality-controlled twin of journey_clinical.vital_sign."""
    df = _qc("clinical_vital_sign", CLINICAL_VITAL_SIGN_SELECT)
    return _with_comments(df, CLINICAL_VITAL_SIGN_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.waiting_list_entry ====

CLINICAL_WAITING_LIST_ENTRY_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 1 of 233,491,987 rows (4.28e-07%) when profiled on 2026-08-24.
    #
    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 2 of 233,491,987 rows (8.57e-07%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE CASE WHEN CAST(`event_datetime` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`event_datetime` AS DATE)) < 9999 THEN NULL ELSE `event_datetime` END END AS `event_datetime`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 8 of 233,491,987 rows (3.43e-06%) when profiled on 2026-08-24.
    #
    # A date before 1901 that is not one of the known placeholders. Nothing in this estate
    # predates the twentieth century, so these are mistyped or mis-scaled rather than early.
    # Hit 1 of 233,491,987 rows (4.28e-07%) when profiled on 2026-08-24.
    #
    # event_end_datetime cannot precede event_datetime. The start is the better-attested of
    # the two, so the end is what goes and the row keeps its event_datetime. Hit 46,059 of
    # 233,491,987 rows (0.0197%) when profiled on 2026-08-24.
    #
    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 2 of 233,491,987 rows (8.57e-07%) when profiled on 2026-08-24.
    "CASE WHEN `event_datetime` IS NOT NULL AND `event_end_datetime` IS NOT NULL AND `event_datetime` > `event_end_datetime` OR CAST(`event_end_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE CASE WHEN CAST(`event_end_datetime` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`event_end_datetime` AS DATE)) < 9999 OR CAST(`event_end_datetime` AS DATE) < DATE'1901-01-01' AND CAST(`event_end_datetime` AS DATE) NOT IN (DATE'1800-01-01', DATE'1899-12-30', DATE'1900-01-01') THEN NULL ELSE `event_end_datetime` END END AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 92,211,378 of 233,491,987 rows
    # (39.5%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`source_code` AS STRING))) = '0' THEN NULL ELSE `source_code` END AS `source_code`",
    "`source_display` AS `source_display`",
    "`pm_wait_list_id` AS `pm_wait_list_id`",
    "`row_source` AS `row_source`",
    "`source_version_id` AS `source_version_id`",
    "`hist_action` AS `hist_action`",
    "`version_datetime` AS `version_datetime`",
    "`is_current` AS `is_current`",
    "`updt_cnt` AS `updt_cnt`",
    "`sch_event_id` AS `sch_event_id`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 27,898,364 of 233,491,987 rows
    # (11.9%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`status_code` AS STRING))) = '0' THEN NULL ELSE `status_code` END AS `status_code`",
    "`status_display` AS `status_display`",
    "`sub_status_display` AS `sub_status_display`",
    "`active_status_display` AS `active_status_display`",
    "`urgency_display` AS `urgency_display`",
    "`stand_by_display` AS `stand_by_display`",
    "`admit_category_display` AS `admit_category_display`",
    "`admit_booking_display` AS `admit_booking_display`",
    "`admit_type_display` AS `admit_type_display`",
    "`admit_offer_outcome_display` AS `admit_offer_outcome_display`",
    "`management_display` AS `management_display`",
    "`attendance_display` AS `attendance_display`",
    "`reason_for_change_display` AS `reason_for_change_display`",
    "`reason_for_removal_display` AS `reason_for_removal_display`",
    "`anesthetic_display` AS `anesthetic_display`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 92,211,378 of 233,491,987 rows
    # (39.5%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`planned_procedure_code` AS STRING))) = '0' THEN NULL ELSE `planned_procedure_code` END AS `planned_procedure_code`",
    "`planned_procedure_display` AS `planned_procedure_display`",
    "`referral_source_display` AS `referral_source_display`",
    "`referral_type_display` AS `referral_type_display`",
    "`service_type_requested_display` AS `service_type_requested_display`",
    "`from_ed_ind` AS `from_ed_ind`",
    "`suspended_days` AS `suspended_days`",

    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 12 of 233,491,987 rows (5.14e-06%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`recommend_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `recommend_datetime` END AS `recommend_datetime`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 1 of 233,491,987 rows (4.28e-07%) when profiled on 2026-08-24.
    #
    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 58 of 233,491,987 rows (2.48e-05%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`referral_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE CASE WHEN CAST(`referral_datetime` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`referral_datetime` AS DATE)) < 9999 THEN NULL ELSE `referral_datetime` END END AS `referral_datetime`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 1 of 233,491,987 rows (4.28e-07%) when profiled on 2026-08-24.
    #
    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 34 of 233,491,987 rows (1.46e-05%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`original_request_received_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE CASE WHEN CAST(`original_request_received_datetime` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`original_request_received_datetime` AS DATE)) < 9999 THEN NULL ELSE `original_request_received_datetime` END END AS `original_request_received_datetime`",

    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 12 of 233,491,987 rows (5.14e-06%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`admit_decision_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `admit_decision_datetime` END AS `admit_decision_datetime`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 4 of 233,491,987 rows (1.71e-06%) when profiled on 2026-08-24.
    #
    # A date before 1901 that is not one of the known placeholders. Nothing in this estate
    # predates the twentieth century, so these are mistyped or mis-scaled rather than early.
    # Hit 16,379 of 233,491,987 rows (0.00701%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`admit_guaranteed_datetime` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`admit_guaranteed_datetime` AS DATE)) < 9999 OR CAST(`admit_guaranteed_datetime` AS DATE) < DATE'1901-01-01' AND CAST(`admit_guaranteed_datetime` AS DATE) NOT IN (DATE'1800-01-01', DATE'1899-12-30', DATE'1900-01-01') THEN NULL ELSE `admit_guaranteed_datetime` END AS `admit_guaranteed_datetime`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 26 of 233,491,987 rows (1.11e-05%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`provisional_admit_datetime` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`provisional_admit_datetime` AS DATE)) < 9999 THEN NULL ELSE `provisional_admit_datetime` END AS `provisional_admit_datetime`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 20 of 233,491,987 rows (8.57e-06%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`previous_provisional_admit_datetime` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`previous_provisional_admit_datetime` AS DATE)) < 9999 THEN NULL ELSE `previous_provisional_admit_datetime` END AS `previous_provisional_admit_datetime`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 4 of 233,491,987 rows (1.71e-06%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`adjusted_waiting_start_datetime` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`adjusted_waiting_start_datetime` AS DATE)) < 9999 THEN NULL ELSE `adjusted_waiting_start_datetime` END AS `adjusted_waiting_start_datetime`",
    "`scheduled_datetime` AS `scheduled_datetime`",

    # 2100-12-31 appears here on a field that is not an end date, so it cannot be the 'still
    # open' marker it is elsewhere and is a placeholder instead. Hit 1 of 233,491,987 rows
    # (4.28e-07%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`requested_datetime` AS DATE) = DATE'2100-12-31' THEN NULL ELSE `requested_datetime` END AS `requested_datetime`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 138 of 233,491,987 rows (5.91e-05%) when profiled on 2026-08-24.
    #
    # A date before 1901 that is not one of the known placeholders. Nothing in this estate
    # predates the twentieth century, so these are mistyped or mis-scaled rather than early.
    # Hit 5 of 233,491,987 rows (2.14e-06%) when profiled on 2026-08-24.
    #
    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 3 of 233,491,987 rows (1.28e-06%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`removal_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE CASE WHEN CAST(`removal_datetime` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`removal_datetime` AS DATE)) < 9999 OR CAST(`removal_datetime` AS DATE) < DATE'1901-01-01' AND CAST(`removal_datetime` AS DATE) NOT IN (DATE'1800-01-01', DATE'1899-12-30', DATE'1900-01-01') THEN NULL ELSE `removal_datetime` END END AS `removal_datetime`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 6 of 233,491,987 rows (2.57e-06%) when profiled on 2026-08-24.
    #
    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 15,753 of 233,491,987 rows (0.00675%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`last_dna_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE CASE WHEN CAST(`last_dna_datetime` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`last_dna_datetime` AS DATE)) < 9999 THEN NULL ELSE `last_dna_datetime` END END AS `last_dna_datetime`",
    "`status_datetime` AS `status_datetime`",

    # 2100-12-31 is the far-future marker the source writes to mean 'no end yet'. The
    # absence is what the row means, and NULL states it without putting a fictional date
    # into a range comparison. Hit 1,224,955 of 233,491,987 rows (0.525%) when profiled on
    # 2026-08-24.
    "CASE WHEN CAST(`status_end_datetime` AS DATE) = DATE'2100-12-31' THEN NULL ELSE `status_end_datetime` END AS `status_end_datetime`",
    "`location_id` AS `location_id`",
    "`location_display` AS `location_display`",
    "`facility_display` AS `facility_display`",
    "`record_status` AS `record_status`",
    "`record_status_effective_from` AS `record_status_effective_from`",

    # 2100-12-31 is the far-future marker the source writes to mean 'no end yet'. The
    # absence is what the row means, and NULL states it without putting a fictional date
    # into a range comparison. Hit 145,217,955 of 233,491,987 rows (62.2%) when profiled on
    # 2026-08-24.
    #
    # record_status_effective_to cannot precede record_status_effective_from. The start is
    # the better-attested of the two, so the end is what goes and the row keeps its
    # record_status_effective_from. Hit 290 of 233,491,987 rows (0.000124%) when profiled on
    # 2026-08-24.
    "CASE WHEN `record_status_effective_from` IS NOT NULL AND `record_status_effective_to` IS NOT NULL AND `record_status_effective_from` > `record_status_effective_to` THEN NULL ELSE CASE WHEN CAST(`record_status_effective_to` AS DATE) = DATE'2100-12-31' THEN NULL ELSE `record_status_effective_to` END END AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`fact_category` AS `fact_category`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
]

CLINICAL_WAITING_LIST_ENTRY_MANDATORY_RULES = {
    # The research surface. identity_status = 'resolved' keeps the 233,491,987 rows of
    # 233,491,987 that are current and attributable. Superseded versions and rows whose
    # identity was never resolved are not research data, and a consumer who wants them has
    # silver.
    "research_surface": "(identity_status = 'resolved')",
}

CLINICAL_WAITING_LIST_ENTRY_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 0 of 233,491,987 at the profile.
    "gold.clinical.waiting_list_entry.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 952 of 233,491,987 rows (0.000408%) when profiled on 2026-08-24.
    "gold.clinical.waiting_list_entry.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 6,193 of 233,491,987 rows (0.00265%) when profiled on
    # 2026-08-24.
    "gold.clinical.waiting_list_entry.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",
}

CLINICAL_WAITING_LIST_ENTRY_COLUMN_COMMENTS = {
    "patient_event_id":
        "Stable entry-grain identifier shared by all physical versions of one waiting-list entry.",
    "fact_row_id":
        "Stable version-grain storage-row identifier; uniqueness belongs here because patient_event_id is shared across versions.",
    "subject_key": "Always-populated peppered subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier when available.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Minted Millennium encounter identifier when supplied.",
    "event_datetime":
        "Waiting-start timestamp. Gold QC transform rules: gold.clinical.waiting_list_entry.event_datetime.beyond_2100_below_9999_owner.",
    "event_end_datetime":
        "Waiting-end timestamp. Gold QC transform rules: gold.clinical.waiting_list_entry.event_end_datetime.beyond_2100_below_9999_owner, gold.clinical.waiting_list_entry.event_end_datetime.pre1901_other_owner.",
    "source_coding_system": "Verbatim Millennium planned-procedure coding system.",
    "source_code": "Verbatim planned-procedure code.",
    "source_display": "Verbatim planned-procedure display.",
    "pm_wait_list_id": "Native waiting-list entry identifier.",
    "row_source": "Physical bronze row source.",
    "source_version_id": "Native source version identifier; CURRENT rows carry -1.",
    "hist_action": "Source history action when the row is historical.",
    "version_datetime":
        "Descriptive fallback version timestamp only; it defines no effective interval.",
    "is_current": "Whether this is the current physical version.",
    "updt_cnt": "Native source update counter.",
    "sch_event_id": "Raw scheduling event linkage evidence.",
    "status_code": "Source waiting-list status code.",
    "status_display": "Source waiting-list status display.",
    "sub_status_display": "Source waiting-list sub-status display.",
    "active_status_display": "Source active-status display.",
    "urgency_display": "Source urgency display.",
    "stand_by_display": "Source stand-by display.",
    "admit_category_display": "Source admission-category display.",
    "admit_booking_display": "Source admission-booking display.",
    "admit_type_display": "Source admission-type display.",
    "admit_offer_outcome_display": "Source admission-offer outcome display.",
    "management_display": "Source management display.",
    "attendance_display": "Source attendance display.",
    "reason_for_change_display": "Source reason-for-change display.",
    "reason_for_removal_display": "Source reason-for-removal display.",
    "anesthetic_display": "Source anesthetic display.",
    "planned_procedure_code": "Source planned-procedure code.",
    "planned_procedure_display": "Source planned-procedure display.",
    "referral_source_display": "Source referral-source display.",
    "referral_type_display": "Source referral-type display.",
    "service_type_requested_display": "Source requested-service-type display.",
    "from_ed_ind": "Raw source indicator for origin in the emergency department.",
    "suspended_days": "Source count of suspended days.",
    "recommend_datetime": "Source recommendation timestamp.",
    "referral_datetime":
        "Source referral timestamp. Gold QC transform rules: gold.clinical.waiting_list_entry.referral_datetime.beyond_2100_below_9999_owner.",
    "original_request_received_datetime":
        "Original request-received timestamp. Gold QC transform rules: gold.clinical.waiting_list_entry.original_request_received_datetime.beyond_2100_below_9999_owner.",
    "admit_decision_datetime": "Source decision-to-admit timestamp.",
    "admit_guaranteed_datetime":
        "Source guaranteed-admission timestamp. Gold QC transform rules: gold.clinical.waiting_list_entry.admit_guaranteed_datetime.beyond_2100_below_9999_owner, gold.clinical.waiting_list_entry.admit_guaranteed_datetime.pre1901_other_owner.",
    "provisional_admit_datetime":
        "Source provisional-admission timestamp. Gold QC transform rules: gold.clinical.waiting_list_entry.provisional_admit_datetime.beyond_2100_below_9999_owner.",
    "previous_provisional_admit_datetime":
        "Previous provisional-admission timestamp. Gold QC transform rules: gold.clinical.waiting_list_entry.previous_provisional_admit_datetime.beyond_2100_below_9999_owner.",
    "adjusted_waiting_start_datetime":
        "Adjusted waiting-start timestamp. Gold QC transform rules: gold.clinical.waiting_list_entry.adjusted_waiting_start_datetime.beyond_2100_below_9999_owner.",
    "scheduled_datetime": "Source scheduled timestamp.",
    "requested_datetime":
        "Source requested timestamp. Gold QC transform rules: gold.clinical.waiting_list_entry.requested_datetime.d2100_12_31_owner.",
    "removal_datetime":
        "Source removal timestamp. Gold QC transform rules: gold.clinical.waiting_list_entry.removal_datetime.beyond_2100_below_9999_owner, gold.clinical.waiting_list_entry.removal_datetime.pre1901_other_owner.",
    "last_dna_datetime":
        "Source last did-not-attend timestamp. Gold QC transform rules: gold.clinical.waiting_list_entry.last_dna_datetime.beyond_2100_below_9999_owner.",
    "status_datetime": "Source status timestamp.",
    "status_end_datetime":
        "Source status-end timestamp. Gold QC transform rules: gold.clinical.waiting_list_entry.status_end_datetime.open_sentinel.",
    "location_id": "Minted nurse-unit location identifier.",
    "location_display": "Source nurse-unit display.",
    "facility_display": "Source facility display.",
    "record_status": "Normalized source lifecycle status.",
    "record_status_effective_from": "Source row effective start.",
    "record_status_effective_to":
        "Source lifecycle end or source-absence detection timestamp. Gold QC transform rules: gold.clinical.waiting_list_entry.record_status_effective_to.open_sentinel.",
    "confidentiality_code": "Source confidentiality code when supplied.",
    "vip_ind": "Source VIP indicator when supplied.",
    "withheld_identity_ind": "Withheld-identity indicator when supplied.",
    "fact_category":
        "Whether this fact is clinical or administrative in the v2 plane merge.",
    "source_feed": "Registered feed owning the fact.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp": "Latest native source update timestamp.",
    "loaded_at": "Latest bronze load timestamp.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_clinical.waiting_list_entry"),
    comment=(
        "Versioned Millennium waiting-list entry fact activated as a recorded contract change "
        "from a zero-row stub with no consumer break. Gold QC twin of the silver product: 19 "
        "columns are repaired or nulled, 1 rule(s) drop rows, 3 check(s) are advisory. Each "
        "rule states its reason in the pipeline notebook, and Lakeflow expectation metrics "
        "report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_WAITING_LIST_ENTRY_MANDATORY_RULES)
@_expect_all(CLINICAL_WAITING_LIST_ENTRY_ADVISORY_RULES)
def gold_clinical_waiting_list_entry():
    """Quality-controlled twin of journey_clinical.waiting_list_entry."""
    # 63 rows point at a person_id the spine does not have. The pointer is nulled so it
    # cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    df = _qc(
        "clinical_waiting_list_entry",
        CLINICAL_WAITING_LIST_ENTRY_SELECT,
        fk_columns=["person_id"],
        date_flags=["event_after_death_30d", "event_before_birth"],
    )
    return _with_comments(df, CLINICAL_WAITING_LIST_ENTRY_COLUMN_COMMENTS)

In [0]:
# ==== journey_clinical.waiting_list_snapshot ====

CLINICAL_WAITING_LIST_SNAPSHOT_SELECT = [
    "`waiting_list_snapshot_id` AS `waiting_list_snapshot_id`",
    "`snapshot_date` AS `snapshot_date`",
    "`snapshot_cutoff_ts` AS `snapshot_cutoff_ts`",
    "`pm_wait_list_id` AS `pm_wait_list_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",
    "`sch_event_id` AS `sch_event_id`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 4,303,056 of 4,771,446 rows
    # (90.2%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`status_code` AS STRING))) = '0' THEN NULL ELSE `status_code` END AS `status_code`",
    "`status_display` AS `status_display`",
    "`sub_status_display` AS `sub_status_display`",
    "`active_status_display` AS `active_status_display`",
    "`urgency_display` AS `urgency_display`",
    "`stand_by_display` AS `stand_by_display`",
    "`admit_category_display` AS `admit_category_display`",
    "`admit_booking_display` AS `admit_booking_display`",
    "`admit_type_display` AS `admit_type_display`",
    "`admit_offer_outcome_display` AS `admit_offer_outcome_display`",
    "`management_display` AS `management_display`",
    "`attendance_display` AS `attendance_display`",
    "`reason_for_change_display` AS `reason_for_change_display`",
    "`reason_for_removal_display` AS `reason_for_removal_display`",
    "`anesthetic_display` AS `anesthetic_display`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 4,190,905 of 4,771,446 rows
    # (87.8%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`planned_procedure_code` AS STRING))) = '0' THEN NULL ELSE `planned_procedure_code` END AS `planned_procedure_code`",
    "`planned_procedure_display` AS `planned_procedure_display`",
    "`referral_source_display` AS `referral_source_display`",
    "`referral_type_display` AS `referral_type_display`",
    "`service_type_requested_display` AS `service_type_requested_display`",
    "`from_ed_ind` AS `from_ed_ind`",
    "`suspended_days` AS `suspended_days`",
    "`recommend_datetime` AS `recommend_datetime`",

    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 36 of 4,771,446 rows (0.000754%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`referral_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `referral_datetime` END AS `referral_datetime`",
    "`original_request_received_datetime` AS `original_request_received_datetime`",
    "`admit_decision_datetime` AS `admit_decision_datetime`",

    # A date before 1901 that is not one of the known placeholders. Nothing in this estate
    # predates the twentieth century, so these are mistyped or mis-scaled rather than early.
    # Hit 24 of 4,771,446 rows (0.000503%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`admit_guaranteed_datetime` AS DATE) < DATE'1901-01-01' AND CAST(`admit_guaranteed_datetime` AS DATE) NOT IN (DATE'1800-01-01', DATE'1899-12-30', DATE'1900-01-01') THEN NULL ELSE `admit_guaranteed_datetime` END AS `admit_guaranteed_datetime`",
    "`provisional_admit_datetime` AS `provisional_admit_datetime`",
    "`previous_provisional_admit_datetime` AS `previous_provisional_admit_datetime`",
    "`waiting_start_datetime` AS `waiting_start_datetime`",

    # waiting_end_datetime cannot precede waiting_start_datetime. The start is the better-
    # attested of the two, so the end is what goes and the row keeps its
    # waiting_start_datetime. Hit 150 of 4,771,446 rows (0.00314%) when profiled on
    # 2026-08-24.
    "CASE WHEN `waiting_start_datetime` IS NOT NULL AND `waiting_end_datetime` IS NOT NULL AND `waiting_start_datetime` > `waiting_end_datetime` THEN NULL ELSE `waiting_end_datetime` END AS `waiting_end_datetime`",
    "`adjusted_waiting_start_datetime` AS `adjusted_waiting_start_datetime`",
    "`scheduled_datetime` AS `scheduled_datetime`",
    "`requested_datetime` AS `requested_datetime`",
    "`removal_datetime` AS `removal_datetime`",

    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 130 of 4,771,446 rows (0.00272%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`last_dna_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `last_dna_datetime` END AS `last_dna_datetime`",
    "`status_datetime` AS `status_datetime`",

    # 2100-12-31 is the far-future marker the source writes to mean 'no end yet'. The
    # absence is what the row means, and NULL states it without putting a fictional date
    # into a range comparison. Hit 12 of 4,771,446 rows (0.000251%) when profiled on
    # 2026-08-24.
    "CASE WHEN CAST(`status_end_datetime` AS DATE) = DATE'2100-12-31' THEN NULL ELSE `status_end_datetime` END AS `status_end_datetime`",
    "`location_id` AS `location_id`",
    "`location_display` AS `location_display`",
    "`facility_display` AS `facility_display`",
    "`fact_category` AS `fact_category`",
    "`source_feed` AS `source_feed`",
    "`loaded_at` AS `loaded_at`",
]

CLINICAL_WAITING_LIST_SNAPSHOT_MANDATORY_RULES = {
    # The research surface. identity_status = 'resolved' keeps the 4,771,446 rows of
    # 4,771,446 that are current and attributable. Superseded versions and rows whose
    # identity was never resolved are not research data, and a consumer who wants them has
    # silver.
    "research_surface": "(identity_status = 'resolved')",
}

CLINICAL_WAITING_LIST_SNAPSHOT_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 0 of 4,771,446 at the profile.
    "gold.clinical.waiting_list_snapshot.identity_status.default_view_resolved":
        "identity_status = 'resolved'",
}

CLINICAL_WAITING_LIST_SNAPSHOT_COLUMN_COMMENTS = {
    "waiting_list_snapshot_id": "Stable census-row identifier.",
    "snapshot_date": "Census snapshot date.",
    "snapshot_cutoff_ts": "Immutable census cutoff timestamp.",
    "pm_wait_list_id": "Native waiting-list entry identifier.",
    "subject_key": "Always-populated peppered subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier when available.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Minted Millennium encounter identifier when supplied.",
    "sch_event_id": "Raw scheduling event linkage evidence.",
    "status_code": "Source waiting-list status code.",
    "status_display": "Source waiting-list status display.",
    "sub_status_display": "Source waiting-list sub-status display.",
    "active_status_display": "Source active-status display.",
    "urgency_display": "Source urgency display.",
    "stand_by_display": "Source stand-by display.",
    "admit_category_display": "Source admission-category display.",
    "admit_booking_display": "Source admission-booking display.",
    "admit_type_display": "Source admission-type display.",
    "admit_offer_outcome_display": "Source admission-offer outcome display.",
    "management_display": "Source management display.",
    "attendance_display": "Source attendance display.",
    "reason_for_change_display": "Source reason-for-change display.",
    "reason_for_removal_display": "Source reason-for-removal display.",
    "anesthetic_display": "Source anesthetic display.",
    "planned_procedure_code": "Source planned-procedure code.",
    "planned_procedure_display": "Source planned-procedure display.",
    "referral_source_display": "Source referral-source display.",
    "referral_type_display": "Source referral-type display.",
    "service_type_requested_display": "Source requested-service-type display.",
    "from_ed_ind": "Raw source indicator for origin in the emergency department.",
    "suspended_days": "Source count of suspended days.",
    "recommend_datetime": "Source recommendation timestamp.",
    "referral_datetime": "Source referral timestamp.",
    "original_request_received_datetime": "Original request-received timestamp.",
    "admit_decision_datetime": "Source decision-to-admit timestamp.",
    "admit_guaranteed_datetime":
        "Source guaranteed-admission timestamp. Gold QC transform rules: gold.clinical.waiting_list_snapshot.admit_guaranteed_datetime.pre1901_other_owner.",
    "provisional_admit_datetime": "Source provisional-admission timestamp.",
    "previous_provisional_admit_datetime": "Previous provisional-admission timestamp.",
    "waiting_start_datetime": "Source waiting-start timestamp.",
    "waiting_end_datetime": "Source waiting-end timestamp.",
    "adjusted_waiting_start_datetime": "Adjusted waiting-start timestamp.",
    "scheduled_datetime": "Source scheduled timestamp.",
    "requested_datetime": "Source requested timestamp.",
    "removal_datetime": "Source removal timestamp.",
    "last_dna_datetime": "Source last did-not-attend timestamp.",
    "status_datetime": "Source status timestamp.",
    "status_end_datetime":
        "Source status-end timestamp. Gold QC transform rules: gold.clinical.waiting_list_snapshot.status_end_datetime.open_sentinel.",
    "location_id": "Minted nurse-unit location identifier.",
    "location_display": "Source nurse-unit display.",
    "facility_display": "Source facility display.",
    "fact_category":
        "Whether this fact is clinical or administrative in the v2 plane merge.",
    "source_feed": "Registered feed owning the census row.",
    "loaded_at": "Census cutoff timestamp used as the immutable load timestamp.",
}

@dp.materialized_view(
    name=_n("gold_clinical.waiting_list_snapshot"),
    comment=(
        "Immutable Millennium waiting-list census activated as a recorded contract change "
        "from a zero-row stub with no consumer break. Gold QC twin of the silver product: 8 "
        "columns are repaired or nulled, 1 rule(s) drop rows, 1 check(s) are advisory. Each "
        "rule states its reason in the pipeline notebook, and Lakeflow expectation metrics "
        "report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(CLINICAL_WAITING_LIST_SNAPSHOT_MANDATORY_RULES)
@_expect_all(CLINICAL_WAITING_LIST_SNAPSHOT_ADVISORY_RULES)
def gold_clinical_waiting_list_snapshot():
    """Quality-controlled twin of journey_clinical.waiting_list_snapshot."""
    # 12 rows point at a person_id the spine does not have. The pointer is nulled so it
    # cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    df = _qc(
        "clinical_waiting_list_snapshot",
        CLINICAL_WAITING_LIST_SNAPSHOT_SELECT,
        fk_columns=["person_id"],
    )
    return _with_comments(df, CLINICAL_WAITING_LIST_SNAPSHOT_COLUMN_COMMENTS)

In [0]:
# ======== Events ========
#
# The dated activity spine: one row per thing that happened to a person.
# 1 products follow.

In [0]:
# ==== journey_events.patient_event ====

EVENTS_PATIENT_EVENT_SELECT = [
    "`patient_event_row_id` AS `patient_event_row_id`",
    "`patient_event_id` AS `patient_event_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",
    "`person_id` AS `person_id`",
    "`identity_status` AS `identity_status`",
    "`encounter_id` AS `encounter_id`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 600 of 14,356,906,440 rows (4.18e-06%) when profiled on 2026-08-24.
    #
    # 1900-01-01 is a placeholder low date rather than a date in 1900; on a non-birth field
    # it carries no more meaning than it does on a birth one. Hit 100 of 14,356,906,440 rows
    # (6.97e-07%) when profiled on 2026-08-24.
    #
    # 1899-12-30 is the zero point of the OLE/Excel date scale, so it is what a spreadsheet
    # or a COM layer writes when the date was left blank. It is not a date anyone recorded.
    # Hit 500 of 14,356,906,440 rows (3.48e-06%) when profiled on 2026-08-24.
    #
    # A date before 1901 that is not one of the known placeholders. Nothing in this estate
    # predates the twentieth century, so these are mistyped or mis-scaled rather than early.
    # Hit 600 of 14,356,906,440 rows (4.18e-06%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`event_datetime` AS DATE) = DATE'1899-12-30' OR CAST(`event_datetime` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`event_datetime` AS DATE)) < 9999 OR CAST(`event_datetime` AS DATE) = DATE'1900-01-01' OR CAST(`event_datetime` AS DATE) < DATE'1901-01-01' AND CAST(`event_datetime` AS DATE) NOT IN (DATE'1800-01-01', DATE'1899-12-30', DATE'1900-01-01') THEN NULL ELSE `event_datetime` END AS `event_datetime`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 600 of 14,356,906,440 rows (4.18e-06%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`event_end_datetime` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`event_end_datetime` AS DATE)) < 9999 THEN NULL ELSE `event_end_datetime` END AS `event_end_datetime`",
    "`event_type` AS `event_type`",
    "`fact_category` AS `fact_category`",
    "`fact_table` AS `fact_table`",
    "`fact_row_id` AS `fact_row_id`",
    "`source_system` AS `source_system`",
    "`source_object` AS `source_object`",
    "`source_row_key` AS `source_row_key`",
    "`source_coding_system` AS `source_coding_system`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 18,890,703 of 14,356,906,440
    # rows (0.132%) when profiled on 2026-08-24.
    #
    # 'UNKNOWN' is a placeholder the source writes when the value was not recorded; it is
    # not a code, so it is nulled rather than passed on as one. Hit 6,827 of 14,356,906,440
    # rows (4.76e-05%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`source_code` AS STRING))) = '0' OR UPPER(TRIM(CAST(`source_code` AS STRING))) = 'UNKNOWN' THEN NULL ELSE `source_code` END AS `source_code`",
    "`source_display` AS `source_display`",
    "`record_status` AS `record_status`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`mapped_coding_system` AS `mapped_coding_system`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 120,392 of 14,356,906,440 rows
    # (0.000839%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`mapped_code` AS STRING))) = '0' THEN NULL ELSE `mapped_code` END AS `mapped_code`",
    "`mapped_display` AS `mapped_display`",
    "`target_domain` AS `target_domain`",
    "`map_source` AS `map_source`",
    "`map_version` AS `map_version`",
    "`load_batch_id` AS `load_batch_id`",
    "`loaded_at` AS `loaded_at`",
]

EVENTS_PATIENT_EVENT_MANDATORY_RULES = {
    # The research surface. identity_status = 'resolved' keeps the 13,235,874,738 rows of
    # 14,356,906,440 that are current and attributable. Superseded versions and rows whose
    # identity was never resolved are not research data, and a consumer who wants them has
    # silver.
    "research_surface": "(identity_status = 'resolved')",
}

EVENTS_PATIENT_EVENT_ADVISORY_RULES = {
    # Counted rather than nulled because the event index carries every booked appointment
    # forward, so a future stamp here is the booking and not an error. Seen on 656,600 of
    # 14,356,906,440 rows (0.00457%) when profiled on 2026-08-24.
    "gold.events.patient_event.event_datetime.future_owner":
        "NOT COALESCE((CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",

    # Counted rather than nulled because the event index carries every booked appointment
    # forward, so a future stamp here is the booking and not an error. Seen on 490,000 of
    # 14,356,906,440 rows (0.00341%) when profiled on 2026-08-24.
    "gold.events.patient_event.event_end_datetime.future_owner":
        "NOT COALESCE((CAST(`event_end_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",

    # Kept as a regression guard on the accepted set ('resolved', 'provisional',
    # 'unresolved'): the research-surface filter already removes every row that fails it, so
    # this expectation should read zero forever and is worth watching for the day it does
    # not. Measured at 203 of 14,356,906,440 rows (1.41e-06%) when profiled on 2026-08-24.
    "gold.events.patient_event.identity_status.accepted_values":
        "NOT COALESCE((`identity_status` IS NOT NULL AND `identity_status` NOT IN ('resolved', 'provisional', 'unresolved')), FALSE)",

    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 1,121,031,702 of 14,356,906,440 at the profile.
    "gold.events.patient_event.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 13,288,735 of 14,356,906,440 rows (0.0926%) when profiled on
    # 2026-08-24.
    "gold.events.patient_event.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 58,507,245 of 14,356,906,440 rows (0.408%) when profiled on
    # 2026-08-24.
    "gold.events.patient_event.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",

    # Left as a warning because it fires on 507,133,196 of 14,356,906,440 rows (3.53%) when
    # profiled on 2026-08-24 -- at that rate the rule's assumption about what event_datetime
    # and event_end_datetime mean is the thing in doubt, not the data. The inverted gaps are
    # mostly minutes, which reads as two clocks rather than two events in the wrong order.
    "gold.events.patient_event.table.ordering_violation_event_datetime_event_end_datetime":
        "NOT COALESCE((`event_datetime` IS NOT NULL AND `event_end_datetime` IS NOT NULL AND `event_datetime` > `event_end_datetime`), FALSE)",
}

EVENTS_PATIENT_EVENT_COLUMN_COMMENTS = {
    "patient_event_row_id":
        "Deterministic row key (the mapping id for mapped rows; a namespaced unmapped id otherwise).",
    "patient_event_id":
        "Immutable fact-grain event identifier shared across a mapped event's N rows.",
    "subject_key": "Peppered subject join key.",
    "subject_id_system": "Identifier system behind subject_key.",
    "person_id": "Resolved Millennium person id when available.",
    "identity_status": "Resolution state.",
    "encounter_id": "Encounter reference when the source supplies one.",
    "event_datetime":
        "Clinical or administrative event time. Gold QC transform rules: gold.events.patient_event.event_datetime.beyond_2100_below_9999_owner, gold.events.patient_event.event_datetime.d1900_01_01_owner, gold.events.patient_event.event_datetime.ole_zero_date, gold.events.patient_event.event_datetime.pre1901_other_owner.",
    "event_end_datetime":
        "Event end when the source supplies one. Gold QC transform rules: gold.events.patient_event.event_end_datetime.beyond_2100_below_9999_owner.",
    "event_type": "Fact kind (condition",
    "fact_category": "Clinical versus administrative fact.",
    "fact_table": "Typed fact table holding the event's values.",
    "fact_row_id": "Row key inside fact_table.",
    "source_system": "Plain-language source system (Millennium",
    "source_object": "Native source entity in plain language",
    "source_row_key": "Native source record key.",
    "source_coding_system": "Source coding system.",
    "source_code": "Source code as recorded.",
    "source_display": "Source display as recorded.",
    "record_status": "Normalized lifecycle status carried as data.",
    "confidentiality_code": "Source confidentiality label.",
    "vip_ind": "Source VIP indicator carried as data.",
    "withheld_identity_ind": "Source withheld-identity indicator.",
    "mapped_coding_system":
        "Standard coding system for this mapping row; null on the unmapped row.",
    "mapped_code": "Standard code.",
    "mapped_display": "Standard display or definition.",
    "target_domain": "Downstream routing domain for this mapping.",
    "map_source": "Governed mapping product that produced the row.",
    "map_version": "Mapping content version.",
    "load_batch_id": "Deterministic batch token from bronze load time.",
    "loaded_at": "Bronze load or update timestamp; never pipeline wall-clock time.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_events.patient_event"),
    comment=(
        "One row per admitted event x standard mapping; unmapped admitted events carry "
        "exactly one row with a null mapping block. Values live in the typed fact tables. "
        "Gold QC twin of the silver product: 6 columns are repaired or nulled, 1 rule(s) drop "
        "rows, 7 check(s) are advisory. Each rule states its reason in the pipeline notebook, "
        "and Lakeflow expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(EVENTS_PATIENT_EVENT_MANDATORY_RULES)
@_expect_all(EVENTS_PATIENT_EVENT_ADVISORY_RULES)
def gold_events_patient_event():
    """Quality-controlled twin of journey_events.patient_event."""
    # 6,302,313 rows point at a encounter_id the spine does not have. The pointer is nulled
    # so it cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    # 10,710 rows point at a person_id the spine does not have. The pointer is nulled so it
    # cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    df = _qc(
        "events_patient_event",
        EVENTS_PATIENT_EVENT_SELECT,
        fk_columns=["encounter_id", "person_id"],
        date_flags=["event_after_death_30d", "event_before_birth"],
    )
    return _with_comments(df, EVENTS_PATIENT_EVENT_COLUMN_COMMENTS)

In [0]:
# ======== Text ========
#
# Documents and the narrative extracted from them.
# 1 products follow.

In [0]:
# ==== journey_text.document ====

TEXT_DOCUMENT_SELECT = [
    "`patient_event_id` AS `patient_event_id`",
    "`fact_row_id` AS `fact_row_id`",
    "`subject_key` AS `subject_key`",
    "`subject_id_system` AS `subject_id_system`",

    # 288 rows point at a person_id the spine does not have. The pointer is nulled so it
    # cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    "CASE WHEN NOT `person_id_resolved` THEN NULL ELSE `person_id` END AS `person_id`",
    "`identity_status` AS `identity_status`",

    # 39,240 rows point at a encounter_id the spine does not have. The pointer is nulled so
    # it cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    "CASE WHEN NOT `encounter_id_resolved` THEN NULL ELSE `encounter_id` END AS `encounter_id`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 5 of 656,894,348 rows (7.61e-07%) when profiled on 2026-08-24.
    #
    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 26 of 656,894,348 rows (3.96e-06%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE CASE WHEN CAST(`event_datetime` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`event_datetime` AS DATE)) < 9999 THEN NULL ELSE `event_datetime` END END AS `event_datetime`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 5 of 656,894,348 rows (7.61e-07%) when profiled on 2026-08-24.
    #
    # event_end_datetime cannot precede event_datetime. The start is the better-attested of
    # the two, so the end is what goes and the row keeps its event_datetime. Hit 1,219,607
    # of 656,894,348 rows (0.186%) when profiled on 2026-08-24.
    #
    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 18 of 656,894,348 rows (2.74e-06%) when profiled on 2026-08-24.
    "CASE WHEN `event_datetime` IS NOT NULL AND `event_end_datetime` IS NOT NULL AND `event_datetime` > `event_end_datetime` OR CAST(`event_end_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE CASE WHEN CAST(`event_end_datetime` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`event_end_datetime` AS DATE)) < 9999 THEN NULL ELSE `event_end_datetime` END END AS `event_end_datetime`",
    "`source_coding_system` AS `source_coding_system`",
    "`source_code` AS `source_code`",
    "`source_display` AS `source_display`",
    "`document_type` AS `document_type`",
    "`title` AS `title`",
    "`author_practitioner_id` AS `author_practitioner_id`",
    "`author_role` AS `author_role`",
    "`service_id` AS `service_id`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 5,250,046 of 656,894,348 rows
    # (0.799%) when profiled on 2026-08-24.
    #
    # 'UNKNOWN' is a placeholder the source writes when the value was not recorded; it is
    # not a code, so it is nulled rather than passed on as one. Hit 110,413,837 of
    # 656,894,348 rows (16.8%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`status_code` AS STRING))) = '0' OR UPPER(TRIM(CAST(`status_code` AS STRING))) = 'UNKNOWN' THEN NULL ELSE `status_code` END AS `status_code`",
    "`version_id` AS `version_id`",
    "`document_thread_id` AS `document_thread_id`",
    "`supersedes_document_id` AS `supersedes_document_id`",
    "`version_ordinal` AS `version_ordinal`",
    "`is_latest_version` AS `is_latest_version`",

    # An empty or whitespace-only string is how the source writes 'nothing here'. It reads
    # as a value in a query and is not one, so it is nulled. Hit 1,371,747 of 656,894,348
    # rows (0.209%) when profiled on 2026-08-24.
    "CASE WHEN TRIM(CAST(`document_text` AS STRING)) = '' THEN NULL ELSE `document_text` END AS `document_text`",
    "`sections` AS `sections`",
    "`parser_version` AS `parser_version`",
    "`decompressor_version` AS `decompressor_version`",
    "`post_processor_version` AS `post_processor_version`",
    "`content_type` AS `content_type`",
    "`encoding` AS `encoding`",
    "`language` AS `language`",
    "`text_sha256` AS `text_sha256`",
    "`raw_content_sha256` AS `raw_content_sha256`",
    "`text_length` AS `text_length`",
    "`content_class` AS `content_class`",
    "`date_quality` AS `date_quality`",
    "`text_is_truncated` AS `text_is_truncated`",
    "`linkage_route` AS `linkage_route`",
    "`source_class` AS `source_class`",
    "`assembly_status` AS `assembly_status`",
    "`chunk_count` AS `chunk_count`",
    "`corpus_frequency` AS `corpus_frequency`",
    "`is_boilerplate` AS `is_boilerplate`",
    "`source_link_event_id` AS `source_link_event_id`",
    "`source_link_system` AS `source_link_system`",
    "`author_id_system` AS `author_id_system`",
    "`author_source_id` AS `author_source_id`",
    "`verified_practitioner_id` AS `verified_practitioner_id`",

    # 1899-12-30 is the zero point of the OLE/Excel date scale, so it is what a spreadsheet
    # or a COM layer writes when the date was left blank. It is not a date anyone recorded.
    # Hit 463 of 656,894,348 rows (7.05e-05%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`verified_datetime` AS DATE) = DATE'1899-12-30' THEN NULL ELSE `verified_datetime` END AS `verified_datetime`",
    "`source_organization_id` AS `source_organization_id`",
    "`source_organization_display` AS `source_organization_display`",
    "`record_status` AS `record_status`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 35 of 656,894,348 rows (5.33e-06%) when profiled on 2026-08-24.
    #
    # 1899-12-30 is the zero point of the OLE/Excel date scale, so it is what a spreadsheet
    # or a COM layer writes when the date was left blank. It is not a date anyone recorded.
    # Hit 2,761 of 656,894,348 rows (0.00042%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`record_status_effective_from` AS DATE) = DATE'1899-12-30' OR CAST(`record_status_effective_from` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`record_status_effective_from` AS DATE)) < 9999 THEN NULL ELSE `record_status_effective_from` END AS `record_status_effective_from`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 3 of 656,894,348 rows (4.57e-07%) when profiled on 2026-08-24.
    #
    # 1899-12-30 is the zero point of the OLE/Excel date scale, so it is what a spreadsheet
    # or a COM layer writes when the date was left blank. It is not a date anyone recorded.
    # Hit 969 of 656,894,348 rows (0.000148%) when profiled on 2026-08-24.
    #
    # 2100-12-31 is the far-future marker the source writes to mean 'no end yet'. The
    # absence is what the row means, and NULL states it without putting a fictional date
    # into a range comparison. Hit 50,536 of 656,894,348 rows (0.00769%) when profiled on
    # 2026-08-24.
    #
    # record_status_effective_to cannot precede record_status_effective_from. The start is
    # the better-attested of the two, so the end is what goes and the row keeps its
    # record_status_effective_from. Hit 3,202 of 656,894,348 rows (0.000487%) when profiled
    # on 2026-08-24.
    "CASE WHEN `record_status_effective_from` IS NOT NULL AND `record_status_effective_to` IS NOT NULL AND `record_status_effective_from` > `record_status_effective_to` THEN NULL ELSE CASE WHEN CAST(`record_status_effective_to` AS DATE) = DATE'1899-12-30' OR CAST(`record_status_effective_to` AS DATE) = DATE'2100-12-31' OR CAST(`record_status_effective_to` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`record_status_effective_to` AS DATE)) < 9999 THEN NULL ELSE `record_status_effective_to` END END AS `record_status_effective_to`",
    "`confidentiality_code` AS `confidentiality_code`",
    "`vip_ind` AS `vip_ind`",
    "`withheld_identity_ind` AS `withheld_identity_ind`",
    "`sensitivity_labels` AS `sensitivity_labels`",
    "`document_class` AS `document_class`",
    "`contributor_system` AS `contributor_system`",
    "`succession_status` AS `succession_status`",
    "`source_parent_event_id` AS `source_parent_event_id`",
    "`parent_relation` AS `parent_relation`",
    "`source_parent_display` AS `source_parent_display`",
    "`source_parent_title` AS `source_parent_title`",
    "`source_parent_tag` AS `source_parent_tag`",
    "`source_tag` AS `source_tag`",
    "`source_record_status` AS `source_record_status`",
    "`document_text_anonymised` AS `document_text_anonymised`",
    "`text_is_anonymised` AS `text_is_anonymised`",
    "`prsb_document_type` AS `prsb_document_type`",
    "`prsb_subtype` AS `prsb_subtype`",
    "`prsb_standard` AS `prsb_standard`",
    "`prsb_setting` AS `prsb_setting`",
    "`prsb_map_method` AS `prsb_map_method`",
    "`prsb_map_score` AS `prsb_map_score`",
    "`prsb_map_version` AS `prsb_map_version`",
    "`source_feed` AS `source_feed`",
    "`load_batch_id` AS `load_batch_id`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 35 of 656,894,348 rows (5.33e-06%) when profiled on 2026-08-24.
    #
    # 1899-12-30 is the zero point of the OLE/Excel date scale, so it is what a spreadsheet
    # or a COM layer writes when the date was left blank. It is not a date anyone recorded.
    # Hit 2,761 of 656,894,348 rows (0.00042%) when profiled on 2026-08-24.
    #
    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 8 of 656,894,348 rows (1.22e-06%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`source_update_timestamp` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE CASE WHEN CAST(`source_update_timestamp` AS DATE) = DATE'1899-12-30' OR CAST(`source_update_timestamp` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`source_update_timestamp` AS DATE)) < 9999 THEN NULL ELSE `source_update_timestamp` END END AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",

    # Silver derived this flag by comparing event_datetime with the person's own dates,
    # before the rules above corrected event_datetime. On the rows where event_datetime
    # changed, the flag describes a timestamp gold no longer publishes. This product carries
    # a VARIANT column, which rules out the spine join gold would need to recompute the
    # flag, so it is nulled where the value beneath it moved rather than left asserting
    # something stale.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `event_before_birth` END AS `event_before_birth`",

    # Silver derived this flag by comparing event_datetime with the person's own dates,
    # before the rules above corrected event_datetime. On the rows where event_datetime
    # changed, the flag describes a timestamp gold no longer publishes. This product carries
    # a VARIANT column, which rules out the spine join gold would need to recompute the
    # flag, so it is nulled where the value beneath it moved rather than left asserting
    # something stale.
    "CASE WHEN CAST(`event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE `event_after_death_30d` END AS `event_after_death_30d`",
]

TEXT_DOCUMENT_MANDATORY_RULES = {
    # The research surface. identity_status = 'resolved' keeps the 319,398,203 rows of
    # 656,894,348 that are current and attributable; is_latest_version = TRUE keeps the
    # 624,022,430 rows of 656,894,348 that are current and attributable; source_feed IN
    # ('ancil_long_blob', 'elective_access_comment', 'endobase_exam',
    # 'neonatal_episode_narrative', 'order_comment', 'pathology_report', 'text_event') OR
    # LOWER(TRIM(succession_status)) IN ('final', 'addendum') keeps the 448,698,905 rows of
    # 656,894,348 that are current and attributable. Superseded versions and rows whose
    # identity was never resolved are not research data, and a consumer who wants them has
    # silver.
    "research_surface":
        "(identity_status = 'resolved') AND (is_latest_version = TRUE) AND (source_feed IN ('ancil_long_blob', 'elective_access_comment', 'endobase_exam', 'neonatal_episode_narrative', 'order_comment', 'pathology_report', 'text_event') OR LOWER(TRIM(succession_status)) IN ('final', 'addendum'))",
}

TEXT_DOCUMENT_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing identity_status = 'resolved'.
    # The mandatory rule above has already dropped them, so this is how many went, not how
    # many survive. 337,496,145 of 656,894,348 at the profile.
    "gold.text.document.identity_status.default_view_resolved":
        "identity_status = 'resolved'",

    # Counts what the research surface removed: rows failing is_latest_version = TRUE. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 32,871,918 of 656,894,348 at the profile.
    "gold.text.document.is_latest_version.default_view_latest": "is_latest_version = TRUE",

    # This bounds a period of validity, and a future end is exactly how the source says a
    # record is still current -- nulling it would assert the record is valid forever, which
    # is a stronger and worse claim than the one being corrected. Seen on 8 of 656,894,348
    # rows (1.22e-06%) when profiled on 2026-08-24.
    "gold.text.document.record_status_effective_from.future_owner":
        "NOT COALESCE((CAST(`record_status_effective_from` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",

    # This bounds a period of validity, and a future end is exactly how the source says a
    # record is still current -- nulling it would assert the record is valid forever, which
    # is a stronger and worse claim than the one being corrected. Seen on 1 of 656,894,348
    # rows (1.52e-07%) when profiled on 2026-08-24.
    "gold.text.document.record_status_effective_to.future_owner":
        "NOT COALESCE((CAST(`record_status_effective_to` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",

    # Counts what the research surface removed: rows failing source_feed IN
    # ('ancil_long_blob', 'elective_access_comment', 'endobase_exam',
    # 'neonatal_episode_narrative', 'order_comment', 'pathology_report', 'text_event') OR
    # LOWER(TRIM(succession_status)) IN ('final', 'addendum'). The mandatory rule above has
    # already dropped them, so this is how many went, not how many survive. 208,195,443 of
    # 656,894,348 at the profile.
    "gold.text.document.succession_status.default_view_completed":
        "source_feed IN ('ancil_long_blob', 'elective_access_comment', 'endobase_exam', 'neonatal_episode_narrative', 'order_comment', 'pathology_report', 'text_event') OR LOWER(TRIM(succession_status)) IN ('final', 'addendum')",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 937,755 of 656,894,348 rows (0.143%) when profiled on
    # 2026-08-24.
    "gold.text.document.table.after_death_30d_event_datetime_person_deceased_datetime":
        "NOT COALESCE(event_after_death_30d, FALSE)",

    # Left as a warning because the comparison has no trustworthy side: 2,051 people in the
    # spine carry a death recorded before their own birth, 14,720 died before 1990 and
    # 188,119 were born before 1920. When an event disagrees with the anchor, the data does
    # not say which of the two is wrong, so nulling either would destroy sound values at an
    # unknown rate. Seen on 176,524 of 656,894,348 rows (0.0269%) when profiled on
    # 2026-08-24.
    "gold.text.document.table.before_birth_event_datetime_person_birth_datetime":
        "NOT COALESCE(event_before_birth, FALSE)",
}

TEXT_DOCUMENT_COLUMN_COMMENTS = {
    "patient_event_id": "Stable product-wide document-version identifier.",
    "fact_row_id": "Storage-row identifier equal to patient_event_id.",
    "subject_key": "Always-populated peppered or per-document subject key.",
    "subject_id_system": "Identifier system used for subject_key.",
    "person_id": "Resolved Millennium person identifier when recoverable.",
    "identity_status": "Subject resolution state.",
    "encounter_id": "Encounter reference when supplied.",
    "event_datetime":
        "Document clinical or version timestamp. Gold QC transform rules: gold.text.document.event_datetime.beyond_2100_below_9999_owner.",
    "event_end_datetime":
        "Source version-validity end. Gold QC transform rules: gold.text.document.event_end_datetime.beyond_2100_below_9999_owner.",
    "source_coding_system": "Verbatim source document-type system.",
    "source_code": "Verbatim source document-type code.",
    "source_display": "Verbatim source document-type display.",
    "document_type": "Source document-type CodeableConcept.",
    "title": "Source document title or label.",
    "author_practitioner_id": "Source author or updater practitioner.",
    "author_role": "Source author role when supplied.",
    "service_id": "Service reference when supplied.",
    "status_code": "Verbatim document extraction or source status.",
    "version_id": "Source document-version token.",
    "document_thread_id":
        "Stable identifier for all versions of the same logical document; source series identifiers are preferred and single-version lanes use patient_event_id.",
    "supersedes_document_id":
        "Stable patient_event_id of the directly superseded document version when the source supplies that relationship.",
    "version_ordinal":
        "Source-supplied version ordinal when available; one for single-version lanes.",
    "is_latest_version":
        "Exactly one row per document_thread_id selected by non-empty text then effective-from and source version ordinal descending with version_id descending as deterministic tiebreak.",
    "document_text":
        "Identifiable source document text under default-deny access. Gold QC transform rules: gold.text.document.document_text.empty_string.",
    "sections": "Deterministically ordered parsed document sections when supplied.",
    "parser_version": "Parser version that produced document_text or sections.",
    "decompressor_version": "Decompressor version used by bronze.",
    "post_processor_version": "Post-processor version used by bronze.",
    "content_type": "Source MIME or content type.",
    "encoding": "Source text encoding",
    "language":
        "Document language defaulted to English (`en`) because no source language field is available; method=default.",
    "text_sha256":
        "SHA-256 of retained document_text only; the anonymous alternative never changes this hash or any stable identifier. NULL is represented by the empty-string digest.",
    "raw_content_sha256":
        "SHA-256 of the original binary payload when supplied by the blob source; distinct from text_sha256.",
    "text_length":
        "Retained document-text character count recomputed after parsing and post-processing.",
    "content_class":
        "Deterministic retained-text quality class; rows are flagged rather than dropped.",
    "date_quality":
        "Event-date quality relative to source provenance: null, pre-1975 epoch sentinel, future beyond source/load time plus one day, or ok.",
    "text_is_truncated":
        "True when retained text length equals a known source or parser cap (1000000, 65535, 32767, or 32000 characters).",
    "linkage_route":
        "Deterministic provenance route used to resolve document subject or encounter linkage.",
    "source_class":
        "Fail-closed source-content class; the public document product contains clinical text only.",
    "assembly_status":
        "Source-row assembly outcome; NULL for parked SCD working-copy rows.",
    "chunk_count":
        "Number of source chunks represented by this document row; NULL for parked SCD working-copy rows.",
    "corpus_frequency":
        "Number of current Journey document rows sharing text_sha256 when the governed release-built frequency is at least 100; NULL means below that storage floor.",
    "is_boilerplate":
        "True exactly when governed corpus_frequency is at least 1000; absent frequency is false.",
    "source_link_event_id":
        "Verbatim linked source event identifier when a governed cross-feed bridge supplies one.",
    "source_link_system": "Identifier system for source_link_event_id.",
    "author_id_system": "Identifier system for the verbatim source author identifier.",
    "author_source_id":
        "Verbatim source author identifier retained separately from any resolved practitioner reference.",
    "verified_practitioner_id":
        "Stable practitioner reference for the source verifier when resolvable.",
    "verified_datetime":
        "Source verification timestamp associated with verified_practitioner_id. Gold QC transform rules: gold.text.document.verified_datetime.ole_zero_date.",
    "source_organization_id":
        "Stable source-organization reference derived from governed source evidence.",
    "source_organization_display":
        "Source-organization display from the governed organization dimension or verbatim PACS institution.",
    "record_status": "Normalized source-record lifecycle.",
    "record_status_effective_from":
        "Source lifecycle start. Gold QC transform rules: gold.text.document.record_status_effective_from.beyond_2100_below_9999_owner, gold.text.document.record_status_effective_from.ole_zero_date.",
    "record_status_effective_to":
        "Source lifecycle end. Gold QC transform rules: gold.text.document.record_status_effective_to.beyond_2100_below_9999_owner, gold.text.document.record_status_effective_to.ole_zero_date, gold.text.document.record_status_effective_to.open_sentinel.",
    "confidentiality_code": "Source confidentiality classification.",
    "vip_ind": "VIP indicator when supplied.",
    "withheld_identity_ind": "Withheld-identity indicator when supplied.",
    "sensitivity_labels": "Ordered source security labels when supplied.",
    "document_class": "Source event-class display such as Document or mdoc.",
    "contributor_system":
        "Source contributor-system display such as PowerChart or BLT_TIE_RAD.",
    "succession_status": "Blob succession status display such as Interim or Final.",
    "source_parent_event_id":
        "Parent clinical-event identifier for document threading when supplied.",
    "parent_relation": "Event relation display such as Root or Child.",
    "source_parent_display":
        "Parent event-code description; the legacy MainEventDesc document-type label.",
    "source_parent_title": "Parent event title text; the legacy MainTitleText.",
    "source_parent_tag": "Parent event tag text; the legacy MainTagText.",
    "source_tag": "Own event tag text; the legacy ChildTagText.",
    "source_record_status":
        "Clinical-event record-status display such as Active or Deleted; the legacy Status.",
    "document_text_anonymised":
        "Hygiene-transformed bronze anonymous alternative; NULL when anonymisation is unavailable.",
    "text_is_anonymised":
        "True only when document_text itself uses the existing blob raw-absent anonymous fallback.",
    "prsb_document_type":
        "PRSB-aligned canonical document type from the governed doc_type_prsb_map lookup.",
    "prsb_subtype": "Canonical subtype qualifier when the mapping supplies one.",
    "prsb_standard": "Source PRSB or Royal-College standard label for the canonical type.",
    "prsb_setting": "Care-setting bucket of the canonical type.",
    "prsb_map_method": "Mapping method provenance.",
    "prsb_map_score": "Embedder cosine score for embedder-method rows.",
    "prsb_map_version": "doc_type_prsb_map version label.",
    "source_feed":
        "Registered owning source route; all document text is IG-sensitive clinical content.",
    "load_batch_id": "Bronze batch token.",
    "source_update_timestamp":
        "Native source update timestamp. Gold QC transform rules: gold.text.document.source_update_timestamp.beyond_2100_below_9999_owner, gold.text.document.source_update_timestamp.ole_zero_date.",
    "loaded_at": "Bronze load timestamp.",
    "event_before_birth":
        "True when event_datetime precedes the resolved person's birth_datetime; published for Gold cross-rule attribution.",
    "event_after_death_30d":
        "True when event_datetime is more than 30 days after the resolved person's deceased_datetime; published for Gold cross-rule attribution.",
}

@dp.materialized_view(
    name=_n("gold_text.document"),
    comment=(
        "One governed clinical document version with identifiable text and parser provenance. "
        "Includes the RDE-parity block and governed PRSB typing; blob-lane document_type "
        "became the clinical event coding (recorded change). Gold QC twin of the silver "
        "product: 12 columns are repaired or nulled, 1 rule(s) drop rows, 7 check(s) are "
        "advisory. Each rule states its reason in the pipeline notebook, and Lakeflow "
        "expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(TEXT_DOCUMENT_MANDATORY_RULES)
@_expect_all(TEXT_DOCUMENT_ADVISORY_RULES)
def gold_text_document():
    """Quality-controlled twin of journey_text.document."""
    df = _qc("text_document", TEXT_DOCUMENT_SELECT)
    return _with_comments(df, TEXT_DOCUMENT_COLUMN_COMMENTS)

In [0]:
# ======== Reference ========
#
# Code lists, catalogues and the quality rules themselves.
# 26 products follow.

In [0]:
# ==== journey_reference.admission_metrics ====

REFERENCE_ADMISSION_METRICS_SELECT = [
    "`feed_id` AS `feed_id`",
    "`route_id` AS `route_id`",
    "`admitted` AS `admitted`",
    "`load_date` AS `load_date`",
    "`row_count` AS `row_count`",
    "`latest_loaded_at` AS `latest_loaded_at`",
]

REFERENCE_ADMISSION_METRICS_COLUMN_COMMENTS = {
    "feed_id": "Registered feed.",
    "route_id": "Route or exclusion reason.",
    "admitted": "Whether these rows entered silver.",
    "load_date": "Bronze load date bucket for per-load accounting.",
    "row_count": "Rows in this bucket.",
    "latest_loaded_at": "Latest bronze load time observed in the bucket.",
}

@dp.materialized_view(
    name=_n("gold_reference.admission_metrics"),
    comment=(
        "Admission-gate accounting per registered route; route_id doubles as the exclusion "
        "reason for admitted=false rows. The product-wide usable-code gate covers every "
        "source_code-bearing fact lane; document is exempt as a narrative route, and "
        "structured admin facts without source_code are exempt; registry_entry rows carry no "
        "source codes by construction and are exempt as a class. Gold QC twin of the silver "
        "product: 0 columns are repaired or nulled, 0 check(s) are advisory. Each rule states "
        "its reason in the pipeline notebook, and Lakeflow expectation metrics report what "
        "every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
def gold_reference_admission_metrics():
    """Quality-controlled twin of journey_reference.admission_metrics."""
    df = _qc("reference_admission_metrics", REFERENCE_ADMISSION_METRICS_SELECT)
    return _with_comments(df, REFERENCE_ADMISSION_METRICS_COLUMN_COMMENTS)

In [0]:
# ==== journey_reference.concept_map ====

REFERENCE_CONCEPT_MAP_SELECT = [
    "`concept_map_id` AS `concept_map_id`",
    "`source_coding_system` AS `source_coding_system`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 23 of 1,080,442 rows
    # (0.00213%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`source_code` AS STRING))) = '0' THEN NULL ELSE `source_code` END AS `source_code`",
    "`source_display` AS `source_display`",
    "`target_coding_system` AS `target_coding_system`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 2 of 1,080,442 rows
    # (0.000185%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`target_code` AS STRING))) = '0' THEN NULL ELSE `target_code` END AS `target_code`",
    "`target_display` AS `target_display`",
    "`target_domain` AS `target_domain`",
    "`equivalence` AS `equivalence`",
    "`map_source` AS `map_source`",
    "`map_version` AS `map_version`",
    "`mapping_rule_id` AS `mapping_rule_id`",
    "`mapping_rank` AS `mapping_rank`",
    "`review_status` AS `review_status`",
    "`valid_from` AS `valid_from`",
    "`valid_to` AS `valid_to`",
    "`source_table` AS `source_table`",
    "`source_row_count` AS `source_row_count`",
    "`loaded_at` AS `loaded_at`",
]

REFERENCE_CONCEPT_MAP_ADVISORY_RULES = {
    # This bounds a period of validity, and a future end is exactly how the source says a
    # record is still current -- nulling it would assert the record is valid forever, which
    # is a stronger and worse claim than the one being corrected. Seen on 33 of 1,080,442
    # rows (0.00305%) when profiled on 2026-08-24.
    "gold.reference.concept_map.valid_to.future_owner":
        "NOT COALESCE((CAST(`valid_to` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",
}

REFERENCE_CONCEPT_MAP_COLUMN_COMMENTS = {
    "concept_map_id": "Deterministic versioned mapping primary key.",
    "source_coding_system": "Source code-system namespace.",
    "source_code": "Source code or mapped source value.",
    "source_display": "Source display carried from bronze.",
    "target_coding_system": "Target code-system namespace.",
    "target_code": "Target code or concept identifier.",
    "target_display": "Target display carried from bronze.",
    "target_domain": "Intended downstream semantic domain.",
    "equivalence": "FHIR equivalence when explicitly supplied; null rather than inferred.",
    "map_source": "Governed mapping product and route.",
    "map_version": "Pinned mapping release.",
    "mapping_rule_id": "Source mapping rule or confidence discriminator.",
    "mapping_rank": "Source-supplied candidate rank where available.",
    "review_status":
        "Review state carried verbatim from the mapping source; source_carried rows are bronze-carried without review",
    "valid_from": "Target mapping validity start where supplied.",
    "valid_to": "Target mapping validity end where supplied.",
    "source_table":
        "Configured bronze source or explicitly authorized development fixture.",
    "source_row_count": "Number of bronze mapping observations represented by the row.",
    "loaded_at": "Latest bronze mapping load timestamp represented.",
}

@dp.materialized_view(
    name=_n("gold_reference.concept_map"),
    comment=(
        "One versioned source-to-target concept mapping carried by a governed bronze product; "
        "one-to-many mappings remain separate rows. Gold QC twin of the silver product: 2 "
        "columns are repaired or nulled, 1 check(s) are advisory. Each rule states its reason "
        "in the pipeline notebook, and Lakeflow expectation metrics report what every rule "
        "matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all(REFERENCE_CONCEPT_MAP_ADVISORY_RULES)
def gold_reference_concept_map():
    """Quality-controlled twin of journey_reference.concept_map."""
    df = _qc("reference_concept_map", REFERENCE_CONCEPT_MAP_SELECT)
    return _with_comments(df, REFERENCE_CONCEPT_MAP_COLUMN_COMMENTS)

In [0]:
# ==== journey_reference.concept_registry ====

REFERENCE_CONCEPT_REGISTRY_SELECT = [
    "`concept_registry_id` AS `concept_registry_id`",
    "`coding_system` AS `coding_system`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 28 of 1,303,944 rows
    # (0.00215%) when profiled on 2026-08-24.
    #
    # 'UNKNOWN' is a placeholder the source writes when the value was not recorded; it is
    # not a code, so it is nulled rather than passed on as one. Hit 9 of 1,303,944 rows
    # (0.00069%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`code` AS STRING))) = '0' OR UPPER(TRIM(CAST(`code` AS STRING))) = 'UNKNOWN' THEN NULL ELSE `code` END AS `code`",
    "`preferred_display` AS `preferred_display`",
    "`status` AS `status`",
    "`source_use_count` AS `source_use_count`",
    "`target_use_count` AS `target_use_count`",
    "`first_observed_at` AS `first_observed_at`",
    "`last_observed_at` AS `last_observed_at`",
]

REFERENCE_CONCEPT_REGISTRY_COLUMN_COMMENTS = {
    "concept_registry_id": "Deterministic concept-registry primary key.",
    "coding_system": "Coding-system namespace.",
    "code": "Observed code.",
    "preferred_display": "Deterministically selected observed display.",
    "status": "Registry observation status.",
    "source_use_count": "Source-side event and mapping observation count.",
    "target_use_count": "Target-side mapping observation count.",
    "first_observed_at": "Earliest represented bronze observation timestamp.",
    "last_observed_at": "Latest represented bronze observation timestamp.",
}

@dp.materialized_view(
    name=_n("gold_reference.concept_registry"),
    comment=(
        "One observed coding-system and code pair across event sources and mapping targets. "
        "Gold QC twin of the silver product: 1 columns are repaired or nulled, 0 check(s) are "
        "advisory. Each rule states its reason in the pipeline notebook, and Lakeflow "
        "expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
def gold_reference_concept_registry():
    """Quality-controlled twin of journey_reference.concept_registry."""
    df = _qc("reference_concept_registry", REFERENCE_CONCEPT_REGISTRY_SELECT)
    return _with_comments(df, REFERENCE_CONCEPT_REGISTRY_COLUMN_COMMENTS)

In [0]:
# ==== journey_reference.cost_line_item ====

REFERENCE_COST_LINE_ITEM_SELECT = [
    "`cost_line_item_id` AS `cost_line_item_id`",
    "`costed_activity_id` AS `costed_activity_id`",
    "`extract_cd` AS `extract_cd`",
    "`activity_record_id` AS `activity_record_id`",
    "`line_hash` AS `line_hash`",
    "`activity_cost_item_cd` AS `activity_cost_item_cd`",
    "`resource_cost_item_cd` AS `resource_cost_item_cd`",
    "`activity_count` AS `activity_count`",
    "`unbundled_subtype_cd` AS `unbundled_subtype_cd`",
    "`unbundled_currency_cd` AS `unbundled_currency_cd`",
    "`unbundled_currency_datetime` AS `unbundled_currency_datetime`",
    "`total_cost` AS `total_cost`",
    "`total_o_cost` AS `total_o_cost`",
    "`source_duplicate_count` AS `source_duplicate_count`",
    "`record_status` AS `record_status`",
    "`loaded_at` AS `loaded_at`",
]

REFERENCE_COST_LINE_ITEM_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 181,295,498 rows of
    # 181,295,498 that are current and attributable. Superseded versions and rows whose
    # identity was never resolved are not research data, and a consumer who wants them has
    # silver.
    "research_surface": "(record_status = 'active')",
}

REFERENCE_COST_LINE_ITEM_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 0 of 181,295,498 at the profile.
    "gold.reference.cost_line_item.record_status.default_view_active":
        "record_status = 'active'",

    # Negative values here are signed by design -- credit lines in the costing feed and
    # below-zero readings on a calibrated scale -- so they are counted, not removed. Seen on
    # 11,112,901 of 181,295,498 rows (6.13%) when profiled on 2026-08-24.
    "gold.reference.cost_line_item.total_cost.negative_mass_scale_owner":
        "NOT COALESCE((`total_cost` < 0), FALSE)",

    # Negative values here are signed by design -- credit lines in the costing feed and
    # below-zero readings on a calibrated scale -- so they are counted, not removed. Seen on
    # 16,457,918 of 181,295,498 rows (9.08%) when profiled on 2026-08-24.
    "gold.reference.cost_line_item.total_o_cost.negative_mass_scale_owner":
        "NOT COALESCE((`total_o_cost` < 0), FALSE)",
}

REFERENCE_COST_LINE_ITEM_COLUMN_COMMENTS = {
    "cost_line_item_id": "Stable row identifier.",
    "costed_activity_id": "Published field.",
    "extract_cd": "Published field.",
    "activity_record_id": "Published field.",
    "line_hash": "Published field.",
    "activity_cost_item_cd": "Published field.",
    "resource_cost_item_cd": "Published field.",
    "activity_count": "Published field.",
    "unbundled_subtype_cd": "Published field.",
    "unbundled_currency_cd": "Published field.",
    "unbundled_currency_datetime": "Published field.",
    "total_cost": "Published field.",
    "total_o_cost": "Published field.",
    "source_duplicate_count": "Published field.",
    "record_status": "Published field.",
    "loaded_at": "Published field.",
}

@dp.materialized_view(
    name=_n("gold_reference.cost_line_item"),
    comment=(
        "One frozen PLICS cost line; excluded from the event plane. Gold QC twin of the "
        "silver product: 0 columns are repaired or nulled, 1 rule(s) drop rows, 3 check(s) "
        "are advisory. Each rule states its reason in the pipeline notebook, and Lakeflow "
        "expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(REFERENCE_COST_LINE_ITEM_MANDATORY_RULES)
@_expect_all(REFERENCE_COST_LINE_ITEM_ADVISORY_RULES)
def gold_reference_cost_line_item():
    """Quality-controlled twin of journey_reference.cost_line_item."""
    df = _qc("reference_cost_line_item", REFERENCE_COST_LINE_ITEM_SELECT)
    return _with_comments(df, REFERENCE_COST_LINE_ITEM_COLUMN_COMMENTS)

In [0]:
# ==== journey_reference.device_mapping ====

REFERENCE_DEVICE_MAPPING_SELECT = [
    "`device_mapping_id` AS `device_mapping_id`",
    "`source_event_id` AS `source_event_id`",
    "`person_id` AS `person_id`",
    "`encounter_id` AS `encounter_id`",
    "`implant_description` AS `implant_description`",
    "`normalized_description` AS `normalized_description`",
    "`cleaned_description` AS `cleaned_description`",
    "`udi_di` AS `udi_di`",
    "`udi_issuer` AS `udi_issuer`",
    "`gs1_identifier` AS `gs1_identifier`",
    "`hibcc_device_id` AS `hibcc_device_id`",
    "`serial_number` AS `serial_number`",
    "`expiry_date` AS `expiry_date`",
    "`gmdn_code` AS `gmdn_code`",
    "`gmdn_name` AS `gmdn_name`",
    "`snomed_concept_id` AS `snomed_concept_id`",
    "`snomed_name` AS `snomed_name`",
    "`standard_concept_id` AS `standard_concept_id`",
    "`standard_concept_name` AS `standard_concept_name`",
    "`standard_vocabulary_id` AS `standard_vocabulary_id`",
    "`device_type` AS `device_type`",
    "`mapping_layer` AS `mapping_layer`",
    "`mapping_status` AS `mapping_status`",
    "`mapping_confidence` AS `mapping_confidence`",
    "`confidence_tier` AS `confidence_tier`",
    "`mapping_rule_id` AS `mapping_rule_id`",
    "`matched_field` AS `matched_field`",
    "`matched_value` AS `matched_value`",
    "`mapping_candidate_count` AS `mapping_candidate_count`",
    "`mapping_distinct_concept_count` AS `mapping_distinct_concept_count`",
    "`mapping_ambiguous_ind` AS `mapping_ambiguous_ind`",
    "`matched_opcs_code` AS `matched_opcs_code`",
    "`procedure_support_ind` AS `procedure_support_ind`",
    "`mapping_schema_version` AS `mapping_schema_version`",
    "`normalization_version` AS `normalization_version`",
    "`brand_rules_version` AS `brand_rules_version`",
    "`mapped_at` AS `mapped_at`",
    "`loaded_at` AS `loaded_at`",
    "`_source_system` AS `_source_system`",
    "`_source_table` AS `_source_table`",
    "`_source_row_id` AS `_source_row_id`",
]

REFERENCE_DEVICE_MAPPING_ADVISORY_RULES = {
    # This bounds a period of validity, and a future end is exactly how the source says a
    # record is still current -- nulling it would assert the record is valid forever, which
    # is a stronger and worse claim than the one being corrected. Seen on 106,883 of 248,449
    # rows (43%) when profiled on 2026-08-24.
    "gold.reference.device_mapping.expiry_date.future_owner":
        "NOT COALESCE((CAST(`expiry_date` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",
}

REFERENCE_DEVICE_MAPPING_COLUMN_COMMENTS = {
    "device_mapping_id": "Published field device_mapping_id.",
    "source_event_id": "Published field source_event_id.",
    "person_id": "Published field person_id.",
    "encounter_id": "Published field encounter_id.",
    "implant_description": "Published field implant_description.",
    "normalized_description": "Published field normalized_description.",
    "cleaned_description": "Published field cleaned_description.",
    "udi_di": "Published field udi_di.",
    "udi_issuer": "Published field udi_issuer.",
    "gs1_identifier": "Published field gs1_identifier.",
    "hibcc_device_id": "Published field hibcc_device_id.",
    "serial_number": "Published field serial_number.",
    "expiry_date": "Published field expiry_date.",
    "gmdn_code": "Published field gmdn_code.",
    "gmdn_name": "Published field gmdn_name.",
    "snomed_concept_id": "Published field snomed_concept_id.",
    "snomed_name": "Published field snomed_name.",
    "standard_concept_id": "Published field standard_concept_id.",
    "standard_concept_name": "Published field standard_concept_name.",
    "standard_vocabulary_id": "Published field standard_vocabulary_id.",
    "device_type": "Published field device_type.",
    "mapping_layer": "Published field mapping_layer.",
    "mapping_status": "Published field mapping_status.",
    "mapping_confidence": "Published field mapping_confidence.",
    "confidence_tier": "Published field confidence_tier.",
    "mapping_rule_id": "Published field mapping_rule_id.",
    "matched_field": "Published field matched_field.",
    "matched_value": "Published field matched_value.",
    "mapping_candidate_count": "Published field mapping_candidate_count.",
    "mapping_distinct_concept_count": "Published field mapping_distinct_concept_count.",
    "mapping_ambiguous_ind": "Published field mapping_ambiguous_ind.",
    "matched_opcs_code": "Published field matched_opcs_code.",
    "procedure_support_ind": "Published field procedure_support_ind.",
    "mapping_schema_version": "Published field mapping_schema_version.",
    "normalization_version": "Published field normalization_version.",
    "brand_rules_version": "Published field brand_rules_version.",
    "mapped_at": "Published field mapped_at.",
    "loaded_at": "Published field loaded_at.",
    "_source_system": "Published field _source_system.",
    "_source_table": "Published field _source_table.",
    "_source_row_id": "Published field _source_row_id.",
}

@dp.materialized_view(
    name=_n("gold_reference.device_mapping"),
    comment=(
        "One best-effort implant-device mapping with source and terminology provenance. Gold "
        "QC twin of the silver product: 0 columns are repaired or nulled, 1 check(s) are "
        "advisory. Each rule states its reason in the pipeline notebook, and Lakeflow "
        "expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all(REFERENCE_DEVICE_MAPPING_ADVISORY_RULES)
def gold_reference_device_mapping():
    """Quality-controlled twin of journey_reference.device_mapping."""
    df = _qc("reference_device_mapping", REFERENCE_DEVICE_MAPPING_SELECT)
    return _with_comments(df, REFERENCE_DEVICE_MAPPING_COLUMN_COMMENTS)

In [0]:
# ==== journey_reference.elective_access_procedure ====

REFERENCE_ELECTIVE_ACCESS_PROCEDURE_SELECT = [
    "`elective_access_procedure_id` AS `elective_access_procedure_id`",
    "`elective_access_entry_id` AS `elective_access_entry_id`",
    "`waiting_list_oid` AS `waiting_list_oid`",
    "`procedure_code` AS `procedure_code`",
    "`procedure_desc` AS `procedure_desc`",
    "`procedure_catalog` AS `procedure_catalog`",
    "`procedure_rvid` AS `procedure_rvid`",
    "`procedure_type_seq` AS `procedure_type_seq`",
    "`procedure_seq` AS `procedure_seq`",
    "`active_ind` AS `active_ind`",
    "`parent_present_ind` AS `parent_present_ind`",
    "`source_system_oid` AS `source_system_oid`",
    "`source_system_oid_inherited_ind` AS `source_system_oid_inherited_ind`",
    "`record_status` AS `record_status`",
    "`loaded_at` AS `loaded_at`",
]

REFERENCE_ELECTIVE_ACCESS_PROCEDURE_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 2,922,493 rows of 2,922,503
    # that are current and attributable. Superseded versions and rows whose identity was
    # never resolved are not research data, and a consumer who wants them has silver.
    "research_surface": "(record_status = 'active')",
}

REFERENCE_ELECTIVE_ACCESS_PROCEDURE_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 10 of 2,922,503 at the profile.
    "gold.reference.elective_access_procedure.record_status.default_view_active":
        "record_status = 'active'",
}

REFERENCE_ELECTIVE_ACCESS_PROCEDURE_COLUMN_COMMENTS = {
    "elective_access_procedure_id": "Stable row identifier.",
    "elective_access_entry_id": "Published field.",
    "waiting_list_oid": "Published field.",
    "procedure_code": "Published field.",
    "procedure_desc": "Published field.",
    "procedure_catalog": "Published field.",
    "procedure_rvid": "Published field.",
    "procedure_type_seq": "Published field.",
    "procedure_seq": "Published field.",
    "active_ind": "Published field.",
    "parent_present_ind": "Published field.",
    "source_system_oid": "Published field.",
    "source_system_oid_inherited_ind": "Published field.",
    "record_status": "Published field.",
    "loaded_at": "Published field.",
}

@dp.materialized_view(
    name=_n("gold_reference.elective_access_procedure"),
    comment=(
        "One LUNA elective-access planned-procedure slot. Gold QC twin of the silver product: "
        "0 columns are repaired or nulled, 1 rule(s) drop rows, 1 check(s) are advisory. Each "
        "rule states its reason in the pipeline notebook, and Lakeflow expectation metrics "
        "report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(REFERENCE_ELECTIVE_ACCESS_PROCEDURE_MANDATORY_RULES)
@_expect_all(REFERENCE_ELECTIVE_ACCESS_PROCEDURE_ADVISORY_RULES)
def gold_reference_elective_access_procedure():
    """Quality-controlled twin of journey_reference.elective_access_procedure."""
    df = _qc("reference_elective_access_procedure", REFERENCE_ELECTIVE_ACCESS_PROCEDURE_SELECT)
    return _with_comments(df, REFERENCE_ELECTIVE_ACCESS_PROCEDURE_COLUMN_COMMENTS)

In [0]:
# ==== journey_reference.encounter_attribute ====

REFERENCE_ENCOUNTER_ATTRIBUTE_SELECT = [
    "`encounter_attribute_id` AS `encounter_attribute_id`",
    "`encounter_id` AS `encounter_id`",
    "`source_encounter_id` AS `source_encounter_id`",
    "`attribute_name` AS `attribute_name`",
    "`value_kind` AS `value_kind`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 10,805 of 87,111,763 rows
    # (0.0124%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`value_code` AS STRING))) = '0' THEN NULL ELSE `value_code` END AS `value_code`",
    "`value_display` AS `value_display`",
    "`value_code_set` AS `value_code_set`",
    "`value_datetime` AS `value_datetime`",
    "`value_numeric` AS `value_numeric`",
    "`answered_ind` AS `answered_ind`",
    "`active_ind` AS `active_ind`",
    "`current_ind` AS `current_ind`",
    "`beg_effective` AS `beg_effective`",
    "`end_effective` AS `end_effective`",
    "`link_status` AS `link_status`",
    "`loaded_at` AS `loaded_at`",
    "`_source_system` AS `_source_system`",
    "`_source_table` AS `_source_table`",
    "`_source_row_id` AS `_source_row_id`",
]

REFERENCE_ENCOUNTER_ATTRIBUTE_MANDATORY_RULES = {
    # The flow nulls encounter_id when it names a parent the spine does not have, and this
    # rule then drops the row, because the row is only an attribute of an encounter, so with
    # no encounter it describes nothing. That was 203,581 orphaned rows plus 0 that already
    # had no encounter_id, out of 87,111,763.
    "gold.reference.encounter_attribute.encounter_id.fk_containment":
        "`encounter_id` IS NOT NULL",
}

REFERENCE_ENCOUNTER_ATTRIBUTE_COLUMN_COMMENTS = {
    "encounter_attribute_id": "Published field encounter_attribute_id.",
    "encounter_id": "Published field encounter_id.",
    "source_encounter_id": "Published field source_encounter_id.",
    "attribute_name": "Published field attribute_name.",
    "value_kind": "Published field value_kind.",
    "value_code": "Published field value_code.",
    "value_display": "Published field value_display.",
    "value_code_set": "Published field value_code_set.",
    "value_datetime": "Published field value_datetime.",
    "value_numeric": "Published field value_numeric.",
    "answered_ind": "Published field answered_ind.",
    "active_ind": "Published field active_ind.",
    "current_ind": "Published field current_ind.",
    "beg_effective": "Published field beg_effective.",
    "end_effective": "Published field end_effective.",
    "link_status": "Published field link_status.",
    "loaded_at": "Published field loaded_at.",
    "_source_system": "Published field _source_system.",
    "_source_table": "Published field _source_table.",
    "_source_row_id": "Published field _source_row_id.",
}

@dp.materialized_view(
    name=_n("gold_reference.encounter_attribute"),
    comment=(
        "One allowlisted encounter attribute with current and effective evidence. Gold QC "
        "twin of the silver product: 2 columns are repaired or nulled, 1 rule(s) drop rows, 0 "
        "check(s) are advisory. Each rule states its reason in the pipeline notebook, and "
        "Lakeflow expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(REFERENCE_ENCOUNTER_ATTRIBUTE_MANDATORY_RULES)
def gold_reference_encounter_attribute():
    """Quality-controlled twin of journey_reference.encounter_attribute."""
    df = _qc(
        "reference_encounter_attribute",
        REFERENCE_ENCOUNTER_ATTRIBUTE_SELECT,
        fk_columns=["encounter_id"],
    )
    return _with_comments(df, REFERENCE_ENCOUNTER_ATTRIBUTE_COLUMN_COMMENTS)

In [0]:
# ==== journey_reference.encounter_bounds ====

REFERENCE_ENCOUNTER_BOUNDS_SELECT = [
    "`encounter_bounds_id` AS `encounter_bounds_id`",
    "`encounter_id` AS `encounter_id`",
    "`source_encounter_id` AS `source_encounter_id`",
    "`first_clinical_event_datetime` AS `first_clinical_event_datetime`",
    "`last_clinical_event_datetime` AS `last_clinical_event_datetime`",
    "`clinical_event_count` AS `clinical_event_count`",
    "`first_contemporaneous_event_datetime` AS `first_contemporaneous_event_datetime`",
    "`last_contemporaneous_event_datetime` AS `last_contemporaneous_event_datetime`",
    "`contemporaneous_event_count` AS `contemporaneous_event_count`",
    "`first_order_datetime` AS `first_order_datetime`",
    "`last_order_datetime` AS `last_order_datetime`",
    "`order_count` AS `order_count`",
    "`ward_move_count` AS `ward_move_count`",
    "`ward_occupancy_minutes` AS `ward_occupancy_minutes`",
    "`last_ward_in_datetime` AS `last_ward_in_datetime`",
    "`last_ward_out_datetime` AS `last_ward_out_datetime`",
    "`last_ward_still_open_ind` AS `last_ward_still_open_ind`",
    "`event_first_datetime` AS `event_first_datetime`",
    "`event_last_datetime` AS `event_last_datetime`",
    "`in_activity_bounds_ind` AS `in_activity_bounds_ind`",
    "`in_event_bounds_ind` AS `in_event_bounds_ind`",
    "`loaded_at` AS `loaded_at`",
    "`_source_system` AS `_source_system`",
    "`_source_table` AS `_source_table`",
    "`_source_row_id` AS `_source_row_id`",
]

REFERENCE_ENCOUNTER_BOUNDS_MANDATORY_RULES = {
    # The flow nulls encounter_id when it names a parent the spine does not have, and this
    # rule then drops the row, because the row is only the start and end of an encounter, so
    # with no encounter it bounds nothing. That was 413,547 orphaned rows plus 0 that
    # already had no encounter_id, out of 47,674,684.
    "gold.reference.encounter_bounds.encounter_id.fk_containment":
        "`encounter_id` IS NOT NULL",
}

REFERENCE_ENCOUNTER_BOUNDS_COLUMN_COMMENTS = {
    "encounter_bounds_id": "Published field encounter_bounds_id.",
    "encounter_id": "Published field encounter_id.",
    "source_encounter_id": "Published field source_encounter_id.",
    "first_clinical_event_datetime": "Published field first_clinical_event_datetime.",
    "last_clinical_event_datetime": "Published field last_clinical_event_datetime.",
    "clinical_event_count": "Published field clinical_event_count.",
    "first_contemporaneous_event_datetime":
        "Published field first_contemporaneous_event_datetime.",
    "last_contemporaneous_event_datetime":
        "Published field last_contemporaneous_event_datetime.",
    "contemporaneous_event_count": "Published field contemporaneous_event_count.",
    "first_order_datetime": "Published field first_order_datetime.",
    "last_order_datetime": "Published field last_order_datetime.",
    "order_count": "Published field order_count.",
    "ward_move_count": "Published field ward_move_count.",
    "ward_occupancy_minutes": "Published field ward_occupancy_minutes.",
    "last_ward_in_datetime": "Published field last_ward_in_datetime.",
    "last_ward_out_datetime": "Published field last_ward_out_datetime.",
    "last_ward_still_open_ind": "Published field last_ward_still_open_ind.",
    "event_first_datetime": "Published field event_first_datetime.",
    "event_last_datetime": "Published field event_last_datetime.",
    "in_activity_bounds_ind": "Published field in_activity_bounds_ind.",
    "in_event_bounds_ind": "Published field in_event_bounds_ind.",
    "loaded_at": "Published field loaded_at.",
    "_source_system": "Published field _source_system.",
    "_source_table": "Published field _source_table.",
    "_source_row_id": "Published field _source_row_id.",
}

@dp.materialized_view(
    name=_n("gold_reference.encounter_bounds"),
    comment=(
        "One full-outer encounter activity and event bounds row. Gold QC twin of the silver "
        "product: 1 columns are repaired or nulled, 1 rule(s) drop rows, 0 check(s) are "
        "advisory. Each rule states its reason in the pipeline notebook, and Lakeflow "
        "expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(REFERENCE_ENCOUNTER_BOUNDS_MANDATORY_RULES)
def gold_reference_encounter_bounds():
    """Quality-controlled twin of journey_reference.encounter_bounds."""
    df = _qc(
        "reference_encounter_bounds",
        REFERENCE_ENCOUNTER_BOUNDS_SELECT,
        fk_columns=["encounter_id"],
    )
    return _with_comments(df, REFERENCE_ENCOUNTER_BOUNDS_COLUMN_COMMENTS)

In [0]:
# ==== journey_reference.gene_tested ====

REFERENCE_GENE_TESTED_SELECT = [
    "`gene_tested_row_id` AS `gene_tested_row_id`",
    "`source_gene_tested_id` AS `source_gene_tested_id`",
    "`genomic_test_id` AS `genomic_test_id`",
    "`source_genetic_test_id` AS `source_genetic_test_id`",
    "`hgnc_id` AS `hgnc_id`",
    "`reported_gene_symbol` AS `reported_gene_symbol`",
    "`normalized_gene_symbol` AS `normalized_gene_symbol`",
    "`alias_match_type` AS `alias_match_type`",
    "`evidence_type` AS `evidence_type`",
    "`test_scope` AS `test_scope`",
    "`panel_version_inferred` AS `panel_version_inferred`",
    "`confidence` AS `confidence`",
    "`loaded_at` AS `loaded_at`",
]

REFERENCE_GENE_TESTED_COLUMN_COMMENTS = {
    "gene_tested_row_id": "Stable assay-gene reference row identifier.",
    "source_gene_tested_id": "Bronze gene-tested identifier.",
    "genomic_test_id": "Parent genomic-test fact identifier.",
    "source_genetic_test_id": "Bronze parent genetic-test identifier.",
    "hgnc_id": "Governed HGNC identifier.",
    "reported_gene_symbol": "Gene symbol as reported or configured.",
    "normalized_gene_symbol": "Current approved HGNC symbol.",
    "alias_match_type": "HGNC symbol-resolution route.",
    "evidence_type": "Source of assay-gene membership.",
    "test_scope": "Reported exon",
    "panel_version_inferred": "Whether membership used an inferred panel version.",
    "confidence": "Deterministic normalization confidence.",
    "loaded_at": "Bronze material-change timestamp.",
}

@dp.materialized_view(
    name=_n("gold_reference.gene_tested"),
    comment=(
        "Assay-gene denominator at gene_tested_id grain. The current build covers 42.6% of "
        "assays; BCRL and FLT3 are complete while MNGS is 10.5% and TNGS is 16.6%. Gold QC "
        "twin of the silver product: 0 columns are repaired or nulled, 0 check(s) are "
        "advisory. Each rule states its reason in the pipeline notebook, and Lakeflow "
        "expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
def gold_reference_gene_tested():
    """Quality-controlled twin of journey_reference.gene_tested."""
    df = _qc("reference_gene_tested", REFERENCE_GENE_TESTED_SELECT)
    return _with_comments(df, REFERENCE_GENE_TESTED_COLUMN_COMMENTS)

In [0]:
# ==== journey_reference.location ====

REFERENCE_LOCATION_SELECT = [
    "`location_id` AS `location_id`",
    "`parent_location_id` AS `parent_location_id`",
    "`location_level` AS `location_level`",
    "`source_location_code` AS `source_location_code`",
    "`name` AS `name`",
    "`status` AS `status`",
    "`organization_id` AS `organization_id`",
    "`physical_type_code` AS `physical_type_code`",
    "`valid_from` AS `valid_from`",

    # 2100-12-31 is the far-future marker the source writes to mean 'no end yet'. The
    # absence is what the row means, and NULL states it without putting a fictional date
    # into a range comparison. Hit 475 of 857 rows (55.4%) when profiled on 2026-08-24.
    #
    # valid_to cannot precede valid_from. The start is the better-attested of the two, so
    # the end is what goes and the row keeps its valid_from. Hit 8 of 857 rows (0.933%) when
    # profiled on 2026-08-24.
    "CASE WHEN `valid_from` IS NOT NULL AND `valid_to` IS NOT NULL AND `valid_from` > `valid_to` THEN NULL ELSE CASE WHEN CAST(`valid_to` AS DATE) = DATE'2100-12-31' THEN NULL ELSE `valid_to` END END AS `valid_to`",
    "`latitude` AS `latitude`",
    "`longitude` AS `longitude`",
    "`address_city` AS `address_city`",
    "`address_postcode_masked` AS `address_postcode_masked`",
    "`source_table` AS `source_table`",
    "`source_row_id` AS `source_row_id`",
    "`load_batch_id` AS `load_batch_id`",
    "`loaded_at` AS `loaded_at`",
]

REFERENCE_LOCATION_COLUMN_COMMENTS = {
    "location_id": "Deterministic location primary key.",
    "parent_location_id": "Parent facility or building location.",
    "location_level": "Derived hierarchy level.",
    "source_location_code": "Source Millennium location code.",
    "name": "Source location name.",
    "status": "Source-derived location status.",
    "organization_id": "Managing organization reference.",
    "physical_type_code": "FHIR physical location type code.",
    "valid_from": "Source validity start.",
    "valid_to":
        "Source validity end. Gold QC transform rules: gold.reference.location.valid_to.open_sentinel.",
    "latitude": "Source address latitude.",
    "longitude": "Source address longitude.",
    "address_city": "Source address city.",
    "address_postcode_masked": "Privacy-aware source postcode.",
    "source_table": "Fully qualified bronze source table.",
    "source_row_id": "Stable source row identifier.",
    "load_batch_id": "Deterministic batch token.",
    "loaded_at": "Bronze load timestamp.",
}

@dp.materialized_view(
    name=_n("gold_reference.location"),
    comment=(
        "Effective-dated facility, building, and nurse-unit hierarchy. Gold QC twin of the "
        "silver product: 1 columns are repaired or nulled, 0 check(s) are advisory. Each rule "
        "states its reason in the pipeline notebook, and Lakeflow expectation metrics report "
        "what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
def gold_reference_location():
    """Quality-controlled twin of journey_reference.location."""
    df = _qc("reference_location", REFERENCE_LOCATION_SELECT)
    return _with_comments(df, REFERENCE_LOCATION_COLUMN_COMMENTS)

In [0]:
# ==== journey_reference.organization ====

REFERENCE_ORGANIZATION_SELECT = [
    "`organization_id` AS `organization_id`",
    "`source_organization_id` AS `source_organization_id`",
    "`name` AS `name`",
    "`status` AS `status`",
    "`valid_from` AS `valid_from`",

    # 2100-12-31 is the far-future marker the source writes to mean 'no end yet'. The
    # absence is what the row means, and NULL states it without putting a fictional date
    # into a range comparison. Hit 24 of 31 rows (77.4%) when profiled on 2026-08-24.
    #
    # valid_to cannot precede valid_from. The start is the better-attested of the two, so
    # the end is what goes and the row keeps its valid_from. Hit 1 of 31 rows (3.23%) when
    # profiled on 2026-08-24.
    "CASE WHEN `valid_from` IS NOT NULL AND `valid_to` IS NOT NULL AND `valid_from` > `valid_to` THEN NULL ELSE CASE WHEN CAST(`valid_to` AS DATE) = DATE'2100-12-31' THEN NULL ELSE `valid_to` END END AS `valid_to`",
    "`address_city` AS `address_city`",
    "`address_postcode_masked` AS `address_postcode_masked`",
    "`source_table` AS `source_table`",
    "`source_row_id` AS `source_row_id`",
    "`load_batch_id` AS `load_batch_id`",
    "`loaded_at` AS `loaded_at`",
]

REFERENCE_ORGANIZATION_COLUMN_COMMENTS = {
    "organization_id": "Deterministic organization primary key.",
    "source_organization_id": "Millennium organization identifier.",
    "name": "Source organization name.",
    "status": "Source-derived organization status.",
    "valid_from": "Source validity start.",
    "valid_to":
        "Source validity end. Gold QC transform rules: gold.reference.organization.valid_to.open_sentinel.",
    "address_city": "Source organization address city.",
    "address_postcode_masked": "Privacy-aware source postcode.",
    "source_table": "Fully qualified bronze source table.",
    "source_row_id": "Stable source organization identifier.",
    "load_batch_id": "Deterministic batch token.",
    "loaded_at": "Bronze load timestamp.",
}

@dp.materialized_view(
    name=_n("gold_reference.organization"),
    comment=(
        "Thin v1 provider organization dimension from care-site bronze. Gold QC twin of the "
        "silver product: 1 columns are repaired or nulled, 0 check(s) are advisory. Each rule "
        "states its reason in the pipeline notebook, and Lakeflow expectation metrics report "
        "what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
def gold_reference_organization():
    """Quality-controlled twin of journey_reference.organization."""
    df = _qc("reference_organization", REFERENCE_ORGANIZATION_SELECT)
    return _with_comments(df, REFERENCE_ORGANIZATION_COLUMN_COMMENTS)

In [0]:
# ==== journey_reference.person_address ====

REFERENCE_PERSON_ADDRESS_SELECT = [
    "`person_address_id` AS `person_address_id`",
    "`source_address_id` AS `source_address_id`",
    "`parent_entity` AS `parent_entity`",
    "`person_id` AS `person_id`",
    "`organization_id` AS `organization_id`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 70 of 17,057,866 rows
    # (0.00041%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`address_type_code` AS STRING))) = '0' THEN NULL ELSE `address_type_code` END AS `address_type_code`",
    "`active_ind` AS `active_ind`",

    # 1900-01-01 is a placeholder low date rather than a date in 1900; on a non-birth field
    # it carries no more meaning than it does on a birth one. Hit 159,543 of 17,057,866 rows
    # (0.935%) when profiled on 2026-08-24.
    #
    # 2100-01-01 is a placeholder high date. Nothing recorded here happens in the twenty-
    # second century. Hit 6 of 17,057,866 rows (3.52e-05%) when profiled on 2026-08-24.
    #
    # 2100-12-31 appears here on a field that is not an end date, so it cannot be the 'still
    # open' marker it is elsewhere and is a placeholder instead. Hit 66 of 17,057,866 rows
    # (0.000387%) when profiled on 2026-08-24.
    #
    # A date before 1901 that is not one of the known placeholders. Nothing in this estate
    # predates the twentieth century, so these are mistyped or mis-scaled rather than early.
    # Hit 136 of 17,057,866 rows (0.000797%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`beg_effective` AS DATE) = DATE'1900-01-01' OR CAST(`beg_effective` AS DATE) = DATE'2100-01-01' OR CAST(`beg_effective` AS DATE) = DATE'2100-12-31' OR CAST(`beg_effective` AS DATE) < DATE'1901-01-01' AND CAST(`beg_effective` AS DATE) NOT IN (DATE'1800-01-01', DATE'1899-12-30', DATE'1900-01-01') THEN NULL ELSE `beg_effective` END AS `beg_effective`",

    # 1900-01-01 is a placeholder low date rather than a date in 1900; on a non-birth field
    # it carries no more meaning than it does on a birth one. Hit 18 of 17,057,866 rows
    # (0.000106%) when profiled on 2026-08-24.
    #
    # 2100-01-01 is a placeholder high date. Nothing recorded here happens in the twenty-
    # second century. Hit 283 of 17,057,866 rows (0.00166%) when profiled on 2026-08-24.
    #
    # 2100-12-31 is the far-future marker the source writes to mean 'no end yet'. The
    # absence is what the row means, and NULL states it without putting a fictional date
    # into a range comparison. Hit 11,031,200 of 17,057,866 rows (64.7%) when profiled on
    # 2026-08-24.
    #
    # A date before 1901 that is not one of the known placeholders. Nothing in this estate
    # predates the twentieth century, so these are mistyped or mis-scaled rather than early.
    # Hit 5 of 17,057,866 rows (2.93e-05%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`end_effective` AS DATE) = DATE'2100-12-31' OR CAST(`end_effective` AS DATE) = DATE'1900-01-01' OR CAST(`end_effective` AS DATE) = DATE'2100-01-01' OR CAST(`end_effective` AS DATE) < DATE'1901-01-01' AND CAST(`end_effective` AS DATE) NOT IN (DATE'1800-01-01', DATE'1899-12-30', DATE'1900-01-01') THEN NULL ELSE `end_effective` END AS `end_effective`",
    "`open_ended_ind` AS `open_ended_ind`",
    "`street_address` AS `street_address`",
    "`city` AS `city`",
    "`postcode` AS `postcode`",
    "`postcode_masked` AS `postcode_masked`",
    "`postcode_outward` AS `postcode_outward`",
    "`uprn` AS `uprn`",
    "`lsoa` AS `lsoa`",
    "`msoa` AS `msoa`",
    "`local_authority_code` AS `local_authority_code`",
    "`imd_decile` AS `imd_decile`",
    "`imd_quintile` AS `imd_quintile`",
    "`latitude` AS `latitude`",
    "`longitude` AS `longitude`",
    "`uprn_match_quality` AS `uprn_match_quality`",
    "`epc_current_energy_rating` AS `epc_current_energy_rating`",
    "`epc_potential_energy_rating` AS `epc_potential_energy_rating`",
    "`epc_property_type` AS `epc_property_type`",
    "`epc_built_form` AS `epc_built_form`",
    "`epc_construction_age_band` AS `epc_construction_age_band`",
    "`epc_tenure` AS `epc_tenure`",
    "`epc_mains_gas_flag` AS `epc_mains_gas_flag`",
    "`epc_total_floor_area` AS `epc_total_floor_area`",
    "`epc_inspection_date` AS `epc_inspection_date`",
    "`epc_lodgement_date` AS `epc_lodgement_date`",
    "`epc_fuel_poverty_risk` AS `epc_fuel_poverty_risk`",
    "`epc_cold_hazard_proxy` AS `epc_cold_hazard_proxy`",
    "`epc_spatial_heating_poverty` AS `epc_spatial_heating_poverty`",
    "`epc_off_gas_grid` AS `epc_off_gas_grid`",
    "`loaded_at` AS `loaded_at`",
    "`_source_system` AS `_source_system`",
    "`_source_table` AS `_source_table`",
    "`_source_row_id` AS `_source_row_id`",
]

REFERENCE_PERSON_ADDRESS_MANDATORY_RULES = {
    # The flow nulls person_id when it names a parent the spine does not have, and this rule
    # then drops the row unless organization_id names the parent instead, because an address
    # belonging to nobody at all cannot be used, and the orphans are a wider population than
    # the spine covers rather than a join defect. That is the 3,803,862 orphaned rows out of
    # 17,057,866. The 42,490 rows with no person_id at all are not orphans: they belong to
    # something other than a person, and they are kept whenever organization_id says what.
    "gold.reference.person_address.person_id.fk_containment":
        "`person_id` IS NOT NULL OR `organization_id` IS NOT NULL",
}

REFERENCE_PERSON_ADDRESS_ADVISORY_RULES = {
    # This bounds a period of validity, and a future end is exactly how the source says a
    # record is still current -- nulling it would assert the record is valid forever, which
    # is a stronger and worse claim than the one being corrected. Seen on 122 of 17,057,866
    # rows (0.000715%) when profiled on 2026-08-24.
    "gold.reference.person_address.beg_effective.future_owner":
        "NOT COALESCE((CAST(`beg_effective` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",

    # This bounds a period of validity, and a future end is exactly how the source says a
    # record is still current -- nulling it would assert the record is valid forever, which
    # is a stronger and worse claim than the one being corrected. Seen on 1,861 of
    # 17,057,866 rows (0.0109%) when profiled on 2026-08-24.
    "gold.reference.person_address.end_effective.future_owner":
        "NOT COALESCE((CAST(`end_effective` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",
}

REFERENCE_PERSON_ADDRESS_COLUMN_COMMENTS = {
    "person_address_id": "Published field person_address_id.",
    "source_address_id": "Published field source_address_id.",
    "parent_entity": "Published field parent_entity.",
    "person_id": "Published field person_id.",
    "organization_id": "Published field organization_id.",
    "address_type_code": "Published field address_type_code.",
    "active_ind": "Published field active_ind.",
    "beg_effective":
        "Published field beg_effective. Gold QC transform rules: gold.reference.person_address.beg_effective.d1900_01_01_owner, gold.reference.person_address.beg_effective.d2100_01_01_owner, gold.reference.person_address.beg_effective.d2100_12_31_owner, gold.reference.person_address.beg_effective.pre1901_other_owner.",
    "end_effective":
        "Published field end_effective. Gold QC transform rules: gold.reference.person_address.end_effective.d1900_01_01_owner, gold.reference.person_address.end_effective.d2100_01_01_owner, gold.reference.person_address.end_effective.open_sentinel, gold.reference.person_address.end_effective.pre1901_other_owner.",
    "open_ended_ind": "Published field open_ended_ind.",
    "street_address": "Published field street_address.",
    "city": "Published field city.",
    "postcode": "Published field postcode.",
    "postcode_masked": "Published field postcode_masked.",
    "postcode_outward": "Published field postcode_outward.",
    "uprn": "Published field uprn.",
    "lsoa": "Published field lsoa.",
    "msoa": "Published field msoa.",
    "local_authority_code": "Published field local_authority_code.",
    "imd_decile": "Published field imd_decile.",
    "imd_quintile": "Published field imd_quintile.",
    "latitude": "Published field latitude.",
    "longitude": "Published field longitude.",
    "uprn_match_quality": "Published field uprn_match_quality.",
    "epc_current_energy_rating": "Published field epc_current_energy_rating.",
    "epc_potential_energy_rating": "Published field epc_potential_energy_rating.",
    "epc_property_type": "Published field epc_property_type.",
    "epc_built_form": "Published field epc_built_form.",
    "epc_construction_age_band": "Published field epc_construction_age_band.",
    "epc_tenure": "Published field epc_tenure.",
    "epc_mains_gas_flag": "Published field epc_mains_gas_flag.",
    "epc_total_floor_area": "Published field epc_total_floor_area.",
    "epc_inspection_date": "Published field epc_inspection_date.",
    "epc_lodgement_date": "Published field epc_lodgement_date.",
    "epc_fuel_poverty_risk": "Published field epc_fuel_poverty_risk.",
    "epc_cold_hazard_proxy": "Published field epc_cold_hazard_proxy.",
    "epc_spatial_heating_poverty": "Published field epc_spatial_heating_poverty.",
    "epc_off_gas_grid": "Published field epc_off_gas_grid.",
    "loaded_at": "Published field loaded_at.",
    "_source_system": "Published field _source_system.",
    "_source_table": "Published field _source_table.",
    "_source_row_id": "Published field _source_row_id.",
}

@dp.materialized_view(
    name=_n("gold_reference.person_address"),
    comment=(
        "One person or organization address assignment; direct address, postcode and UPRN "
        "fields are IG-sensitive. Gold QC twin of the silver product: 4 columns are repaired "
        "or nulled, 1 rule(s) drop rows, 2 check(s) are advisory. Each rule states its reason "
        "in the pipeline notebook, and Lakeflow expectation metrics report what every rule "
        "matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(REFERENCE_PERSON_ADDRESS_MANDATORY_RULES)
@_expect_all(REFERENCE_PERSON_ADDRESS_ADVISORY_RULES)
def gold_reference_person_address():
    """Quality-controlled twin of journey_reference.person_address."""
    df = _qc(
        "reference_person_address",
        REFERENCE_PERSON_ADDRESS_SELECT,
        fk_columns=["person_id"],
    )
    return _with_comments(df, REFERENCE_PERSON_ADDRESS_COLUMN_COMMENTS)

In [0]:
# ==== journey_reference.person_attribute ====

REFERENCE_PERSON_ATTRIBUTE_SELECT = [
    "`person_attribute_id` AS `person_attribute_id`",
    "`person_id` AS `person_id`",
    "`source_person_id` AS `source_person_id`",
    "`attribute_name` AS `attribute_name`",
    "`value_kind` AS `value_kind`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 1,409,608 of 9,775,723 rows
    # (14.4%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`value_code` AS STRING))) = '0' THEN NULL ELSE `value_code` END AS `value_code`",
    "`value_display` AS `value_display`",
    "`value_code_set` AS `value_code_set`",
    "`value_datetime` AS `value_datetime`",
    "`value_numeric` AS `value_numeric`",
    "`answered_ind` AS `answered_ind`",
    "`active_ind` AS `active_ind`",
    "`current_ind` AS `current_ind`",
    "`beg_effective` AS `beg_effective`",
    "`end_effective` AS `end_effective`",
    "`link_status` AS `link_status`",
    "`loaded_at` AS `loaded_at`",
    "`_source_system` AS `_source_system`",
    "`_source_table` AS `_source_table`",
    "`_source_row_id` AS `_source_row_id`",
]

REFERENCE_PERSON_ATTRIBUTE_MANDATORY_RULES = {
    # The flow nulls person_id when it names a parent the spine does not have, and this rule
    # then drops the row, because the row is only an attribute of a person, so with no
    # person it describes nothing. That was 331 orphaned rows plus 0 that already had no
    # person_id, out of 9,775,723.
    "gold.reference.person_attribute.person_id.fk_containment": "`person_id` IS NOT NULL",
}

REFERENCE_PERSON_ATTRIBUTE_COLUMN_COMMENTS = {
    "person_attribute_id": "Published field person_attribute_id.",
    "person_id": "Published field person_id.",
    "source_person_id": "Published field source_person_id.",
    "attribute_name": "Published field attribute_name.",
    "value_kind": "Published field value_kind.",
    "value_code": "Published field value_code.",
    "value_display": "Published field value_display.",
    "value_code_set": "Published field value_code_set.",
    "value_datetime": "Published field value_datetime.",
    "value_numeric": "Published field value_numeric.",
    "answered_ind": "Published field answered_ind.",
    "active_ind": "Published field active_ind.",
    "current_ind": "Published field current_ind.",
    "beg_effective": "Published field beg_effective.",
    "end_effective": "Published field end_effective.",
    "link_status": "Published field link_status.",
    "loaded_at": "Published field loaded_at.",
    "_source_system": "Published field _source_system.",
    "_source_table": "Published field _source_table.",
    "_source_row_id": "Published field _source_row_id.",
}

@dp.materialized_view(
    name=_n("gold_reference.person_attribute"),
    comment="One allowlisted person attribute; no-fixed-abode and public-care flags are IG-sensitive. Gold QC twin of the silver product: 2 columns are repaired or nulled, 1 rule(s) drop rows, 0 check(s) are advisory. Each rule states its reason in the pipeline notebook, and Lakeflow expectation metrics report what every rule matched on each update.",
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(REFERENCE_PERSON_ATTRIBUTE_MANDATORY_RULES)
def gold_reference_person_attribute():
    """Quality-controlled twin of journey_reference.person_attribute."""
    df = _qc(
        "reference_person_attribute",
        REFERENCE_PERSON_ATTRIBUTE_SELECT,
        fk_columns=["person_id"],
    )
    return _with_comments(df, REFERENCE_PERSON_ATTRIBUTE_COLUMN_COMMENTS)

In [0]:
# ==== journey_reference.person_death_evidence ====

REFERENCE_PERSON_DEATH_EVIDENCE_SELECT = [
    "`person_death_evidence_id` AS `person_death_evidence_id`",
    "`person_id` AS `person_id`",
    "`deceased_datetime_raw` AS `deceased_datetime_raw`",
    "`deceased_datetime` AS `deceased_datetime`",
    "`calculated_death_date` AS `calculated_death_date`",
    "`precision_flag` AS `precision_flag`",
    "`precision_desc` AS `precision_desc`",
    "`source_desc` AS `source_desc`",
    "`method_desc` AS `method_desc`",
    "`death_date_estimate_source` AS `death_date_estimate_source`",
    "`cause_of_death` AS `cause_of_death`",
    "`autopsy_desc` AS `autopsy_desc`",
    "`age_at_death` AS `age_at_death`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 3 of 584,657 rows (0.000513%) when profiled on 2026-08-24.
    #
    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 1 of 584,657 rows (0.000171%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`last_encounter_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS THEN NULL ELSE CASE WHEN CAST(`last_encounter_datetime` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`last_encounter_datetime` AS DATE)) < 9999 THEN NULL ELSE `last_encounter_datetime` END END AS `last_encounter_datetime`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 47 of 584,657 rows (0.00804%) when profiled on 2026-08-24.
    #
    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 40 of 584,657 rows (0.00684%) when profiled on 2026-08-24.
    #
    # Stamps landing in January 1970 carry the same millisecond-epoch defect found on
    # clinical_score and vital_sign, but nothing here corroborates a rescaled value the way
    # the encounter window does there, so the stamp is nulled rather than reconstructed. The
    # catalogue's date rules stop at 1901 and do not reach this.
    "CASE WHEN CAST(`last_clinical_event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS OR YEAR(CAST(`last_clinical_event_datetime` AS TIMESTAMP)) = 1970 THEN NULL ELSE CASE WHEN CAST(`last_clinical_event_datetime` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`last_clinical_event_datetime` AS DATE)) < 9999 THEN NULL ELSE `last_clinical_event_datetime` END END AS `last_clinical_event_datetime`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 50 of 584,657 rows (0.00855%) when profiled on 2026-08-24.
    #
    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 41 of 584,657 rows (0.00701%) when profiled on 2026-08-24.
    #
    # Stamps landing in January 1970 carry the same millisecond-epoch defect found on
    # clinical_score and vital_sign, but nothing here corroborates a rescaled value the way
    # the encounter window does there, so the stamp is nulled rather than reconstructed. The
    # catalogue's date rules stop at 1901 and do not reach this.
    "CASE WHEN CAST(`last_known_activity_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS OR YEAR(CAST(`last_known_activity_datetime` AS TIMESTAMP)) = 1970 THEN NULL ELSE CASE WHEN CAST(`last_known_activity_datetime` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`last_known_activity_datetime` AS DATE)) < 9999 THEN NULL ELSE `last_known_activity_datetime` END END AS `last_known_activity_datetime`",
    "`clinical_event_count` AS `clinical_event_count`",
    "`record_status` AS `record_status`",
    "`loaded_at` AS `loaded_at`",
    "`_source_system` AS `_source_system`",
    "`_source_table` AS `_source_table`",
    "`_source_row_id` AS `_source_row_id`",
]

REFERENCE_PERSON_DEATH_EVIDENCE_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 584,657 rows of 584,657 that
    # are current and attributable. Superseded versions and rows whose identity was never
    # resolved are not research data, and a consumer who wants them has silver.
    "research_surface": "(record_status = 'active')",

    # The flow nulls person_id when it names a parent the spine does not have, and this rule
    # then drops the row, because evidence that someone died is unusable without knowing
    # who. That was 226 orphaned rows plus 0 that already had no person_id, out of 584,657.
    "gold.reference.person_death_evidence.person_id.fk_containment":
        "`person_id` IS NOT NULL",
}

REFERENCE_PERSON_DEATH_EVIDENCE_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 0 of 584,657 at the profile.
    "gold.reference.person_death_evidence.record_status.default_view_active":
        "record_status = 'active'",
}

REFERENCE_PERSON_DEATH_EVIDENCE_COLUMN_COMMENTS = {
    "person_death_evidence_id": "Published field person_death_evidence_id.",
    "person_id": "Published field person_id.",
    "deceased_datetime_raw": "Published field deceased_datetime_raw.",
    "deceased_datetime": "Published field deceased_datetime.",
    "calculated_death_date": "Published field calculated_death_date.",
    "precision_flag": "Published field precision_flag.",
    "precision_desc": "Published field precision_desc.",
    "source_desc": "Published field source_desc.",
    "method_desc": "Published field method_desc.",
    "death_date_estimate_source": "Published field death_date_estimate_source.",
    "cause_of_death": "Published field cause_of_death.",
    "autopsy_desc": "Published field autopsy_desc.",
    "age_at_death": "Published field age_at_death.",
    "last_encounter_datetime":
        "Published field last_encounter_datetime. Gold QC transform rules: gold.reference.person_death_evidence.last_encounter_datetime.beyond_2100_below_9999_owner.",
    "last_clinical_event_datetime":
        "Published field last_clinical_event_datetime. Gold QC transform rules: gold.reference.person_death_evidence.last_clinical_event_datetime.beyond_2100_below_9999_owner.",
    "last_known_activity_datetime":
        "Published field last_known_activity_datetime. Gold QC transform rules: gold.reference.person_death_evidence.last_known_activity_datetime.beyond_2100_below_9999_owner.",
    "clinical_event_count": "Published field clinical_event_count.",
    "record_status": "Published field record_status.",
    "loaded_at": "Published field loaded_at.",
    "_source_system": "Published field _source_system.",
    "_source_table": "Published field _source_table.",
    "_source_row_id": "Published field _source_row_id.",
}

@dp.materialized_view(
    name=_n("gold_reference.person_death_evidence"),
    comment=(
        "One corroborating death-evidence row; cause of death is IG-sensitive. Gold QC twin "
        "of the silver product: 4 columns are repaired or nulled, 2 rule(s) drop rows, 1 "
        "check(s) are advisory. Each rule states its reason in the pipeline notebook, and "
        "Lakeflow expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(REFERENCE_PERSON_DEATH_EVIDENCE_MANDATORY_RULES)
@_expect_all(REFERENCE_PERSON_DEATH_EVIDENCE_ADVISORY_RULES)
def gold_reference_person_death_evidence():
    """Quality-controlled twin of journey_reference.person_death_evidence."""
    df = _qc(
        "reference_person_death_evidence",
        REFERENCE_PERSON_DEATH_EVIDENCE_SELECT,
        fk_columns=["person_id"],
    )
    return _with_comments(df, REFERENCE_PERSON_DEATH_EVIDENCE_COLUMN_COMMENTS)

In [0]:
# ==== journey_reference.practitioner ====

REFERENCE_PRACTITIONER_SELECT = [
    "`practitioner_id` AS `practitioner_id`",
    "`source_practitioner_id` AS `source_practitioner_id`",
    "`active` AS `active`",
    "`name` AS `name`",
    "`physician_ind` AS `physician_ind`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 234,768 of 351,201 rows
    # (66.8%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`position_code` AS STRING))) = '0' THEN NULL ELSE `position_code` END AS `position_code`",
    "`position_display` AS `position_display`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 1 of 351,201 rows (0.000285%)
    # when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`practitioner_type_code` AS STRING))) = '0' THEN NULL ELSE `practitioner_type_code` END AS `practitioner_type_code`",
    "`practitioner_type_display` AS `practitioner_type_display`",
    "`primary_location_id` AS `primary_location_id`",
    "`medical_service_id` AS `medical_service_id`",
    "`npi` AS `npi`",
    "`doctor_number` AS `doctor_number`",
    "`gdp_number` AS `gdp_number`",
    "`external_provider_id` AS `external_provider_id`",
    "`record_status` AS `record_status`",
    "`valid_from` AS `valid_from`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 1 of 351,201 rows (0.000285%) when profiled on 2026-08-24.
    #
    # 2100-01-01 is a placeholder high date. Nothing recorded here happens in the twenty-
    # second century. Hit 5 of 351,201 rows (0.00142%) when profiled on 2026-08-24.
    #
    # 2100-12-31 is the far-future marker the source writes to mean 'no end yet'. The
    # absence is what the row means, and NULL states it without putting a fictional date
    # into a range comparison. Hit 307,551 of 351,201 rows (87.6%) when profiled on
    # 2026-08-24.
    #
    # valid_to cannot precede valid_from. The start is the better-attested of the two, so
    # the end is what goes and the row keeps its valid_from. Hit 28 of 351,201 rows
    # (0.00797%) when profiled on 2026-08-24.
    "CASE WHEN `valid_from` IS NOT NULL AND `valid_to` IS NOT NULL AND `valid_from` > `valid_to` THEN NULL ELSE CASE WHEN CAST(`valid_to` AS DATE) = DATE'2100-12-31' OR CAST(`valid_to` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`valid_to` AS DATE)) < 9999 OR CAST(`valid_to` AS DATE) = DATE'2100-01-01' THEN NULL ELSE `valid_to` END END AS `valid_to`",
    "`source_table` AS `source_table`",
    "`source_row_id` AS `source_row_id`",
    "`load_batch_id` AS `load_batch_id`",
    "`loaded_at` AS `loaded_at`",
]

REFERENCE_PRACTITIONER_ADVISORY_RULES = {
    # This bounds a period of validity, and a future end is exactly how the source says a
    # record is still current -- nulling it would assert the record is valid forever, which
    # is a stronger and worse claim than the one being corrected. Seen on 2 of 351,201 rows
    # (0.000569%) when profiled on 2026-08-24.
    "gold.reference.practitioner.valid_from.future_owner":
        "NOT COALESCE((CAST(`valid_from` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",

    # This bounds a period of validity, and a future end is exactly how the source says a
    # record is still current -- nulling it would assert the record is valid forever, which
    # is a stronger and worse claim than the one being corrected. Seen on 373 of 351,201
    # rows (0.106%) when profiled on 2026-08-24.
    "gold.reference.practitioner.valid_to.future_owner":
        "NOT COALESCE((CAST(`valid_to` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",
}

REFERENCE_PRACTITIONER_COLUMN_COMMENTS = {
    "practitioner_id": "Deterministic practitioner primary key.",
    "source_practitioner_id": "Millennium personnel identifier.",
    "active": "Source active indicator.",
    "name": "Source formatted practitioner name.",
    "physician_ind": "Source physician indicator.",
    "position_code": "Source position code.",
    "position_display": "Source position display.",
    "practitioner_type_code": "Source personnel type code.",
    "practitioner_type_display": "Source personnel type display.",
    "primary_location_id": "Primary assigned or inferred location reference.",
    "medical_service_id": "Selected medical service reference.",
    "npi": "Selected NPI identifier.",
    "doctor_number": "Selected organization doctor number.",
    "gdp_number": "Selected dental practitioner number.",
    "external_provider_id": "Selected external provider identifier.",
    "record_status": "Normalized lifecycle status.",
    "valid_from": "Source validity start.",
    "valid_to":
        "Source validity end. Gold QC transform rules: gold.reference.practitioner.valid_to.beyond_2100_below_9999_owner, gold.reference.practitioner.valid_to.d2100_01_01_owner, gold.reference.practitioner.valid_to.open_sentinel.",
    "source_table": "Fully qualified bronze source table.",
    "source_row_id": "Stable source row identifier.",
    "load_batch_id": "Deterministic batch token.",
    "loaded_at": "Bronze load timestamp.",
}

@dp.materialized_view(
    name=_n("gold_reference.practitioner"),
    comment=(
        "One effective-dated Millennium practitioner or personnel record. Gold QC twin of the "
        "silver product: 3 columns are repaired or nulled, 2 check(s) are advisory. Each rule "
        "states its reason in the pipeline notebook, and Lakeflow expectation metrics report "
        "what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all(REFERENCE_PRACTITIONER_ADVISORY_RULES)
def gold_reference_practitioner():
    """Quality-controlled twin of journey_reference.practitioner."""
    df = _qc("reference_practitioner", REFERENCE_PRACTITIONER_SELECT)
    return _with_comments(df, REFERENCE_PRACTITIONER_COLUMN_COMMENTS)

In [0]:
# ==== journey_reference.practitioner_group ====

REFERENCE_PRACTITIONER_GROUP_SELECT = [
    "`practitioner_group_id` AS `practitioner_group_id`",
    "`practitioner_id` AS `practitioner_id`",
    "`practitioner_person_id` AS `practitioner_person_id`",
    "`in_practitioner_dimension_ind` AS `in_practitioner_dimension_ind`",
    "`source_group_id` AS `source_group_id`",
    "`group_name` AS `group_name`",
    "`group_label` AS `group_label`",
    "`group_type_meaning` AS `group_type_meaning`",
    "`group_type_display` AS `group_type_display`",
    "`primary_ind` AS `primary_ind`",
    "`relation_active_ind` AS `relation_active_ind`",
    "`relation_beg_effective` AS `relation_beg_effective`",

    # 2100-12-31 is the far-future marker the source writes to mean 'no end yet'. The
    # absence is what the row means, and NULL states it without putting a fictional date
    # into a range comparison. Hit 248,163 of 251,212 rows (98.8%) when profiled on
    # 2026-08-24.
    "CASE WHEN CAST(`relation_end_effective` AS DATE) = DATE'2100-12-31' THEN NULL ELSE `relation_end_effective` END AS `relation_end_effective`",
    "`group_active_ind` AS `group_active_ind`",
    "`group_beg_effective` AS `group_beg_effective`",

    # 2100-12-31 is the far-future marker the source writes to mean 'no end yet'. The
    # absence is what the row means, and NULL states it without putting a fictional date
    # into a range comparison. Hit 151,125 of 251,212 rows (60.2%) when profiled on
    # 2026-08-24.
    "CASE WHEN CAST(`group_end_effective` AS DATE) = DATE'2100-12-31' THEN NULL ELSE `group_end_effective` END AS `group_end_effective`",
    "`record_status` AS `record_status`",
    "`loaded_at` AS `loaded_at`",
    "`_source_system` AS `_source_system`",
    "`_source_table` AS `_source_table`",
    "`_source_row_id` AS `_source_row_id`",
]

REFERENCE_PRACTITIONER_GROUP_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 251,212 rows of 251,212 that
    # are current and attributable. Superseded versions and rows whose identity was never
    # resolved are not research data, and a consumer who wants them has silver.
    "research_surface": "(record_status = 'active')",
}

REFERENCE_PRACTITIONER_GROUP_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 0 of 251,212 at the profile.
    "gold.reference.practitioner_group.record_status.default_view_active":
        "record_status = 'active'",
}

REFERENCE_PRACTITIONER_GROUP_COLUMN_COMMENTS = {
    "practitioner_group_id": "Published field practitioner_group_id.",
    "practitioner_id": "Published field practitioner_id.",
    "practitioner_person_id": "Published field practitioner_person_id.",
    "in_practitioner_dimension_ind": "Published field in_practitioner_dimension_ind.",
    "source_group_id": "Published field source_group_id.",
    "group_name": "Published field group_name.",
    "group_label": "Published field group_label.",
    "group_type_meaning": "Published field group_type_meaning.",
    "group_type_display": "Published field group_type_display.",
    "primary_ind": "Published field primary_ind.",
    "relation_active_ind": "Published field relation_active_ind.",
    "relation_beg_effective": "Published field relation_beg_effective.",
    "relation_end_effective":
        "Published field relation_end_effective. Gold QC transform rules: gold.reference.practitioner_group.relation_end_effective.open_sentinel.",
    "group_active_ind": "Published field group_active_ind.",
    "group_beg_effective": "Published field group_beg_effective.",
    "group_end_effective":
        "Published field group_end_effective. Gold QC transform rules: gold.reference.practitioner_group.group_end_effective.open_sentinel.",
    "record_status": "Published field record_status.",
    "loaded_at": "Published field loaded_at.",
    "_source_system": "Published field _source_system.",
    "_source_table": "Published field _source_table.",
    "_source_row_id": "Published field _source_row_id.",
}

@dp.materialized_view(
    name=_n("gold_reference.practitioner_group"),
    comment=(
        "One staff-group membership with practitioner-dimension orphan evidence. Gold QC twin "
        "of the silver product: 2 columns are repaired or nulled, 1 rule(s) drop rows, 1 "
        "check(s) are advisory. Each rule states its reason in the pipeline notebook, and "
        "Lakeflow expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(REFERENCE_PRACTITIONER_GROUP_MANDATORY_RULES)
@_expect_all(REFERENCE_PRACTITIONER_GROUP_ADVISORY_RULES)
def gold_reference_practitioner_group():
    """Quality-controlled twin of journey_reference.practitioner_group."""
    df = _qc("reference_practitioner_group", REFERENCE_PRACTITIONER_GROUP_SELECT)
    return _with_comments(df, REFERENCE_PRACTITIONER_GROUP_COLUMN_COMMENTS)

In [0]:
# ==== journey_reference.practitioner_identifier ====

REFERENCE_PRACTITIONER_IDENTIFIER_SELECT = [
    "`practitioner_identifier_id` AS `practitioner_identifier_id`",
    "`practitioner_id` AS `practitioner_id`",
    "`practitioner_person_id` AS `practitioner_person_id`",
    "`source_alias_id` AS `source_alias_id`",
    "`alias_type_meaning` AS `alias_type_meaning`",
    "`alias_type_display` AS `alias_type_display`",
    "`alias` AS `alias`",
    "`active_ind` AS `active_ind`",
    "`effective_now_ind` AS `effective_now_ind`",
    "`beg_effective` AS `beg_effective`",

    # 2100-01-01 is a placeholder high date. Nothing recorded here happens in the twenty-
    # second century. Hit 7 of 519,155 rows (0.00135%) when profiled on 2026-08-24.
    #
    # 2100-12-31 is the far-future marker the source writes to mean 'no end yet'. The
    # absence is what the row means, and NULL states it without putting a fictional date
    # into a range comparison. Hit 488,152 of 519,155 rows (94%) when profiled on
    # 2026-08-24.
    "CASE WHEN CAST(`end_effective` AS DATE) = DATE'2100-12-31' OR CAST(`end_effective` AS DATE) = DATE'2100-01-01' THEN NULL ELSE `end_effective` END AS `end_effective`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 519,154 of 519,155 rows (100%)
    # when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`contributor_system_code` AS STRING))) = '0' THEN NULL ELSE `contributor_system_code` END AS `contributor_system_code`",
    "`record_status` AS `record_status`",
    "`loaded_at` AS `loaded_at`",
    "`_source_system` AS `_source_system`",
    "`_source_table` AS `_source_table`",
    "`_source_row_id` AS `_source_row_id`",
]

REFERENCE_PRACTITIONER_IDENTIFIER_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 519,155 rows of 519,155 that
    # are current and attributable. Superseded versions and rows whose identity was never
    # resolved are not research data, and a consumer who wants them has silver.
    "research_surface": "(record_status = 'active')",
}

REFERENCE_PRACTITIONER_IDENTIFIER_ADVISORY_RULES = {
    # This bounds a period of validity, and a future end is exactly how the source says a
    # record is still current -- nulling it would assert the record is valid forever, which
    # is a stronger and worse claim than the one being corrected. Seen on 2 of 519,155 rows
    # (0.000385%) when profiled on 2026-08-24.
    "gold.reference.practitioner_identifier.beg_effective.future_owner":
        "NOT COALESCE((CAST(`beg_effective` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",

    # This bounds a period of validity, and a future end is exactly how the source says a
    # record is still current -- nulling it would assert the record is valid forever, which
    # is a stronger and worse claim than the one being corrected. Seen on 2 of 519,155 rows
    # (0.000385%) when profiled on 2026-08-24.
    "gold.reference.practitioner_identifier.end_effective.future_owner":
        "NOT COALESCE((CAST(`end_effective` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",

    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 0 of 519,155 at the profile.
    "gold.reference.practitioner_identifier.record_status.default_view_active":
        "record_status = 'active'",
}

REFERENCE_PRACTITIONER_IDENTIFIER_COLUMN_COMMENTS = {
    "practitioner_identifier_id": "Published field practitioner_identifier_id.",
    "practitioner_id": "Published field practitioner_id.",
    "practitioner_person_id": "Published field practitioner_person_id.",
    "source_alias_id": "Published field source_alias_id.",
    "alias_type_meaning": "Published field alias_type_meaning.",
    "alias_type_display": "Published field alias_type_display.",
    "alias": "Published field alias.",
    "active_ind": "Published field active_ind.",
    "effective_now_ind": "Published field effective_now_ind.",
    "beg_effective": "Published field beg_effective.",
    "end_effective":
        "Published field end_effective. Gold QC transform rules: gold.reference.practitioner_identifier.end_effective.d2100_01_01_owner, gold.reference.practitioner_identifier.end_effective.open_sentinel.",
    "contributor_system_code": "Published field contributor_system_code.",
    "record_status": "Published field record_status.",
    "loaded_at": "Published field loaded_at.",
    "_source_system": "Published field _source_system.",
    "_source_table": "Published field _source_table.",
    "_source_row_id": "Published field _source_row_id.",
}

@dp.materialized_view(
    name=_n("gold_reference.practitioner_identifier"),
    comment=(
        "One staff alias assignment; alias values can include IG-sensitive staff postcodes. "
        "Gold QC twin of the silver product: 2 columns are repaired or nulled, 1 rule(s) drop "
        "rows, 3 check(s) are advisory. Each rule states its reason in the pipeline notebook, "
        "and Lakeflow expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(REFERENCE_PRACTITIONER_IDENTIFIER_MANDATORY_RULES)
@_expect_all(REFERENCE_PRACTITIONER_IDENTIFIER_ADVISORY_RULES)
def gold_reference_practitioner_identifier():
    """Quality-controlled twin of journey_reference.practitioner_identifier."""
    df = _qc("reference_practitioner_identifier", REFERENCE_PRACTITIONER_IDENTIFIER_SELECT)
    return _with_comments(df, REFERENCE_PRACTITIONER_IDENTIFIER_COLUMN_COMMENTS)

In [0]:
# ==== journey_reference.practitioner_location_evidence ====

REFERENCE_PRACTITIONER_LOCATION_EVIDENCE_SELECT = [
    "`practitioner_location_evidence_id` AS `practitioner_location_evidence_id`",
    "`practitioner_id` AS `practitioner_id`",
    "`practitioner_person_id` AS `practitioner_person_id`",
    "`location_code` AS `location_code`",
    "`event_count` AS `event_count`",

    # 1899-12-30 is the zero point of the OLE/Excel date scale, so it is what a spreadsheet
    # or a COM layer writes when the date was left blank. It is not a date anyone recorded.
    # Hit 192 of 2,468,651 rows (0.00778%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`first_event_datetime_raw` AS DATE) = DATE'1899-12-30' THEN NULL ELSE `first_event_datetime_raw` END AS `first_event_datetime_raw`",

    # Stamps landing in January 1970 carry the same millisecond-epoch defect found on
    # clinical_score and vital_sign, but nothing here corroborates a rescaled value the way
    # the encounter window does there, so the stamp is nulled rather than reconstructed. The
    # catalogue's date rules stop at 1901 and do not reach this.
    "CASE WHEN YEAR(CAST(`first_event_datetime` AS TIMESTAMP)) = 1970 THEN NULL ELSE `first_event_datetime` END AS `first_event_datetime`",

    # A year past 2100 but short of the 9999 'never' marker -- a typo in the year, most
    # often a transposed or extra digit. The true date is not recoverable from the wrong
    # one. Hit 7 of 2,468,651 rows (0.000284%) when profiled on 2026-08-24.
    #
    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 12 of 2,468,651 rows (0.000486%) when profiled on 2026-08-24.
    #
    # Stamps landing in January 1970 carry the same millisecond-epoch defect found on
    # clinical_score and vital_sign, but nothing here corroborates a rescaled value the way
    # the encounter window does there, so the stamp is nulled rather than reconstructed. The
    # catalogue's date rules stop at 1901 and do not reach this.
    "CASE WHEN CAST(`last_event_datetime_raw` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS OR YEAR(CAST(`last_event_datetime_raw` AS TIMESTAMP)) = 1970 THEN NULL ELSE CASE WHEN CAST(`last_event_datetime_raw` AS DATE) > DATE'2100-12-31' AND YEAR(CAST(`last_event_datetime_raw` AS DATE)) < 9999 THEN NULL ELSE `last_event_datetime_raw` END END AS `last_event_datetime_raw`",

    # This records something that already happened, so a stamp more than 90 days past the
    # moment the row was loaded cannot be right, and nothing in the row says what the real
    # date was. Hit 12 of 2,468,651 rows (0.000486%) when profiled on 2026-08-24.
    #
    # Stamps landing in January 1970 carry the same millisecond-epoch defect found on
    # clinical_score and vital_sign, but nothing here corroborates a rescaled value the way
    # the encounter window does there, so the stamp is nulled rather than reconstructed. The
    # catalogue's date rules stop at 1901 and do not reach this.
    "CASE WHEN CAST(`last_event_datetime` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS OR YEAR(CAST(`last_event_datetime` AS TIMESTAMP)) = 1970 THEN NULL ELSE `last_event_datetime` END AS `last_event_datetime`",
    "`location_rank` AS `location_rank`",
    "`top_count_tie_count` AS `top_count_tie_count`",
    "`loaded_at` AS `loaded_at`",
    "`_source_system` AS `_source_system`",
    "`_source_table` AS `_source_table`",
    "`_source_row_id` AS `_source_row_id`",
]

REFERENCE_PRACTITIONER_LOCATION_EVIDENCE_COLUMN_COMMENTS = {
    "practitioner_location_evidence_id":
        "Published field practitioner_location_evidence_id.",
    "practitioner_id": "Published field practitioner_id.",
    "practitioner_person_id": "Published field practitioner_person_id.",
    "location_code": "Published field location_code.",
    "event_count": "Published field event_count.",
    "first_event_datetime_raw":
        "Published field first_event_datetime_raw. Gold QC transform rules: gold.reference.practitioner_location_evidence.first_event_datetime_raw.ole_zero_date.",
    "first_event_datetime": "Published field first_event_datetime.",
    "last_event_datetime_raw":
        "Published field last_event_datetime_raw. Gold QC transform rules: gold.reference.practitioner_location_evidence.last_event_datetime_raw.beyond_2100_below_9999_owner.",
    "last_event_datetime": "Published field last_event_datetime.",
    "location_rank": "Published field location_rank.",
    "top_count_tie_count": "Published field top_count_tie_count.",
    "loaded_at": "Published field loaded_at.",
    "_source_system": "Published field _source_system.",
    "_source_table": "Published field _source_table.",
    "_source_row_id": "Published field _source_row_id.",
}

@dp.materialized_view(
    name=_n("gold_reference.practitioner_location_evidence"),
    comment=(
        "One staff location-evidence summary with sentinel-clamped bounds. Gold QC twin of "
        "the silver product: 4 columns are repaired or nulled, 0 check(s) are advisory. Each "
        "rule states its reason in the pipeline notebook, and Lakeflow expectation metrics "
        "report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
def gold_reference_practitioner_location_evidence():
    """Quality-controlled twin of journey_reference.practitioner_location_evidence."""
    df = _qc("reference_practitioner_location_evidence", REFERENCE_PRACTITIONER_LOCATION_EVIDENCE_SELECT)
    return _with_comments(df, REFERENCE_PRACTITIONER_LOCATION_EVIDENCE_COLUMN_COMMENTS)

In [0]:
# ==== journey_reference.pregnancy_reconciliation ====

REFERENCE_PREGNANCY_RECONCILIATION_SELECT = [
    "`pregnancy_reconciliation_id` AS `pregnancy_reconciliation_id`",
    "`unmatched_reason` AS `unmatched_reason`",
    "`pregnancy_id_raw` AS `pregnancy_id_raw`",
    "`pregnancy_id` AS `pregnancy_id`",
    "`lpid_mother` AS `lpid_mother`",
    "`antenatal_appointment_date` AS `antenatal_appointment_date`",
    "`pregnancy_first_contact_date` AS `pregnancy_first_contact_date`",
    "`expected_delivery_date` AS `expected_delivery_date`",
    "`last_mens_period_date` AS `last_mens_period_date`",
    "`folic_acid_supplement_cd` AS `folic_acid_supplement_cd`",
    "`previous_live_births` AS `previous_live_births`",
    "`previous_still_births` AS `previous_still_births`",
    "`previous_losses_under_24_weeks` AS `previous_losses_under_24_weeks`",
    "`previous_caesarean_sections` AS `previous_caesarean_sections`",
    "`source_system_code` AS `source_system_code`",
    "`is_valid` AS `is_valid`",
    "`msds_source_version` AS `msds_source_version`",
    "`record_status` AS `record_status`",
    "`source_update_timestamp` AS `source_update_timestamp`",
    "`loaded_at` AS `loaded_at`",
]

REFERENCE_PREGNANCY_RECONCILIATION_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 59 rows of 59 that are
    # current and attributable. Superseded versions and rows whose identity was never
    # resolved are not research data, and a consumer who wants them has silver.
    "research_surface": "(record_status = 'active')",
}

REFERENCE_PREGNANCY_RECONCILIATION_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 0 of 59 at the profile.
    "gold.reference.pregnancy_reconciliation.record_status.default_view_active":
        "record_status = 'active'",
}

REFERENCE_PREGNANCY_RECONCILIATION_COLUMN_COMMENTS = {
    "pregnancy_reconciliation_id": "Stable row identifier.",
    "unmatched_reason": "Published field.",
    "pregnancy_id_raw": "Published field.",
    "pregnancy_id": "Published field.",
    "lpid_mother": "Direct identifier published and IG-governed at serve time",
    "antenatal_appointment_date": "Published field.",
    "pregnancy_first_contact_date": "Published field.",
    "expected_delivery_date": "Published field.",
    "last_mens_period_date": "Published field.",
    "folic_acid_supplement_cd": "Published field.",
    "previous_live_births": "Published field.",
    "previous_still_births": "Published field.",
    "previous_losses_under_24_weeks": "Published field.",
    "previous_caesarean_sections": "Published field.",
    "source_system_code": "Published field.",
    "is_valid": "Published field.",
    "msds_source_version": "Published field.",
    "record_status": "Published field.",
    "source_update_timestamp": "Published field.",
    "loaded_at": "Published field.",
}

@dp.materialized_view(
    name=_n("gold_reference.pregnancy_reconciliation"),
    comment=(
        "Unmatched MSDS pregnancy evidence documenting a person-spine coverage gap. Gold QC "
        "twin of the silver product: 0 columns are repaired or nulled, 1 rule(s) drop rows, 1 "
        "check(s) are advisory. Each rule states its reason in the pipeline notebook, and "
        "Lakeflow expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(REFERENCE_PREGNANCY_RECONCILIATION_MANDATORY_RULES)
@_expect_all(REFERENCE_PREGNANCY_RECONCILIATION_ADVISORY_RULES)
def gold_reference_pregnancy_reconciliation():
    """Quality-controlled twin of journey_reference.pregnancy_reconciliation."""
    df = _qc("reference_pregnancy_reconciliation", REFERENCE_PREGNANCY_RECONCILIATION_SELECT)
    return _with_comments(df, REFERENCE_PREGNANCY_RECONCILIATION_COLUMN_COMMENTS)

In [0]:
# ==== journey_reference.research_study ====

REFERENCE_RESEARCH_STUDY_SELECT = [
    "`research_study_id` AS `research_study_id`",
    "`source_protocol_id` AS `source_protocol_id`",
    "`study_mnemonic` AS `study_mnemonic`",
    "`study_mnemonic_key` AS `study_mnemonic_key`",
    "`protocol_type_code` AS `protocol_type_code`",
    "`protocol_type_desc` AS `protocol_type_desc`",
    "`protocol_phase_code` AS `protocol_phase_code`",
    "`protocol_phase_desc` AS `protocol_phase_desc`",
    "`protocol_status_code` AS `protocol_status_code`",
    "`protocol_status_desc` AS `protocol_status_desc`",
    "`protocol_purpose_code` AS `protocol_purpose_code`",
    "`protocol_purpose_desc` AS `protocol_purpose_desc`",
    "`parent_protocol_id` AS `parent_protocol_id`",
    "`previous_protocol_id` AS `previous_protocol_id`",
    "`root_protocol_ind` AS `root_protocol_ind`",
    "`beg_effective` AS `beg_effective`",

    # 2100-12-31 is the far-future marker the source writes to mean 'no end yet'. The
    # absence is what the row means, and NULL states it without putting a fictional date
    # into a range comparison. Hit 156 of 1,055 rows (14.8%) when profiled on 2026-08-24.
    "CASE WHEN CAST(`end_effective` AS DATE) = DATE'2100-12-31' THEN NULL ELSE `end_effective` END AS `end_effective`",
    "`open_ended_ind` AS `open_ended_ind`",
    "`display_ind` AS `display_ind`",
    "`record_status` AS `record_status`",
    "`loaded_at` AS `loaded_at`",
    "`_source_system` AS `_source_system`",
    "`_source_table` AS `_source_table`",
    "`_source_row_id` AS `_source_row_id`",
]

REFERENCE_RESEARCH_STUDY_MANDATORY_RULES = {
    # The research surface. record_status = 'active' keeps the 1,055 rows of 1,055 that are
    # current and attributable. Superseded versions and rows whose identity was never
    # resolved are not research data, and a consumer who wants them has silver.
    "research_surface": "(record_status = 'active')",
}

REFERENCE_RESEARCH_STUDY_ADVISORY_RULES = {
    # Counts what the research surface removed: rows failing record_status = 'active'. The
    # mandatory rule above has already dropped them, so this is how many went, not how many
    # survive. 0 of 1,055 at the profile.
    "gold.reference.research_study.record_status.default_view_active":
        "record_status = 'active'",
}

REFERENCE_RESEARCH_STUDY_COLUMN_COMMENTS = {
    "research_study_id": "Published field research_study_id.",
    "source_protocol_id": "Published field source_protocol_id.",
    "study_mnemonic": "Published field study_mnemonic.",
    "study_mnemonic_key": "Published field study_mnemonic_key.",
    "protocol_type_code": "Published field protocol_type_code.",
    "protocol_type_desc": "Published field protocol_type_desc.",
    "protocol_phase_code": "Published field protocol_phase_code.",
    "protocol_phase_desc": "Published field protocol_phase_desc.",
    "protocol_status_code": "Published field protocol_status_code.",
    "protocol_status_desc": "Published field protocol_status_desc.",
    "protocol_purpose_code": "Published field protocol_purpose_code.",
    "protocol_purpose_desc": "Published field protocol_purpose_desc.",
    "parent_protocol_id": "Published field parent_protocol_id.",
    "previous_protocol_id": "Published field previous_protocol_id.",
    "root_protocol_ind": "Published field root_protocol_ind.",
    "beg_effective": "Published field beg_effective.",
    "end_effective":
        "Published field end_effective. Gold QC transform rules: gold.reference.research_study.end_effective.open_sentinel.",
    "open_ended_ind": "Published field open_ended_ind.",
    "display_ind": "Published field display_ind.",
    "record_status": "Published field record_status.",
    "loaded_at": "Published field loaded_at.",
    "_source_system": "Published field _source_system.",
    "_source_table": "Published field _source_table.",
    "_source_row_id": "Published field _source_row_id.",
}

@dp.materialized_view(
    name=_n("gold_reference.research_study"),
    comment=(
        "One Cerner research protocol amendment with parent and previous-protocol lineage. "
        "Gold QC twin of the silver product: 1 columns are repaired or nulled, 1 rule(s) drop "
        "rows, 1 check(s) are advisory. Each rule states its reason in the pipeline notebook, "
        "and Lakeflow expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all_or_drop(REFERENCE_RESEARCH_STUDY_MANDATORY_RULES)
@_expect_all(REFERENCE_RESEARCH_STUDY_ADVISORY_RULES)
def gold_reference_research_study():
    """Quality-controlled twin of journey_reference.research_study."""
    df = _qc("reference_research_study", REFERENCE_RESEARCH_STUDY_SELECT)
    return _with_comments(df, REFERENCE_RESEARCH_STUDY_COLUMN_COMMENTS)

In [0]:
# ==== journey_reference.service ====

REFERENCE_SERVICE_SELECT = [
    "`service_id` AS `service_id`",
    "`source_service_code` AS `source_service_code`",
    "`name` AS `name`",
    "`status` AS `status`",
    "`source_table` AS `source_table`",
    "`source_row_id` AS `source_row_id`",
    "`load_batch_id` AS `load_batch_id`",
    "`loaded_at` AS `loaded_at`",
]

REFERENCE_SERVICE_COLUMN_COMMENTS = {
    "service_id": "Deterministic HealthcareService primary key.",
    "source_service_code": "Source personnel-group service code.",
    "name": "Source service name.",
    "status": "Source-derived service status.",
    "source_table": "Fully qualified bronze source table.",
    "source_row_id": "Stable source service identifier.",
    "load_batch_id": "Deterministic batch token.",
    "loaded_at": "Bronze load timestamp.",
}

@dp.materialized_view(
    name=_n("gold_reference.service"),
    comment=(
        "Thin v1 medical-service dimension from practitioner assignments. Gold QC twin of the "
        "silver product: 0 columns are repaired or nulled, 0 check(s) are advisory. Each rule "
        "states its reason in the pipeline notebook, and Lakeflow expectation metrics report "
        "what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
def gold_reference_service():
    """Quality-controlled twin of journey_reference.service."""
    df = _qc("reference_service", REFERENCE_SERVICE_SELECT)
    return _with_comments(df, REFERENCE_SERVICE_COLUMN_COMMENTS)

In [0]:
# ==== journey_reference.theatre_attendance ====

REFERENCE_THEATRE_ATTENDANCE_SELECT = [
    "`theatre_attendance_id` AS `theatre_attendance_id`",
    "`theatre_case_id` AS `theatre_case_id`",
    "`case_link_status` AS `case_link_status`",
    "`attendee_practitioner_id` AS `attendee_practitioner_id`",
    "`attendee_personnel_id` AS `attendee_personnel_id`",
    "`role_description` AS `role_description`",
    "`signing_attendee_ind` AS `signing_attendee_ind`",
    "`in_datetime` AS `in_datetime`",
    "`in_datetime_quality` AS `in_datetime_quality`",
    "`out_datetime` AS `out_datetime`",
    "`out_datetime_quality` AS `out_datetime_quality`",
    "`person_id` AS `person_id`",
    "`encounter_id` AS `encounter_id`",
    "`surgical_area_desc` AS `surgical_area_desc`",
    "`active_ind` AS `active_ind`",
    "`loaded_at` AS `loaded_at`",
    "`_source_system` AS `_source_system`",
    "`_source_table` AS `_source_table`",
    "`_source_row_id` AS `_source_row_id`",
]

REFERENCE_THEATRE_ATTENDANCE_COLUMN_COMMENTS = {
    "theatre_attendance_id": "Published field theatre_attendance_id.",
    "theatre_case_id": "Published field theatre_case_id.",
    "case_link_status": "Published field case_link_status.",
    "attendee_practitioner_id": "Published field attendee_practitioner_id.",
    "attendee_personnel_id": "Published field attendee_personnel_id.",
    "role_description": "Published field role_description.",
    "signing_attendee_ind": "Published field signing_attendee_ind.",
    "in_datetime": "Published field in_datetime.",
    "in_datetime_quality": "Published field in_datetime_quality.",
    "out_datetime": "Published field out_datetime.",
    "out_datetime_quality": "Published field out_datetime_quality.",
    "person_id": "Published field person_id.",
    "encounter_id": "Published field encounter_id.",
    "surgical_area_desc": "Published field surgical_area_desc.",
    "active_ind": "Published field active_ind.",
    "loaded_at": "Published field loaded_at.",
    "_source_system": "Published field _source_system.",
    "_source_table": "Published field _source_table.",
    "_source_row_id": "Published field _source_row_id.",
}

@dp.materialized_view(
    name=_n("gold_reference.theatre_attendance"),
    comment=(
        "One SurgiNet theatre attendance row with parent-case linkage evidence. Gold QC twin "
        "of the silver product: 2 columns are repaired or nulled, 0 check(s) are advisory. "
        "Each rule states its reason in the pipeline notebook, and Lakeflow expectation "
        "metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
def gold_reference_theatre_attendance():
    """Quality-controlled twin of journey_reference.theatre_attendance."""
    # 101,593 rows point at a encounter_id the spine does not have. The pointer is nulled so
    # it cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    # 7 rows point at a person_id the spine does not have. The pointer is nulled so it
    # cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    df = _qc(
        "reference_theatre_attendance",
        REFERENCE_THEATRE_ATTENDANCE_SELECT,
        fk_columns=["encounter_id", "person_id"],
    )
    return _with_comments(df, REFERENCE_THEATRE_ATTENDANCE_COLUMN_COMMENTS)

In [0]:
# ==== journey_reference.theatre_case_milestone ====

REFERENCE_THEATRE_CASE_MILESTONE_SELECT = [
    "`theatre_case_milestone_id` AS `theatre_case_milestone_id`",
    "`theatre_case_id` AS `theatre_case_id`",
    "`case_link_status` AS `case_link_status`",

    # '0' is a placeholder the source writes when the value was not recorded; it is not a
    # code, so it is nulled rather than passed on as one. Hit 1 of 6,897,057 rows
    # (1.45e-05%) when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`task_assay_code` AS STRING))) = '0' THEN NULL ELSE `task_assay_code` END AS `task_assay_code`",
    "`task_assay_description` AS `task_assay_description`",
    "`stage_description` AS `stage_description`",
    "`case_time_datetime` AS `case_time_datetime`",
    "`case_time_quality` AS `case_time_quality`",
    "`person_id` AS `person_id`",
    "`encounter_id` AS `encounter_id`",
    "`active_ind` AS `active_ind`",
    "`loaded_at` AS `loaded_at`",
    "`_source_system` AS `_source_system`",
    "`_source_table` AS `_source_table`",
    "`_source_row_id` AS `_source_row_id`",
]

REFERENCE_THEATRE_CASE_MILESTONE_COLUMN_COMMENTS = {
    "theatre_case_milestone_id": "Published field theatre_case_milestone_id.",
    "theatre_case_id": "Published field theatre_case_id.",
    "case_link_status": "Published field case_link_status.",
    "task_assay_code": "Published field task_assay_code.",
    "task_assay_description": "Published field task_assay_description.",
    "stage_description": "Published field stage_description.",
    "case_time_datetime": "Published field case_time_datetime.",
    "case_time_quality": "Published field case_time_quality.",
    "person_id": "Published field person_id.",
    "encounter_id": "Published field encounter_id.",
    "active_ind": "Published field active_ind.",
    "loaded_at": "Published field loaded_at.",
    "_source_system": "Published field _source_system.",
    "_source_table": "Published field _source_table.",
    "_source_row_id": "Published field _source_row_id.",
}

@dp.materialized_view(
    name=_n("gold_reference.theatre_case_milestone"),
    comment=(
        "One SurgiNet theatre case-time milestone with parent-case linkage evidence. Gold QC "
        "twin of the silver product: 3 columns are repaired or nulled, 0 check(s) are "
        "advisory. Each rule states its reason in the pipeline notebook, and Lakeflow "
        "expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
def gold_reference_theatre_case_milestone():
    """Quality-controlled twin of journey_reference.theatre_case_milestone."""
    # 122,160 rows point at a encounter_id the spine does not have. The pointer is nulled so
    # it cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    # 8 rows point at a person_id the spine does not have. The pointer is nulled so it
    # cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    df = _qc(
        "reference_theatre_case_milestone",
        REFERENCE_THEATRE_CASE_MILESTONE_SELECT,
        fk_columns=["encounter_id", "person_id"],
    )
    return _with_comments(df, REFERENCE_THEATRE_CASE_MILESTONE_COLUMN_COMMENTS)

In [0]:
# ==== journey_reference.theatre_implant ====

REFERENCE_THEATRE_IMPLANT_SELECT = [
    "`theatre_implant_id` AS `theatre_implant_id`",
    "`theatre_case_id` AS `theatre_case_id`",
    "`case_link_status` AS `case_link_status`",
    "`item_id` AS `item_id`",
    "`manufacturer` AS `manufacturer`",
    "`model_number` AS `model_number`",
    "`catalog_number` AS `catalog_number`",
    "`serial_number` AS `serial_number`",
    "`lot_number` AS `lot_number`",
    "`batch_number` AS `batch_number`",
    "`implant_site` AS `implant_site`",
    "`implant_size` AS `implant_size`",
    "`quantity` AS `quantity`",
    "`expiry_date` AS `expiry_date`",
    "`free_text_item_desc` AS `free_text_item_desc`",
    "`implanted_by_practitioner_id` AS `implanted_by_practitioner_id`",
    "`mill_implant_event_id` AS `mill_implant_event_id`",
    "`implant_link_method` AS `implant_link_method`",
    "`person_id` AS `person_id`",
    "`encounter_id` AS `encounter_id`",
    "`document_type_desc` AS `document_type_desc`",
    "`loaded_at` AS `loaded_at`",
    "`_source_system` AS `_source_system`",
    "`_source_table` AS `_source_table`",
    "`_source_row_id` AS `_source_row_id`",
]

REFERENCE_THEATRE_IMPLANT_ADVISORY_RULES = {
    # This bounds a period of validity, and a future end is exactly how the source says a
    # record is still current -- nulling it would assert the record is valid forever, which
    # is a stronger and worse claim than the one being corrected. Seen on 55,523 of 205,632
    # rows (27%) when profiled on 2026-08-24.
    "gold.reference.theatre_implant.expiry_date.future_owner":
        "NOT COALESCE((CAST(`expiry_date` AS TIMESTAMP) > `loaded_at` + INTERVAL 90 DAYS), FALSE)",
}

REFERENCE_THEATRE_IMPLANT_COLUMN_COMMENTS = {
    "theatre_implant_id": "Published field theatre_implant_id.",
    "theatre_case_id": "Published field theatre_case_id.",
    "case_link_status": "Published field case_link_status.",
    "item_id": "Published field item_id.",
    "manufacturer": "Published field manufacturer.",
    "model_number": "Published field model_number.",
    "catalog_number": "Published field catalog_number.",
    "serial_number": "Published field serial_number.",
    "lot_number": "Published field lot_number.",
    "batch_number": "Published field batch_number.",
    "implant_site": "Published field implant_site.",
    "implant_size": "Published field implant_size.",
    "quantity": "Published field quantity.",
    "expiry_date": "Published field expiry_date.",
    "free_text_item_desc": "Published field free_text_item_desc.",
    "implanted_by_practitioner_id": "Published field implanted_by_practitioner_id.",
    "mill_implant_event_id": "Published field mill_implant_event_id.",
    "implant_link_method": "Published field implant_link_method.",
    "person_id": "Published field person_id.",
    "encounter_id": "Published field encounter_id.",
    "document_type_desc": "Published field document_type_desc.",
    "loaded_at": "Published field loaded_at.",
    "_source_system": "Published field _source_system.",
    "_source_table": "Published field _source_table.",
    "_source_row_id": "Published field _source_row_id.",
}

@dp.materialized_view(
    name=_n("gold_reference.theatre_implant"),
    comment=(
        "One SurgiNet theatre implant row; serial, lot and batch fields are IG-sensitive. "
        "Gold QC twin of the silver product: 1 columns are repaired or nulled, 1 check(s) are "
        "advisory. Each rule states its reason in the pipeline notebook, and Lakeflow "
        "expectation metrics report what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
@_expect_all(REFERENCE_THEATRE_IMPLANT_ADVISORY_RULES)
def gold_reference_theatre_implant():
    """Quality-controlled twin of journey_reference.theatre_implant."""
    # 3,332 rows point at a encounter_id the spine does not have. The pointer is nulled so
    # it cannot be followed to nothing, and the row is kept because it still describes
    # something in its own right.
    df = _qc(
        "reference_theatre_implant",
        REFERENCE_THEATRE_IMPLANT_SELECT,
        fk_columns=["encounter_id"],
    )
    return _with_comments(df, REFERENCE_THEATRE_IMPLANT_COLUMN_COMMENTS)

In [0]:
# ==== journey_reference.value_set ====

REFERENCE_VALUE_SET_SELECT = [
    "`value_set_id` AS `value_set_id`",
    "`canonical_url` AS `canonical_url`",
    "`version` AS `version`",
    "`member_system` AS `member_system`",

    # 'UNKNOWN' is a placeholder the source writes when the value was not recorded; it is
    # not a code, so it is nulled rather than passed on as one. Hit 4 of 114 rows (3.51%)
    # when profiled on 2026-08-24.
    "CASE WHEN UPPER(TRIM(CAST(`member_code` AS STRING))) = 'UNKNOWN' THEN NULL ELSE `member_code` END AS `member_code`",
    "`member_display` AS `member_display`",
    "`status` AS `status`",
    "`source_table` AS `source_table`",
    "`source_row_id` AS `source_row_id`",
    "`load_batch_id` AS `load_batch_id`",
    "`loaded_at` AS `loaded_at`",
]

REFERENCE_VALUE_SET_COLUMN_COMMENTS = {
    "value_set_id": "Deterministic value-set membership primary key.",
    "canonical_url": "Canonical value-set URL.",
    "version": "Pinned value-set version.",
    "member_system": "Member coding system.",
    "member_code": "Member code.",
    "member_display": "Member display.",
    "status": "Value-set membership status.",
    "source_table": "Governed bronze value-set source.",
    "source_row_id": "Stable source membership identity.",
    "load_batch_id": "Bronze batch token.",
    "loaded_at": "Bronze load timestamp.",
}

@dp.materialized_view(
    name=_n("gold_reference.value_set"),
    comment=(
        "Pinned value-set membership materialized from the governed value_set_release lookup "
        "asset (FHIR R4 4.0.1 core sets, UK Core ethnic category, NHS RTT period status); "
        "v3-ServiceDeliveryLocationRoleType is a recorded v1 exclusion. Gold QC twin of the "
        "silver product: 1 columns are repaired or nulled, 0 check(s) are advisory. Each rule "
        "states its reason in the pipeline notebook, and Lakeflow expectation metrics report "
        "what every rule matched on each update."
    ),
    table_properties={"quality": "gold"},
    refresh_policy="incremental",
)
def gold_reference_value_set():
    """Quality-controlled twin of journey_reference.value_set."""
    df = _qc("reference_value_set", REFERENCE_VALUE_SET_SELECT)
    return _with_comments(df, REFERENCE_VALUE_SET_COLUMN_COMMENTS)
